# Echolocating bats homing over kilometer scales: circular stats on goal-relative bearings

This notebook revisits the free-flying bat homing data of **Goldshtein et al. (2024, *Science*)**
(*"Acoustic cognitive map--based navigation in echolocating bats"*: echolocating bats translocated over
kilometer scales and tracked back to a known goal). The biological
question is whether sensory deprivation (temporary occlusion of vision, disruption of
magnetoreception / olfaction) produces detectable changes in **goal-relative orientation**
(directional bias and/or directional precision), and how robust homing is when some sensory
modalities are degraded.

Each flight is a GPS track recorded with an on-animal tag. We reduce each track to a sequence of
fixed-length movement **steps**, compute a heading per step, and express that heading **relative
to the instantaneous direction toward the animal's known destination** (its roost, or the
Kinneret/Agamon lake region). On this goal-relative scale, a step aimed straight at the goal is
near $0^\circ$ and a step heading away is near $\pm180^\circ$, so fitted von Mises parameters map
directly onto goal-directed travel.

**Why a mixture:** stepwise headings in these tracks are multimodal. A bat may wander before
committing to goal-directed travel, and the original study had to split trajectories into a
localization phase and a homing phase using *post-treatment* features (straightness,
distance-to-goal). Selecting on such features can entangle treatment effects with the selection
rule. To avoid that, the main analysis below fits a **von Mises--Uniform mixture (vMU)**: each step
is drawn either from a directed von Mises component (goal-directed "exploitation") or from an
undirected component (wandering "exploration"), with treatment and per-animal (tag) random
effects. This separates trajectory structure from treatment comparisons while using the **full**
trajectory, with no post-treatment filtering.

The notebook also fits a simpler **single von Mises treatment model** (Bambi) first, so the vMU
result can be read against a non-mixed baseline, and compares the two out-of-sample (PSIS-LOO
ELPD).

**Environment (run once from the repository root, in a terminal):**

```bash
uv venv --python 3.12
uv pip install -e ".[notebook]" "pandas[excel]" geopandas shapely contextily osmnx pyproj xlrd
uv run --no-project jupyter lab notebooks/goldshtein_bats.ipynb
```

Then select that kernel and run the cells top to bottom. This notebook loads its
track/metadata tables and the Agamon–Hula lake outline from the public [jgraving/unwrap-data](https://github.com/jgraving/unwrap-data) repository via `unwrap.datasets` (`load_bats_tracks`, `load_bats_info`, `bats_lake_outline_path`); the first run clones it to `~/.cache/unwrap-data`,
after which no further download happens.


In [ ]:
import pandas as pd
from unwrap.datasets import load_bats_tracks, load_bats_info, bats_lake_outline_path

import goldshtein_bats_plots as plots

plots.set_style()

Figures will be saved to: figures/bats


## 1. Load the data

Per-tag metadata and the raw track file. `meta_df` carries the treatment assignment and the
destination flags; `raw_track_df` carries the georeferenced track per tag.


In [ ]:
meta_df = load_bats_info()
meta_df

,tag,year,month,treatment,translocationPoint,treatmentName,flyHome,flyAgamon,returnHomeFirstNight,distStartTracking,...,localizationEntropyAfter_mean,localizationEntropyBefore_max,localizationEntropyAfter_max,localizationKlBefore_min,localizationKlAfter_min,localizationKlBefore_mean,localizationKlAfter_mean,meanSampleRate,medianSampleRate,maxSampleRate
0,7602,2022,3,1,1,control,False,True,True,584.071210,...,1.729793,2.625646,5.089097,0.170992,0.137833,0.248733,0.266432,3,2,128
1,7588,2022,3,4,1,"eye cover, magnet, anosmia",True,False,True,456.920672,...,1.371117,4.168098,4.385098,0.141696,0.136776,0.242352,0.203693,2,2,2
2,7606,2022,3,1,1,control,False,True,True,241.173726,...,1.441880,1.300453,3.846068,0.176578,0.132219,0.186483,0.205889,2,2,2
3,7603,2022,3,1,1,control,False,True,True,188.544908,...,1.672788,2.029828,5.363843,0.144022,0.140116,0.202933,0.268299,3,2,182
4,7604,2022,3,1,1,control,True,False,True,353.239091,...,1.738866,1.891083,5.710083,0.230500,0.147754,0.249878,0.276228,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,7615,2022,6,2,2,eye cover,False,True,True,77.354964,...,1.529624,5.486835,4.499099,0.138425,0.142599,0.332106,0.253474,2,2,2
63,7736,2022,6,4,2,"eye cover, magnet, anosmia",True,False,True,166.838186,...,1.506678,6.441877,4.221379,0.142300,0.135485,0.309039,0.226304,2,2,130
64,7611,2022,6,2,2,eye cover,False,True,True,35.613200,...,1.335386,2.268668,3.734204,0.149794,0.138183,0.225696,0.204692,2,2,2
65,7616,2022,6,4,1,"eye cover, magnet, anosmia",True,False,True,96.886674,...,1.504526,3.588204,3.861477,0.139762,0.137833,0.206260,0.235293,2,2,2


In [ ]:
raw_track_df = load_bats_tracks()
raw_track_df

,tag,time,x,y,lon,lat,alt,si,distance,speed,flightAngle,turningAngle,distDeparture,distDest,flightAngDiffDest,afterLocalization,illumination
0,7602,738608.957596,255310.931959,781769.662341,35.588670,33.130642,70,NaN,NaN,NaN,57.211712,NaN,584.071210,2530.181024,106.157718,0,0.0
1,7602,738608.959077,255426.557671,781844.745535,35.589912,33.131315,70,0.621477,137.865121,1.077139,73.378430,-0.000012,466.711117,2571.957726,92.942905,1,0.0
2,7602,738608.959100,255451.910406,781852.415572,35.590184,33.131384,70,0.631036,26.487556,13.243783,73.378418,88.135931,450.891423,2573.453643,93.532022,1,0.0
3,7602,738608.959123,255477.263141,781860.085608,35.590456,33.131452,70,0.640223,26.487556,13.243783,161.514349,37.133251,436.107859,2575.221144,5.984448,1,0.0
4,7602,738608.959146,255484.496121,781838.712500,35.590533,33.131259,70,0.649045,22.563815,11.281912,198.647600,0.000020,454.912920,2552.781382,-31.095961,1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28345,7776,738690.976348,256467.608776,779512.134513,35.600973,33.110249,70,0.884859,10.214910,5.107457,120.268189,6.809826,2902.158754,49.444468,8.514294,1,0.0
28346,7776,738690.976371,256478.820666,779505.647848,35.601093,33.110190,70,0.921553,12.953119,6.476562,127.078015,4.580946,2911.729526,36.684269,4.701223,1,0.0
28347,7776,738690.976395,256489.747840,779497.455004,35.601210,33.110116,70,0.913627,13.657446,6.828725,131.658961,5.534591,2922.863382,23.099908,2.897833,1,0.0
28348,7776,738690.976418,256498.007384,779490.162401,35.601298,33.110050,70,0.924582,11.018264,5.509134,137.193551,NaN,2932.339709,12.108553,0.000000,1,0.0


## 2. Spatially resample to fixed step length

`rediscretize_latlon_parallel` resamples each tag's lat/lon trajectory to a **constant step length**
(60 m) along-track, computing step headings and distances. Gaps in the raw track (a jump larger than
`gap_factor \times spacing_m`, or a time gap larger than `gap_time_s`) split the track into
contiguous `segment_id`s, and the first resampled row after a gap is flagged `gap_before=1`. Working
with fixed step lengths (rather than variable GPS fix intervals) is what lets us treat each step's
bearing as one circular observation and keeps distance along-track `s_along_m` well defined.


In [ ]:
import goldshtein_bats_preprocess as pre

# Deterministic geometry (previously ~1,300 lines inline): spatial rediscretization
# with gap detection -> per-tag metadata/destination -> metric projection ->
# harmonic/smoothed-distance goal-relative bearings -> flightAngDiffDest.
# Lifted to notebooks/goldshtein_bats_preprocess.py; bit-identical to the old
# pipeline on the current unwrap-data tracks (all 29 columns, 5,072 steps).
_pipeline = pre.build_tracking_table(raw_track_df, meta_df)
track_w_treat = _pipeline["track_w_treat"]
meta_by_tag   = _pipeline["meta_by_tag"]


### Per-tag metadata + destination label

Collapse the metadata to one row per tag, coerce the destination flags to 0/1, and build a
`destination` label: `Roost` (flyHome) vs `Agamon` (flyAgamon, the Kinneret/lake region) vs
Unknown. The per-tag metadata are merged back onto the resampled tracks so every step knows its
treatment and goal.


## 3. Goal-relative bearings (distance + bearing to the destination)

For every resampled step, compute:

- its local metric coordinates (`x_m, y_m`, a projected plane around the arena),
- its step `bearing_to_...`/`direction`,
- the **instantaneous direction toward the goal**: for the lake (Agamon) this is the
  negative-gradient of a smoothed-distance (or harmonic) distance field (`-\nabla u`), so the
  "best way to reduce distance" from each point, not just a straight line to a single point; for
  the roost it is the bearing to the roost point,
- the **goal-relative angle** `flightAngDiffDest_{deg,rad}`: the animal's step bearing **relative to**
  that goal direction. This is the circular response used in the models below (0 = aimed at goal).

This is what places every flight on a common circular scale: steps toward the goal lie near $0^\circ$,
steps away near $\pm180^\circ$.


## 4. Modeling table and descriptive summaries

Assemble the modeling frame and check the design.


Observations and animals per treatment, then steps per animal, so we can see the repeated-measures
structure (many steps within each tag) that motivates the per-tag random effects.


In [ ]:
combo_summary = (
    track_w_treat.groupby(["treatment", "treatmentName"], dropna=False)
                 .agg(
                     n_rows=("tag", "size"),       # total trajectory rows/steps
                     n_tags=("tag", "nunique"),    # number of animals/tags
                 )
                 .sort_values(["n_tags", "n_rows"], ascending=False)
                 .reset_index()
)

combo_summary


,treatment,treatmentName,n_rows,n_tags
0,1,control,1601,29
1,3,"eye cover, magnet",1179,15
2,2,eye cover,1213,14
3,4,"eye cover, magnet, anosmia",1079,9


In [ ]:
steps_per_tag = (
    track_w_treat.groupby(["treatment", "treatmentName", "tag"], dropna=False)
                 .size()
                 .rename("n_steps")
                 .reset_index()
)

steps_summary = (
    steps_per_tag.groupby(["treatment", "treatmentName"], dropna=False)["n_steps"]
                 .agg(n_tags="count", mean="mean", median="median", min="min", max="max")
                 .sort_values("n_tags", ascending=False)
                 .reset_index()
)

steps_summary


,treatment,treatmentName,n_tags,mean,median,min,max
0,1,control,29,55.206897,55.0,17,120
1,3,"eye cover, magnet",15,78.600000,78.0,40,151
2,2,eye cover,14,86.642857,88.5,25,150
3,4,"eye cover, magnet, anosmia",9,119.888889,90.0,45,236


## 5. Trajectory maps (goal-relative context)

Panels of the resampled tracks, colored by destination (roost vs Agamon) and by treatment, with the
lake polygon (Agamon) overlaid. Useful ground truth for what "aimed at the goal" looks like in each
condition before we fit any model.


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString, Point
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import matplotlib as mpl
import contextily as cx
from matplotlib.transforms import offset_copy
from matplotlib.patches import FancyArrowPatch, ArrowStyle

# ------------------------------------------------------------
# Assumes you already defined these from your other figure:
#   - hide_axes(ax)
#   - add_scale_bar_dual(...)
#   - add_compass(...)
# ------------------------------------------------------------

# --- fonts + pdf embedding (match your style) ---
mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "pdf.compression": 9,
})

def hide_axes(ax):
    """Remove axis labels, ticks, tick labels, and spines."""
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


def add_scale_bar_dual(
    ax,
    total_length_m=20.0,
    n_segments=2,
    segment_colors=("#f2f2f2", "#222222"),  # UPDATED for dark basemap
    segment_height_pt=8,
    edgecolor="#f2f2f2",                    # UPDATED for dark basemap
    edgewidth=0.9,                          # slightly thicker for visibility
    label=True,
    label_above=False,
    label_kwargs=None,
    # precise placement:
    ax_xy=(0.95, 0.08),     # anchor in axes coords
    offset_pt=(0, -41),     # extra (dx, dy) in points from the anchor
    align="right",          # bar aligned to anchor’s right edge or left edge
    direction="left",       # bar grows to the left or right from its aligned edge
    pad_text_pt=4,
    return_info=False
):
    """
    Two-tone horizontal scale bar (meters, data units).
    Placement uses axes coords + point offset. Drawn as rectangles in data coords.

    If return_info=True, returns dict including artists so caller can measure/adjust.
    """
    if label_kwargs is None:
        label_kwargs = dict(
            color="#f2f2f2", fontsize=10, ha="center", va="top",  # UPDATED for dark basemap
            fontname="Arial", fontstretch="normal"
        )

    fig = ax.figure
    if fig.canvas is not None:
        fig.canvas.draw()

    base = offset_copy(ax.transAxes, fig=fig, x=offset_pt[0], y=offset_pt[1], units='points')

    x_disp, y_disp = base.transform(ax_xy)
    x_anchor, y_anchor = ax.transData.inverted().transform((x_disp, y_disp))

    seg_len = total_length_m / float(max(n_segments, 1))
    if direction == "left":
        x0 = (x_anchor if align == "left" else x_anchor - total_length_m)
        xdir = +1
    elif direction == "right":
        x0 = (x_anchor if align == "left" else x_anchor + total_length_m)
        if align == "right":
            x0 = x_anchor - total_length_m
        xdir = +1
    else:
        raise ValueError("direction must be 'left' or 'right'")

    # Convert bar height (points) to data units using axes pixel height
    bbox = ax.get_window_extent()
    axpix_h = bbox.height
    y_min, y_max = ax.get_ylim()
    pt_to_data_y = (y_max - y_min) / axpix_h
    half_h = 0.5 * segment_height_pt * pt_to_data_y
    yb, yt = y_anchor - half_h, y_anchor + half_h

    colors = (segment_colors * ((n_segments + len(segment_colors) - 1) // len(segment_colors)))[:n_segments]

    patches = []
    for i in range(n_segments):
        xa = x0 + xdir * (i * seg_len)
        xb = x0 + xdir * ((i + 1) * seg_len)
        p = plt.Rectangle(
            (min(xa, xb), yb),
            abs(xb - xa),
            yt - yb,
            facecolor=colors[i],
            edgecolor=edgecolor,
            linewidth=edgewidth,
            zorder=5
        )
        ax.add_patch(p)
        patches.append(p)

    x_center = x0 + xdir * (total_length_m / 2.0)
    txt = None
    if label:
        txt = ax.text(x_center, y_anchor, f"{int(total_length_m)} m", **label_kwargs)
        dy = pad_text_pt if label_above else -pad_text_pt
        txt.set_transform(offset_copy(ax.transData, fig=fig, x=0, y=dy, units='points'))

    if return_info:
        return {
            "center_data": (x_center, y_anchor),
            "y_anchor_data": y_anchor,
            "segment_height_pt": segment_height_pt,
            "offset_pt": offset_pt,
            "ax_xy": ax_xy,
            "total_length_m": total_length_m,
            "patches": patches,
            "text": txt
        }
    return None


def add_compass(
    ax,
    ax_xy=(0.92, 0.18),
    offset_pt=(0, -95),
    arrow_len_pt=42,
    head_width_pt=12,
    head_length_pt=14,
    tail_width_pt=2.5,
    color="#f2f2f2",         # UPDATED for dark basemap
    edgecolor="#f2f2f2",     # UPDATED for dark basemap
    edgewidth=1.0,
    alpha=1.0,
    draw_N=True,
    N_fontsize=12,
    N_offset_pt=(0, 4)
):
    """
    North arrow drawn in AXES coordinates so it stays upright and consistent size.
    Points UP; position set by axes coords + point offset.
    """
    fig = ax.figure
    if fig.canvas is not None:
        fig.canvas.draw()

    base = offset_copy(ax.transAxes, fig=fig, x=offset_pt[0], y=offset_pt[1], units='points')

    bbox = ax.get_window_extent()
    axpix_h = bbox.height
    dy_axes = (arrow_len_pt * fig.dpi / 72.0) / axpix_h

    x_axes, y_axes = ax_xy
    start = (x_axes, y_axes)
    end   = (x_axes, y_axes + dy_axes)

    style = ArrowStyle.Simple(head_length=head_length_pt, head_width=head_width_pt, tail_width=tail_width_pt)
    patch = FancyArrowPatch(
        start, end,
        transform=ax.transAxes,
        arrowstyle=style,
        mutation_scale=1.0,
        linewidth=edgewidth,
        edgecolor=edgecolor,
        facecolor=color,
        alpha=alpha,
        joinstyle="miter", capstyle="butt",
        zorder=6
    )
    patch.set_transform(base)
    ax.add_patch(patch)

    if draw_N:
        tx = ax.text(
            x_axes, y_axes + dy_axes, "N",
            transform=ax.transAxes, ha="center", va="bottom",
            fontsize=N_fontsize, color=color, zorder=6,
            fontname="Arial"
        )
        tx.set_transform(offset_copy(
            ax.transAxes, fig=fig,
            x=N_offset_pt[0] + offset_pt[0],
            y=N_offset_pt[1] + offset_pt[1],
            units='points'
        ))


def _format_treatment_name(name: str) -> str:
    if pd.isna(name):
        return "Unknown"
    s = str(name).strip().lower()
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if "control" in parts and len(parts) > 1:
        parts = [p for p in parts if p != "control"]
    if len(parts) == 0:
        parts = ["control"]
    parts = sorted(parts)
    parts = [p[:1].upper() + p[1:] for p in parts]
    return ", ".join(parts)

def _nice_scale_length_m(width_m: float) -> float:
    # choose a nice 1/2/5 * 10^k length around ~20% of map width
    target = max(width_m * 0.2, 1.0)
    exp = np.floor(np.log10(target))
    base = target / (10 ** exp)
    if base < 1.5:
        nice = 1.0
    elif base < 3.5:
        nice = 2.0
    elif base < 7.5:
        nice = 5.0
    else:
        nice = 10.0
    return float(nice * (10 ** exp))

def _add_scale_and_compass(ax):
    """
    Bottom-right scalebar + compass centered above it, using your exact placement style.
    Updated colors for DARK basemap.
    """
    fig = ax.figure
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    corner_pad_px = 22
    pad_pt = corner_pad_px * 72.0 / fig.dpi
    seg_h_pt = 8
    scale_text_breathe_pt = 10

    # scale length from axis width (meters; EPSG:3857)
    x0, x1 = ax.get_xlim()
    scale_len_m = _nice_scale_length_m(abs(x1 - x0))

    # UPDATED label color for dark basemap
    label_kwargs = dict(
        color="#f2f2f2", fontsize=10, ha="center", va="top",
        fontname="Arial", fontstretch="normal"
    )

    def _draw_scale(offset_y_pt):
        return add_scale_bar_dual(
            ax,
            total_length_m=scale_len_m,
            n_segments=2,
            segment_colors=("#f2f2f2", "#222222"),  # UPDATED (light/dark)
            segment_height_pt=seg_h_pt,
            edgecolor="#f2f2f2",                   # UPDATED
            edgewidth=0.9,
            ax_xy=(1.0, 0.0),
            offset_pt=(-pad_pt, offset_y_pt),
            align="right",
            direction="left",
            label=True,
            label_above=False,
            pad_text_pt=scale_text_breathe_pt,
            label_kwargs=label_kwargs,
            return_info=True
        )

    # initial guess then correct to exact pixel padding like your function
    offset_y_pt = pad_pt + (seg_h_pt / 2.0) + scale_text_breathe_pt + 6.0
    sb = _draw_scale(offset_y_pt)
    fig.canvas.draw()

    txt = sb["text"]
    if txt is not None:
        ax_bbox = ax.get_window_extent(renderer=renderer)
        txt_bbox = txt.get_window_extent(renderer=renderer)
        current_bottom_pad_px = txt_bbox.y0 - ax_bbox.y0
        dy_px = corner_pad_px - current_bottom_pad_px
        dy_pt = dy_px * 72.0 / fig.dpi

        if abs(dy_pt) > 0.25:
            for p in sb["patches"]:
                p.remove()
            txt.remove()
            sb = _draw_scale(offset_y_pt + dy_pt)
            fig.canvas.draw()

    # compass centered above scale bar
    (cx_data, y_anchor_data) = sb["center_data"]
    cx_disp, cy_disp = ax.transData.transform((cx_data, y_anchor_data))
    compass_gap_pt = (seg_h_pt / 2.0) + 14.0
    cy_disp_up = cy_disp + compass_gap_pt * fig.dpi / 72.0
    cx_axes, cy_axes = ax.transAxes.inverted().transform((cx_disp, cy_disp_up))

    add_compass(
        ax,
        ax_xy=(cx_axes, cy_axes),
        offset_pt=(0, 0),
        arrow_len_pt=42,
        head_width_pt=12, head_length_pt=14, tail_width_pt=2.5,
        color="#f2f2f2", edgecolor="#f2f2f2", edgewidth=1.0,  # UPDATED
        draw_N=True, N_fontsize=12, N_offset_pt=(0, 4)
    )


Full-track panels (all steps) by treatment \times destination.


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString, Point
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import contextily as cx
import seaborn as sns
# ------------------------------------------------------------
# Assumes you already defined (from your style code):
#   - hide_axes(ax)
#   - _add_scale_and_compass(ax)
#   - _format_treatment_name(name)
# ------------------------------------------------------------

LAKE_GEOJSON = bats_lake_outline_path()
WEBMERC_EPSG = 3857

# Special locations
ROOST_LAT, ROOST_LON = 33.112373, 35.584135
TRANS_POINTS = [
    (33.135136, 35.592712),
    (33.081932, 35.604462),
]

mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",     # keep text as text in SVG (not paths)
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
})
def _ensure_dest_cols(track_df, meta_df, tag_col="tag"):
    """Ensure flyHome/flyAgamon exist on track_df without creating _x/_y duplicates."""
    out = track_df.copy()

    need = ["flyHome", "flyAgamon"]
    missing = [c for c in need if c not in out.columns]
    if missing:
        meta_dest = (
            meta_df[[tag_col, "flyHome", "flyAgamon"]]
            .drop_duplicates(subset=[tag_col], keep="first")
        )
        out = out.merge(meta_dest, on=tag_col, how="left", validate="many_to_one")

    out["flyHome"] = pd.to_numeric(out["flyHome"], errors="coerce").fillna(0).astype(int)
    out["flyAgamon"] = pd.to_numeric(out["flyAgamon"], errors="coerce").fillna(0).astype(int)
    return out

def _dest_label_from_flags(flyHome, flyAgamon):
    fh = int(flyHome == 1)
    fa = int(flyAgamon == 1)
    if fh and not fa:
        return "Home"
    if fa and not fh:
        return "Agamon"
    if fh and fa:
        return "Ambiguous"
    return "Unknown"

def _pick_order_cols(df):
    """Pick the first reasonable ordering column that exists."""
    for c in ["frame", "step", "t", "time", "timestamp"]:
        if c in df.columns:
            return c
    return None

def _make_segment_and_gap_lines_from_df(
    df,
    *,
    tag_col="tag", lat_col="lat", lon_col="lon",
    treat_col="treatment", treat_name_col="treatmentName",
    segment_col="segment_id",
    gap_col="gap_before",
):
    """
    Build:
      (1) One LineString per contiguous segment (solid)
      (2) One LineString per detected gap (dashed connector between last point before gap and first after)

    Expects resampled df to include `segment_id` and `gap_before` (from your gap-aware rediscretization).
    If they don't exist, falls back to one LineString per tag and no gaps.
    """
    d = df.dropna(subset=[tag_col, lat_col, lon_col]).copy()
    d[lat_col] = pd.to_numeric(d[lat_col], errors="coerce")
    d[lon_col] = pd.to_numeric(d[lon_col], errors="coerce")
    d = d[np.isfinite(d[lat_col]) & np.isfinite(d[lon_col])].copy()

    order_col = _pick_order_cols(d)

    seg_records = []
    gap_records = []

    has_segments = segment_col in d.columns
    has_gaps = gap_col in d.columns

    for tag, g_tag in d.groupby(tag_col, sort=False):
        g_tag = g_tag.sort_values(order_col) if order_col else g_tag.sort_index()

        flyHome = int(g_tag["flyHome"].iloc[0]) if "flyHome" in g_tag.columns else 0
        flyAgamon = int(g_tag["flyAgamon"].iloc[0]) if "flyAgamon" in g_tag.columns else 0
        dest = _dest_label_from_flags(flyHome, flyAgamon)

        trt = g_tag[treat_col].iloc[0] if treat_col in g_tag.columns else np.nan
        trt_name = g_tag[treat_name_col].iloc[0] if treat_name_col in g_tag.columns else "Unknown"

        # ---- SOLID segment lines ----
        if has_segments:
            for seg_id, g_seg in g_tag.groupby(segment_col, sort=True):
                coords = list(zip(g_seg[lon_col].to_numpy(), g_seg[lat_col].to_numpy()))
                if len(coords) < 2:
                    continue
                seg_records.append({
                    "tag": tag,
                    "treatment": trt,
                    "treatmentName": trt_name,
                    "dest": dest,
                    "segment_id": int(seg_id),
                    "geometry": LineString(coords),
                })
        else:
            coords = list(zip(g_tag[lon_col].to_numpy(), g_tag[lat_col].to_numpy()))
            if len(coords) >= 2:
                seg_records.append({
                    "tag": tag,
                    "treatment": trt,
                    "treatmentName": trt_name,
                    "dest": dest,
                    "segment_id": 0,
                    "geometry": LineString(coords),
                })

        # ---- DASHED gap connectors ----
        if has_gaps:
            gb = pd.to_numeric(g_tag[gap_col], errors="coerce").fillna(0).astype(int).to_numpy()
            # indices where THIS row is first point after a gap
            gap_starts = np.where(gb == 1)[0]
            for j in gap_starts:
                if j <= 0:
                    continue
                x0, y0 = float(g_tag.iloc[j-1][lon_col]), float(g_tag.iloc[j-1][lat_col])
                x1, y1 = float(g_tag.iloc[j][lon_col]), float(g_tag.iloc[j][lat_col])
                if not (np.isfinite(x0) and np.isfinite(y0) and np.isfinite(x1) and np.isfinite(y1)):
                    continue
                gap_records.append({
                    "tag": tag,
                    "treatment": trt,
                    "treatmentName": trt_name,
                    "dest": dest,
                    "geometry": LineString([(x0, y0), (x1, y1)]),
                })

    seg_gdf = gpd.GeoDataFrame(seg_records, geometry="geometry", crs="EPSG:4326") if len(seg_records) else \
        gpd.GeoDataFrame({"tag": [], "treatment": [], "treatmentName": [], "dest": [], "segment_id": [], "geometry": []},
                         geometry="geometry", crs="EPSG:4326")

    gap_gdf = gpd.GeoDataFrame(gap_records, geometry="geometry", crs="EPSG:4326") if len(gap_records) else \
        gpd.GeoDataFrame({"tag": [], "treatment": [], "treatmentName": [], "dest": [], "geometry": []},
                         geometry="geometry", crs="EPSG:4326")

    seg_gdf = seg_gdf.to_crs(epsg=WEBMERC_EPSG)
    gap_gdf = gap_gdf.to_crs(epsg=WEBMERC_EPSG)
    return seg_gdf, gap_gdf


def plot_rediscretized_destination_panels(
    resampled_df,
    meta_df,
    *,
    tag_col="tag",
    lat_col="lat",
    lon_col="lon",
    treat_col="treatment",
    treat_name_col="treatmentName",
    out_pdf="rediscretized_destination_panels.pdf",
    out_svg="rediscretized_destination_panels.svg",
    lake_facecolor="tab:cyan",
    lake_alpha=0.35,
    # style knobs
    res_lw: float = 1.6,
    res_alpha: float = 0.95,
    basemap_alpha: float = 1.0,
    # NEW: gap style
    gap_lw: float | None = None,
    gap_alpha: float = 0.85,
    gap_linestyle: str = "--",
):
    # 1) Ensure dest flags exist on rediscretized df
    res_df = _ensure_dest_cols(resampled_df, meta_df, tag_col=tag_col)

    # 2) Build GeoDataFrames: solid segments + dashed gaps
    res_segs_gdf, res_gaps_gdf = _make_segment_and_gap_lines_from_df(
        res_df,
        tag_col=tag_col, lat_col=lat_col, lon_col=lon_col,
        treat_col=treat_col, treat_name_col=treat_name_col,
        segment_col="segment_id",
        gap_col="gap_before",
    )

    print("rediscretized segment-lines:", len(res_segs_gdf))
    print("gap connectors:", len(res_gaps_gdf))

    if len(res_segs_gdf) == 0:
        raise ValueError("No lines to plot. Check that lat/lon are present and not all NaN.")

    if gap_lw is None:
        gap_lw = max(0.9, res_lw * 0.9)

    # 3) Points layer (reference + translocations)
    pts = [{"site": "Reference", "geometry": Point(ROOST_LON, ROOST_LAT)}]
    pts += [{"site": "Translocation", "geometry": Point(lon, lat)} for (lat, lon) in TRANS_POINTS]
    pts_gdf = gpd.GeoDataFrame(pts, crs="EPSG:4326").to_crs(epsg=WEBMERC_EPSG)

    # 4) Lake polygon
    lake_gdf_3857 = gpd.read_file(LAKE_GEOJSON).to_crs(epsg=WEBMERC_EPSG)

    # 5) Treatment colors (seaborn colorblind)
    treatments = sorted(pd.unique(res_segs_gdf[treat_col].dropna())) if treat_col in res_segs_gdf.columns else []

    if len(treatments):
        treat_colors = sns.color_palette("colorblind", n_colors=len(treatments))
        treat_to_color = {t: mcolors.to_hex(treat_colors[i]) for i, t in enumerate(treatments)}
    else:
        treat_to_color = {}
    # labels (use treatmentName if present)
    if treat_name_col in res_segs_gdf.columns and len(treatments):
        name_map = res_segs_gdf.groupby(treat_col)[treat_name_col].first().to_dict()
        treat_to_label = {t: _format_treatment_name(name_map.get(t, "Unknown")) for t in treatments}
    else:
        treat_to_label = {t: str(t) for t in treatments}

    # 6) Figure panels
    fig, axes = plt.subplots(1, 2, figsize=(7, 8), constrained_layout=False)
    panels = [("Home", axes[0], "Home"), ("Agamon", axes[1], "Agamon Hula Lake")]

    # unified extent across everything
    layers_for_bounds = [g for g in [res_segs_gdf, res_gaps_gdf, pts_gdf, lake_gdf_3857] if len(g)]
    minx = min(g.total_bounds[0] for g in layers_for_bounds)
    miny = min(g.total_bounds[1] for g in layers_for_bounds)
    maxx = max(g.total_bounds[2] for g in layers_for_bounds)
    maxy = max(g.total_bounds[3] for g in layers_for_bounds)

    pad_x = (maxx - minx) * 0.06 if maxx > minx else 1000
    pad_y = (maxy - miny) * 0.06 if maxy > miny else 1000
    xlim = (minx - pad_x, maxx + pad_x)
    ylim = (miny - pad_y, maxy + pad_y)

    for dest, ax, title in panels:
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)

        # basemap
        try:
            cx.add_basemap(ax, source=cx.providers.CartoDB.DarkMatter, attribution=False, alpha=basemap_alpha)
        except Exception as e:
            print("Basemap failed (continuing without tiles):", e)

        # lake polygon ONLY on Agamon panel
        if dest == "Agamon" and len(lake_gdf_3857):
            lake_gdf_3857.plot(
                ax=ax,
                facecolor=lake_facecolor,
                edgecolor=lake_facecolor,
                alpha=lake_alpha,
                linewidth=1.2,
                zorder=12
            )
            lake_gdf_3857.boundary.plot(
                ax=ax,
                color=lake_facecolor,
                linewidth=1.4,
                alpha=1.0,
                zorder=13
            )

        # --- Dashed gap connectors (draw FIRST, then solid segments on top) ---
        if len(res_gaps_gdf):
            sub_gap = res_gaps_gdf.loc[res_gaps_gdf["dest"] == dest]
            for t in treatments:
                sub_gap_t = sub_gap.loc[sub_gap[treat_col] == t]
                if len(sub_gap_t):
                    sub_gap_t.plot(
                        ax=ax,
                        color=treat_to_color[t],
                        linewidth=gap_lw,
                        alpha=gap_alpha,
                        linestyle=gap_linestyle,
                        zorder=26
                    )

        # --- Solid rediscretized segments (colored) ---
        sub_seg = res_segs_gdf.loc[res_segs_gdf["dest"] == dest]
        for t in treatments:
            sub_seg_t = sub_seg.loc[sub_seg[treat_col] == t]
            if len(sub_seg_t):
                sub_seg_t.plot(
                    ax=ax,
                    color=treat_to_color[t],
                    linewidth=res_lw,
                    alpha=res_alpha,
                    zorder=28
                )

        # reference + translocation markers
        ref = pts_gdf.loc[pts_gdf["site"] == "Reference"]
        ax.scatter(ref.geometry.x, ref.geometry.y,
                   marker="*", s=220, facecolors="white", edgecolors="black",
                   linewidths=0.9, zorder=50)

        trans = pts_gdf.loc[pts_gdf["site"] == "Translocation"]
        ax.scatter(trans.geometry.x, trans.geometry.y,
                   marker="X", s=140, facecolors="white", edgecolors="black",
                   linewidths=0.9, zorder=50)

        ax.set_aspect("equal", adjustable="box")
        ax.set_title(title, fontsize=16)
        hide_axes(ax)

    # --- Legend ---
    treat_handles = []#[Patch(color=treat_to_color[t], label=treat_to_label[t]) for t in treatments]
    other_handles = [
        Line2D([0], [0], marker="*", color="none", markerfacecolor="white",
               markeredgecolor="black", markersize=12, label="Roost"),
        Line2D([0], [0], marker="X", color="none", markerfacecolor="white",
               markeredgecolor="black", markersize=9, label="Translocation"),
    ]
    # Optional: include a generic dashed "gap" legend entry if gaps exist
    if len(res_gaps_gdf):
        other_handles.append(Line2D([0, 1], [0, 0], color="black", lw=gap_lw,
                                    linestyle=gap_linestyle, alpha=gap_alpha, label="Signal Dropout"))

    fig.legend(
        handles=treat_handles + other_handles,
        loc="center",
        bbox_to_anchor=(1.0, 0.5),
        frameon=True,
        framealpha=0.9,
    )

    plt.tight_layout(rect=[0, 0, 0.86, 1])

    # scale + compass on RIGHT panel (Agamon)
    _add_scale_and_compass(axes[1])

    fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.02, dpi=300)
    fig.savefig(out_svg, bbox_inches="tight", pad_inches=0.02, dpi=300)
    plt.show()


# -------------------------
# RUN
# -------------------------
# track_w_treat is your downstream rediscretized df (should include gap_before + segment_id)
plot_rediscretized_destination_panels(
    resampled_df=track_w_treat,
    meta_df=meta_df,
    out_pdf="../figures/bats/rediscretized_destination_panels.pdf",
    out_svg="../figures/bats/rediscretized_destination_panels.svg",
)


rediscretized segment-lines: 88
gap connectors: 24


/home/jake/.venvs/unwrap-env/lib/python3.12/site-packages/contextily/tile.py:298: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tile_urls = [provider.build_url(x=tile.x, y=tile.y, z=tile.z) for tile in tiles]


/home/jake/.venvs/unwrap-env/lib/python3.12/site-packages/contextily/tile.py:298: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tile_urls = [provider.build_url(x=tile.x, y=tile.y, z=tile.z) for tile in tiles]


Compact per-panel trajectory plots (one panel per treatment / destination combination).


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString, Point
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import matplotlib as mpl
import contextily as cx
from matplotlib.transforms import offset_copy
from matplotlib.patches import FancyArrowPatch, ArrowStyle
import seaborn as sns

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
LAKE_GEOJSON = bats_lake_outline_path()
WEBMERC_EPSG = 3857

# Special locations
ROOST_LAT, ROOST_LON = 33.112373, 35.584135
TRANS_POINTS = [
    (33.135136, 35.592712),
    (33.081932, 35.604462),
]

# ------------------------------------------------------------
# STYLE (match your pipeline + compact paper figure)
# ------------------------------------------------------------
DPI = 300
mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "pdf.compression": 9,
    "svg.fonttype": "none",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
})

# ------------------------------------------------------------
# SMALL UTILITIES
# ------------------------------------------------------------
def hide_axes(ax):
    """Remove axis labels, ticks, tick labels, and spines."""
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

def _format_treatment_name(name: str) -> str:
    if pd.isna(name):
        return "Unknown"
    s = str(name).strip().lower()
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if "control" in parts and len(parts) > 1:
        parts = [p for p in parts if p != "control"]
    if len(parts) == 0:
        parts = ["control"]
    parts = sorted(parts)
    parts = [p[:1].upper() + p[1:] for p in parts]
    return ", ".join(parts)

def _nice_scale_length_m(width_m: float) -> float:
    # choose a nice 1/2/5 * 10^k length around ~20% of map width
    target = max(width_m * 0.2, 1.0)
    exp = np.floor(np.log10(target))
    base = target / (10 ** exp)
    if base < 1.5:
        nice = 1.0
    elif base < 3.5:
        nice = 2.0
    elif base < 7.5:
        nice = 5.0
    else:
        nice = 10.0
    return float(nice * (10 ** exp))

def _pick_order_cols(df):
    """Pick the first reasonable ordering column that exists."""
    for c in ["frame", "step", "t", "time", "timestamp"]:
        if c in df.columns:
            return c
    return None

# ------------------------------------------------------------
# SCALE BAR + COMPASS (dark basemap colors)
# ------------------------------------------------------------
def add_scale_bar_dual(
    ax,
    total_length_m=20.0,
    n_segments=2,
    segment_colors=("#f2f2f2", "#222222"),  # good on dark basemap
    segment_height_pt=8,
    edgecolor="#f2f2f2",
    edgewidth=0.9,
    label=True,
    label_above=False,
    label_kwargs=None,
    ax_xy=(0.95, 0.08),
    offset_pt=(0, -41),
    align="right",
    direction="left",
    pad_text_pt=4,
    return_info=False
):
    """
    Two-tone horizontal scale bar (meters, data units).
    Placement uses axes coords + point offset. Drawn as rectangles in data coords.
    """
    if label_kwargs is None:
        label_kwargs = dict(
            color="#f2f2f2",
            fontsize=9.5,  # slightly smaller for compact figure
            ha="center",
            va="top",
            fontname="Arial",
            fontstretch="normal",
        )

    fig = ax.figure
    if fig.canvas is not None:
        fig.canvas.draw()

    base = offset_copy(ax.transAxes, fig=fig, x=offset_pt[0], y=offset_pt[1], units="points")
    x_disp, y_disp = base.transform(ax_xy)
    x_anchor, y_anchor = ax.transData.inverted().transform((x_disp, y_disp))

    seg_len = total_length_m / float(max(n_segments, 1))
    if direction == "left":
        x0 = (x_anchor if align == "left" else x_anchor - total_length_m)
        xdir = +1
    elif direction == "right":
        x0 = (x_anchor if align == "left" else x_anchor + total_length_m)
        if align == "right":
            x0 = x_anchor - total_length_m
        xdir = +1
    else:
        raise ValueError("direction must be 'left' or 'right'")

    # Convert bar height (points) to data units using axes pixel height
    bbox = ax.get_window_extent()
    axpix_h = bbox.height
    y_min, y_max = ax.get_ylim()
    pt_to_data_y = (y_max - y_min) / axpix_h
    half_h = 0.5 * segment_height_pt * pt_to_data_y
    yb, yt = y_anchor - half_h, y_anchor + half_h

    colors = (segment_colors * ((n_segments + len(segment_colors) - 1) // len(segment_colors)))[:n_segments]

    patches = []
    for i in range(n_segments):
        xa = x0 + xdir * (i * seg_len)
        xb = x0 + xdir * ((i + 1) * seg_len)
        p = plt.Rectangle(
            (min(xa, xb), yb),
            abs(xb - xa),
            yt - yb,
            facecolor=colors[i],
            edgecolor=edgecolor,
            linewidth=edgewidth,
            zorder=5
        )
        ax.add_patch(p)
        patches.append(p)

    x_center = x0 + xdir * (total_length_m / 2.0)
    txt = None
    if label:
        txt = ax.text(x_center, y_anchor, f"{int(total_length_m)} m", **label_kwargs)
        dy = pad_text_pt if label_above else -pad_text_pt
        txt.set_transform(offset_copy(ax.transData, fig=fig, x=0, y=dy, units="points"))
        txt.set_zorder(6)

    if return_info:
        return {
            "center_data": (x_center, y_anchor),
            "y_anchor_data": y_anchor,
            "segment_height_pt": segment_height_pt,
            "offset_pt": offset_pt,
            "ax_xy": ax_xy,
            "total_length_m": total_length_m,
            "patches": patches,
            "text": txt,
        }
    return None

def add_compass(
    ax,
    ax_xy=(0.92, 0.18),
    offset_pt=(0, -95),
    arrow_len_pt=36,   # reduced for compact figure
    head_width_pt=10,
    head_length_pt=12,
    tail_width_pt=2.1,
    color="#f2f2f2",
    edgecolor="#f2f2f2",
    edgewidth=1.0,
    alpha=1.0,
    draw_N=True,
    N_fontsize=10,     # reduced for compact figure
    N_offset_pt=(0, 3)
):
    """
    North arrow drawn in AXES coordinates so it stays upright and consistent size.
    """
    fig = ax.figure
    if fig.canvas is not None:
        fig.canvas.draw()

    base = offset_copy(ax.transAxes, fig=fig, x=offset_pt[0], y=offset_pt[1], units="points")

    bbox = ax.get_window_extent()
    axpix_h = bbox.height
    dy_axes = (arrow_len_pt * fig.dpi / 72.0) / axpix_h

    x_axes, y_axes = ax_xy
    start = (x_axes, y_axes)
    end   = (x_axes, y_axes + dy_axes)

    style = ArrowStyle.Simple(head_length=head_length_pt, head_width=head_width_pt, tail_width=tail_width_pt)
    patch = FancyArrowPatch(
        start, end,
        transform=ax.transAxes,
        arrowstyle=style,
        mutation_scale=1.0,
        linewidth=edgewidth,
        edgecolor=edgecolor,
        facecolor=color,
        alpha=alpha,
        joinstyle="miter",
        capstyle="butt",
        zorder=6
    )
    patch.set_transform(base)
    ax.add_patch(patch)

    if draw_N:
        tx = ax.text(
            x_axes, y_axes + dy_axes, "N",
            transform=ax.transAxes,
            ha="center",
            va="bottom",
            fontsize=N_fontsize,
            color=color,
            zorder=6,
            fontname="Arial"
        )
        tx.set_transform(offset_copy(
            ax.transAxes, fig=fig,
            x=N_offset_pt[0] + offset_pt[0],
            y=N_offset_pt[1] + offset_pt[1],
            units="points"
        ))

def _add_scale_and_compass(ax):
    """
    Bottom-right scalebar + compass centered above it.
    Updated for compact figure on dark basemap.
    """
    fig = ax.figure
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    corner_pad_px = 16  # tighter for compact layout
    pad_pt = corner_pad_px * 72.0 / fig.dpi
    seg_h_pt = 7
    scale_text_breathe_pt = 7

    # scale length from axis width (meters; EPSG:3857)
    x0, x1 = ax.get_xlim()
    scale_len_m = _nice_scale_length_m(abs(x1 - x0))

    label_kwargs = dict(
        color="#f2f2f2",
        fontsize=9.0,
        ha="center",
        va="top",
        fontname="Arial",
        fontstretch="normal"
    )

    def _draw_scale(offset_y_pt):
        return add_scale_bar_dual(
            ax,
            total_length_m=scale_len_m,
            n_segments=2,
            segment_colors=("#f2f2f2", "#222222"),
            segment_height_pt=seg_h_pt,
            edgecolor="#f2f2f2",
            edgewidth=0.9,
            ax_xy=(1.0, 0.0),
            offset_pt=(-pad_pt, offset_y_pt),
            align="right",
            direction="left",
            label=True,
            label_above=False,
            pad_text_pt=scale_text_breathe_pt,
            label_kwargs=label_kwargs,
            return_info=True
        )

    # initial guess then adjust to exact pixel padding
    offset_y_pt = pad_pt + (seg_h_pt / 2.0) + scale_text_breathe_pt + 4.0
    sb = _draw_scale(offset_y_pt)
    fig.canvas.draw()

    txt = sb["text"]
    if txt is not None:
        ax_bbox = ax.get_window_extent(renderer=renderer)
        txt_bbox = txt.get_window_extent(renderer=renderer)
        current_bottom_pad_px = txt_bbox.y0 - ax_bbox.y0
        dy_px = corner_pad_px - current_bottom_pad_px
        dy_pt = dy_px * 72.0 / fig.dpi

        if abs(dy_pt) > 0.25:
            for p in sb["patches"]:
                p.remove()
            txt.remove()
            sb = _draw_scale(offset_y_pt + dy_pt)
            fig.canvas.draw()

    # compass centered above scale bar
    (cx_data, y_anchor_data) = sb["center_data"]
    cx_disp, cy_disp = ax.transData.transform((cx_data, y_anchor_data))
    compass_gap_pt = (seg_h_pt / 2.0) + 8.0
    cy_disp_up = cy_disp + compass_gap_pt * fig.dpi / 72.0
    cx_axes, cy_axes = ax.transAxes.inverted().transform((cx_disp, cy_disp_up))

    add_compass(
        ax,
        ax_xy=(cx_axes, cy_axes),
        offset_pt=(0, 0),
        arrow_len_pt=30,
        head_width_pt=9,
        head_length_pt=10,
        tail_width_pt=1.9,
        color="#f2f2f2",
        edgecolor="#f2f2f2",
        edgewidth=0.9,
        draw_N=True,
        N_fontsize=9,
        N_offset_pt=(0, 2),
    )

# ------------------------------------------------------------
# DATA PREP HELPERS
# ------------------------------------------------------------
def _ensure_dest_cols(track_df, meta_df, tag_col="tag"):
    """Ensure flyHome/flyAgamon exist on track_df without creating _x/_y duplicates."""
    out = track_df.copy()

    need = ["flyHome", "flyAgamon"]
    missing = [c for c in need if c not in out.columns]
    if missing:
        meta_dest = (
            meta_df[[tag_col, "flyHome", "flyAgamon"]]
            .drop_duplicates(subset=[tag_col], keep="first")
        )
        out = out.merge(meta_dest, on=tag_col, how="left", validate="many_to_one")

    out["flyHome"] = pd.to_numeric(out["flyHome"], errors="coerce").fillna(0).astype(int)
    out["flyAgamon"] = pd.to_numeric(out["flyAgamon"], errors="coerce").fillna(0).astype(int)
    return out

def _dest_label_from_flags(flyHome, flyAgamon):
    fh = int(flyHome == 1)
    fa = int(flyAgamon == 1)
    if fh and not fa:
        return "Home"
    if fa and not fh:
        return "Agamon"
    if fh and fa:
        return "Ambiguous"
    return "Unknown"

def _make_segment_and_gap_lines_from_df(
    df,
    *,
    tag_col="tag", lat_col="lat", lon_col="lon",
    treat_col="treatment", treat_name_col="treatmentName",
    segment_col="segment_id",
    gap_col="gap_before",
):
    """
    Build:
      (1) One LineString per contiguous segment (solid)
      (2) One LineString per detected gap (dashed connector)
    """
    d = df.dropna(subset=[tag_col, lat_col, lon_col]).copy()
    d[lat_col] = pd.to_numeric(d[lat_col], errors="coerce")
    d[lon_col] = pd.to_numeric(d[lon_col], errors="coerce")
    d = d[np.isfinite(d[lat_col]) & np.isfinite(d[lon_col])].copy()

    order_col = _pick_order_cols(d)

    seg_records = []
    gap_records = []

    has_segments = segment_col in d.columns
    has_gaps = gap_col in d.columns

    for tag, g_tag in d.groupby(tag_col, sort=False):
        g_tag = g_tag.sort_values(order_col) if order_col else g_tag.sort_index()

        flyHome = int(g_tag["flyHome"].iloc[0]) if "flyHome" in g_tag.columns else 0
        flyAgamon = int(g_tag["flyAgamon"].iloc[0]) if "flyAgamon" in g_tag.columns else 0
        dest = _dest_label_from_flags(flyHome, flyAgamon)

        trt = g_tag[treat_col].iloc[0] if treat_col in g_tag.columns else np.nan
        trt_name = g_tag[treat_name_col].iloc[0] if treat_name_col in g_tag.columns else "Unknown"

        # solid segment lines
        if has_segments:
            for seg_id, g_seg in g_tag.groupby(segment_col, sort=True):
                coords = list(zip(g_seg[lon_col].to_numpy(), g_seg[lat_col].to_numpy()))
                if len(coords) < 2:
                    continue
                seg_records.append({
                    "tag": tag,
                    "treatment": trt,
                    "treatmentName": trt_name,
                    "dest": dest,
                    "segment_id": int(seg_id),
                    "geometry": LineString(coords),
                })
        else:
            coords = list(zip(g_tag[lon_col].to_numpy(), g_tag[lat_col].to_numpy()))
            if len(coords) >= 2:
                seg_records.append({
                    "tag": tag,
                    "treatment": trt,
                    "treatmentName": trt_name,
                    "dest": dest,
                    "segment_id": 0,
                    "geometry": LineString(coords),
                })

        # dashed gap connectors
        if has_gaps:
            gb = pd.to_numeric(g_tag[gap_col], errors="coerce").fillna(0).astype(int).to_numpy()
            gap_starts = np.where(gb == 1)[0]  # this row is first point after a gap
            for j in gap_starts:
                if j <= 0:
                    continue
                x0, y0 = float(g_tag.iloc[j - 1][lon_col]), float(g_tag.iloc[j - 1][lat_col])
                x1, y1 = float(g_tag.iloc[j][lon_col]), float(g_tag.iloc[j][lat_col])
                if not (np.isfinite(x0) and np.isfinite(y0) and np.isfinite(x1) and np.isfinite(y1)):
                    continue
                gap_records.append({
                    "tag": tag,
                    "treatment": trt,
                    "treatmentName": trt_name,
                    "dest": dest,
                    "geometry": LineString([(x0, y0), (x1, y1)]),
                })

    if len(seg_records):
        seg_gdf = gpd.GeoDataFrame(seg_records, geometry="geometry", crs="EPSG:4326")
    else:
        seg_gdf = gpd.GeoDataFrame(
            {"tag": [], "treatment": [], "treatmentName": [], "dest": [], "segment_id": [], "geometry": []},
            geometry="geometry", crs="EPSG:4326"
        )

    if len(gap_records):
        gap_gdf = gpd.GeoDataFrame(gap_records, geometry="geometry", crs="EPSG:4326")
    else:
        gap_gdf = gpd.GeoDataFrame(
            {"tag": [], "treatment": [], "treatmentName": [], "dest": [], "geometry": []},
            geometry="geometry", crs="EPSG:4326"
        )

    seg_gdf = seg_gdf.to_crs(epsg=WEBMERC_EPSG)
    gap_gdf = gap_gdf.to_crs(epsg=WEBMERC_EPSG)
    return seg_gdf, gap_gdf

# ------------------------------------------------------------
# MAIN PLOT
# ------------------------------------------------------------
def plot_rediscretized_destination_panels(
    resampled_df,
    meta_df,
    *,
    tag_col="tag",
    lat_col="lat",
    lon_col="lon",
    treat_col="treatment",
    treat_name_col="treatmentName",
    out_pdf="../figures/bats/rediscretized_destination_panels_compact.pdf",
    out_svg="../figures/bats/rediscretized_destination_panels_compact.svg",
    lake_facecolor="tab:cyan",
    lake_alpha=0.35,
    # line style
    res_lw=1.35,
    res_alpha=0.95,
    basemap_alpha=1.0,
    # gap style
    gap_lw=None,
    gap_alpha=0.85,
    gap_linestyle="--",
    # compact figure + larger text
    fig_w=5.6,
    fig_h=3.8,
    title_fs=12.5,
    legend_fs=7.2,
):
    # 1) Ensure destination flags exist
    res_df = _ensure_dest_cols(resampled_df, meta_df, tag_col=tag_col)

    # 2) Build segment/gap geodataframes
    res_segs_gdf, res_gaps_gdf = _make_segment_and_gap_lines_from_df(
        res_df,
        tag_col=tag_col, lat_col=lat_col, lon_col=lon_col,
        treat_col=treat_col, treat_name_col=treat_name_col,
        segment_col="segment_id",
        gap_col="gap_before",
    )

    print("rediscretized segment-lines:", len(res_segs_gdf))
    print("gap connectors:", len(res_gaps_gdf))

    if len(res_segs_gdf) == 0:
        raise ValueError("No lines to plot. Check that lat/lon are present and not all NaN.")

    if gap_lw is None:
        gap_lw = max(0.8, res_lw * 0.85)

    # 3) Reference + translocation points
    pts = [{"site": "Reference", "geometry": Point(ROOST_LON, ROOST_LAT)}]
    pts += [{"site": "Translocation", "geometry": Point(lon, lat)} for (lat, lon) in TRANS_POINTS]
    pts_gdf = gpd.GeoDataFrame(pts, crs="EPSG:4326").to_crs(epsg=WEBMERC_EPSG)

    # 4) Lake polygon
    lake_gdf_3857 = gpd.read_file(LAKE_GEOJSON).to_crs(epsg=WEBMERC_EPSG)

    # 5) Treatment colors (seaborn colorblind)
    treatments = sorted(pd.unique(res_segs_gdf[treat_col].dropna())) if treat_col in res_segs_gdf.columns else []
    if len(treatments):
        treat_colors = sns.color_palette("colorblind", n_colors=len(treatments))
        treat_to_color = {t: mcolors.to_hex(treat_colors[i]) for i, t in enumerate(treatments)}
    else:
        treat_to_color = {}

    if treat_name_col in res_segs_gdf.columns and len(treatments):
        name_map = res_segs_gdf.groupby(treat_col)[treat_name_col].first().to_dict()
        treat_to_label = {t: _format_treatment_name(name_map.get(t, "Unknown")) for t in treatments}
    else:
        treat_to_label = {t: str(t) for t in treatments}

    # 6) Figure: compact + larger fonts relative to size
    fig, axes = plt.subplots(1, 2, figsize=(fig_w, fig_h), dpi=DPI, constrained_layout=False)
    panels = [("Home", axes[0], "Home"), ("Agamon", axes[1], "Agamon Hula Lake")]

    # Unified extent
    layers_for_bounds = [g for g in [res_segs_gdf, res_gaps_gdf, pts_gdf, lake_gdf_3857] if len(g)]
    minx = min(g.total_bounds[0] for g in layers_for_bounds)
    miny = min(g.total_bounds[1] for g in layers_for_bounds)
    maxx = max(g.total_bounds[2] for g in layers_for_bounds)
    maxy = max(g.total_bounds[3] for g in layers_for_bounds)

    pad_x = (maxx - minx) * 0.05 if maxx > minx else 1000
    pad_y = (maxy - miny) * 0.05 if maxy > miny else 1000
    xlim = (minx - pad_x, maxx + pad_x)
    ylim = (miny - pad_y, maxy + pad_y)

    for dest, ax, title in panels:
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)

        # basemap
        try:
            cx.add_basemap(ax, source=cx.providers.CartoDB.DarkMatter, attribution=False, alpha=basemap_alpha)
        except Exception as e:
            print("Basemap failed (continuing without tiles):", e)

        # lake ONLY on Agamon panel
        if dest == "Agamon" and len(lake_gdf_3857):
            lake_gdf_3857.plot(
                ax=ax,
                facecolor=lake_facecolor,
                edgecolor=lake_facecolor,
                alpha=lake_alpha,
                linewidth=1.0,
                zorder=12,
            )
            lake_gdf_3857.boundary.plot(
                ax=ax,
                color=lake_facecolor,
                linewidth=1.1,
                alpha=1.0,
                zorder=13,
            )

        # dashed gap connectors (first)
        if len(res_gaps_gdf):
            sub_gap = res_gaps_gdf.loc[res_gaps_gdf["dest"] == dest]
            for t in treatments:
                sub_gap_t = sub_gap.loc[sub_gap[treat_col] == t]
                if len(sub_gap_t):
                    sub_gap_t.plot(
                        ax=ax,
                        color=treat_to_color[t],
                        linewidth=gap_lw,
                        alpha=gap_alpha,
                        linestyle=gap_linestyle,
                        zorder=26,
                    )

        # solid rediscretized segments
        sub_seg = res_segs_gdf.loc[res_segs_gdf["dest"] == dest]
        for t in treatments:
            sub_seg_t = sub_seg.loc[sub_seg[treat_col] == t]
            if len(sub_seg_t):
                sub_seg_t.plot(
                    ax=ax,
                    color=treat_to_color[t],
                    linewidth=res_lw,
                    alpha=res_alpha,
                    zorder=28,
                )

        # reference + translocation markers
        ref = pts_gdf.loc[pts_gdf["site"] == "Reference"]
        ax.scatter(
            ref.geometry.x, ref.geometry.y,
            marker="*", s=150, facecolors="white", edgecolors="black",
            linewidths=0.8, zorder=50
        )

        trans = pts_gdf.loc[pts_gdf["site"] == "Translocation"]
        ax.scatter(
            trans.geometry.x, trans.geometry.y,
            marker="X", s=90, facecolors="white", edgecolors="black",
            linewidths=0.8, zorder=50
        )

        ax.set_aspect("equal", adjustable="box")
        ax.set_title(title, fontsize=title_fs, pad=2.0)
        hide_axes(ax)

    # --- Legend: top-right INSIDE right plot, compact and properly fitting ---
    # Treatment handles are intentionally omitted (as in your current code).
    other_handles = [
        Line2D([0], [0], marker="*", color="none",
               markerfacecolor="white", markeredgecolor="black",
               markeredgewidth=0.8, markersize=7.2, label="Roost"),
        Line2D([0], [0], marker="X", color="none",
               markerfacecolor="white", markeredgecolor="black",
               markeredgewidth=0.8, markersize=6.2, label="Translocation"),
    ]
    if len(res_gaps_gdf):
        other_handles.append(
            Line2D([0, 1], [0, 0], color="black", lw=1.0,
                   linestyle=gap_linestyle, alpha=0.95, label="Signal dropout")
        )

    leg = axes[1].legend(
        handles=other_handles,
        loc="upper right",
        bbox_to_anchor=(0.985, 0.985),
        borderaxespad=0.2,
        frameon=True,
        framealpha=0.92,
        facecolor="white",
        edgecolor="0.8",
        fontsize=legend_fs,
        handlelength=1.4,
        handletextpad=0.45,
        labelspacing=0.35,
        borderpad=0.35,
        columnspacing=0.7,
        markerscale=0.9,  # prevent marker overflow
    )

    # Tight/compact layout (no external legend margin needed)
    fig.subplots_adjust(left=0.01, right=0.995, bottom=0.02, top=0.90, wspace=0.02)

    # scale + compass on RIGHT panel
    _add_scale_and_compass(axes[1])

    fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.01, dpi=DPI)
    fig.savefig(out_svg, bbox_inches="tight", pad_inches=0.01, dpi=DPI)
    plt.show()

    return fig, axes

# -------------------------
# RUN
# -------------------------
# track_w_treat should include gap_before + segment_id
fig, axes = plot_rediscretized_destination_panels(
    resampled_df=track_w_treat,
    meta_df=meta_df,
    out_pdf="../figures/bats/rediscretized_destination_panels_compact.pdf",
    out_svg="../figures/bats/rediscretized_destination_panels_compact.svg",
    fig_w=2.5,   # compact
    fig_h=3.8,   # compact
    title_fs=9, #larger relative to figure size
    legend_fs=6,  # compact legend that fits inside box
)

rediscretized segment-lines: 88
gap connectors: 24


/home/jake/.venvs/unwrap-env/lib/python3.12/site-packages/contextily/tile.py:298: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tile_urls = [provider.build_url(x=tile.x, y=tile.y, z=tile.z) for tile in tiles]


/home/jake/.venvs/unwrap-env/lib/python3.12/site-packages/contextily/tile.py:298: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tile_urls = [provider.build_url(x=tile.x, y=tile.y, z=tile.z) for tile in tiles]


Per-treatment destination panels (compact).


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString, Point
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import matplotlib as mpl
import contextily as cx
from matplotlib.transforms import offset_copy
from matplotlib.patches import FancyArrowPatch, ArrowStyle
import seaborn as sns

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
LAKE_GEOJSON = bats_lake_outline_path()
WEBMERC_EPSG = 3857

# Special locations
ROOST_LAT, ROOST_LON = 33.112373, 35.584135
TRANS_POINTS = [
    (33.135136, 35.592712),
    (33.081932, 35.604462),
]

# ------------------------------------------------------------
# STYLE (match your pipeline + compact paper figure)
# ------------------------------------------------------------
DPI = 300
mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "pdf.compression": 9,
    "svg.fonttype": "none",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
    # Helps with huge paths in Agg rasterization (no visual change)
    "agg.path.chunksize": 20000,
})

# ------------------------------------------------------------
# SMALL UTILITIES
# ------------------------------------------------------------
def hide_axes(ax):
    """Remove axis labels, ticks, tick labels, and spines."""
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

def _format_treatment_name(name: str) -> str:
    if pd.isna(name):
        return "Unknown"
    s = str(name).strip().lower()
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if "control" in parts and len(parts) > 1:
        parts = [p for p in parts if p != "control"]
    if len(parts) == 0:
        parts = ["control"]
    parts = sorted(parts)
    parts = [p[:1].upper() + p[1:] for p in parts]
    return ", ".join(parts)

def _nice_scale_length_m(width_m: float) -> float:
    # choose a nice 1/2/5 * 10^k length around ~20% of map width
    target = max(width_m * 0.2, 1.0)
    exp = np.floor(np.log10(target))
    base = target / (10 ** exp)
    if base < 1.5:
        nice = 1.0
    elif base < 3.5:
        nice = 2.0
    elif base < 7.5:
        nice = 5.0
    else:
        nice = 10.0
    return float(nice * (10 ** exp))

def _pick_order_cols(df):
    """Pick the first reasonable ordering column that exists."""
    for c in ["frame", "step", "t", "time", "timestamp"]:
        if c in df.columns:
            return c
    return None

# ------------------------------------------------------------
# ARTIST CAPTURE / RASTERIZATION HELPERS
# ------------------------------------------------------------
def _plot_gdf_capture_artists(gdf, ax, **kwargs):
    """
    Plot a GeoDataFrame and return the matplotlib artists added by this call.
    Useful for selectively rasterizing only the heavy trajectory layers.
    """
    n_col = len(ax.collections)
    n_line = len(ax.lines)
    n_patch = len(ax.patches)

    gdf.plot(ax=ax, **kwargs)

    new_artists = []
    new_artists.extend(list(ax.collections[n_col:]))
    new_artists.extend(list(ax.lines[n_line:]))
    new_artists.extend(list(ax.patches[n_patch:]))
    return new_artists

def _set_rasterized(artists, rasterized=True):
    for a in artists:
        try:
            a.set_rasterized(rasterized)
        except Exception:
            pass

def _style_trajectory_artists(artists):
    """
    Make rasterized trajectories look smoother in vector outputs (PDF/SVG).
    """
    for a in artists:
        for method_name, val in [
            ("set_antialiased", True),
            ("set_antialiaseds", [True]),
            ("set_capstyle", "round"),
            ("set_joinstyle", "round"),
            ("set_snap", False),
        ]:
            if hasattr(a, method_name):
                try:
                    getattr(a, method_name)(val)
                except Exception:
                    pass

# ------------------------------------------------------------
# SCALE BAR + COMPASS
# ------------------------------------------------------------
def add_scale_bar_dual(
    ax,
    total_length_m=20.0,
    n_segments=2,
    segment_colors=("#f2f2f2", "#222222"),
    segment_height_pt=8,
    edgecolor="#f2f2f2",
    edgewidth=0.9,
    label=True,
    label_above=False,
    label_kwargs=None,
    ax_xy=(0.95, 0.08),
    offset_pt=(0, -41),
    align="right",
    direction="left",
    pad_text_pt=4,
    return_info=False
):
    """
    Two-tone horizontal scale bar (meters, data units).
    Placement uses axes coords + point offset. Drawn as rectangles in data coords.
    """
    if label_kwargs is None:
        label_kwargs = dict(
            color="#f2f2f2",
            fontsize=9.5,
            ha="center",
            va="top",
            fontname="Arial",
            fontstretch="normal",
        )

    fig = ax.figure
    if fig.canvas is not None:
        fig.canvas.draw()

    base = offset_copy(ax.transAxes, fig=fig, x=offset_pt[0], y=offset_pt[1], units="points")
    x_disp, y_disp = base.transform(ax_xy)
    x_anchor, y_anchor = ax.transData.inverted().transform((x_disp, y_disp))

    seg_len = total_length_m / float(max(n_segments, 1))
    if direction == "left":
        x0 = (x_anchor if align == "left" else x_anchor - total_length_m)
        xdir = +1
    elif direction == "right":
        x0 = (x_anchor if align == "left" else x_anchor + total_length_m)
        if align == "right":
            x0 = x_anchor - total_length_m
        xdir = +1
    else:
        raise ValueError("direction must be 'left' or 'right'")

    # Convert bar height (points) to data units using axes pixel height
    bbox = ax.get_window_extent()
    axpix_h = bbox.height
    y_min, y_max = ax.get_ylim()
    pt_to_data_y = (y_max - y_min) / axpix_h
    half_h = 0.5 * segment_height_pt * pt_to_data_y
    yb, yt = y_anchor - half_h, y_anchor + half_h

    colors = (segment_colors * ((n_segments + len(segment_colors) - 1) // len(segment_colors)))[:n_segments]

    patches = []
    for i in range(n_segments):
        xa = x0 + xdir * (i * seg_len)
        xb = x0 + xdir * ((i + 1) * seg_len)
        p = plt.Rectangle(
            (min(xa, xb), yb),
            abs(xb - xa),
            yt - yb,
            facecolor=colors[i],
            edgecolor=edgecolor,
            linewidth=edgewidth,
            zorder=5
        )
        ax.add_patch(p)
        patches.append(p)

    x_center = x0 + xdir * (total_length_m / 2.0)
    txt = None
    if label:
        txt = ax.text(x_center, y_anchor, f"{int(total_length_m)} m", **label_kwargs)
        dy = pad_text_pt if label_above else -pad_text_pt
        txt.set_transform(offset_copy(ax.transData, fig=fig, x=0, y=dy, units="points"))
        txt.set_zorder(6)

    if return_info:
        return {
            "center_data": (x_center, y_anchor),
            "y_anchor_data": y_anchor,
            "segment_height_pt": segment_height_pt,
            "offset_pt": offset_pt,
            "ax_xy": ax_xy,
            "total_length_m": total_length_m,
            "patches": patches,
            "text": txt,
        }
    return None

def add_compass(
    ax,
    ax_xy=(0.92, 0.18),
    offset_pt=(0, -95),
    arrow_len_pt=36,
    head_width_pt=10,
    head_length_pt=12,
    tail_width_pt=2.1,
    color="#f2f2f2",
    edgecolor="#f2f2f2",
    edgewidth=1.0,
    alpha=1.0,
    draw_N=True,
    N_fontsize=10,
    N_offset_pt=(0, 3)
):
    """
    North arrow drawn in AXES coordinates so it stays upright and consistent size.
    """
    fig = ax.figure
    if fig.canvas is not None:
        fig.canvas.draw()

    base = offset_copy(ax.transAxes, fig=fig, x=offset_pt[0], y=offset_pt[1], units="points")

    bbox = ax.get_window_extent()
    axpix_h = bbox.height
    dy_axes = (arrow_len_pt * fig.dpi / 72.0) / axpix_h

    x_axes, y_axes = ax_xy
    start = (x_axes, y_axes)
    end = (x_axes, y_axes + dy_axes)

    style = ArrowStyle.Simple(head_length=head_length_pt, head_width=head_width_pt, tail_width=tail_width_pt)
    patch = FancyArrowPatch(
        start, end,
        transform=ax.transAxes,
        arrowstyle=style,
        mutation_scale=1.0,
        linewidth=edgewidth,
        edgecolor=edgecolor,
        facecolor=color,
        alpha=alpha,
        joinstyle="miter",
        capstyle="butt",
        zorder=6
    )
    patch.set_transform(base)
    ax.add_patch(patch)

    if draw_N:
        tx = ax.text(
            x_axes, y_axes + dy_axes, "N",
            transform=ax.transAxes,
            ha="center",
            va="bottom",
            fontsize=N_fontsize,
            color=color,
            zorder=6,
            fontname="Arial"
        )
        tx.set_transform(offset_copy(
            ax.transAxes, fig=fig,
            x=N_offset_pt[0] + offset_pt[0],
            y=N_offset_pt[1] + offset_pt[1],
            units="points"
        ))

def _add_scale_and_compass(ax):
    """
    Bottom-right scalebar + compass centered above it.
    Styled for a LIGHT basemap (e.g., Positron): dark annotation colors.
    """
    fig = ax.figure
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    corner_pad_px = 16  # tighter for compact layout
    pad_pt = corner_pad_px * 72.0 / fig.dpi
    seg_h_pt = 7
    scale_text_breathe_pt = 7

    # scale length from axis width (meters; EPSG:3857)
    x0, x1 = ax.get_xlim()
    scale_len_m = _nice_scale_length_m(abs(x1 - x0))

    # DARK (black) text for Positron/light basemap
    label_kwargs = dict(
        color="#000000",
        fontsize=9.0,
        ha="center",
        va="top",
        fontname="Arial",
        fontstretch="normal"
    )

    def _draw_scale(offset_y_pt):
        return add_scale_bar_dual(
            ax,
            total_length_m=scale_len_m,
            n_segments=2,
            # black/white scalebar for light background
            segment_colors=("#ffffff", "#000000"),
            segment_height_pt=seg_h_pt,
            edgecolor="#000000",
            edgewidth=0.9,
            ax_xy=(1.0, 0.0),
            offset_pt=(-pad_pt, offset_y_pt),
            align="right",
            direction="left",
            label=True,
            label_above=False,
            pad_text_pt=scale_text_breathe_pt,
            label_kwargs=label_kwargs,
            return_info=True
        )

    # initial guess then adjust to exact pixel padding
    offset_y_pt = pad_pt + (seg_h_pt / 2.0) + scale_text_breathe_pt + 4.0
    sb = _draw_scale(offset_y_pt)
    fig.canvas.draw()

    txt = sb["text"]
    if txt is not None:
        ax_bbox = ax.get_window_extent(renderer=renderer)
        txt_bbox = txt.get_window_extent(renderer=renderer)
        current_bottom_pad_px = txt_bbox.y0 - ax_bbox.y0
        dy_px = corner_pad_px - current_bottom_pad_px
        dy_pt = dy_px * 72.0 / fig.dpi

        if abs(dy_pt) > 0.25:
            for p in sb["patches"]:
                p.remove()
            txt.remove()
            sb = _draw_scale(offset_y_pt + dy_pt)
            fig.canvas.draw()

    # compass centered above scale bar
    (cx_data, y_anchor_data) = sb["center_data"]
    cx_disp, cy_disp = ax.transData.transform((cx_data, y_anchor_data))
    compass_gap_pt = (seg_h_pt / 2.0) + 8.0
    cy_disp_up = cy_disp + compass_gap_pt * fig.dpi / 72.0
    cx_axes, cy_axes = ax.transAxes.inverted().transform((cx_disp, cy_disp_up))

    add_compass(
        ax,
        ax_xy=(cx_axes, cy_axes),
        offset_pt=(0, 0),
        arrow_len_pt=30,
        head_width_pt=9,
        head_length_pt=10,
        tail_width_pt=1.9,
        # DARK compass for light basemap
        color="#000000",
        edgecolor="#000000",
        edgewidth=0.9,
        draw_N=True,
        N_fontsize=9,
        N_offset_pt=(0, 2),
    )

# ------------------------------------------------------------
# DATA PREP HELPERS
# ------------------------------------------------------------
def _ensure_dest_cols(track_df, meta_df, tag_col="tag"):
    """Ensure flyHome/flyAgamon exist on track_df without creating _x/_y duplicates."""
    out = track_df.copy()

    need = ["flyHome", "flyAgamon"]
    missing = [c for c in need if c not in out.columns]
    if missing:
        meta_dest = (
            meta_df[[tag_col, "flyHome", "flyAgamon"]]
            .drop_duplicates(subset=[tag_col], keep="first")
        )
        out = out.merge(meta_dest, on=tag_col, how="left", validate="many_to_one")

    out["flyHome"] = pd.to_numeric(out["flyHome"], errors="coerce").fillna(0).astype(int)
    out["flyAgamon"] = pd.to_numeric(out["flyAgamon"], errors="coerce").fillna(0).astype(int)
    return out

def _dest_label_from_flags(flyHome, flyAgamon):
    fh = int(flyHome == 1)
    fa = int(flyAgamon == 1)
    if fh and not fa:
        return "Home"
    if fa and not fh:
        return "Agamon"
    if fh and fa:
        return "Ambiguous"
    return "Unknown"

def _make_segment_and_gap_lines_from_df(
    df,
    *,
    tag_col="tag", lat_col="lat", lon_col="lon",
    treat_col="treatment", treat_name_col="treatmentName",
    segment_col="segment_id",
    gap_col="gap_before",
):
    """
    Build:
      (1) One LineString per contiguous segment (solid)
      (2) One LineString per detected gap (dashed connector)
    """
    d = df.dropna(subset=[tag_col, lat_col, lon_col]).copy()
    d[lat_col] = pd.to_numeric(d[lat_col], errors="coerce")
    d[lon_col] = pd.to_numeric(d[lon_col], errors="coerce")
    d = d[np.isfinite(d[lat_col]) & np.isfinite(d[lon_col])].copy()

    order_col = _pick_order_cols(d)

    seg_records = []
    gap_records = []

    has_segments = segment_col in d.columns
    has_gaps = gap_col in d.columns

    for tag, g_tag in d.groupby(tag_col, sort=False):
        g_tag = g_tag.sort_values(order_col) if order_col else g_tag.sort_index()

        flyHome = int(g_tag["flyHome"].iloc[0]) if "flyHome" in g_tag.columns else 0
        flyAgamon = int(g_tag["flyAgamon"].iloc[0]) if "flyAgamon" in g_tag.columns else 0
        dest = _dest_label_from_flags(flyHome, flyAgamon)

        trt = g_tag[treat_col].iloc[0] if treat_col in g_tag.columns else np.nan
        trt_name = g_tag[treat_name_col].iloc[0] if treat_name_col in g_tag.columns else "Unknown"

        # solid segment lines
        if has_segments:
            for seg_id, g_seg in g_tag.groupby(segment_col, sort=True):
                coords = list(zip(g_seg[lon_col].to_numpy(), g_seg[lat_col].to_numpy()))
                if len(coords) < 2:
                    continue
                seg_records.append({
                    "tag": tag,
                    "treatment": trt,
                    "treatmentName": trt_name,
                    "dest": dest,
                    "segment_id": int(seg_id),
                    "geometry": LineString(coords),
                })
        else:
            coords = list(zip(g_tag[lon_col].to_numpy(), g_tag[lat_col].to_numpy()))
            if len(coords) >= 2:
                seg_records.append({
                    "tag": tag,
                    "treatment": trt,
                    "treatmentName": trt_name,
                    "dest": dest,
                    "segment_id": 0,
                    "geometry": LineString(coords),
                })

        # dashed gap connectors
        if has_gaps:
            gb = pd.to_numeric(g_tag[gap_col], errors="coerce").fillna(0).astype(int).to_numpy()
            gap_starts = np.where(gb == 1)[0]  # this row is first point after a gap
            for j in gap_starts:
                if j <= 0:
                    continue
                x0, y0 = float(g_tag.iloc[j - 1][lon_col]), float(g_tag.iloc[j - 1][lat_col])
                x1, y1 = float(g_tag.iloc[j][lon_col]), float(g_tag.iloc[j][lat_col])
                if not (np.isfinite(x0) and np.isfinite(y0) and np.isfinite(x1) and np.isfinite(y1)):
                    continue
                gap_records.append({
                    "tag": tag,
                    "treatment": trt,
                    "treatmentName": trt_name,
                    "dest": dest,
                    "geometry": LineString([(x0, y0), (x1, y1)]),
                })

    if len(seg_records):
        seg_gdf = gpd.GeoDataFrame(seg_records, geometry="geometry", crs="EPSG:4326")
    else:
        seg_gdf = gpd.GeoDataFrame(
            {"tag": [], "treatment": [], "treatmentName": [], "dest": [], "segment_id": [], "geometry": []},
            geometry="geometry", crs="EPSG:4326"
        )

    if len(gap_records):
        gap_gdf = gpd.GeoDataFrame(gap_records, geometry="geometry", crs="EPSG:4326")
    else:
        gap_gdf = gpd.GeoDataFrame(
            {"tag": [], "treatment": [], "treatmentName": [], "dest": [], "geometry": []},
            geometry="geometry", crs="EPSG:4326"
        )

    seg_gdf = seg_gdf.to_crs(epsg=WEBMERC_EPSG)
    gap_gdf = gap_gdf.to_crs(epsg=WEBMERC_EPSG)
    return seg_gdf, gap_gdf

# ------------------------------------------------------------
# MAIN PLOT
# ------------------------------------------------------------
def plot_rediscretized_destination_panels(
    resampled_df,
    meta_df,
    *,
    tag_col="tag",
    lat_col="lat",
    lon_col="lon",
    treat_col="treatment",
    treat_name_col="treatmentName",
    out_pdf="../figures/bats/rediscretized_destination_panels_compact.pdf",
    out_svg="../figures/bats/rediscretized_destination_panels_compact.svg",
    lake_facecolor="tab:cyan",
    lake_alpha=0.35,
    # line style
    res_lw=1.35,
    res_alpha=0.95,
    basemap_alpha=1.0,
    # gap style
    gap_lw=None,
    gap_alpha=0.85,
    gap_linestyle="--",
    # compact figure + larger text
    fig_w=5.6,
    fig_h=3.8,
    title_fs=12.5,
    legend_fs=7.2,
    # selective rasterization controls
    rasterize_trajectories_pdf=True,
    rasterize_trajectories_svg=False,   # keep SVG trajectories vector by default (nicest)
    pdf_raster_dpi=300,                 # dpi used for rasterized layers inside PDF
    svg_raster_dpi=600,                 # only matters if rasterize_trajectories_svg=True
):
    # 1) Ensure destination flags exist
    res_df = _ensure_dest_cols(resampled_df, meta_df, tag_col=tag_col)

    # 2) Build segment/gap geodataframes
    res_segs_gdf, res_gaps_gdf = _make_segment_and_gap_lines_from_df(
        res_df,
        tag_col=tag_col, lat_col=lat_col, lon_col=lon_col,
        treat_col=treat_col, treat_name_col=treat_name_col,
        segment_col="segment_id",
        gap_col="gap_before",
    )

    print("rediscretized segment-lines:", len(res_segs_gdf))
    print("gap connectors:", len(res_gaps_gdf))

    if len(res_segs_gdf) == 0:
        raise ValueError("No lines to plot. Check that lat/lon are present and not all NaN.")

    if gap_lw is None:
        gap_lw = max(0.8, res_lw * 0.85)

    # 3) Reference + translocation points
    pts = [{"site": "Reference", "geometry": Point(ROOST_LON, ROOST_LAT)}]
    pts += [{"site": "Translocation", "geometry": Point(lon, lat)} for (lat, lon) in TRANS_POINTS]
    pts_gdf = gpd.GeoDataFrame(pts, crs="EPSG:4326").to_crs(epsg=WEBMERC_EPSG)

    # 4) Lake polygon
    lake_gdf_3857 = gpd.read_file(LAKE_GEOJSON).to_crs(epsg=WEBMERC_EPSG)

    # 5) Treatment colors (seaborn colorblind)
    treatments = sorted(pd.unique(res_segs_gdf[treat_col].dropna())) if treat_col in res_segs_gdf.columns else []
    if len(treatments):
        treat_colors = sns.color_palette("colorblind", n_colors=len(treatments))
        treat_to_color = {t: mcolors.to_hex(treat_colors[i]) for i, t in enumerate(treatments)}
    else:
        treat_to_color = {}

    if treat_name_col in res_segs_gdf.columns and len(treatments):
        name_map = res_segs_gdf.groupby(treat_col)[treat_name_col].first().to_dict()
        treat_to_label = {t: _format_treatment_name(name_map.get(t, "Unknown")) for t in treatments}
    else:
        treat_to_label = {t: str(t) for t in treatments}

    # 6) Figure: compact + larger fonts relative to size
    fig, axes = plt.subplots(1, 2, figsize=(fig_w, fig_h), dpi=DPI, constrained_layout=False)
    panels = [("Home", axes[0], "Home"), ("Agamon", axes[1], "Agamon Hula Lake")]

    # Track only trajectory artists (solid segments + gap connectors)
    trajectory_artists = []

    # Unified extent
    layers_for_bounds = [g for g in [res_segs_gdf, res_gaps_gdf, pts_gdf, lake_gdf_3857] if len(g)]
    minx = min(g.total_bounds[0] for g in layers_for_bounds)
    miny = min(g.total_bounds[1] for g in layers_for_bounds)
    maxx = max(g.total_bounds[2] for g in layers_for_bounds)
    maxy = max(g.total_bounds[3] for g in layers_for_bounds)

    pad_x = (maxx - minx) * 0.05 if maxx > minx else 1000
    pad_y = (maxy - miny) * 0.05 if maxy > miny else 1000
    xlim = (minx - pad_x, maxx + pad_x)
    ylim = (miny - pad_y, maxy + pad_y)

    for dest, ax, title in panels:
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)

        # basemap (already raster by nature)
        try:
            cx.add_basemap(ax, source=cx.providers.CartoDB.PositronNoLabels, attribution=False, alpha=basemap_alpha)
        except Exception as e:
            print("Basemap failed (continuing without tiles):", e)

        # lake ONLY on Agamon panel (keep vector)
        if dest == "Agamon" and len(lake_gdf_3857):
            lake_gdf_3857.plot(
                ax=ax,
                facecolor=lake_facecolor,
                edgecolor=lake_facecolor,
                alpha=lake_alpha,
                linewidth=1.0,
                zorder=12,
            )
            lake_gdf_3857.boundary.plot(
                ax=ax,
                color=lake_facecolor,
                linewidth=1.1,
                alpha=1.0,
                zorder=13,
            )

        # dashed gap connectors (heavy layer -> candidate for rasterization)
        if len(res_gaps_gdf):
            sub_gap = res_gaps_gdf.loc[res_gaps_gdf["dest"] == dest]
            for t in treatments:
                sub_gap_t = sub_gap.loc[sub_gap[treat_col] == t]
                if len(sub_gap_t):
                    artists = _plot_gdf_capture_artists(
                        sub_gap_t,
                        ax=ax,
                        color=treat_to_color[t],
                        linewidth=gap_lw,
                        alpha=gap_alpha,
                        linestyle=gap_linestyle,
                        zorder=26,
                    )
                    trajectory_artists.extend(artists)

        # solid rediscretized segments (heavy layer -> candidate for rasterization)
        sub_seg = res_segs_gdf.loc[res_segs_gdf["dest"] == dest]
        for t in treatments:
            sub_seg_t = sub_seg.loc[sub_seg[treat_col] == t]
            if len(sub_seg_t):
                artists = _plot_gdf_capture_artists(
                    sub_seg_t,
                    ax=ax,
                    color=treat_to_color[t],
                    linewidth=res_lw,
                    alpha=res_alpha,
                    zorder=28,
                )
                trajectory_artists.extend(artists)

        # Improve trajectory appearance whether vector or rasterized
        _style_trajectory_artists(trajectory_artists)

        # reference + translocation markers (keep vector)
        ref = pts_gdf.loc[pts_gdf["site"] == "Reference"]
        ax.scatter(
            ref.geometry.x, ref.geometry.y,
            marker="*", s=150, facecolors="white", edgecolors="black",
            linewidths=0.8, zorder=50
        )

        trans = pts_gdf.loc[pts_gdf["site"] == "Translocation"]
        ax.scatter(
            trans.geometry.x, trans.geometry.y,
            marker="X", s=90, facecolors="white", edgecolors="black",
            linewidths=0.8, zorder=50
        )

        ax.set_aspect("equal", adjustable="box")
        ax.set_title(title, fontsize=title_fs, pad=2.0)
        hide_axes(ax)
        
        # black outline around the map panel
        rect = plt.Rectangle(
            (0, 0), 1, 1,
            transform=ax.transAxes,
            fill=False,
            edgecolor="black",
            linewidth=1.0,
            zorder=100,
            clip_on=False
        )
        ax.add_patch(rect)

    # --- Legend: top-right INSIDE right plot, compact and properly fitting ---
    other_handles = [
        Line2D([0], [0], marker="*", color="none",
               markerfacecolor="white", markeredgecolor="black",
               markeredgewidth=0.8, markersize=7.2, label="Roost"),
        Line2D([0], [0], marker="X", color="none",
               markerfacecolor="white", markeredgecolor="black",
               markeredgewidth=0.8, markersize=6.2, label="Translocation"),
    ]
    if len(res_gaps_gdf):
        other_handles.append(
            Line2D([0, 1], [0, 0], color="black", lw=1.0,
                   linestyle=gap_linestyle, alpha=0.95, label="Signal dropout")
        )

    axes[1].legend(
        handles=other_handles,
        loc="upper right",
        bbox_to_anchor=(0.985, 0.985),
        borderaxespad=0.2,
        frameon=True,
        framealpha=0.92,
        facecolor="white",
        edgecolor="0.8",
        fontsize=legend_fs,
        handlelength=1.4,
        handletextpad=0.45,
        labelspacing=0.35,
        borderpad=0.35,
        columnspacing=0.7,
        markerscale=0.9,
    )

    # Tight/compact layout
    fig.subplots_adjust(left=0.01, right=0.995, bottom=0.02, top=0.90, wspace=0.02)

    # scale + compass on RIGHT panel (now styled dark for Positron)
    _add_scale_and_compass(axes[0])

    # --------------------------------------------------------
    # SAVE: selectively rasterize ONLY trajectories per export
    # --------------------------------------------------------
    if out_pdf:
        _set_rasterized(trajectory_artists, rasterize_trajectories_pdf)
        fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.01, dpi=pdf_raster_dpi)

    if out_svg:
        _set_rasterized(trajectory_artists, rasterize_trajectories_svg)
        fig.savefig(out_svg, bbox_inches="tight", pad_inches=0.01, dpi=svg_raster_dpi)

    plt.show()
    return fig, axes

# -------------------------
# RUN
# -------------------------
# track_w_treat should include gap_before + segment_id
fig, axes = plot_rediscretized_destination_panels(
    resampled_df=track_w_treat,
    meta_df=meta_df,
    out_pdf="../figures/bats/rediscretized_destination_panels_compact.pdf",
    out_svg="../figures/bats/rediscretized_destination_panels_compact.svg",
    fig_w=2.5,      # compact
    fig_h=3.8,      # compact
    title_fs=9,     # larger relative to figure size
    legend_fs=6,    # compact legend that fits inside box

    # Key behavior:
    rasterize_trajectories_pdf=True,   # fast/light PDF
    rasterize_trajectories_svg=False,  # keep SVG trajectories vector (best-looking/editable)
    pdf_raster_dpi=300,
    svg_raster_dpi=600,                # only used if rasterize_trajectories_svg=True
)

rediscretized segment-lines: 88
gap connectors: 24


/home/jake/.venvs/unwrap-env/lib/python3.12/site-packages/contextily/tile.py:298: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tile_urls = [provider.build_url(x=tile.x, y=tile.y, z=tile.z) for tile in tiles]


/home/jake/.venvs/unwrap-env/lib/python3.12/site-packages/contextily/tile.py:298: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tile_urls = [provider.build_url(x=tile.x, y=tile.y, z=tile.z) for tile in tiles]


After-localisation tracking and destination panels.


In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString, Point
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import seaborn as sns
import contextily as cx

# ------------------------------------------------------------
# Put matplotlib back to "normal" (defaults) for the rest of your notebook
# ------------------------------------------------------------
mpl.rcdefaults()

# ------------------------------------------------------------
# Assumes you already defined (from your style code):
#   - hide_axes(ax)
#   - _add_scale_and_compass(ax)
#   - _format_treatment_name(name)
# ------------------------------------------------------------

LAKE_GEOJSON = bats_lake_outline_path()
WEBMERC_EPSG = 3857

# Special locations
ROOST_LAT, ROOST_LON = 33.112373, 35.584135
TRANS_POINTS = [
    (33.135136, 35.592712),
    (33.081932, 35.604462),
]

def _ensure_dest_cols(track_df, meta_df, tag_col="tag"):
    """Ensure flyHome/flyAgamon exist on track_df without creating _x/_y duplicates."""
    out = track_df.copy()

    need = ["flyHome", "flyAgamon"]
    missing = [c for c in need if c not in out.columns]
    if missing:
        meta_dest = (
            meta_df[[tag_col, "flyHome", "flyAgamon"]]
            .drop_duplicates(subset=[tag_col], keep="first")
        )
        out = out.merge(meta_dest, on=tag_col, how="left", validate="many_to_one")

    out["flyHome"] = pd.to_numeric(out["flyHome"], errors="coerce").fillna(0).astype(int)
    out["flyAgamon"] = pd.to_numeric(out["flyAgamon"], errors="coerce").fillna(0).astype(int)
    return out

def _dest_label_from_flags(flyHome, flyAgamon):
    fh = int(flyHome == 1)
    fa = int(flyAgamon == 1)
    if fh and not fa:
        return "Home"
    if fa and not fh:
        return "Agamon"
    if fh and fa:
        return "Ambiguous"
    return "Unknown"

def _pick_order_cols(df):
    """Pick the first reasonable ordering column that exists."""
    for c in ["frame", "step", "t", "time", "timestamp"]:
        if c in df.columns:
            return c
    return None

def _make_segment_and_gap_lines_from_df(
    df,
    *,
    tag_col="tag", lat_col="lat", lon_col="lon",
    treat_col="treatment", treat_name_col="treatmentName",
    segment_col="segment_id",
    gap_col="gap_before",
):
    """
    Build:
      (1) One LineString per contiguous segment (solid)
      (2) One LineString per detected gap (dashed connector between last point before gap and first after)

    Expects resampled df to include `segment_id` and `gap_before` (from your gap-aware rediscretization).
    If they don't exist, falls back to one LineString per tag and no gaps.
    """
    d = df.dropna(subset=[tag_col, lat_col, lon_col]).copy()
    d[lat_col] = pd.to_numeric(d[lat_col], errors="coerce")
    d[lon_col] = pd.to_numeric(d[lon_col], errors="coerce")
    d = d[np.isfinite(d[lat_col]) & np.isfinite(d[lon_col])].copy()

    order_col = _pick_order_cols(d)

    seg_records = []
    gap_records = []

    has_segments = segment_col in d.columns
    has_gaps = gap_col in d.columns

    for tag, g_tag in d.groupby(tag_col, sort=False):
        g_tag = g_tag.sort_values(order_col) if order_col else g_tag.sort_index()

        flyHome = int(g_tag["flyHome"].iloc[0]) if "flyHome" in g_tag.columns else 0
        flyAgamon = int(g_tag["flyAgamon"].iloc[0]) if "flyAgamon" in g_tag.columns else 0
        dest = _dest_label_from_flags(flyHome, flyAgamon)

        trt = g_tag[treat_col].iloc[0] if treat_col in g_tag.columns else np.nan
        trt_name = g_tag[treat_name_col].iloc[0] if treat_name_col in g_tag.columns else "Unknown"

        # ---- SOLID segment lines ----
        if has_segments:
            for seg_id, g_seg in g_tag.groupby(segment_col, sort=True):
                coords = list(zip(g_seg[lon_col].to_numpy(), g_seg[lat_col].to_numpy()))
                if len(coords) < 2:
                    continue
                seg_records.append({
                    "tag": tag,
                    "treatment": trt,
                    "treatmentName": trt_name,
                    "dest": dest,
                    "segment_id": int(seg_id),
                    "geometry": LineString(coords),
                })
        else:
            coords = list(zip(g_tag[lon_col].to_numpy(), g_tag[lat_col].to_numpy()))
            if len(coords) >= 2:
                seg_records.append({
                    "tag": tag,
                    "treatment": trt,
                    "treatmentName": trt_name,
                    "dest": dest,
                    "segment_id": 0,
                    "geometry": LineString(coords),
                })

        # ---- DASHED gap connectors ----
        if has_gaps:
            gb = pd.to_numeric(g_tag[gap_col], errors="coerce").fillna(0).astype(int).to_numpy()
            gap_starts = np.where(gb == 1)[0]
            for j in gap_starts:
                if j <= 0:
                    continue
                x0, y0 = float(g_tag.iloc[j-1][lon_col]), float(g_tag.iloc[j-1][lat_col])
                x1, y1 = float(g_tag.iloc[j][lon_col]), float(g_tag.iloc[j][lat_col])
                if not (np.isfinite(x0) and np.isfinite(y0) and np.isfinite(x1) and np.isfinite(y1)):
                    continue
                gap_records.append({
                    "tag": tag,
                    "treatment": trt,
                    "treatmentName": trt_name,
                    "dest": dest,
                    "geometry": LineString([(x0, y0), (x1, y1)]),
                })

    seg_gdf = gpd.GeoDataFrame(seg_records, geometry="geometry", crs="EPSG:4326") if len(seg_records) else \
        gpd.GeoDataFrame({"tag": [], "treatment": [], "treatmentName": [], "dest": [], "segment_id": [], "geometry": []},
                         geometry="geometry", crs="EPSG:4326")

    gap_gdf = gpd.GeoDataFrame(gap_records, geometry="geometry", crs="EPSG:4326") if len(gap_records) else \
        gpd.GeoDataFrame({"tag": [], "treatment": [], "treatmentName": [], "dest": [], "geometry": []},
                         geometry="geometry", crs="EPSG:4326")

    seg_gdf = seg_gdf.to_crs(epsg=WEBMERC_EPSG)
    gap_gdf = gap_gdf.to_crs(epsg=WEBMERC_EPSG)
    return seg_gdf, gap_gdf


def plot_rediscretized_destination_panels(
    resampled_df,
    meta_df,
    *,
    tag_col="tag",
    lat_col="lat",
    lon_col="lon",
    treat_col="treatment",
    treat_name_col="treatmentName",
    out_pdf="rediscretized_destination_panels.pdf",
    out_svg="rediscretized_destination_panels.svg",
    # colorblind-ish cyan replacement for tab:cyan
    lake_facecolor="#56B4E9",
    lake_alpha=0.35,
    # style knobs
    res_lw: float = 1.6,
    res_alpha: float = 0.95,
    # basemap knobs
    basemap_alpha: float = 1.0,
    basemap_zoom: int | None = None,         # higher zoom => more detailed tiles (but more tiles)
    basemap_retina: bool = True,             # use @2x tiles when provider supports {r}
    basemap_interpolation: str = "bilinear", # smoothing of raster display
    save_dpi: int = 300,                     # affects raster layers in PDF/SVG export
    # gap style
    gap_lw: float | None = None,
    gap_alpha: float = 0.85,
    gap_linestyle: str = "--",
    # legend
    show_treatment_legend: bool = True,
):
    # Apply your preferred export/style settings ONLY inside this plot,
    # so other figures in the notebook keep "normal" defaults.
    style_rc = {
        "font.family": "Arial",
        "font.sans-serif": ["Arial"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",  # keep text as text in SVG
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
    }

    with mpl.rc_context(rc=style_rc):
        # 1) Ensure dest flags exist on rediscretized df
        res_df = _ensure_dest_cols(resampled_df, meta_df, tag_col=tag_col)

        # 2) Build GeoDataFrames: solid segments + dashed gaps
        res_segs_gdf, res_gaps_gdf = _make_segment_and_gap_lines_from_df(
            res_df,
            tag_col=tag_col, lat_col=lat_col, lon_col=lon_col,
            treat_col=treat_col, treat_name_col=treat_name_col,
            segment_col="segment_id",
            gap_col="gap_before",
        )

        # (optional) debug prints
        # print("rediscretized segment-lines:", len(res_segs_gdf))
        # print("gap connectors:", len(res_gaps_gdf))

        if len(res_segs_gdf) == 0:
            raise ValueError("No lines to plot. Check that lat/lon are present and not all NaN.")

        if gap_lw is None:
            gap_lw = max(0.9, res_lw * 0.9)

        # 3) Points layer (reference + translocations)
        pts = [{"site": "Reference", "geometry": Point(ROOST_LON, ROOST_LAT)}]
        pts += [{"site": "Translocation", "geometry": Point(lon, lat)} for (lat, lon) in TRANS_POINTS]
        pts_gdf = gpd.GeoDataFrame(pts, crs="EPSG:4326").to_crs(epsg=WEBMERC_EPSG)

        # 4) Lake polygon
        lake_gdf_3857 = gpd.read_file(LAKE_GEOJSON).to_crs(epsg=WEBMERC_EPSG)

        # 5) Treatment colors (seaborn colorblind)
        treatments = sorted(pd.unique(res_segs_gdf[treat_col].dropna())) if treat_col in res_segs_gdf.columns else []
        if len(treatments):
            treat_colors = sns.color_palette("colorblind", n_colors=len(treatments))
            treat_to_color = {t: mcolors.to_hex(treat_colors[i]) for i, t in enumerate(treatments)}
        else:
            treat_to_color = {}

        # labels (use treatmentName if present)
        if treat_name_col in res_segs_gdf.columns and len(treatments):
            name_map = res_segs_gdf.groupby(treat_col)[treat_name_col].first().to_dict()
            treat_to_label = {t: _format_treatment_name(name_map.get(t, "Unknown")) for t in treatments}
        else:
            treat_to_label = {t: str(t) for t in treatments}

        # basemap provider, optionally "retina" (@2x) if supported
        basemap_source = cx.providers.CartoDB.DarkMatter
        if basemap_retina and hasattr(basemap_source, "build_url"):
            try:
                # returns a URL string with {r} filled (provider supports retina via {r})
                basemap_source = basemap_source.build_url(r="@2x")
            except Exception:
                pass

        # 6) Figure panels
        fig, axes = plt.subplots(1, 2, figsize=(7, 8), constrained_layout=False)
        panels = [("Home", axes[0], "Home"), ("Agamon", axes[1], "Agamon Hula Lake")]

        # unified extent across everything
        layers_for_bounds = [g for g in [res_segs_gdf, res_gaps_gdf, pts_gdf, lake_gdf_3857] if len(g)]
        minx = min(g.total_bounds[0] for g in layers_for_bounds)
        miny = min(g.total_bounds[1] for g in layers_for_bounds)
        maxx = max(g.total_bounds[2] for g in layers_for_bounds)
        maxy = max(g.total_bounds[3] for g in layers_for_bounds)

        pad_x = (maxx - minx) * 0.06 if maxx > minx else 1000
        pad_y = (maxy - miny) * 0.06 if maxy > miny else 1000
        xlim = (minx - pad_x, maxx + pad_x)
        ylim = (miny - pad_y, maxy + pad_y)

        for dest, ax, title in panels:
            ax.set_xlim(*xlim)
            ax.set_ylim(*ylim)

            # basemap (higher zoom and/or retina improves apparent raster detail)
            try:
                cx.add_basemap(
                    ax,
                    source=basemap_source,
                    attribution=False,
                    alpha=basemap_alpha,
                    zoom=basemap_zoom,              # None = auto; set e.g. 14/15/16 for sharper tiles
                    interpolation=basemap_interpolation,
                )
            except Exception as e:
                print("Basemap failed (continuing without tiles):", e)

            # lake polygon ONLY on Agamon panel
            if dest == "Agamon" and len(lake_gdf_3857):
                lake_gdf_3857.plot(
                    ax=ax,
                    facecolor=lake_facecolor,
                    edgecolor=lake_facecolor,
                    alpha=lake_alpha,
                    linewidth=1.2,
                    zorder=12
                )
                lake_gdf_3857.boundary.plot(
                    ax=ax,
                    color=lake_facecolor,
                    linewidth=1.4,
                    alpha=1.0,
                    zorder=13
                )

            # --- Dashed gap connectors (draw FIRST, then solid segments on top) ---
            if len(res_gaps_gdf):
                sub_gap = res_gaps_gdf.loc[res_gaps_gdf["dest"] == dest]
                for t in treatments:
                    sub_gap_t = sub_gap.loc[sub_gap[treat_col] == t]
                    if len(sub_gap_t):
                        sub_gap_t.plot(
                            ax=ax,
                            color=treat_to_color[t],
                            linewidth=gap_lw,
                            alpha=gap_alpha,
                            linestyle=gap_linestyle,
                            zorder=26
                        )

            # --- Solid rediscretized segments (colored) ---
            sub_seg = res_segs_gdf.loc[res_segs_gdf["dest"] == dest]
            for t in treatments:
                sub_seg_t = sub_seg.loc[sub_seg[treat_col] == t]
                if len(sub_seg_t):
                    sub_seg_t.plot(
                        ax=ax,
                        color=treat_to_color[t],
                        linewidth=res_lw,
                        alpha=res_alpha,
                        zorder=28
                    )

            # reference + translocation markers
            ref = pts_gdf.loc[pts_gdf["site"] == "Reference"]
            ax.scatter(ref.geometry.x, ref.geometry.y,
                       marker="*", s=220, facecolors="white", edgecolors="black",
                       linewidths=0.9, zorder=50)

            trans = pts_gdf.loc[pts_gdf["site"] == "Translocation"]
            ax.scatter(trans.geometry.x, trans.geometry.y,
                       marker="X", s=140, facecolors="white", edgecolors="black",
                       linewidths=0.9, zorder=50)

            ax.set_aspect("equal", adjustable="box")
            ax.set_title(title, fontsize=16)
            hide_axes(ax)

        # --- Legend ---
        treat_handles = (
            [Patch(color=treat_to_color[t], label=treat_to_label[t]) for t in treatments]
            if show_treatment_legend and len(treatments) else
            []
        )

        other_handles = [
            Line2D([0], [0], marker="*", color="none", markerfacecolor="white",
                   markeredgecolor="black", markersize=12, label="Roost"),
            Line2D([0], [0], marker="X", color="none", markerfacecolor="white",
                   markeredgecolor="black", markersize=9, label="Translocation"),
        ]
        if len(res_gaps_gdf):
            other_handles.append(Line2D([0, 1], [0, 0], color="black", lw=gap_lw,
                                        linestyle=gap_linestyle, alpha=gap_alpha, label="Signal Dropout"))

        fig.legend(
            handles=treat_handles + other_handles,
            loc="center",
            bbox_to_anchor=(1.02, 0.5),
            frameon=True,
            framealpha=0.9,
        )

        plt.tight_layout(rect=[0, 0, 0.86, 1])

        # scale + compass on RIGHT panel (Agamon)
        _add_scale_and_compass(axes[1])

        os.makedirs(os.path.dirname(out_pdf), exist_ok=True)
        os.makedirs(os.path.dirname(out_svg), exist_ok=True)

        # Note: for raster tiles, real detail mainly comes from basemap_zoom / retina.
        fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.02, dpi=save_dpi)
        fig.savefig(out_svg, bbox_inches="tight", pad_inches=0.02, dpi=save_dpi)

        plt.show()
        plt.close(fig)


# -------------------------
# RUN
# -------------------------
# track_w_treat is your downstream rediscretized df (should include gap_before + segment_id)
plot_rediscretized_destination_panels(
    resampled_df=track_w_treat,
    meta_df=meta_df,
    out_pdf="../figures/bats/rediscretized_destination_panels.pdf",
    out_svg="../figures/bats/rediscretized_destination_panels.svg",
    # suggested tweaks for sharper basemap:
    basemap_zoom=15,      # try 14–16; higher = sharper but heavier
    basemap_retina=True,  # use @2x tiles if supported
    save_dpi=400,         # helps raster export a bit (esp. pdf/svg embedding)
    show_treatment_legend=False
)

/tmp/ipykernel_3898688/1100424530.py:262: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  basemap_source = basemap_source.build_url(r="@2x")


Destination flow fields: smoothed-distance and harmonic distance fields over the arena, with the
negative-gradient (goal-approach) direction field that supplies `bearing_to_dest` in the goal-relative
constructions above.


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString, Polygon, MultiPolygon
from shapely.prepared import prep
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import contextily as cx

# ------------------------------------------------------------
# Assumes you already defined (from your style code):
#   - hide_axes(ax)
#   - _add_scale_and_compass(ax)   (now optional via flag)
#
# If _add_scale_and_compass is NOT defined, we add a small fallback.
# ------------------------------------------------------------

LAKE_GEOJSON = bats_lake_outline_path()
WEBMERC_EPSG = 3857

ROOST_LAT, ROOST_LON = 33.112373, 35.584135
TRANS_POINTS = [
    (33.135136, 35.592712),
    (33.081932, 35.604462),
]

# -------------------------
# Fallback UI: scale + compass (only used if you don't have yours)
# -------------------------
def _nice_scale_length_m(width_m: float, target_frac: float = 0.22) -> float:
    width_m = float(abs(width_m))
    if not np.isfinite(width_m) or width_m <= 0:
        return 1000.0
    target = max(width_m * float(target_frac), 1.0)
    exp = np.floor(np.log10(target))
    base = target / (10 ** exp)
    if base < 1.5:
        nice = 1.0
    elif base < 3.5:
        nice = 2.0
    elif base < 7.5:
        nice = 5.0
    else:
        nice = 10.0
    return float(nice * (10 ** exp))

def _format_scale_label(total_m: float) -> str:
    total_m = float(total_m)
    if np.isfinite(total_m) and total_m >= 1000 and abs(total_m % 1000) < 1e-6:
        return f"{int(round(total_m/1000))} km"
    return f"{int(round(total_m))} m"

def _fallback_add_scale_and_compass(ax, *, target_frac=0.22):
    x0, x1 = ax.get_xlim()
    width_m = abs(x1 - x0)
    L = _nice_scale_length_m(width_m, target_frac=target_frac)

    # scale bar (axes coords)
    xA0, yA0 = 0.05, 0.06
    frac = (L / width_m) if width_m > 0 else 0.2
    bar = min(0.40, max(0.14, frac))

    ax.plot([xA0, xA0 + bar], [yA0, yA0], transform=ax.transAxes, color="white",
            lw=6, solid_capstyle="butt", zorder=300)
    ax.plot([xA0, xA0 + bar / 2], [yA0, yA0], transform=ax.transAxes, color="black",
            lw=6, solid_capstyle="butt", zorder=301)
    ax.text(xA0 + bar / 2, yA0 - 0.02, _format_scale_label(L),
            transform=ax.transAxes, ha="center", va="top", color="white", fontsize=9, zorder=302)

    ax.annotate("N", xy=(0.92, 0.20), xytext=(0.92, 0.08),
                xycoords="axes fraction", textcoords="axes fraction",
                ha="center", va="bottom", color="white", fontsize=10,
                arrowprops=dict(arrowstyle="-|>", color="white", lw=1.4),
                zorder=303)

if "_add_scale_and_compass" not in globals():
    def _add_scale_and_compass(ax):
        _fallback_add_scale_and_compass(ax)

# -------------------------
# Small helpers
# -------------------------
def _safe_normalize(dx, dy):
    nrm = np.hypot(dx, dy)
    outx = np.full_like(dx, np.nan, dtype=float)
    outy = np.full_like(dy, np.nan, dtype=float)
    m = np.isfinite(nrm) & (nrm > 0)
    outx[m] = dx[m] / nrm[m]
    outy[m] = dy[m] / nrm[m]
    return outx, outy

def _ensure_dest_cols(track_df, meta_df, tag_col="tag"):
    out = track_df.copy()
    need = ["flyHome", "flyAgamon"]
    missing = [c for c in need if c not in out.columns]
    if missing:
        meta_dest = (
            meta_df[[tag_col, "flyHome", "flyAgamon"]]
            .drop_duplicates(subset=[tag_col], keep="first")
        )
        out = out.merge(meta_dest, on=tag_col, how="left", validate="many_to_one")
    out["flyHome"] = pd.to_numeric(out["flyHome"], errors="coerce").fillna(0).astype(int)
    out["flyAgamon"] = pd.to_numeric(out["flyAgamon"], errors="coerce").fillna(0).astype(int)
    return out

def _dest_label_from_flags(flyHome, flyAgamon):
    fh = int(flyHome == 1)
    fa = int(flyAgamon == 1)
    if fh and not fa:
        return "Home"
    if fa and not fh:
        return "Agamon"
    if fh and fa:
        return "Ambiguous"
    return "Unknown"

def _pick_order_cols(df):
    for c in ["frame", "step", "t", "time", "timestamp"]:
        if c in df.columns:
            return c
    return None

def _make_lines_from_df(df, *, tag_col="tag", lat_col="lat", lon_col="lon"):
    d = df.dropna(subset=[tag_col, lat_col, lon_col]).copy()
    d[lat_col] = pd.to_numeric(d[lat_col], errors="coerce")
    d[lon_col] = pd.to_numeric(d[lon_col], errors="coerce")
    d = d[np.isfinite(d[lat_col]) & np.isfinite(d[lon_col])].copy()

    order_col = _pick_order_cols(d)
    recs = []
    for tag, g_tag in d.groupby(tag_col, sort=False):
        g_tag = g_tag.sort_values(order_col) if order_col else g_tag.sort_index()
        flyHome = int(g_tag["flyHome"].iloc[0])
        flyAgamon = int(g_tag["flyAgamon"].iloc[0])
        dest = _dest_label_from_flags(flyHome, flyAgamon)
        coords = list(zip(g_tag[lon_col].to_numpy(), g_tag[lat_col].to_numpy()))
        if len(coords) < 2:
            continue
        recs.append({"tag": tag, "dest": dest, "geometry": LineString(coords)})

    if recs:
        return gpd.GeoDataFrame(recs, geometry="geometry", crs="EPSG:4326").to_crs(epsg=WEBMERC_EPSG)

    return gpd.GeoDataFrame({"tag": [], "dest": [], "geometry": []}, geometry="geometry", crs="EPSG:4326").to_crs(
        epsg=WEBMERC_EPSG
    )

# -------------------------
# Lake polygon for calculations: largest outer polygon only (drop holes)
# -------------------------
def lake_outer_polygon_largest(geom):
    """
    Return a Polygon using the EXTERIOR ring of the largest polygon in `geom`.
    This is used ONLY for mask/field calculations (stable outer boundary).
    """
    if geom is None:
        return None

    if isinstance(geom, Polygon):
        main = geom
    elif isinstance(geom, MultiPolygon):
        main = max(list(geom.geoms), key=lambda p: float(p.area))
    else:
        try:
            return lake_outer_polygon_largest(geom.buffer(0))
        except Exception:
            return None

    return Polygon(main.exterior.coords)  # drop interiors/holes

# -------------------------
# Mask on grid
# -------------------------
def polygon_mask_on_grid(poly, x_centers, y_centers, chunk=200_000):
    X, Y = np.meshgrid(x_centers, y_centers)
    xx, yy = X.ravel(), Y.ravel()

    try:
        import shapely
        pts = shapely.points(xx, yy)
        inside = shapely.contains(poly, pts)
        return np.asarray(inside, dtype=bool).reshape(Y.shape)
    except Exception:
        pass

    poly_p = prep(poly)
    inside = np.zeros(xx.size, dtype=bool)
    for a in range(0, xx.size, int(chunk)):
        b = min(xx.size, a + int(chunk))
        inside[a:b] = [poly_p.contains(Point(x, y)) for x, y in zip(xx[a:b], yy[a:b])]
    return inside.reshape(Y.shape)

def _adaptive_grid_res(xlim, ylim, base_res_m, max_n=420):
    x0, x1 = xlim
    y0, y1 = ylim
    w = float(abs(x1 - x0))
    h = float(abs(y1 - y0))
    res = float(base_res_m)
    if not np.isfinite(res) or res <= 0:
        res = max(w, h) / max_n if max(w, h) > 0 else 200.0
    res = max(res, w / max_n if w > 0 else res, h / max_n if h > 0 else res)
    return res

# -------------------------
# Harmonic field (Laplace) in 3857
# -------------------------
def solve_laplace_dirichlet(mask_lake, *, tol=1e-5, max_iter=20000):
    ny, nx = mask_lake.shape
    u = np.ones((ny, nx), float)
    u[mask_lake] = 0.0
    u[0, :] = u[-1, :] = u[:, 0] = u[:, -1] = 1.0

    free = (~mask_lake).copy()
    free[0, :] = free[-1, :] = free[:, 0] = free[:, -1] = False
    free_flat = np.flatnonzero(free.ravel())

    try:
        import scipy.sparse as sp
        import scipy.sparse.linalg as spla

        idx = -np.ones((ny, nx), int)
        idx.ravel()[free_flat] = np.arange(free_flat.size)
        n = free_flat.size
        if n == 0:
            return u

        A = sp.lil_matrix((n, n), float)
        b = np.zeros(n, float)

        def add(eq, r, c):
            j = idx[r, c]
            if j >= 0:
                A[eq, j] -= 1.0
            else:
                b[eq] += u[r, c]

        rr, cc = np.where(free)
        for r, c in zip(rr, cc):
            eq = idx[r, c]
            A[eq, eq] = 4.0
            add(eq, r - 1, c)
            add(eq, r + 1, c)
            add(eq, r, c - 1)
            add(eq, r, c + 1)

        sol, _ = spla.cg(A.tocsr(), b, atol=0.0, rtol=1e-8, maxiter=3000)
        u2 = u.ravel()
        u2[free_flat] = sol
        return u2.reshape((ny, nx))
    except Exception:
        u_old = u.copy()
        for _ in range(int(max_iter)):
            u_new = u_old.copy()
            u_new[1:-1, 1:-1] = 0.25 * (
                u_old[:-2, 1:-1] + u_old[2:, 1:-1] + u_old[1:-1, :-2] + u_old[1:-1, 2:]
            )
            u_new[mask_lake] = 0.0
            u_new[0, :] = u_new[-1, :] = u_new[:, 0] = u_new[:, -1] = 1.0
            if np.nanmax(np.abs(u_new - u_old)) < tol:
                return u_new
            u_old = u_new
        return u_old

def harmonic_field_on_extent(xlim, ylim, lake_poly_calc_3857, *, grid_res_m=160.0, tol=1e-5, max_grid_n=420):
    grid_res_m = _adaptive_grid_res(xlim, ylim, grid_res_m, max_n=max_grid_n)
    x0, x1 = xlim
    y0, y1 = ylim
    x_centers = np.arange(min(x0, x1), max(x0, x1) + grid_res_m, grid_res_m)
    y_centers = np.arange(min(y0, y1), max(y0, y1) + grid_res_m, grid_res_m)

    mask = polygon_mask_on_grid(lake_poly_calc_3857, x_centers, y_centers)
    u = solve_laplace_dirichlet(mask, tol=tol)

    du_dy, du_dx = np.gradient(u, grid_res_m, grid_res_m)
    tx, ty = _safe_normalize(-du_dx, -du_dy)

    X, Y = np.meshgrid(x_centers, y_centers)
    return X, Y, tx, ty, mask, grid_res_m

def smoothed_distance_field_on_extent(
    xlim, ylim, lake_poly_calc_3857, *, grid_res_m=160.0, sigma_pix=3.0, max_grid_n=420
):
    grid_res_m = _adaptive_grid_res(xlim, ylim, grid_res_m, max_n=max_grid_n)
    x0, x1 = xlim
    y0, y1 = ylim
    x_centers = np.arange(min(x0, x1), max(x0, x1) + grid_res_m, grid_res_m)
    y_centers = np.arange(min(y0, y1), max(y0, y1) + grid_res_m, grid_res_m)

    mask = polygon_mask_on_grid(lake_poly_calc_3857, x_centers, y_centers)

    from scipy.ndimage import distance_transform_edt, gaussian_filter

    dist = distance_transform_edt(~mask, sampling=(grid_res_m, grid_res_m)).astype(float)
    dist[mask] = 0.0
    dist_s = gaussian_filter(dist, sigma=float(sigma_pix), mode="nearest")

    du_dy, du_dx = np.gradient(dist_s, grid_res_m, grid_res_m)
    tx, ty = _safe_normalize(-du_dx, -du_dy)

    X, Y = np.meshgrid(x_centers, y_centers)
    return X, Y, tx, ty, mask, grid_res_m

# -------------------------
# Roost radial field (toward roost point)
# -------------------------
def roost_field_on_extent(xlim, ylim, roost_xy_3857, *, grid_res_m=220.0, max_grid_n=420):
    grid_res_m = _adaptive_grid_res(xlim, ylim, grid_res_m, max_n=max_grid_n)
    x0, x1 = xlim
    y0, y1 = ylim
    x_centers = np.arange(min(x0, x1), max(x0, x1) + grid_res_m, grid_res_m)
    y_centers = np.arange(min(y0, y1), max(y0, y1) + grid_res_m, grid_res_m)

    X, Y = np.meshgrid(x_centers, y_centers)
    rx, ry = float(roost_xy_3857[0]), float(roost_xy_3857[1])
    dx = rx - X
    dy = ry - Y
    tx, ty = _safe_normalize(dx, dy)
    return X, Y, tx, ty, grid_res_m

# ============================================================
# MAIN: same xlim/ylim/figsize as trajectory panels, flow only
# ============================================================
def plot_destination_flow_fields_panels(
    resampled_df,
    meta_df,
    *,
    tag_col="tag",
    lat_col="lat",
    lon_col="lon",
    out_pdf="destination_flow_fields.pdf",
    out_svg="destination_flow_fields.svg",
    figsize=(8, 9),                     # EXACT match to your trajectory panels
    lake_facecolor="tab:cyan",
    lake_alpha=0.35,
    lake_edge_lw=1.4,
    basemap_alpha: float = 1.0,
    basemap_source=cx.providers.CartoDB.DarkMatter,
    show_basemap: bool = True,
    show_ui_scale_compass: bool = True, # NEW: turn off scale+compass
    show_legend: bool = True,

    # ---- choose lake field ----
    lake_field: str = "harmonic",       # {"harmonic","smoothed_distance"}

    # ---- field resolution knobs (separate for lake vs roost) ----
    lake_grid_res_m: float = 160.0,
    lake_max_grid_n: int = 520,
    lake_jacobi_tol: float = 1e-5,
    lake_smooth_sigma_pix: float = 3.0,

    roost_grid_res_m: float = 220.0,
    roost_max_grid_n: int = 520,

    # ---- quiver display knobs (bigger arrows -> lower scale, larger width/head sizes) ----
    quiver_stride_lake: int = 1,
    quiver_stride_roost: int = 1,

    quiver_scale_lake: float = 22.0,    # LOWER = BIGGER arrows
    quiver_scale_roost: float = 22.0,   # LOWER = BIGGER arrows

    quiver_width_lake: float = 0.0042,
    quiver_width_roost: float = 0.0042,

    quiver_alpha_lake: float = 0.95,
    quiver_alpha_roost: float = 0.95,

    quiver_color_lake: str = "white",
    quiver_color_roost: str = "white",

    quiver_headwidth: float = 5.5,
    quiver_headlength: float = 7.0,
    quiver_headaxislength: float = 6.2,
    quiver_minshaft: float = 2.2,
):
    # 1) Ensure dest flags exist (extent matching)
    res_df = _ensure_dest_cols(resampled_df, meta_df, tag_col=tag_col)

    # 2) Build GeoDataFrame lines ONLY to replicate extent computation (same as trajectory plot)
    res_lines_gdf = _make_lines_from_df(res_df, tag_col=tag_col, lat_col=lat_col, lon_col=lon_col)
    if len(res_lines_gdf) == 0:
        raise ValueError("No rediscretized lines found for extent. Check lat/lon.")

    # 3) Points layer (reference + translocations)
    pts = [{"site": "Reference", "geometry": Point(ROOST_LON, ROOST_LAT)}]
    pts += [{"site": "Translocation", "geometry": Point(lon, lat)} for (lat, lon) in TRANS_POINTS]
    pts_gdf = gpd.GeoDataFrame(pts, crs="EPSG:4326").to_crs(epsg=WEBMERC_EPSG)

    # 4) Lake polygon (PLOT FULL geometry; COMPUTE field on largest outer polygon)
    lake_gdf_3857 = gpd.read_file(LAKE_GEOJSON).to_crs(epsg=WEBMERC_EPSG)
    if len(lake_gdf_3857) == 0:
        raise ValueError("Lake GeoJSON is empty or could not be read.")

    lake_full_union = (
        lake_gdf_3857.geometry.union_all()
        if hasattr(lake_gdf_3857.geometry, "union_all")
        else lake_gdf_3857.unary_union
    )
    lake_poly_calc_3857 = lake_outer_polygon_largest(lake_full_union)
    if lake_poly_calc_3857 is None:
        raise ValueError("Could not derive a largest outer polygon for lake calculations.")

    # 5) unified extent (SAME logic as trajectory plot)
    layers_for_bounds = [g for g in [res_lines_gdf, pts_gdf, lake_gdf_3857] if len(g)]
    minx = min(g.total_bounds[0] for g in layers_for_bounds)
    miny = min(g.total_bounds[1] for g in layers_for_bounds)
    maxx = max(g.total_bounds[2] for g in layers_for_bounds)
    maxy = max(g.total_bounds[3] for g in layers_for_bounds)

    pad_x = (maxx - minx) * 0.06 if maxx > minx else 1000
    pad_y = (maxy - miny) * 0.06 if maxy > miny else 1000
    xlim = (minx - pad_x, maxx + pad_x)
    ylim = (miny - pad_y, maxy + pad_y)

    # roost in 3857
    roost_3857 = gpd.GeoSeries([Point(ROOST_LON, ROOST_LAT)], crs="EPSG:4326").to_crs(epsg=WEBMERC_EPSG).iloc[0]
    roost_xy = (roost_3857.x, roost_3857.y)

    # 6) compute flow fields on SAME extent
    XR, YR, txR, tyR, _ = roost_field_on_extent(
        xlim, ylim, roost_xy, grid_res_m=float(roost_grid_res_m), max_grid_n=int(roost_max_grid_n)
    )

    field = str(lake_field).strip().lower()
    if field in ("harmonic", "laplace", "potential"):
        XL, YL, txL, tyL, maskL, _ = harmonic_field_on_extent(
            xlim, ylim, lake_poly_calc_3857,
            grid_res_m=float(lake_grid_res_m),
            tol=float(lake_jacobi_tol),
            max_grid_n=int(lake_max_grid_n),
        )
        lake_title = "Agamon Hula Lake"
    elif field in ("smoothed_distance", "smoothed", "edt", "distance"):
        XL, YL, txL, tyL, maskL, _ = smoothed_distance_field_on_extent(
            xlim, ylim, lake_poly_calc_3857,
            grid_res_m=float(lake_grid_res_m),
            sigma_pix=float(lake_smooth_sigma_pix),
            max_grid_n=int(lake_max_grid_n),
        )
        lake_title = f"Agamon Hula Lake"
    else:
        raise ValueError("lake_field must be one of {'harmonic','smoothed_distance'}")

    # 7) plot (same layout/size)
    fig, axes = plt.subplots(1, 2, figsize=figsize, constrained_layout=False)
    panels = [("Home", axes[0], "Home (roost flow)"), ("Agamon", axes[1], lake_title)]

    for dest, ax, title in panels:
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)

        if show_basemap:
            try:
                cx.add_basemap(ax, source=basemap_source, attribution=False, alpha=basemap_alpha)
            except Exception as e:
                print("Basemap failed (continuing without tiles):", e)

        # Plot FULL lake polygon only on Agamon panel (as before)
        if dest == "Agamon":
            lake_gdf_3857.plot(
                ax=ax,
                facecolor=lake_facecolor,
                edgecolor=lake_facecolor,
                alpha=lake_alpha,
                linewidth=1.2,
                zorder=12,
            )
            lake_gdf_3857.boundary.plot(
                ax=ax,
                color=lake_facecolor,
                linewidth=lake_edge_lw,
                alpha=1.0,
                zorder=13,
            )

        # --- flow fields ---
        if dest == "Home":
            s = max(1, int(quiver_stride_roost))
            ax.quiver(
                XR[::s, ::s], YR[::s, ::s],
                txR[::s, ::s], tyR[::s, ::s],
                scale=float(quiver_scale_roost),
                color=quiver_color_roost,
                width=float(quiver_width_roost),
                alpha=float(quiver_alpha_roost),
                headwidth=float(quiver_headwidth),
                headlength=float(quiver_headlength),
                headaxislength=float(quiver_headaxislength),
                minshaft=float(quiver_minshaft),
                zorder=25,
            )
        else:
            s = max(1, int(quiver_stride_lake))
            # IMPORTANT: do NOT draw inside lake (mask from the *largest outer polygon*)
            m = maskL[::s, ::s]
            tx_plot = np.ma.array(txL[::s, ::s], mask=m)
            ty_plot = np.ma.array(tyL[::s, ::s], mask=m)
            ax.quiver(
                XL[::s, ::s], YL[::s, ::s],
                tx_plot, ty_plot,
                scale=float(quiver_scale_lake),
                color=quiver_color_lake,
                width=float(quiver_width_lake),
                alpha=float(quiver_alpha_lake),
                headwidth=float(quiver_headwidth),
                headlength=float(quiver_headlength),
                headaxislength=float(quiver_headaxislength),
                minshaft=float(quiver_minshaft),
                zorder=25,
            )

        # reference + translocation markers
        ref = pts_gdf.loc[pts_gdf["site"] == "Reference"]
        ax.scatter(ref.geometry.x, ref.geometry.y,
                   marker="*", s=220, facecolors="white", edgecolors="black",
                   linewidths=0.9, zorder=50)

        trans = pts_gdf.loc[pts_gdf["site"] == "Translocation"]
        ax.scatter(trans.geometry.x, trans.geometry.y,
                   marker="X", s=140, facecolors="white", edgecolors="black",
                   linewidths=0.9, zorder=50)

        ax.set_aspect("equal", adjustable="box")
        ax.set_title(title)
        hide_axes(ax)

    # Legend (markers only)
    if show_legend:
        other_handles = [
            Line2D([0], [0], marker="*", color="none", markerfacecolor="white",
                   markeredgecolor="black", markersize=12, label="Roost"),
            Line2D([0], [0], marker="X", color="none", markerfacecolor="white",
                   markeredgecolor="black", markersize=9, label="Translocation"),
        ]
        fig.legend(
            handles=other_handles,
            loc="center",
            bbox_to_anchor=(1.0, 0.5),
            frameon=True,
            framealpha=0.9,
        )

    plt.tight_layout(rect=[0, 0, 0.86, 1])

    # Optional scale + compass on RIGHT panel
    if show_ui_scale_compass:
        _add_scale_and_compass(axes[1])

    fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.02, dpi=300)
    fig.savefig(out_svg, bbox_inches="tight", pad_inches=0.02, dpi=300)
    plt.show()


# -------------------------
# RUN (examples)
# -------------------------

# # Harmonic lake flow + roost flow (bigger arrows by default)
# plot_destination_flow_fields_panels(
#     resampled_df=track_w_treat,
#     meta_df=meta_df,
#     out_pdf="destination_flow_fields_harmonic.pdf",
#     out_svg="destination_flow_fields_harmonic.svg",
#     lake_field="harmonic",
#     lake_grid_res_m=140.0,
#     lake_jacobi_tol=1e-5,
#     roost_grid_res_m=180.0,
#     # big arrows:
#     quiver_scale_lake=18,
#     quiver_scale_roost=18,
#     quiver_width_lake=0.0045,
#     quiver_width_roost=0.0045,
#     quiver_stride_lake=1,
#     quiver_stride_roost=1,
#     show_ui_scale_compass=True,
# )

# Smoothed-distance lake flow (outer polygon for mask), UI turned OFF
plot_destination_flow_fields_panels(
    resampled_df=track_w_treat,
    meta_df=meta_df,
    out_pdf="../figures/bats/destination_flow_fields_smoothed_distance.pdf",
    out_svg="../figures/bats/destination_flow_fields_smoothed_distance.svg",
    lake_field="smoothed_distance",
    lake_grid_res_m=200.0,
    lake_smooth_sigma_pix=1.0,
    roost_grid_res_m=200.0,
    quiver_scale_lake=30,
    quiver_scale_roost=30,
    show_ui_scale_compass=False,   # <-- turn off scale+compass
)


/home/jake/.venvs/unwrap-env/lib/python3.12/site-packages/contextily/tile.py:298: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tile_urls = [provider.build_url(x=tile.x, y=tile.y, z=tile.z) for tile in tiles]


### Descriptive circular summaries by treatment

Per-treatment concentration summaries (descriptive, no model): estimated population concentration
($\hat\kappa$), mean resultant length ($\vec{R}$), and the alignment of each individual's mean
bearing to $0^\circ$ (goal direction). These give a first, model-free read on whether treatment
changes directional bias or precision.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

def _format_treatment_name(name: str) -> str:
    if pd.isna(name):
        return "Unknown"
    s = str(name).strip().lower()
    if not s:
        return "Unknown"
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if "control" in parts and len(parts) > 1:
        parts = [p for p in parts if p != "control"]
    if not parts:
        parts = ["control"]
    parts = sorted(parts)
    parts = [p[:1].upper() + p[1:] for p in parts]
    return ", ".join(parts)

def plot_polar_vonmises_kde_all(
    track_w_treat,
    angle_col="flightAngDiffDest_deg",
    tag_col="tag",
    treat_col="treatment",
    treat_name_col="treatmentName",
    kappa=30,
    grid_n=360,
    r0=1.0,              # density=0 -> r=1.0
    r_scale=1.0,         # density=1 -> r=2.0
    lw=2.0,
    ring_color="0.60",
    ring_lw=0.9,
    ring_alpha=0.65,
    spoke_color="0.75",
    spoke_lw=0.8,
    spoke_alpha=0.7,
    zero_seg_color="0.65",
    zero_seg_lw=1.6,
    zero_seg_alpha=0.95,
    title_fs=9,
):
    """
    - theta shown as (-180, 180]
    - ONLY two density rings: r=1 (dens=0) and r=2 (dens=1)
    - keep ONLY the 4 main direction spokes + tick labels
    - add a gray segment at 0° from r=1 to r=2
    - remove polar spine + default grid + radial ticks
    """
    df = track_w_treat[[tag_col, treat_col, treat_name_col, angle_col]].copy()
    df = df.dropna(subset=[tag_col, treat_col, angle_col])

    # degrees -> radians, wrapped to (-pi, pi]
    ang = df[angle_col].astype(float).to_numpy()
    ang_wrapped = ((ang + 180.0) % 360.0) - 180.0
    df["_theta"] = np.deg2rad(ang_wrapped)

    tag_meta = (
        df.groupby(tag_col, as_index=False)
          .agg(**{
              treat_col: (treat_col, "first"),
              treat_name_col: (treat_name_col, "first"),
          })
          .sort_values([treat_col, tag_col])
    )
    tags_ordered = tag_meta[tag_col].tolist()

    treatments = sorted(tag_meta[treat_col].unique())
    cmap = plt.get_cmap("tab10")
    treat_to_color = {t: cmap(i % 10) for i, t in enumerate(treatments)}

    n = len(tags_ordered)
    ncols = math.ceil(math.sqrt(n))
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(
        nrows, ncols,
        subplot_kw={"projection": "polar"},
        figsize=(3.2 * ncols, 3.2 * nrows),
    )
    axes = np.array(axes).reshape(-1)

    # KDE grid in (-pi, pi]
    theta_grid = np.linspace(-np.pi, np.pi, grid_n, endpoint=False)
    norm = 2 * np.pi * np.i0(kappa)

    def vonmises_kde(sample_theta: np.ndarray) -> np.ndarray:
        if sample_theta.size == 0:
            return np.zeros_like(theta_grid)
        return np.exp(kappa * np.cos(theta_grid[:, None] - sample_theta[None, :])).mean(axis=1) / norm

    dens_by_tag = {}
    max_d_global = 0.0
    for tag in tags_ordered:
        th = df.loc[df[tag_col] == tag, "_theta"].to_numpy()
        dens = vonmises_kde(th)
        dens_by_tag[tag] = dens
        md = float(np.nanmax(dens)) if dens.size else 0.0
        if np.isfinite(md):
            max_d_global = max(max_d_global, md)

    # only two rings
    ring_radii = [r0, r0 + r_scale * 1.0]
    ring_theta = np.linspace(-np.pi, np.pi, 500)

    top_density = max(1.0, float(max_d_global))
    rmax = r0 + r_scale * top_density + 0.20

    # 4 main directions in (-180, 180]: -90, 0, 90, 180
    xticks_deg = [-90, 0, 90, 180]
    xticks = np.deg2rad(xticks_deg)
    xtick_labels = [r"$-90^\circ$", r"$0^\circ$", r"$90^\circ$", r"$\pm180^\circ$"]

    for i, tag in enumerate(tags_ordered):
        ax = axes[i]
        trt = tag_meta.loc[tag_meta[tag_col] == tag, treat_col].iloc[0]
        color = treat_to_color[trt]

        dens = dens_by_tag[tag]
        r = r0 + r_scale * dens

        # KDE curve
        ax.plot(theta_grid, r, color=color, linewidth=lw, alpha=0.95)

        # density rings at r=1 and r=2 only
        for rr in ring_radii:
            ax.plot(
                ring_theta, np.full_like(ring_theta, rr),
                color=ring_color, linewidth=ring_lw, alpha=ring_alpha, zorder=0
            )

        # remove default decorations
        ax.grid(False)
        ax.spines["polar"].set_visible(False)
        ax.set_rlim(0, rmax)
        ax.set_rticks([])
        ax.set_yticklabels([])

        # show -180..180
        ax.set_thetamin(-180)
        ax.set_thetamax(180)

        # tick labels
        ax.set_xticks(xticks)
        ax.set_xticklabels(xtick_labels)

        # draw the 4 angle spokes (minimal)
        for th in xticks:
            ax.plot([th, th], [r0, rmax], color=spoke_color, linewidth=spoke_lw, alpha=spoke_alpha, zorder=0)

        # gray segment at 0° from r=1 to r=2
        ax.plot([0.0, 0.0], [r0, r0 + r_scale * 1.0],
                color=zero_seg_color, linewidth=zero_seg_lw, alpha=zero_seg_alpha, zorder=1)

        ax.set_title(f"tag {tag}", fontsize=title_fs, va="bottom")

    for j in range(n, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

# run it
plot_polar_vonmises_kde_all(track_w_treat, kappa=15, r0=1.0, r_scale=1.0)


In [ ]:
import numpy as np
import pandas as pd

def estimate_kappa_from_Rbar(Rbar: float, n: int | None = None) -> float:
    """
    Approximate kappa for von Mises from mean resultant length Rbar in [0,1].
    Optionally applies a common small-sample correction when n is provided.
    """
    if Rbar is None or not np.isfinite(Rbar):
        return np.nan
    Rbar = float(np.clip(Rbar, 0.0, 0.999999999))  # avoid 1 exactly

    if Rbar < 0.53:
        kappa = 2*Rbar + Rbar**3 + (5/6)*Rbar**5
    elif Rbar < 0.85:
        kappa = -0.4 + 1.39*Rbar + 0.43/(1 - Rbar)
    else:
        kappa = 1.0 / (Rbar**3 - 4*Rbar**2 + 3*Rbar)

    # small-sample correction (common in circstats implementations)
    if n is not None and np.isfinite(n) and n > 1:
        n = int(n)
        if kappa < 2:
            kappa = max(kappa - 2/(n*kappa), 0.0) if kappa > 0 else 0.0
        else:
            kappa = (n - 1)**3 * kappa / (n**3 + n)

    return float(kappa)

def mean_resultant_length(theta_rad: np.ndarray) -> float:
    """Mean resultant length Rbar for angles in radians."""
    theta = np.asarray(theta_rad, float)
    theta = theta[np.isfinite(theta)]
    if theta.size == 0:
        return np.nan
    C = np.mean(np.cos(theta))
    S = np.mean(np.sin(theta))
    return float(np.hypot(C, S))

def kappa_by_tag_full_trajectory(
    track_w_treat: pd.DataFrame,
    *,
    angle_col="flightAngDiffDest_deg",   # degrees
    tag_col="tag",
    treat_col="treatment",
    treat_name_col="treatmentName",
) -> pd.DataFrame:
    """
    Per-tag circular concentration on the FULL trajectory (no phase split).

    Returns columns:
      tag, n, Rbar, kappa_hat, treatment, treatmentName
    """
    cols = [tag_col, angle_col]
    if treat_col in track_w_treat.columns:
        cols.append(treat_col)
    if treat_name_col in track_w_treat.columns:
        cols.append(treat_name_col)

    df = track_w_treat[cols].copy()
    df = df.dropna(subset=[tag_col, angle_col])

    # degrees -> radians, wrapped to [0, 2pi)
    df["_theta"] = np.deg2rad(np.mod(df[angle_col].astype(float), 360.0))

    out_rows = []
    for tag, sub in df.groupby(tag_col, sort=False):
        th = sub["_theta"].to_numpy()
        n = int(th.size)
        Rbar = mean_resultant_length(th)
        kappa_hat = estimate_kappa_from_Rbar(Rbar, n=n)

        row = {
            tag_col: tag,
            "n": n,
            "Rbar": Rbar,
            "kappa_hat": kappa_hat,
        }
        if treat_col in sub.columns:
            row[treat_col] = sub[treat_col].iloc[0]
        if treat_name_col in sub.columns:
            row[treat_name_col] = sub[treat_name_col].iloc[0]

        out_rows.append(row)

    out = pd.DataFrame(out_rows)

    # nice ordering
    if treat_col in out.columns:
        out = out.sort_values([treat_col, tag_col]).reset_index(drop=True)
    else:
        out = out.sort_values([tag_col]).reset_index(drop=True)

    return out

# Example:
kappa_df = kappa_by_tag_full_trajectory(track_w_treat)
kappa_df.head()


,tag,n,Rbar,kappa_hat,treatment,treatmentName
0,7537,34,0.799076,2.604358,1,control
1,7538,40,0.804660,2.704525,1,control
2,7539,39,0.679909,1.861288,1,control
3,7541,119,0.542137,1.279716,1,control
4,7544,54,0.476243,1.046652,1,control


$\hat\kappa$ (concentration) by treatment, median with percentile bands.


In [ ]:
plots.plot_kappa_hat(kappa_df)


/tmp/ipykernel_3898688/2172931376.py:38: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(dfp["treatment"]):


Mean resultant length $\vec{R}$ by treatment, median with percentile bands.


In [ ]:
plots.plot_rbar(kappa_df)


/tmp/ipykernel_3898688/3854879063.py:35: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(dfp["treatment"]):


In [ ]:
import numpy as np
import pandas as pd

def mean_alignment_to_zero(theta_rad: np.ndarray, *, ref_rad: float = 0.0) -> float:
    """
    Alignment to reference direction (default 0): E[cos(theta - ref)] in [-1, 1].
      1   = perfectly aligned with ref
      0   = no net alignment (e.g. roughly uniform / symmetric)
     -1   = perfectly opposite ref
    """
    theta = np.asarray(theta_rad, float)
    theta = theta[np.isfinite(theta)]
    if theta.size == 0:
        return np.nan
    return float(np.mean(np.cos(theta - ref_rad)))

def alignment_by_tag_full_trajectory(
    track_w_treat: pd.DataFrame,
    *,
    angle_col="flightAngDiffDest_deg",   # degrees (diff from destination), so 0 = aligned
    tag_col="tag",
    treat_col="treatment",
    treat_name_col="treatmentName",
    ref_deg=0.0,
) -> pd.DataFrame:
    """
    Per-tag alignment to ref_deg (default 0) on the FULL trajectory (no phase split).

    Returns columns:
      tag, n, alignment0, treatment, treatmentName
    """
    cols = [tag_col, angle_col]
    if treat_col in track_w_treat.columns:
        cols.append(treat_col)
    if treat_name_col in track_w_treat.columns:
        cols.append(treat_name_col)

    df = track_w_treat[cols].copy().dropna(subset=[tag_col, angle_col])

    # degrees -> radians; wrap to (-pi, pi] (not strictly required for cos, but keeps things tidy)
    theta = np.deg2rad(np.mod(df[angle_col].astype(float), 360.0))
    df["_theta"] = (theta + np.pi) % (2 * np.pi) - np.pi

    ref_rad = np.deg2rad(ref_deg)

    out_rows = []
    for tag, sub in df.groupby(tag_col, sort=False):
        th = sub["_theta"].to_numpy()
        n = int(th.size)

        row = {
            tag_col: tag,
            "n": n,
            "alignment0": mean_alignment_to_zero(th, ref_rad=ref_rad),
        }
        if treat_col in sub.columns:
            row[treat_col] = sub[treat_col].iloc[0]
        if treat_name_col in sub.columns:
            row[treat_name_col] = sub[treat_name_col].iloc[0]

        out_rows.append(row)

    out = pd.DataFrame(out_rows)

    # nice ordering
    if treat_col in out.columns:
        out = out.sort_values([treat_col, tag_col]).reset_index(drop=True)
    else:
        out = out.sort_values([tag_col]).reset_index(drop=True)

    return out

# Example:
align_df = alignment_by_tag_full_trajectory(track_w_treat)
align_df.head()


,tag,n,alignment0,treatment,treatmentName
0,7537,34,0.796481,1,control
1,7538,40,0.632850,1,control
2,7539,39,0.663519,1,control
3,7541,119,0.535650,1,control
4,7544,54,0.448879,1,control


Alignment of individual mean bearings to $0^\circ$ by treatment, median with percentile bands.


In [ ]:
plots.plot_alignment(align_df)


/tmp/ipykernel_3898688/2867026468.py:38: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(dfp["treatment"]):


## 6. Single von Mises treatment model (baseline, Bambi)

Build the modeling table for a **single** von Mises likelihood: response `HeadingRadians` (the
goal-relative bearing wrapped to $(-\pi, \pi]$), a **treatment** fixed effect, and a **per-tag random
intercept** (repeated measures). Predictors (`z_log_dist`) are standardized. This is the
non-mixed baseline; the mixture below will be judged against it.


In [ ]:
angle_col = "flightAngDiffDest_deg"   # degrees
after_col = "afterLocalization"       # 0/1 (or bool)  (kept, NOT used to filter)
tag_col   = "tag"
dist_col  = "dist_to_dest_grad_m"     # meters (already computed in df)

# -------------------------
# 1) Preprocess (FULL trajectory)
# -------------------------
df_track = track_w_treat.copy()

# Keep only usable rows
need = [tag_col, angle_col, "treatment"]
if dist_col in df_track.columns:
    need.append(dist_col)

df_track = df_track.dropna(subset=need).copy()

# Degrees -> radians; wrap to (-pi, pi]
theta = np.deg2rad(np.mod(pd.to_numeric(df_track[angle_col], errors="coerce"), 360.0))
df_track["HeadingRadians"] = (theta + np.pi) % (2 * np.pi) - np.pi
df_track = df_track.dropna(subset=["HeadingRadians"]).copy()

# Categorical columns
df_track[tag_col] = df_track[tag_col].astype("category")
df_track["treatment"] = df_track["treatment"].astype("category")

# Keep afterLocalization around as int if present (not filtering here)
if after_col in df_track.columns:
    df_track[after_col] = pd.to_numeric(df_track[after_col], errors="coerce").fillna(0).astype(int)

# Optional: standardize log distance if available
if dist_col in df_track.columns:
    df_track["log_dist"] = np.log1p(pd.to_numeric(df_track[dist_col], errors="coerce"))
    log_mean = float(df_track["log_dist"].mean())
    log_sd   = float(df_track["log_dist"].std(ddof=0))
    df_track["z_log_dist"] = (df_track["log_dist"] - log_mean) / (log_sd if log_sd > 0 else 1.0)

    print("log_dist mean:", log_mean, "sd:", log_sd)
else:
    print(f"Note: {dist_col} not found, skipping log_dist / z_log_dist.")

print("Full-trajectory rows:", len(df_track))
print("Full-trajectory tags:", df_track[tag_col].nunique())
print("Treatments:", df_track["treatment"].cat.categories.tolist())


log_dist mean: 7.23823917723698 sd: 0.8609174780452783
Full-trajectory rows: 4942
Full-trajectory tags: 67
Treatments: [1, 2, 3, 4]


Specified and fit in Bambi: `HeadingRadians ~ 1 + treatment + (1|tag)` for the mean direction, and the
matching `kappa ~ 1 + treatment + (1|tag)` submodel for concentration, using the unwrap
`circ_mod` (modulo) and `softplus` links. Fit with 2000+2000 draws, `target_accept=0.9`.


In [ ]:
import numpy as np
import pandas as pd
import bambi as bmb
import arviz as az
import matplotlib.pyplot as plt

from unwrap.link.numpy import inverse_softplus
from unwrap.prior import CircularUniformPrior
from unwrap.link.bambi import LINKS

# ============================================================
# AFTER localization only + Bambi VonMises GLMM
# Fixed effects: treatment (categorical)
# Random effects: random intercepts by tag ONLY (no random slopes)
# Uses YOUR custom links: LINKS["circ_mod"] and LINKS["softplus"]
# Priors: same idea as your other model for slopes (fixed effects),
#         but NO random effects for slopes.
# ============================================================

# -------------------------
# 2) Model formula
#   - treatment as fixed effect in BOTH mu and kappa
#   - random intercept by tag ONLY
# -------------------------
formula = bmb.Formula(
    "HeadingRadians ~ 1 + treatment + (1 | tag)",  # mu submodel
    "kappa ~ 1 + treatment + (1 | tag)",           # kappa submodel
)

# -------------------------
# 3) Priors
# -------------------------
priors = {
    # ---- mu (mean direction) ----
    "Intercept": bmb.Prior("VonMises", mu=0, kappa=1.0),

    # treatment fixed effects on mu (all dummies share this prior)
    "treatment": bmb.Prior("VonMises", mu=0, kappa=2.0),

    # random intercept on mu.
    # Original intent: tight circular (VonMises, kappa≈30 ⇒ angular sd ≈ 1/sqrt(30) ≈ 0.18) RE.
    # bambi's non-centered group-specific terms only support Normal priors, so we
    # use the small-angle Normal analogue on the pre-link scale, with a HalfNormal
    # hyperprior centred at the same angular sd (~0.18). The circular_modulo link
    # wraps the linear predictor back to the circle, so the RE still enters mu as
    # a per-animal angular shift (matching the kappa RE below).
    "1|tag": bmb.Prior(
        "Normal", mu=0.0,
        sigma=bmb.Prior("HalfNormal", sigma=1.0 / np.sqrt(30.0)),
    ),

    # ---- kappa (concentration) submodel ----
    "kappa": {
        "Intercept": bmb.Prior("Normal", mu=inverse_softplus(3), sigma=1.0),

        # treatment fixed effects on kappa (all dummies share this prior)
        "treatment": bmb.Prior("Normal", mu=0, sigma=1.0),

        # random intercept on kappa
        "1|tag": bmb.Prior("Normal", mu=0, sigma=bmb.Prior("HalfNormal", sigma=1.0)),
    },
}

# -------------------------
# 4) Build + fit (with YOUR links)
# -------------------------
model = bmb.Model(
    formula,
    df_track,
    family="vonmises",
    priors=priors,
    link={"mu": LINKS["circ_mod"], "kappa": LINKS["softplus"]},
    center_predictors=False,
    categorical=["tag", "treatment"],
    dropna=True,
)

print(model)

idata_treatment = model.fit(tune=2000, draws=2000, target_accept=0.9, random_seed=20261030, idata_kwargs={"log_likelihood": True}
)


       Formula: HeadingRadians ~ 1 + treatment + (1 | tag)
                kappa ~ 1 + treatment + (1 | tag)
        Family: vonmises
          Link: mu = circ_mod
                kappa = softplus
  Observations: 4942
        Priors: 
    target = mu
        Common-level effects
            Intercept ~ VonMises(mu: 0.0, kappa: 1.0)
            treatment ~ VonMises(mu: 0.0, kappa: 2.0)
        
        Group-level effects
            1|tag ~ Normal(mu: 0.0, sigma: HalfNormal(sigma: 0.1826))
    target = kappa
        Common-level effects
            kappa_Intercept ~ Normal(mu: 2.9489, sigma: 1.0)
            kappa_treatment ~ Normal(mu: 0.0, sigma: 1.0)
        
        Group-level effects
            kappa_1|tag ~ Normal(mu: 0.0, sigma: HalfNormal(sigma: 1.0))


Initializing NUTS using jitter+adapt_diag...


/home/jake/.venvs/unwrap-env/lib/python3.12/site-packages/pytensor/link/c/cmodule.py:2986: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [Intercept, treatment, 1|tag_sigma, 1|tag_offset, kappa_Intercept, kappa_treatment, kappa_1|tag_sigma, kappa_1|tag_offset]


Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 324 seconds.


### Diagnostics and treatment coefficients

ArviZ summary of the vM treatment fit; the following cells flag any variables with poor ESS or
$\hat{R}$.


In [ ]:
summary_df = az.summary(idata_treatment)


In [ ]:
summary_df[(summary_df["ess_bulk"]<400) | (summary_df["ess_tail"]<400)]


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat


In [ ]:
summary_df[(summary_df["r_hat"]>1.01)]

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat


In [ ]:
summary_df

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
Intercept,-0.064,0.086,-0.229,0.093,0.003,0.002,822.0,1532.0,1.0
treatment[2],0.155,0.158,-0.154,0.433,0.005,0.003,918.0,1872.0,1.0
treatment[3],-0.165,0.140,-0.424,0.099,0.004,0.002,1029.0,2201.0,1.0
treatment[4],-0.080,0.173,-0.399,0.254,0.005,0.002,1348.0,2508.0,1.0
1|tag_sigma,0.443,0.039,0.371,0.515,0.001,0.001,1449.0,2759.0,1.0
...,...,...,...,...,...,...,...,...,...
kappa_1|tag[7770],-1.468,0.379,-2.200,-0.773,0.007,0.004,3017.0,4479.0,1.0
kappa_1|tag[7772],-0.850,0.475,-1.733,0.040,0.007,0.005,4161.0,5157.0,1.0
kappa_1|tag[7773],0.415,0.447,-0.428,1.258,0.010,0.005,1889.0,3320.0,1.0
kappa_1|tag[7774],-0.937,0.449,-1.803,-0.111,0.010,0.005,1977.0,3803.0,1.0


Treatment coefficients for the mu and kappa submodels (vM baseline).


In [ ]:
# ============================================================
# Flexible coefficient plot:
#   - Works if treatment coefficients are absent for μ and/or κ
#   - Works if only intercepts exist for μ and/or κ
#   - Rows include treatments ONLY if at least one of (μ, κ) has treatment effects
#   - Missing parameters are simply not plotted for that row
#   - Legend includes only the parameters that are actually plotted
# ============================================================

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import arviz as az
from matplotlib.lines import Line2D

mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

post = idata_treatment.posterior

def _has(varname: str) -> bool:
    return varname in post.data_vars

def _stack_1d(varname: str) -> np.ndarray:
    if not _has(varname):
        raise KeyError(f"Missing '{varname}' in posterior. Available: {list(post.data_vars)[:40]} ...")
    return post[varname].stack(sample=("chain", "draw")).values.astype(float)

def circular_modulo(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def circular_mean(theta):
    z = np.exp(1j * np.asarray(theta, float))
    return float(circular_modulo(np.angle(z.mean())))

def circular_hdi(theta, hdi_prob=0.95):
    theta = np.asarray(theta, float)
    mu_c = circular_mean(theta)
    centered = circular_modulo(theta - mu_c)
    h = az.hdi(centered, hdi_prob=hdi_prob, skipna=True)
    low_c, high_c = float(h[0]), float(h[1])
    low = float(circular_modulo(low_c + mu_c))
    high = float(circular_modulo(high_c + mu_c))
    return low, high, mu_c

def linear_hdi_mean(x, hdi_prob=0.95):
    x = np.asarray(x, float)
    h = az.hdi(x, hdi_prob=hdi_prob, skipna=True)
    return float(np.nanmean(x)), float(h[0]), float(h[1])

def circ_dist(a, b):
    return float(circular_modulo(b - a))

def _cat_dim(varname: str):
    da = post[varname]
    dims = [d for d in da.dims if d not in ("chain", "draw")]
    if len(dims) != 1:
        raise ValueError(f"Expected 1 non-(chain/draw) dim for '{varname}', got {da.dims}")
    return dims[0]

def _sel_level(varname: str, lvl, nonref_levels):
    dim = _cat_dim(varname)
    coords = list(post[varname].coords[dim].values)
    if lvl in coords:
        return {dim: lvl}
    idx = nonref_levels.index(lvl)
    return {dim: coords[idx]}

def _format_treatment_name(name: str) -> str:
    s = str(name).strip().lower()
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if "control" in parts and len(parts) > 1:
        parts = [p for p in parts if p != "control"]
    if not parts:
        parts = ["control"]
    parts = sorted(parts)
    parts = [p[:1].upper() + p[1:] for p in parts]
    return ", ".join(parts)

# ---- treatment levels in fit order (for consistent colors/labels) ----
treat_levels = (
    df_track["treatment"].cat.categories.tolist()
    if hasattr(df_track["treatment"], "cat")
    else sorted(df_track["treatment"].unique())
)
ref = treat_levels[0]
nonref = treat_levels[1:]

# ---- nice labels ----
nice = {t: str(t) for t in treat_levels}
if "treatmentName" in df_track.columns:
    name_map = df_track.groupby("treatment")["treatmentName"].first().to_dict()
    for t in treat_levels:
        nice[t] = _format_treatment_name(name_map.get(t, t))
else:
    for t in treat_levels:
        nice[t] = _format_treatment_name(t)

# ---- detect which coefficient blocks exist ----
mu_has_intercept = _has("Intercept")
mu_has_treat     = _has("treatment")

k_has_intercept  = _has("kappa_Intercept")
k_has_treat      = _has("kappa_treatment")

have_mu_any = mu_has_intercept or mu_has_treat
have_k_any  = k_has_intercept or k_has_treat
if not (have_mu_any or have_k_any):
    raise ValueError(
        "Neither μ nor κ coefficients found in posterior. "
        "Expected at least one of: 'Intercept', 'treatment', 'kappa_Intercept', 'kappa_treatment'."
    )

# ---- rows: include treatments if either μ or κ has treatment effects ----
rows_have_treat = (mu_has_treat or k_has_treat) and (len(nonref) > 0)
if rows_have_treat:
    terms_order = ["Control (Intercept)"] + [nice[t] for t in nonref]
    row_treatment = [ref] + nonref
else:
    terms_order = ["Control (Intercept)"]
    row_treatment = [ref]

y_base = np.arange(len(terms_order), dtype=float)

# ---- colors by treatment row ----
cmap = mpl.cm.get_cmap("tab10")
treat_to_color = {t: cmap(i % 10) for i, t in enumerate(treat_levels)}

# ---- allocate summaries (NaN = not present / not plotted) ----
n = len(terms_order)
mu_mean = np.full(n, np.nan)
mu_lo   = np.full(n, np.nan)
mu_hi   = np.full(n, np.nan)

k_mean  = np.full(n, np.nan)
k_lo    = np.full(n, np.nan)
k_hi    = np.full(n, np.nan)

# ---- fill intercept row (if present) ----
if mu_has_intercept:
    mu_draws = circular_modulo(_stack_1d("Intercept"))
    mu_lo[0], mu_hi[0], mu_mean[0] = circular_hdi(mu_draws, hdi_prob=0.95)

if k_has_intercept:
    k_draws = _stack_1d("kappa_Intercept")
    k_mean[0], k_lo[0], k_hi[0] = linear_hdi_mean(k_draws, hdi_prob=0.95)

# ---- fill treatment rows (only for the parameters that exist) ----
if rows_have_treat:
    for i, lvl in enumerate(nonref, start=1):
        if mu_has_treat:
            mu_draws = (
                post["treatment"]
                .sel(**_sel_level("treatment", lvl, nonref))
                .stack(sample=("chain", "draw"))
                .values.astype(float)
            )
            mu_draws = circular_modulo(mu_draws)
            mu_lo[i], mu_hi[i], mu_mean[i] = circular_hdi(mu_draws, hdi_prob=0.95)

        if k_has_treat:
            k_draws = (
                post["kappa_treatment"]
                .sel(**_sel_level("kappa_treatment", lvl, nonref))
                .stack(sample=("chain", "draw"))
                .values.astype(float)
            )
            k_mean[i], k_lo[i], k_hi[i] = linear_hdi_mean(k_draws, hdi_prob=0.95)

# ---- xerr (NaN-safe; we’ll skip plotting where missing) ----
mu_xerr = np.full((2, n), np.nan)
for i in range(n):
    if np.isfinite(mu_mean[i]) and np.isfinite(mu_lo[i]) and np.isfinite(mu_hi[i]):
        mu_xerr[:, i] = [
            abs(circ_dist(mu_lo[i], mu_mean[i])),
            abs(circ_dist(mu_mean[i], mu_hi[i])),
        ]

k_xerr = np.full((2, n), np.nan)
for i in range(n):
    if np.isfinite(k_mean[i]) and np.isfinite(k_lo[i]) and np.isfinite(k_hi[i]):
        k_xerr[:, i] = [k_mean[i] - k_lo[i], k_hi[i] - k_mean[i]]

# ---- y offsets: only separate μ/κ if both are present anywhere ----
if have_mu_any and have_k_any:
    dy = 0.16
    y_mu = y_base - dy
    y_k  = y_base + dy
else:
    dy = 0.0
    y_mu = y_base
    y_k  = y_base

# ---- plot ----
plt.figure(figsize=(1.8, 0.40 * len(terms_order)), dpi=300)
ax = plt.gca()

for i in range(n):
    c = treat_to_color[row_treatment[i]]

    # μ
    if np.isfinite(mu_mean[i]) and np.all(np.isfinite(mu_xerr[:, i])):
        ax.errorbar(
            mu_mean[i], y_mu[i], xerr=mu_xerr[:, [i]],
            fmt="D", color=c, ecolor=c,
            capsize=3, elinewidth=1, capthick=1,
            markersize=5,
            label=None,
        )

    # κ
    if np.isfinite(k_mean[i]) and np.all(np.isfinite(k_xerr[:, i])):
        ax.errorbar(
            k_mean[i], y_k[i], xerr=k_xerr[:, [i]],
            fmt="^", color=c, ecolor=c,
            capsize=3, elinewidth=1, capthick=1,
            markersize=5,
            label=None,
        )

ax.axvline(0.0, linestyle="--", linewidth=1, color="k", zorder=0)
ax.set_yticks(y_base)
ax.set_yticklabels(terms_order)
ax.invert_yaxis()
ax.set_xlabel("Coefficient value")

# ---- legend: only include what we actually plotted ----
leg_c = "0.2"
handles = []
if have_mu_any:
    handles.append(Line2D([0], [0], marker="D", linestyle="None", color=leg_c, markersize=5, label="μ"))
if have_k_any:
    handles.append(Line2D([0], [0], marker="^", linestyle="None", color=leg_c, markersize=5, label="κ"))
if handles:
    ax.legend(handles=handles, loc="lower right", frameon=True)

plt.savefig("../figures/bats/treatment_coefficients_mu_kappa.pdf", dpi=300, bbox_inches="tight", pad_inches=0.01)
plt.show()


/tmp/ipykernel_3898688/2453464849.py:132: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = mpl.cm.get_cmap("tab10")


Posterior predictive density at the goal direction ($0^\circ$, i.e. straight toward the goal) by
treatment for the vM baseline.


In [ ]:
# ============================================================
# Flexible predictive DENSITY-at-theta plot (population-level)
#   - Computes Von Mises predictive density at theta0 (default 0°)
#   - Uses ONLY population-level effects
#   - Works if treatment effects are absent for μ and/or κ
#   - If BOTH μ and κ lack treatment effects -> plots only Control row
#   - Adds uniform baseline density (1/(2π)) as dashed black line
#
# Style update:
#   - Seaborn "colorblind" palette
#   - Match other plots (Arial, svg.fonttype none, white background)
#   - Same horizontal mean ± HDI style (line + end caps + dot)
#   - Optional hide y ticks/labels while keeping y-axis spine
#   - Auto-narrow width when y ticks/labels are hidden
# ============================================================

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import arviz as az

# ----------------------
# style (match other plots)
# ----------------------
DPI = 300
mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
})

post = idata_treatment.posterior

# ----------------------
# user knobs
# ----------------------
theta0_deg = 0.0
theta0 = np.deg2rad(theta0_deg)  # radians

# hide ONLY y ticks + labels (keep y-axis spine)
HIDE_TREATMENT_TICKS_AND_LABELS = True
show_treatment_labels = not HIDE_TREATMENT_TICKS_AND_LABELS
show_treatment_ticks  = not HIDE_TREATMENT_TICKS_AND_LABELS

# figure sizing (auto-narrow if both labels + ticks hidden)
fig_width = None            # set a float to force width (e.g., 2.1)
fig_width_with_labels = 3.3
fig_width_no_labels   = 2.3

# marker/interval style to match your other plots
CAP = 0.18
LW  = 1.2
MS  = 5
HDI_PROB = 0.95
PALETTE_NAME = "colorblind"

# ----------------------
# helpers
# ----------------------
def _has(varname: str) -> bool:
    return varname in post.data_vars

def circular_modulo(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def softplus(x):
    x = np.asarray(x, float)
    return np.where(x > 30, x, np.log1p(np.exp(x)))

def vonmises_pdf(theta, mu, kappa):
    # pdf(theta | mu, kappa) = exp(kappa cos(theta-mu)) / (2π I0(kappa))
    return np.exp(kappa * np.cos(theta - mu)) / (2.0 * np.pi * np.i0(kappa))

def _stack_samples(da):
    return da.stack(sample=("chain", "draw")).values.astype(float)

def _format_treatment_name(name: str) -> str:
    s = str(name).strip().lower()
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if "control" in parts and len(parts) > 1:
        parts = [p for p in parts if p != "control"]
    if not parts:
        parts = ["control"]
    parts = sorted(parts)
    parts = [p[:1].upper() + p[1:] for p in parts]
    return ", ".join(parts)

def _cat_dim(varname: str):
    da = post[varname]
    dims = [d for d in da.dims if d not in ("chain", "draw")]
    if len(dims) != 1:
        raise ValueError(f"Expected 1 non-(chain/draw) dim for '{varname}', got {da.dims}")
    return dims[0]

def _sel_level(varname: str, lvl, nonref_levels):
    dim = _cat_dim(varname)
    coords = list(post[varname].coords[dim].values)
    if lvl in coords:
        return {dim: lvl}
    idx = nonref_levels.index(lvl)
    return {dim: coords[idx]}

# ----------------------
# require intercepts for μ and κ (density needs both)
# ----------------------
if not _has("Intercept"):
    raise KeyError(f"Missing 'Intercept' (needed for μ). Available: {list(post.data_vars)[:40]} ...")
if not _has("kappa_Intercept"):
    raise KeyError(f"Missing 'kappa_Intercept' (needed for κ). Available: {list(post.data_vars)[:40]} ...")

mu_has_treat = _has("treatment")
k_has_treat  = _has("kappa_treatment")

# ----------------------
# treatment levels (fit order)
# ----------------------
treat_levels = (
    df_track["treatment"].cat.categories.tolist()
    if hasattr(df_track["treatment"], "cat")
    else sorted(df_track["treatment"].unique())
)
ref = treat_levels[0]
nonref = treat_levels[1:]

# ----------------------
# nice labels
# ----------------------
nice = {t: str(t) for t in treat_levels}
if "treatmentName" in df_track.columns:
    # observed=False avoids dropping unused categories if treatment is categorical
    name_map = df_track.groupby("treatment", observed=False)["treatmentName"].first().to_dict()
    for t in treat_levels:
        nice[t] = _format_treatment_name(name_map.get(t, t))
else:
    for t in treat_levels:
        nice[t] = _format_treatment_name(t)

# ----------------------
# rows to plot
# include treatment rows if either μ or κ has treatment effects
# ----------------------
rows_have_treat = (mu_has_treat or k_has_treat) and (len(nonref) > 0)
if rows_have_treat:
    labels = ["Control"] + [nice[t] for t in nonref]
    row_treatment = [ref] + nonref
else:
    labels = ["Control"]
    row_treatment = [ref]

# ----------------------
# pull population-level intercept draws
# ----------------------
b0_mu = _stack_samples(post["Intercept"])        # (S,)
b0_k  = _stack_samples(post["kappa_Intercept"])  # (S,)

# ----------------------
# compute density at theta0 for each row
# ----------------------
dens_draws_by_row = []
for t in row_treatment:
    eta_mu = b0_mu.copy()
    eta_k  = b0_k.copy()

    if t != ref:
        if mu_has_treat:
            b_trt_mu = _stack_samples(post["treatment"].sel(**_sel_level("treatment", t, nonref)))
            eta_mu = eta_mu + b_trt_mu
        if k_has_treat:
            b_trt_k = _stack_samples(post["kappa_treatment"].sel(**_sel_level("kappa_treatment", t, nonref)))
            eta_k = eta_k + b_trt_k

    mu    = circular_modulo(eta_mu)
    kappa = softplus(eta_k)

    dens0 = vonmises_pdf(theta0, mu, kappa)  # (S,)
    dens_draws_by_row.append(dens0)

# ----------------------
# summaries: mean + HDI
# ----------------------
mean = np.array([d.mean() for d in dens_draws_by_row], float)
hdi  = np.array([az.hdi(d, hdi_prob=HDI_PROB) for d in dens_draws_by_row], float)  # (K, 2)
lo, hi = hdi[:, 0], hdi[:, 1]

# ----------------------
# seaborn colorblind palette (stable order by treat_levels)
# ----------------------
palette = sns.color_palette(PALETTE_NAME, n_colors=len(treat_levels))
treat_to_color = {t: palette[i] for i, t in enumerate(treat_levels)}
colors = [treat_to_color[t] for t in row_treatment]

# ----------------------
# figure width (auto-narrow if hiding both y labels + y ticks)
# ----------------------
if fig_width is None:
    if (not show_treatment_labels) and (not show_treatment_ticks):
        fig_width = fig_width_no_labels
    else:
        fig_width = fig_width_with_labels

fig_height = 0.35 * len(labels)

# ----------------------
# plot (match horizontal mean ± HDI style)
# ----------------------
y = np.arange(len(labels), dtype=float)

fig = plt.figure(figsize=(fig_width, fig_height), dpi=DPI)
ax = plt.gca()

for i in range(len(labels)):
    c = colors[i]
    ax.hlines(y[i], lo[i], hi[i], color=c, linewidth=LW, zorder=2)
    ax.vlines([lo[i], hi[i]], y[i] - CAP, y[i] + CAP, color=c, linewidth=LW, zorder=2)
    ax.plot(mean[i], y[i], "o", color=c, markersize=MS, markeredgewidth=0.0, zorder=3)

# uniform reference (constant density 1/(2π))
unif = 1.0 / (2.0 * np.pi)
ax.axvline(unif, color="black", linestyle="--", linewidth=1)

# y-axis: keep spine, optionally hide ONLY ticks/labels
ax.set_yticks(y)
if show_treatment_labels:
    ax.set_yticklabels(labels, fontsize=8)
else:
    ax.set_yticklabels([""] * len(y))

ax.tick_params(
    axis="y",
    which="both",
    left=show_treatment_ticks,
    length=3 if show_treatment_ticks else 0,
    labelleft=show_treatment_labels,
)

ax.invert_yaxis()
ax.margins(y=0.12)
ax.set_xlabel("Predictive density")

# x-lims (include uniform line)
xmin = float(min(np.min(lo), unif))
xmax = float(max(np.max(hi), unif))
pad = 0.10 * (xmax - xmin if xmax > xmin else 1.0)
ax.set_xlim(xmin - pad, xmax + pad)

# style clean-up to match other plot
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(False)

plt.tight_layout()
plt.savefig(
    "../figures/bats/treatment_pred_density_at0_mu_kappa.pdf",
    dpi=DPI,
    bbox_inches="tight",
    pad_inches=0.01
)
plt.show()

### PSIS-LOO comparison (vMU vs vM)

Helper for turning a Normal on the logit scale into sigmoid-space percentiles (used to express the
mixture weight $p$ in interpretable $[0,1]$ terms), and to compute the out-of-sample comparison.


In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.special import expit, logit  # stable sigmoid

def sigmoid_normal_percentiles(mu, sigma, qs=(0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99)):
    """
    Percentiles of P = sigmoid(X), where X ~ Normal(mu, sigma).
    qs are quantiles in [0,1].
    """
    z = norm.ppf(qs)              # standard normal quantiles
    xq = mu + sigma * z           # corresponding X quantiles
    pq = expit(xq)                # transform to P quantiles
    return np.asarray(qs), pq

# example
mu = logit(0.9)  # logit(0.99)
sigma = 1.0

qs, pqs = sigmoid_normal_percentiles(mu, sigma)
for q, p in zip(qs, pqs):
    print(f"{q:>5.2%}: {p:.6f}")


1.00%: 0.467764
5.00%: 0.634685
10.00%: 0.714160
25.00%: 0.820941
50.00%: 0.900000
75.00%: 0.946430
90.00%: 0.970078
95.00%: 0.979001
99.00%: 0.989266


In [ ]:
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt

from unwrap.link.pymc import circular_modulo  # wraps angles to (-pi, pi] or similar
from scipy.special import logit

LOG_UNIF = -np.log(2*np.pi)


# -------------------------
# Data prep (shared)
# -------------------------
y = df_track["HeadingRadians"].to_numpy()

# tag indices
tag_codes, tag_idx = np.unique(df_track["tag"].to_numpy(), return_inverse=True)
n_tag = len(tag_codes)

# treatment indices (first category is reference)
treat_cat = pd.Categorical(df_track["treatment"])
treat_idx = treat_cat.codes
treat_levels = list(treat_cat.categories)
n_treat = len(treat_levels)

def inverse_softplus(x):
    return np.log(np.exp(x) - 1.0)

LOG_UNIF = -np.log(2*np.pi)


def fit_vmum_model_full(
    *,
    include_treatment_in_p: bool = True,
    include_treatment_in_mu: bool = True,
    include_treatment_in_kappa: bool = True,
    seed: int = 123,
):
    """
    vMU mixture:
      Heading ~ p * VonMises(mu, kappa) + (1-p) * VonMises(0, kappa_uniform)

    Adds optional treatment effects on:
      - p (mixture weight) via logit(p)
      - mu (goal-directed mean direction)
      - kappa (goal-directed concentration, via softplus link)

    Tag random intercepts are included for mu, kappa, and p (same as your prior versions).
    """

    with pm.Model() as vmum_full:

        # Data containers
        y_obs = pm.Data("y_obs", y)
        tag = pm.Data("tag", tag_idx)
        treat = pm.Data("treat", treat_idx)

        # ============================================================
        # 1) mu submodel (optionally treatment-dependent)
        #    mu = wrap(mu0 + mu_treat[treat] + mu_tag[tag])
        # ============================================================
        mu0 = pm.VonMises("mu0", mu=0.0, kappa=1.0)

        # tag random effect on mu (angular)
        mu_tag_kappa = pm.LogNormal("mu_tag_kappa", mu=np.log(30.0), sigma=0.1)
        mu_tag = pm.VonMises("mu_tag", mu=0.0, kappa=mu_tag_kappa, shape=n_tag)

        if include_treatment_in_mu and (n_treat > 1):
            # treatment contrasts on the circle: (n_treat-1) angular offsets vs ref
            mu_treat = pm.VonMises("mu_treat", mu=0.0, kappa=2.0, shape=n_treat - 1)
            mu_treat_full = pt.concatenate([pt.zeros((1,)), mu_treat])  # ref = 0
            mu = pm.Deterministic("mu", circular_modulo(mu0 + mu_treat_full[treat] + mu_tag[tag]))
        else:
            mu = pm.Deterministic("mu", circular_modulo(mu0 + mu_tag[tag]))

        # ============================================================
        # 2) kappa submodel (optionally treatment-dependent)
        #    eta_kappa = eta_kappa0 + kappa_treat[treat] + eta_kappa_tag[tag]*sigma
        #    kappa = softplus(eta_kappa)
        # ============================================================
        eta_kappa0 = pm.Normal("eta_kappa0", mu=inverse_softplus(5.0), sigma=1.0)

        sigma_kappa_tag = pm.HalfNormal("sigma_kappa_tag", sigma=1.0)
        eta_kappa_tag = pm.Normal("eta_kappa_tag", mu=0.0, sigma=1.0, shape=n_tag)

        eta_kappa_base = eta_kappa0 + eta_kappa_tag[tag] * sigma_kappa_tag

        if include_treatment_in_kappa and (n_treat > 1):
            # treatment contrasts on eta_kappa (real-valued)
            kappa_treat = pm.Normal("kappa_treat", mu=0.0, sigma=1.0, shape=n_treat - 1)
            kappa_treat_full = pt.concatenate([pt.zeros((1,)), kappa_treat])
            eta_kappa = eta_kappa_base + kappa_treat_full[treat]
        else:
            eta_kappa = eta_kappa_base

        kappa = pm.Deterministic("kappa", pt.softplus(eta_kappa))

        # ============================================================
        # 3) Mixture weight p (optionally treatment-dependent)
        #    logit(p) = logit_p0 + beta_treat[treat] + logit_p_tag[tag]*sigma
        #    w = sigmoid([logit_p, -logit_p])  (2-component simplex)
        # ============================================================
        logit_p0 = pm.Normal("logit_p0", mu=logit(0.8), sigma=1.0)

        sigma_p_tag = pm.HalfNormal("sigma_p_tag", sigma=1.0)
        logit_p_tag = pm.Normal("logit_p_tag", mu=0.0, sigma=1.0, shape=n_tag)

        logit_p_base = logit_p0 + logit_p_tag[tag] * sigma_p_tag

        if include_treatment_in_p and (n_treat > 1):
            beta_treat = pm.Normal("beta_treat", mu=0.0, sigma=1.0, shape=n_treat - 1)
            beta_treat_full = pt.concatenate([pt.zeros((1,)), beta_treat])
            logit_p = logit_p_base + beta_treat_full[treat]
        else:
            logit_p = logit_p_base

        w = pm.Deterministic("p_goal", pm.math.sigmoid(pt.stack([logit_p, -logit_p], axis=1)))

        # ============================================================
        # 4) Likelihood: mixture of goal-directed VM + near-uniform VM
        # ============================================================
        # kappa_uniform = 1e-6

        comp_goal = pm.VonMises.dist(mu=mu, kappa=kappa)
        # comp_unif = pm.VonMises.dist(mu=0.0, kappa=kappa_uniform)
        comp_unif = pm.DensityDist.dist(
            logp=lambda v: pt.full_like(v, LOG_UNIF),
        )

        pm.Mixture(
            "Heading_obs",
            w=w,
            comp_dists=[comp_goal, comp_unif],
            observed=y_obs,
        )

        idata = pm.sample(
            tune=2000,
            draws=2000,
            target_accept=0.9,
            random_seed=seed,
            return_inferencedata=True,
            idata_kwargs={"log_likelihood": True},
        )

    return idata, vmum_full


# ======================
# EXAMPLES
# ======================

# 1) Full: treatment effects on p, mu, and kappa
idata_mix_full, model_full = fit_vmum_model_full(
    include_treatment_in_p=True,
    include_treatment_in_mu=True,
    include_treatment_in_kappa=True,
    seed=123,
)

print("Treatment levels (reference first):", treat_levels)
print("Full model vars:", list(idata_mix_full.posterior.data_vars))


Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [mu0, mu_tag_kappa, mu_tag, mu_treat, eta_kappa0, sigma_kappa_tag, eta_kappa_tag, kappa_treat, logit_p0, sigma_p_tag, logit_p_tag, beta_treat]


### Fit the full von Mises--Uniform mixture (PyMC)

The centerpiece. `fit_vmum_model_full` specifies **each step** as
$\text{Heading} \sim p\,\mathrm{VM}(\mu, \kappa) + (1-p)\,\mathrm{Uniform}$, with

- $\mu$ = goal-directed mean direction (population `mu0` + per-tag von Mises offsets + treatment
  offsets),
- $\kappa$ = goal-directed concentration (softplus-linked, per-tag offsets + treatment offsets),
- $p$ = probability a step is goal-directed (logit-linked, per-tag offsets + treatment offset).

The undirected (uniform) component explicitly represents exploratory movement, so a single
concentration no longer has to absorb both dispersion during homing and wandering — and no
post-treatment "homing" filter is needed.


In [ ]:
import arviz as az
import numpy as np

# -------------------------
# Convergence checks
# -------------------------
# az.summary returns r_hat, ess_bulk, ess_tail for each parameter by default.
# Use var_names to focus on the key parameters (and avoid huge outputs from deterministics).
vars_to_check = [
    "mu0",
    "mu_tag_kappa",
    "mu_tag",
    "mu_treat",
    "eta_kappa0",
    "sigma_kappa_tag",
    "eta_kappa_tag",
    "kappa_treat",
    "logit_p0",
    "sigma_p_tag",
    "logit_p_tag",
    "beta_treat",
]

# Keep only those that exist in this model run (depends on include_treatment_* flags)
present = [v for v in vars_to_check if v in idata_mix_full.posterior.data_vars]

summ = az.summary(
    idata_mix_full,
    var_names=present,
    round_to=3,
    kind="all",
)

print(summ)

# Flag problematic parameters
bad_rhat = summ.loc[summ["r_hat"] > 1.01].sort_values("r_hat", ascending=False)
bad_ess  = summ.loc[(summ["ess_bulk"] < 400) | (summ["ess_tail"] < 400)].sort_values(
    ["ess_bulk", "ess_tail"], ascending=True
)

print("\n==============================")
print("Convergence flags")
print("==============================")
print(f"Total params in summary: {len(summ)}")

if len(bad_rhat):
    print("\nRHAT > 1.01:")
    print(bad_rhat[["r_hat", "ess_bulk", "ess_tail"]])
else:
    print("\nRHAT > 1.01: none ✅")

if len(bad_ess):
    print("\nESS bulk or tail < 400:")
    print(bad_ess[["r_hat", "ess_bulk", "ess_tail"]])
else:
    print("\nESS bulk or tail < 400: none ✅")


                   mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  \
mu0              -0.081  0.049  -0.168    0.014      0.001    0.001  1827.230   
mu_tag_kappa     16.986  1.301  14.598   19.392      0.014    0.014  9031.902   
mu_tag[0]        -0.886  0.116  -1.103   -0.672      0.002    0.003  3679.548   
mu_tag[1]         0.015  0.111  -0.188    0.226      0.002    0.001  5299.310   
mu_tag[2]        -0.500  0.108  -0.703   -0.295      0.001    0.001  5367.086   
...                 ...    ...     ...      ...        ...      ...       ...   
logit_p_tag[65]  -1.013  0.345  -1.652   -0.371      0.006    0.004  3901.199   
logit_p_tag[66]   0.179  0.600  -0.861    1.325      0.008    0.008  5598.151   
beta_treat[0]    -0.345  0.570  -1.400    0.744      0.010    0.007  3190.305   
beta_treat[1]    -0.403  0.515  -1.372    0.563      0.009    0.006  3026.841   
beta_treat[2]     0.448  0.776  -1.001    1.894      0.013    0.008  3871.800   

                 ess_tail  

Sampling summary for the vMU fit (check ESS / $\hat{R}$ before interpreting).


In [ ]:
import numpy as np
import pandas as pd
import arviz as az

def _loo_pointwise_elpd(idata, *, var_name=None):
    loo = az.loo(idata, pointwise=True, var_name=var_name)
    loo_i = np.asarray(loo.loo_i).reshape(-1)
    k = np.asarray(loo.pareto_k).reshape(-1)
    return loo_i, k, loo

def _se_sum(v: np.ndarray) -> float:
    """Rough SE for a SUM: sqrt(n * var(v))."""
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    n = v.size
    if n <= 1:
        return np.nan
    return float(np.sqrt(n * np.var(v, ddof=1)))

def _pareto_report(k: np.ndarray, threshold: float = 0.7) -> dict:
    k = np.asarray(k, float)
    k = k[np.isfinite(k)]
    if k.size == 0:
        return {"n": 0, "n_gt": 0, "max_k": np.nan, "mean_excess": np.nan, "max_excess": np.nan}

    excess = k - threshold
    gt = excess > 0
    n_gt = int(np.sum(gt))

    if n_gt == 0:
        return {
            "n": int(k.size),
            "n_gt": 0,
            "max_k": float(np.max(k)),
            "mean_excess": 0.0,
            "max_excess": 0.0,
        }

    return {
        "n": int(k.size),
        "n_gt": n_gt,
        "max_k": float(np.max(k)),
        "mean_excess": float(np.mean(excess[gt])),
        "max_excess": float(np.max(excess[gt])),
    }

def compute_psisloo_elpd_summary(
    models,          # dict: {name: idata}  (insertion order matters)
    df_fit,          # EXACT df used in all fits (same rows/order)
    *,
    var_name=None,
    pareto_thresh=0.7,
    order_best_first=True,
):
    """
    Compute PSIS-LOO pointwise ELPD and return a summary table with *absolute* ELPD
    (no deltas).

    Columns:
      - elpd: sum of pointwise elpd (loo_i)
      - se_elpd: rough SE for that sum (based on pointwise variability)
      - is_best: True for max elpd model (optional ordering)
      - pareto diagnostics summary
    """
    if not isinstance(models, dict) or len(models) < 1:
        raise ValueError("models must be a non-empty dict: {name: idata}")

    model_order = list(models.keys())  # preserves insertion order
    n = len(df_fit)

    loo_i_map = {}
    pareto_k_map = {}
    loo_obj_map = {}
    total_elpd = {}
    se_elpd = {}

    for name in model_order:
        idata = models[name]
        loo_i, k, loo = _loo_pointwise_elpd(idata, var_name=var_name)

        if loo_i.size != n:
            raise ValueError(
                f"Row mismatch for model '{name}': df_fit has {n} rows but loo_i has {loo_i.size}. "
                "df_fit MUST be the exact df used in all fits (same rows/order)."
            )

        loo_i_map[name] = loo_i
        pareto_k_map[name] = k
        loo_obj_map[name] = loo

        total_elpd[name] = float(np.sum(loo_i))
        se_elpd[name] = _se_sum(loo_i)

    best_name = max(total_elpd, key=total_elpd.get)

    if order_best_first:
        ordered_names = [best_name] + [m for m in model_order if m != best_name]
    else:
        ordered_names = model_order

    rows = []
    for name in ordered_names:
        rep = _pareto_report(pareto_k_map[name], threshold=pareto_thresh)
        rows.append({
            "model": name,
            "elpd": total_elpd[name],
            "se_elpd": se_elpd[name],
            "is_best": (name == best_name),
            "pareto_n": rep["n"],
            f"pareto_n_gt_{pareto_thresh}": rep["n_gt"],
            "pareto_max_k": rep["max_k"],
            f"pareto_mean_excess_over_{pareto_thresh}": rep["mean_excess"],
            f"pareto_max_excess_over_{pareto_thresh}": rep["max_excess"],
        })

    out = pd.DataFrame(rows)

    cache = {
        "loo_i_map": loo_i_map,
        "pareto_k_map": pareto_k_map,
        "loo_obj_map": loo_obj_map,
        "total_elpd": total_elpd,
        "se_elpd": se_elpd,
        "model_order": model_order,
        "ordered_names": ordered_names,
        "best_name": best_name,
    }
    return out, best_name, cache


# ======================
# RUN (Cell 1)
# ======================
models = {
    "von Mises-Uniform": idata_mix_full,
    "von Mises": idata_treatment,
}

res_total, best_model, loo_cache = compute_psisloo_elpd_summary(
    models=models,
    df_fit=df_track,
    var_name=None,
    pareto_thresh=0.7,
    order_best_first=True,
)

print("Best model:", best_model)
print(res_total)


import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

def plot_total_elpd_from_summary(
    summary_df,      # output from compute_psisloo_elpd_summary()
    *,
    savepath=None,
    title=None,
    xlabel="PSIS-LOO ELPD",
    dpi=300,
    vline_at_best=True,
):
    out = summary_df.copy()

    # ensure a stable display order (assumes your summary already best-first)
    out = out.reset_index(drop=True)

    y = np.arange(len(out), dtype=float)

    fig = plt.figure(figsize=(3.0, 1.25), dpi=dpi)
    ax = plt.gca()

    if vline_at_best and "is_best" in out.columns and out["is_best"].any():
        best_elpd = float(out.loc[out["is_best"], "elpd"].iloc[0])
        ax.axvline(best_elpd, linestyle="--", linewidth=1.0, color="k", zorder=0)

    # plot ALL models with error bars (same style, no colors)
    for yy, (_, r) in zip(y, out.iterrows()):
        x = float(r["elpd"])
        se = float(r["se_elpd"]) if np.isfinite(r.get("se_elpd", np.nan)) else np.nan

        if np.isfinite(se):
            ax.errorbar(
                x, yy, xerr=se,
                fmt="o",
                color="black", ecolor="black",
                capsize=3, elinewidth=1, capthick=1,
                markersize=6, markeredgewidth=0.8,
                markeredgecolor="black",
                zorder=3
            )
        else:
            ax.plot(
                x, yy, "o",
                color="black",
                markersize=6,
                markeredgecolor="black",
                markeredgewidth=0.8,
                zorder=3
            )

    ax.set_yticks(y)
    ax.set_yticklabels(out["model"].tolist(), fontsize=8)
    ax.margins(y=0.4)
    ax.invert_yaxis()

    ax.set_xlabel(xlabel)
    if title:
        ax.set_title(title)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

    plt.tight_layout()
    if savepath:
        fig.savefig(savepath, dpi=dpi, bbox_inches="tight", pad_inches=0.01)
    plt.show()
    return fig, ax


# ======================
# RUN
# ======================
fig, ax = plot_total_elpd_from_summary(
    res_total,
    savepath="../figures/bats/psisloo_total_elpd_all_models.pdf",
    title=None,
    xlabel="ELPD",
    vline_at_best=False,
)


Best model: von Mises-Uniform
               model         elpd    se_elpd  is_best  pareto_n  \
0  von Mises-Uniform -6710.858045  60.268675     True      4942   
1          von Mises -6829.434630  59.330875    False      4942   

   pareto_n_gt_0.7  pareto_max_k  pareto_mean_excess_over_0.7  \
0                0      0.439459                          0.0   
1                0      0.462966                          0.0   

   pareto_max_excess_over_0.7  
0                         0.0  
1                         0.0  


Compute **$\Delta$ELPD = vMU $-$ vM** with PSIS-LOO, total and stratified by treatment, with the SE of
the total. A positive value means the mixture predicts goal-relative headings better out of sample;
it is the evidence for a non-trivial undirected component.


In [ ]:
import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

def _format_treatment_name(name: str) -> str:
    if name is None or (isinstance(name, float) and np.isnan(name)):
        return "Unknown"
    s = str(name).strip().lower()
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if "control" in parts and len(parts) > 1:
        parts = [p for p in parts if p != "control"]
    if not parts:
        parts = ["control"]
    parts = sorted(parts)
    parts = [p[:1].upper() + p[1:] for p in parts]
    return ", ".join(parts)

def _loo_pointwise_elpd(idata, *, var_name=None):
    loo = az.loo(idata, pointwise=True, var_name=var_name)
    loo_i = np.asarray(loo.loo_i).reshape(-1)
    k = np.asarray(loo.pareto_k).reshape(-1)
    return loo_i, k

def _se_sum(v: np.ndarray) -> float:
    """Rough SE for a SUM: sqrt(n * var(v))."""
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    n = v.size
    if n <= 1:
        return np.nan
    return float(np.sqrt(n * np.var(v, ddof=1)))

def elpd_diff_total_and_by_treatment(
    idata_vmu,
    idata_vm,
    df_fit,
    *,
    treat_col="treatment",
    treat_name_col="treatmentName",
    var_name=None,
):
    """
    Compute ΔELPD = vMU - vM:
      - total across all rows
      - stratified by treatment group
    df_fit MUST be the exact rows/order used in both fits.
    """
    vmu_i, vmu_k = _loo_pointwise_elpd(idata_vmu, var_name=var_name)
    vm_i,  vm_k  = _loo_pointwise_elpd(idata_vm,  var_name=var_name)

    n = len(df_fit)
    if vmu_i.size != n or vm_i.size != n:
        raise ValueError(
            f"Row mismatch: df_fit has {n} rows but loo_i has "
            f"{vmu_i.size} (vMU) and {vm_i.size} (vM)."
        )

    delta_i = vmu_i - vm_i
    k_max_i = np.maximum(vmu_k, vm_k)

    cols = [treat_col]
    if treat_name_col in df_fit.columns:
        cols.append(treat_name_col)

    tmp = df_fit[cols].copy().reset_index(drop=True)
    tmp["delta_i"] = delta_i
    tmp["k_max_i"] = k_max_i

    rows = []

    # TOTAL row
    rows.append({
        "row_type": "total",
        "treatment": None,
        "label": "Total",
        "n_obs": int(len(tmp)),
        "delta_elpd": float(np.sum(tmp["delta_i"].to_numpy())),
        "se_delta_elpd": _se_sum(tmp["delta_i"].to_numpy()),
        "pareto_k_max": float(np.max(tmp["k_max_i"].to_numpy())) if len(tmp) else np.nan,
    })

    # BY TREATMENT rows
    for treat, sub in tmp.groupby(treat_col, sort=False):
        tname = sub[treat_name_col].iloc[0] if treat_name_col in sub.columns else str(treat)
        rows.append({
            "row_type": "treatment",
            "treatment": treat,
            "label": _format_treatment_name(tname),
            "n_obs": int(len(sub)),
            "delta_elpd": float(np.sum(sub["delta_i"].to_numpy())),
            "se_delta_elpd": _se_sum(sub["delta_i"].to_numpy()),
            "pareto_k_max": float(np.max(sub["k_max_i"].to_numpy())) if len(sub) else np.nan,
        })

    return pd.DataFrame(rows)

def plot_delta_total_and_by_treatment(
    delta_df,
    *,
    treatment_order=None,
    savepath=None,
    title=None,
    xlabel="ELPD difference (vMU - vM)",
    dpi=300,
    vline_at_zero=True,
    y_step=1.35,
    y_margins=0.10,
):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    out = delta_df.copy().reset_index(drop=True)

    treat_rows = out[out["row_type"] == "treatment"].copy()
    total_row = out[out["row_type"] == "total"].copy()

    if treatment_order is None:
        treatment_order = (
            treat_rows["treatment"].cat.categories.tolist()
            if len(treat_rows) and hasattr(treat_rows["treatment"], "cat")
            else sorted(treat_rows["treatment"].dropna().unique())
        )

    treat_rows["treatment"] = pd.Categorical(
        treat_rows["treatment"], categories=list(treatment_order), ordered=True
    )
    treat_rows = treat_rows.sort_values("treatment")

    out = pd.concat([total_row, treat_rows], ignore_index=True)

    cmap = plt.get_cmap("tab10")
    treat_to_color = {t: cmap(i % 10) for i, t in enumerate(treatment_order)}

    y = np.arange(len(out), dtype=float) * float(y_step)

    fig_h = max(1.35, (0.20 * y_step) * len(out) + 0.20)
    fig = plt.figure(figsize=(3.0, fig_h), dpi=dpi)
    ax = plt.gca()

    if vline_at_zero:
        ax.axvline(0.0, linestyle="--", linewidth=1.0, color="k", zorder=0)

    for yy, (_, r) in zip(y, out.iterrows()):
        x = float(r["delta_elpd"])
        se = float(r["se_delta_elpd"]) if np.isfinite(r.get("se_delta_elpd", np.nan)) else np.nan

        if r["row_type"] == "total":
            color = "black"
            ecolor = "black"
        else:
            t = r["treatment"]
            color = treat_to_color.get(t, "black")
            ecolor = color

        if np.isfinite(se):
            # error bars + caps (NO marker here)
            ax.errorbar(
                x, yy, xerr=se,
                fmt="none",
                ecolor=ecolor,
                capsize=3, capthick=1,   # <-- caps WILL show
                elinewidth=1,
                zorder=2
            )

        # marker (no edge)
        ax.plot(
            x, yy, "o",
            color=color,
            markersize=6,
            markeredgecolor="none",     # <-- no outline
            markeredgewidth=0,
            zorder=3
        )

    ax.set_yticks(y)
    ax.set_yticklabels(out["label"].tolist(), fontsize=8)
    ax.margins(y=float(y_margins))
    ax.invert_yaxis()

    ax.set_xlabel(xlabel)
    if title:
        ax.set_title(title)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

    plt.tight_layout()
    if savepath:
        fig.savefig(savepath, dpi=dpi, bbox_inches="tight", pad_inches=0.01)
    plt.show()
    return fig, ax

# ======================
# RUN
# ======================
delta_df = elpd_diff_total_and_by_treatment(
    idata_vmu=idata_mix_full,      # von Mises-Uniform
    idata_vm=idata_treatment,      # von Mises
    df_fit=df_track,
    treat_col="treatment",
    treat_name_col="treatmentName",
    var_name=None,
)

treatment_order = (
    df_track["treatment"].cat.categories.tolist()
    if hasattr(df_track["treatment"], "cat")
    else sorted(delta_df.loc[delta_df["row_type"] == "treatment", "treatment"].dropna().unique())
)

fig, ax = plot_delta_total_and_by_treatment(
    delta_df,
    treatment_order=treatment_order,
    savepath="../figures/bats/psisloo_delta_total_and_by_treatment_vMU_minus_vM.pdf",
    title=None,
    xlabel="ELPD difference",
    vline_at_zero=True,
)


In [ ]:
import numpy as np
import pandas as pd
import arviz as az

def _se_sum(v: np.ndarray) -> float:
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    n = v.size
    if n <= 1:
        return np.nan
    return float(np.sqrt(n * np.var(v, ddof=1)))

def _pareto_report(k: np.ndarray, threshold: float = 0.7) -> dict:
    k = np.asarray(k, float)
    k = k[np.isfinite(k)]
    if k.size == 0:
        return {"n": 0, "n_gt": 0, "max_k": np.nan, "mean_excess": np.nan, "max_excess": np.nan}

    excess = k - threshold
    gt = excess > 0
    n_gt = int(np.sum(gt))

    if n_gt == 0:
        return {
            "n": int(k.size),
            "n_gt": 0,
            "max_k": float(np.max(k)),
            "mean_excess": 0.0,
            "max_excess": 0.0,
        }

    return {
        "n": int(k.size),
        "n_gt": n_gt,
        "max_k": float(np.max(k)),
        "mean_excess": float(np.mean(excess[gt])),
        "max_excess": float(np.max(excess[gt])),
    }

def compute_psisloo_elpd_summary_safe(
    models, df_fit, *, var_name=None, pareto_thresh=0.7, order_best_first=True
):
    if not isinstance(models, dict) or len(models) < 1:
        raise ValueError("models must be a non-empty dict: {name: idata}")

    model_order = list(models.keys())
    n = len(df_fit)

    loo_i_map, pareto_k_map, loo_obj_map = {}, {}, {}
    total_elpd, se_elpd = {}, {}

    for name in model_order:
        loo = az.loo(models[name], pointwise=True, var_name=var_name)
        loo_i = np.asarray(loo.loo_i).reshape(-1)
        k = np.asarray(loo.pareto_k).reshape(-1)

        if loo_i.size != n:
            raise ValueError(
                f"Row mismatch for model '{name}': df_fit has {n} rows but loo_i has {loo_i.size}. "
                "df_fit MUST be the exact df used in all fits (same rows/order)."
            )

        loo_i_map[name] = loo_i
        pareto_k_map[name] = k
        loo_obj_map[name] = loo
        total_elpd[name] = float(np.sum(loo_i))
        se_elpd[name] = _se_sum(loo_i)

    best_name = max(total_elpd, key=total_elpd.get)

    if order_best_first:
        ordered_names = [best_name] + [m for m in model_order if m != best_name]
    else:
        ordered_names = model_order

    rows = []
    for name in ordered_names:
        rep = _pareto_report(pareto_k_map[name], threshold=pareto_thresh)
        rows.append({
            "model": name,
            "elpd": total_elpd[name],
            "se_elpd": se_elpd[name],
            "is_best": (name == best_name),
            "pareto_n": rep["n"],
            f"pareto_n_gt_{pareto_thresh}": rep["n_gt"],
            "pareto_max_k": rep["max_k"],
            f"pareto_mean_excess_over_{pareto_thresh}": rep["mean_excess"],
            f"pareto_max_excess_over_{pareto_thresh}": rep["max_excess"],
        })

    out = pd.DataFrame(rows)
    cache = {
        "loo_i_map": loo_i_map,
        "pareto_k_map": pareto_k_map,
        "loo_obj_map": loo_obj_map,
        "total_elpd": total_elpd,
        "se_elpd": se_elpd,
        "model_order": model_order,
        "ordered_names": ordered_names,
        "best_name": best_name,
    }
    return out, best_name, cache


# -----------------------
# RUN COMPUTE (once)
# -----------------------
models = {
    "von Mises-Uniform": idata_mix_full,
    "von Mises": idata_treatment,
}

res_total, best_model, loo_cache = compute_psisloo_elpd_summary_safe(
    models=models,
    df_fit=df_track,
    var_name=None,
    pareto_thresh=0.7,
    order_best_first=True,
)

delta_df = elpd_diff_total_and_by_treatment(
    idata_vmu=idata_mix_full,
    idata_vm=idata_treatment,
    df_fit=df_track,
    treat_col="treatment",
    treat_name_col="treatmentName",
    var_name=None,
)

treatment_order = (
    df_track["treatment"].cat.categories.tolist()
    if hasattr(df_track["treatment"], "cat")
    else sorted(delta_df.loc[delta_df["row_type"] == "treatment", "treatment"].dropna().unique())
)

Total ELPD by model (vMU vs vM), best first.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

def plot_total_elpd_from_summary_ax_black(
    summary_df, ax, *, xlabel="ELPD", vline_at_best=False, show_model_names=True
):
    out = summary_df.copy().reset_index(drop=True)
    y = np.arange(len(out), dtype=float)

    if vline_at_best and "is_best" in out.columns and out["is_best"].any():
        best_elpd = float(out.loc[out["is_best"], "elpd"].iloc[0])
        ax.axvline(best_elpd, linestyle="--", linewidth=1.0, color="k", zorder=0)

    for yy, (_, r) in zip(y, out.iterrows()):
        x = float(r["elpd"])
        se = float(r["se_elpd"]) if np.isfinite(r.get("se_elpd", np.nan)) else np.nan

        if np.isfinite(se):
            ax.errorbar(
                x, yy, xerr=se,
                fmt="o",
                color="black", ecolor="black",
                capsize=3, elinewidth=1, capthick=1,
                markersize=6, markeredgewidth=0.8,
                markeredgecolor="black",
                zorder=3
            )
        else:
            ax.plot(x, yy, "o", color="black", markersize=6, zorder=3)

    ax.set_yticks(y)
    if show_model_names:
        ax.set_yticklabels(out["model"].tolist(), fontsize=8)
    else:
        ax.set_yticklabels([])
        ax.tick_params(axis="y", left=False, length=0)

    ax.margins(y=0.4)
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

def plot_delta_total_and_by_treatment_ax_colorblind_bracket(
    delta_df,
    ax,
    *,
    treatment_order=None,
    xlabel="ELPD difference",
    vline_at_zero=True,
    y_step=1.35,
    y_margins=0.10,
    bracket_label="Within-treatment",
    # NEW DEFAULTS:
    bracket_x=0.0,         # touch the y-axis spine line
    bracket_cap=0.020,     # cap length INTO the plot
    bracket_tick=0.030,    # tick length OUT to the label (left)
    label_pad=0.006,       # additional left padding for label (axes fraction)
):
    out = delta_df.copy().reset_index(drop=True)

    treat_rows = out[out["row_type"] == "treatment"].copy()
    total_row = out[out["row_type"] == "total"].copy()

    if treatment_order is None:
        treatment_order = (
            treat_rows["treatment"].cat.categories.tolist()
            if len(treat_rows) and hasattr(treat_rows["treatment"], "cat")
            else sorted(treat_rows["treatment"].dropna().unique())
        )

    treat_rows["treatment"] = pd.Categorical(
        treat_rows["treatment"], categories=list(treatment_order), ordered=True
    )
    treat_rows = treat_rows.sort_values("treatment")
    out = pd.concat([total_row, treat_rows], ignore_index=True)

    n_treat = max(1, len(treatment_order))
    treat_colors = sns.color_palette("colorblind", n_colors=n_treat)
    treat_to_color = {t: treat_colors[i % len(treat_colors)] for i, t in enumerate(treatment_order)}

    y = np.arange(len(out), dtype=float) * float(y_step)

    if vline_at_zero:
        ax.axvline(0.0, linestyle="--", linewidth=1.0, color="k", zorder=0)

    for yy, (_, r) in zip(y, out.iterrows()):
        x = float(r["delta_elpd"])
        se = float(r["se_delta_elpd"]) if np.isfinite(r.get("se_delta_elpd", np.nan)) else np.nan

        color = "black" if r["row_type"] == "total" else treat_to_color.get(r["treatment"], "black")

        if np.isfinite(se):
            ax.errorbar(
                x, yy, xerr=se,
                fmt="none",
                ecolor=color,
                capsize=3, capthick=1,
                elinewidth=1,
                zorder=2
            )
        ax.plot(
            x, yy, "o",
            color=color,
            markersize=6,
            markeredgecolor="none",
            markeredgewidth=0,
            zorder=3
        )

    # Total tick+label (only tick we keep)
    if len(y):
        ax.set_yticks([float(y[0])])
        ax.set_yticklabels(["Total"], fontsize=8)
    else:
        ax.set_yticks([])
        ax.set_yticklabels([])

    # bracket over treatment rows, with endpoints exactly at the treatment row centers (y[1]..y[-1])
    if len(y) >= 2:
        y0 = float(y[1])     # first treatment row center
        y1 = float(y[-1])    # last treatment row center
        ymid = 0.5 * (y0 + y1)

        # x in axes-fraction, y in data coords:
        trans = ax.get_yaxis_transform()

        x = float(bracket_x)       # 0.0 touches y-axis line
        cap = float(bracket_cap)   # into plot (right)
        tick = float(bracket_tick) # out to label (left)

        # square bracket "[" sitting on the y-axis spine line
        ax.plot([x, x], [y0, y1], transform=trans, color="black", lw=1.0, clip_on=False, zorder=10)
        ax.plot([x, x + cap], [y0, y0], transform=trans, color="black", lw=1.0, clip_on=False, zorder=10)
        ax.plot([x, x + cap], [y1, y1], transform=trans, color="black", lw=1.0, clip_on=False, zorder=10)

        # tick from bracket to the label (left)
        ax.plot([x, x - tick], [ymid, ymid], transform=trans, color="black", lw=1.0, clip_on=False, zorder=10)

        # label at the end of the tick
        ax.text(
            x - tick - label_pad, ymid, bracket_label,
            transform=trans,
            ha="right", va="center",
            fontsize=8, color="black",
            clip_on=False, zorder=11
        )

    ax.tick_params(axis="y", length=3)  # keep Total tick visible
    ax.margins(y=float(y_margins))
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)


# ----------------------
# PLOT (edit freely)
# ----------------------
SHOW_MODEL_NAMES = True

top_h = 1.25
y_step = 1.35
bottom_h = max(1.35, 0.20 * y_step * len(delta_df) + 0.20)
fig_h = top_h + bottom_h

fig, (ax1, ax2) = plt.subplots(
    nrows=2,
    figsize=(3.2, fig_h),
    dpi=300,
    gridspec_kw={"height_ratios": [top_h, bottom_h]},
)

plot_total_elpd_from_summary_ax_black(
    res_total, ax1,
    xlabel="ELPD",
    vline_at_best=False,
    show_model_names=SHOW_MODEL_NAMES,
)

plot_delta_total_and_by_treatment_ax_colorblind_bracket(
    delta_df, ax2,
    treatment_order=treatment_order,
    xlabel="ELPD difference",
    vline_at_zero=True,
    y_step=y_step,
    bracket_label="Within-treatment",
    bracket_x=-0.02,       # touches y-axis line
    bracket_cap=0.020,   # caps into plot
    bracket_tick=0.030,  # tick out to label
    label_pad=0.006,
)

plt.tight_layout()

savepath = "../figures/bats/psisloo_combined.svg"
os.makedirs(os.path.dirname(savepath), exist_ok=True)
fig.savefig(savepath, bbox_inches="tight", pad_inches=0.01)

plt.show()

Pointwise $\Delta$ELPD (vMU $-$ vM) by animal and by treatment — the vMU advantage can be concentrated
in a subset of animals.


In [ ]:
import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns  # NEW: seaborn for colorblind palette
import os              # NEW: for svg filename

# ---- aesthetics (same style) ----
mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",   # NEW: keep text editable in SVG (same style as your other plots)
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",  # NEW: ensure white background on saved SVG/PDF
})

def _format_treatment_name(name: str) -> str:
    if name is None or (isinstance(name, float) and np.isnan(name)):
        return "Unknown"
    s = str(name).strip().lower()
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if "control" in parts and len(parts) > 1:
        parts = [p for p in parts if p != "control"]
    if not parts:
        parts = ["control"]
    parts = sorted(parts)
    parts = [p[:1].upper() + p[1:] for p in parts]
    return ", ".join(parts)

def _loo_pointwise_elpd(idata, *, var_name=None):
    loo = az.loo(idata, pointwise=True, var_name=var_name)
    loo_i = np.asarray(loo.loo_i).reshape(-1)
    k = np.asarray(loo.pareto_k).reshape(-1)
    return loo_i, k

def _se_sum(v: np.ndarray) -> float:
    """Rough SE for a SUM: sqrt(n * var(v))."""
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    n = v.size
    if n <= 1:
        return np.nan
    return float(np.sqrt(n * np.var(v, ddof=1)))

def _guess_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def delta_by_individual_treatment(
    idata_vmu,
    idata_vm,
    df_fit,
    *,
    individual_col=None,
    treat_col="treatment",
    treat_name_col="treatmentName",
    var_name=None,
):
    """
    Pointwise Δelpd = vMU - vM, aggregated per (individual, treatment).
    df_fit MUST be the exact rows/order used to fit BOTH models.
    """
    if individual_col is None:
        individual_col = _guess_col(df_fit, ["individual", "participant", "subject", "id", "ID"])
        if individual_col is None:
            raise ValueError(
                "Couldn't guess individual column. Pass individual_col='...'. "
                f"Available columns: {list(df_fit.columns)}"
            )

    vmu_i, vmu_k = _loo_pointwise_elpd(idata_vmu, var_name=var_name)
    vm_i,  vm_k  = _loo_pointwise_elpd(idata_vm,  var_name=var_name)

    n = len(df_fit)
    if vmu_i.size != n or vm_i.size != n:
        raise ValueError(
            f"Row mismatch: df_fit has {n} rows but loo_i has "
            f"{vmu_i.size} (vMU) and {vm_i.size} (vM)."
        )

    delta_i = vmu_i - vm_i
    k_max_i = np.maximum(vmu_k, vm_k)

    cols = [individual_col, treat_col]
    if treat_name_col in df_fit.columns:
        cols.append(treat_name_col)

    tmp = df_fit[cols].copy().reset_index(drop=True)
    tmp["delta_i"] = delta_i
    tmp["k_max_i"] = k_max_i

    out = []
    gb_cols = [treat_col, individual_col]
    for (treat, indiv), sub in tmp.groupby(gb_cols, sort=False):
        v = sub["delta_i"].to_numpy()
        tname = sub[treat_name_col].iloc[0] if treat_name_col in sub.columns else str(treat)
        out.append({
            "treatment": treat,
            "treatment_label": _format_treatment_name(tname),
            "individual": indiv,
            "n_obs": int(v.size),
            "delta_elpd": float(np.sum(v)),
            "se_delta_elpd": _se_sum(v),
            "pareto_k_max": float(np.max(sub["k_max_i"].to_numpy())) if len(sub) else np.nan,
        })

    return pd.DataFrame(out)

def plot_delta_subplots_by_treatment(res_ind, treatment_order, *, savepath=None):
    """
    One row of subplots, one per treatment.
    Within each subplot, y-axis = individuals (sorted by delta_elpd).
    """
    res = res_ind.copy()
    res["treatment"] = pd.Categorical(res["treatment"], categories=list(treatment_order), ordered=True)

    # seaborn colorblind palette (stable order)
    palette = sns.color_palette("colorblind", n_colors=len(treatment_order))
    treat_to_color = {t: palette[i] for i, t in enumerate(treatment_order)}
    k = len(treatment_order)

    # global x-limits incl. errorbars
    global_min, global_max = np.inf, -np.inf
    for t in treatment_order:
        sub = res[res["treatment"] == t].sort_values("delta_elpd", ascending=False)
        if sub.empty:
            continue
        x = sub["delta_elpd"].to_numpy()
        xe = sub["se_delta_elpd"].to_numpy()
        xe = np.where(np.isfinite(xe), xe, 0.0)
        global_min = min(global_min, float(np.min(x - xe)))
        global_max = max(global_max, float(np.max(x + xe)))

    if not np.isfinite(global_min):
        global_min, global_max = -1.0, 1.0
    pad = 0.06 * (global_max - global_min if global_max > global_min else 1.0)
    xlim = (global_min - pad, global_max + pad)

    # compact sizing
    max_n = int(res.groupby("treatment").size().max()) if len(res) else 1
    fig_w = max(5.5, 1.6 * k)
    fig_h = max(2.0, 0.18 * max_n + 0.8)

    fig, axes = plt.subplots(1, k, figsize=(fig_w, fig_h), dpi=300, sharex=True)
    if k == 1:
        axes = [axes]

    marker_size = 3
    cap_size = 2
    err_lw = 0.8
    title_fs = 9
    ytick_fs = 7
    xtick_fs = 8

    for i, t in enumerate(treatment_order):
        ax = axes[i]
        sub = (res[res["treatment"] == t]
               .sort_values("delta_elpd", ascending=False)
               .reset_index(drop=True))

        label = sub["treatment_label"].iloc[0] if len(sub) else str(t)
        ax.set_title(label, fontsize=title_fs)
        ax.axvline(0.0, linestyle="--", linewidth=0.9, color="k", alpha=0.8, zorder=0)

        if len(sub):
            y = np.arange(len(sub))
            x = sub["delta_elpd"].to_numpy()
            xe = sub["se_delta_elpd"].to_numpy()
            color = treat_to_color.get(t, "black")

            for yy, xx, xerr in zip(y, x, xe):
                if np.isfinite(xerr):
                    ax.errorbar(
                        xx, yy, xerr=xerr,
                        fmt="o", capsize=cap_size, elinewidth=err_lw,
                        markersize=marker_size,
                        color=color, markeredgecolor=color,
                        markeredgewidth=0.5, zorder=3
                    )
                else:
                    ax.plot(
                        xx, yy, "o",
                        markersize=marker_size,
                        color=color, markeredgecolor=color,
                        markeredgewidth=0.5, zorder=3
                    )

            ax.set_yticks(y)
            ax.set_yticklabels(sub["individual"].astype(str).tolist(), fontsize=ytick_fs)
            ax.invert_yaxis()
        else:
            ax.set_yticks([])

        ax.set_xlim(*xlim)
        ax.tick_params(axis="x", labelsize=xtick_fs)
        ax.grid(False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        if i != 0:
            ax.tick_params(axis="y", length=0)
            ax.set_ylabel("")

    fig.supxlabel("ELPD difference (vMU - vM)", fontsize=9)
    plt.tight_layout()

    if savepath:
        # save original requested file
        fig.savefig(savepath, dpi=300, bbox_inches="tight")

        # ALSO save SVG with same basename/style
        root, _ = os.path.splitext(savepath)
        fig.savefig(f"{root}.svg", bbox_inches="tight", pad_inches=0.01)

    plt.show()

# =========================
# RUN
# =========================
# df_track MUST be EXACT df used in both fits (same rows/order)
res_ind = delta_by_individual_treatment(
    idata_vmu=idata_mix_full,     # von Mises-Uniform
    idata_vm=idata_treatment,     # von Mises
    df_fit=df_track,
    individual_col="tag",          # auto-guess; or set e.g. "subject_id"
    treat_col="treatment",
    treat_name_col="treatmentName",
    var_name=None,
)

treatment_order = (
    df_track["treatment"].cat.categories.tolist()
    if hasattr(df_track["treatment"], "cat")
    else sorted(res_ind["treatment"].unique())
)

plot_delta_subplots_by_treatment(
    res_ind,
    treatment_order=treatment_order,
    savepath="../figures/bats/psisloo_delta_by_individual_by_treatment_vMU_minus_vM.pdf",
)

## 7. vMU population-level summaries

Population-level vMU summaries: per-treatment polar densities of the goal-relative bearings, showing
the directed + undirected mixture.


In [ ]:
# ============================================================
# Polar plot for the FULL mixture model:
# treatment effects on p, mu, and kappa (population-level only)
#
# Expects idata_mix_full from fit_vmum_model_full(...)
# Variables used (population-level):
#   mu0, mu_treat (if present)
#   eta_kappa0, kappa_treat (if present)   -> kappa = softplus(eta_kappa)
#   logit_p0, beta_treat (if present)      -> p = sigmoid(logit_p)
#
# IMPORTANT: This is population-level only (no tag REs).
# ============================================================

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.lines import Line2D
import seaborn as sns

# ----------------------------
# style (match yours)
# ----------------------------
DPI = 300
AX_LW     = 0.8
TICK_LW   = 0.8
TICK_LEN  = 2.5
LINE_LW   = 0.7
FONT_SIZE = 6

mpl.rcParams.update({
    "font.size": FONT_SIZE,
    "axes.labelsize": FONT_SIZE,
    "axes.titlesize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,
    "font.family": "Arial",
    "svg.fonttype": "none",
    "axes.linewidth": AX_LW,
    "lines.linewidth": LINE_LW,
    "xtick.major.width": TICK_LW,
    "ytick.major.width": TICK_LW,
    "xtick.major.size":  TICK_LEN,
    "ytick.major.size":  TICK_LEN,
    "savefig.dpi": DPI,
    "figure.dpi": DPI,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# ----------------------------
# knobs
# ----------------------------
HDI_PROB      = 0.95
MAX_DRAWS     = 4000
EDGE_RADIUS   = 1.0
AMP           = 1.0
BAND_ALPHA    = 0.35
OUTER_LW      = 0.6
KAPPA_UNIFORM = 1e-6
THETA_N       = 900

# NEW: legend toggle
SHOW_LEGEND   = False

# ----------------------------
# helpers
# ----------------------------
def circular_modulo(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def softplus(x):
    x = np.asarray(x, float)
    return np.where(x > 30, x, np.log1p(np.exp(x)))

def sigmoid(x):
    x = np.asarray(x, float)
    return 1.0 / (1.0 + np.exp(-x))

def vonmises_pdf(theta, mu, kappa):
    return np.exp(kappa * np.cos(theta - mu)) / (2 * np.pi * np.i0(kappa))

def _format_treatment_name(name: str) -> str:
    s = str(name).strip().lower()
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if "control" in parts and len(parts) > 1:
        parts = [p for p in parts if p != "control"]
    if not parts:
        parts = ["control"]
    parts = sorted(parts)
    parts = [p[:1].upper() + p[1:] for p in parts]
    return ", ".join(parts)

def _stack_samples_first(da):
    da2 = da.stack(sample=("chain", "draw"))
    other = [d for d in da2.dims if d != "sample"]
    da2 = da2.transpose("sample", *other)
    return da2.values.astype(float)

def hdi_1d(x, prob=0.95):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    n = x.size
    if n == 0:
        return np.nan, np.nan
    x = np.sort(x)
    m = int(np.floor(prob * n))
    if m < 1:
        return np.nan, np.nan
    widths = x[m:] - x[: n - m]
    j = int(np.argmin(widths))
    return float(x[j]), float(x[j + m])

def hdi_along_samples(arr, prob=0.95):
    arr = np.asarray(arr, float)
    S, T = arr.shape
    lo = np.empty(T, dtype=float)
    hi = np.empty(T, dtype=float)
    for t in range(T):
        lo[t], hi[t] = hdi_1d(arr[:, t], prob=prob)
    return lo, hi

def format_polar(ax, *, rmax):
    ax.spines["polar"].set_visible(False)
    ax.set_xticks([])
    ax.set_xticklabels([])
    ax.grid(False)
    ax.set_yticks([])
    ax.set_yticklabels([])

    ax.plot([0, 0], [EDGE_RADIUS, rmax], color="0.80", lw=0.9, ls="-", zorder=0)

    ax.add_artist(Circle((0, 0), EDGE_RADIUS,       transform=ax.transData._b,
                         color="0.75", fill=False, linewidth=OUTER_LW))
    ax.add_artist(Circle((0, 0), EDGE_RADIUS + AMP, transform=ax.transData._b,
                         color="0.85", fill=False, linewidth=OUTER_LW))

# ============================================================
# MAIN
# ============================================================

# <-- CHANGE THIS IF NEEDED
post = idata_mix_full.posterior

# required base vars
required = ["mu0", "eta_kappa0", "logit_p0"]
missing = [v for v in required if v not in post.data_vars]
if missing:
    raise KeyError(f"Missing {missing}. Available: {list(post.data_vars)[:120]} ...")

# optional treatment vars (present in full model)
has_mu_treat    = "mu_treat" in post.data_vars
has_kappa_treat = "kappa_treat" in post.data_vars
has_beta_treat  = "beta_treat" in post.data_vars

# treatment levels/order (reference = first level)
treat_levels = (
    df_track["treatment"].cat.categories.tolist()
    if hasattr(df_track["treatment"], "cat")
    else sorted(df_track["treatment"].unique())
)
ref = treat_levels[0]
nonref = treat_levels[1:]

# labels
nice = {t: _format_treatment_name(t) for t in treat_levels}
if "treatmentName" in df_track.columns:
    name_map = df_track.groupby("treatment", observed=False)["treatmentName"].first().to_dict()
    for t in treat_levels:
        nice[t] = _format_treatment_name(name_map.get(t, t))

# panels: include all treatments if any treatment effect exists; else just control
has_any_treat = (has_mu_treat or has_kappa_treat or has_beta_treat) and (len(nonref) > 0)
if has_any_treat:
    panel_levels = [ref] + nonref
    panel_labels = ["Control"] + [nice[t] for t in nonref]
else:
    panel_levels = [ref]
    panel_labels = ["Control"]

# colors: seaborn colorblind (stable ordering by treat_levels)
palette = sns.color_palette("colorblind", n_colors=max(3, len(treat_levels)))
treat_to_color = {t: palette[i % len(palette)] for i, t in enumerate(treat_levels)}
panel_colors = [treat_to_color[t] for t in panel_levels]

# posterior draws (population-level only)
rng = np.random.default_rng(123)
theta = np.linspace(-np.pi, np.pi, THETA_N, endpoint=False)

mu0 = circular_modulo(_stack_samples_first(post["mu0"]))          # (S,)
ek0 = _stack_samples_first(post["eta_kappa0"]).reshape(-1)        # (S,)
lp0 = _stack_samples_first(post["logit_p0"]).reshape(-1)          # (S,)

if has_mu_treat:
    mu_tr = _stack_samples_first(post["mu_treat"])                # (S, Kmu)
    if mu_tr.ndim == 1:
        mu_tr = mu_tr[:, None]

if has_kappa_treat:
    kt = _stack_samples_first(post["kappa_treat"])                # (S, Kk)
    if kt.ndim == 1:
        kt = kt[:, None]

if has_beta_treat:
    bt = _stack_samples_first(post["beta_treat"])                 # (S, Kp)
    if bt.ndim == 1:
        bt = bt[:, None]

# subsample once
S = mu0.shape[0]
if MAX_DRAWS is not None and S > MAX_DRAWS:
    idx = rng.choice(S, size=MAX_DRAWS, replace=False)
    mu0 = mu0[idx]
    ek0 = ek0[idx]
    lp0 = lp0[idx]
    if has_mu_treat:
        mu_tr = mu_tr[idx, :]
    if has_kappa_treat:
        kt = kt[idx, :]
    if has_beta_treat:
        bt = bt[idx, :]
    S = MAX_DRAWS

# uniform component
pdf_unif = vonmises_pdf(theta[None, :], 0.0, KAPPA_UNIFORM)  # (1, T)

summ = {}
peak_max = 0.0

for t, lab in zip(panel_levels, panel_labels):
    is_ref = (t == ref)

    # mu(t): circular add on linear scale then wrap
    mu_t = mu0.copy()
    if (not is_ref) and has_mu_treat:
        j = nonref.index(t)
        mu_t = circular_modulo(mu_t + mu_tr[:, j])

    # kappa(t): eta_kappa0 + kappa_treat then softplus
    ek_t = ek0.copy()
    if (not is_ref) and has_kappa_treat:
        j = nonref.index(t)
        ek_t = ek_t + kt[:, j]
    k_t = softplus(ek_t)

    # p(t): logit_p0 + beta_treat then sigmoid
    lp_t = lp0.copy()
    if (not is_ref) and has_beta_treat:
        j = nonref.index(t)
        lp_t = lp_t + bt[:, j]
    p_goal = sigmoid(lp_t)

    # mixture pdf
    pdf_goal = vonmises_pdf(theta[None, :], mu_t[:, None], k_t[:, None])  # (S, T)
    pdf_mix  = p_goal[:, None] * pdf_goal + (1.0 - p_goal[:, None]) * pdf_unif  # (S, T)

    med = np.median(pdf_mix, axis=0)
    lo, hi = hdi_along_samples(pdf_mix, prob=HDI_PROB)

    summ[t] = dict(med=med, lo=lo, hi=hi, label=lab)
    peak_max = max(peak_max, float(np.nanmax(hi)))

def to_r(y):
    return EDGE_RADIUS + AMP * (y / (peak_max + 1e-12))

RMAX = EDGE_RADIUS + AMP * 1.03

# plot
k = len(panel_levels)
PANEL_W = 0.85
FIG_H = 1.2

fig, axes = plt.subplots(
    1, k,
    figsize=(PANEL_W * k, FIG_H),
    dpi=DPI,
    subplot_kw={"projection": "polar"},
    squeeze=True
)
if k == 1:
    axes = [axes]

for ax, t, col, lab in zip(axes, panel_levels, panel_colors, panel_labels):
    format_polar(ax, rmax=RMAX)
    ax.set_ylim(0, RMAX)

    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)

    res = summ[t]
    ax.fill_between(theta, to_r(res["lo"]), to_r(res["hi"]),
                    facecolor=col, edgecolor="none", alpha=BAND_ALPHA, zorder=1)
    ax.plot(theta, to_r(res["med"]), color=col, lw=LINE_LW, zorder=3)

# legend (optional)
if SHOW_LEGEND:
    handles = [Line2D([0], [0], color=treat_to_color[t], lw=1.6) for t in panel_levels]
    labels  = panel_labels
    fig.legend(handles, labels,
               loc="center", bbox_to_anchor=(0.5, 0.02),
               ncol=k, frameon=True, handlelength=1.2, columnspacing=0.9)

    bottom = 0.12
else:
    bottom = 0.06

fig.subplots_adjust(left=0.03, right=0.99, top=0.97, bottom=bottom, wspace=0.18)

fig.savefig("../figures/bats/vmum_mixture_full_polar_pdfs_population_only.pdf",
            bbox_inches="tight", pad_inches=0.01)
fig.savefig("../figures/bats/vmum_mixture_full_polar_pdfs_population_only.svg",
            bbox_inches="tight", pad_inches=0.01)
plt.show()

In [ ]:
# ============================================================
# Polar plot for the FULL mixture model:
# treatment effects on p, mu, and kappa (population-level only)
#
# Expects idata_mix_full from fit_vmum_model_full(...)
# Variables used (population-level):
#   mu0, mu_treat (if present)
#   eta_kappa0, kappa_treat (if present)   -> kappa = softplus(eta_kappa)
#   logit_p0, beta_treat (if present)      -> p = sigmoid(logit_p)
#
# IMPORTANT: This is population-level only (no tag REs).
# UPDATED: fixed 2x2 layout, compact figure, slightly larger visual elements
# ============================================================

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.lines import Line2D
import seaborn as sns

# ----------------------------
# style (slightly larger elements)
# ----------------------------
DPI = 300
AX_LW     = 0.9   # was 0.8
TICK_LW   = 0.9   # was 0.8
TICK_LEN  = 3.0   # was 2.5
LINE_LW   = 0.9   # was 0.7
FONT_SIZE = 7     # was 6

mpl.rcParams.update({
    "font.size": FONT_SIZE,
    "axes.labelsize": FONT_SIZE,
    "axes.titlesize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,
    "font.family": "Arial",
    "svg.fonttype": "none",
    "axes.linewidth": AX_LW,
    "lines.linewidth": LINE_LW,
    "xtick.major.width": TICK_LW,
    "ytick.major.width": TICK_LW,
    "xtick.major.size":  TICK_LEN,
    "ytick.major.size":  TICK_LEN,
    "savefig.dpi": DPI,
    "figure.dpi": DPI,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# ----------------------------
# knobs
# ----------------------------
HDI_PROB      = 0.95
MAX_DRAWS     = 4000
EDGE_RADIUS   = 1.0
AMP           = 1.0
BAND_ALPHA    = 0.35
OUTER_LW      = 0.8   # was 0.6
KAPPA_UNIFORM = 1e-6
THETA_N       = 900

# NEW: legend toggle
SHOW_LEGEND   = False

# ----------------------------
# helpers
# ----------------------------
def circular_modulo(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def softplus(x):
    x = np.asarray(x, float)
    return np.where(x > 30, x, np.log1p(np.exp(x)))

def sigmoid(x):
    x = np.asarray(x, float)
    return 1.0 / (1.0 + np.exp(-x))

def vonmises_pdf(theta, mu, kappa):
    return np.exp(kappa * np.cos(theta - mu)) / (2 * np.pi * np.i0(kappa))

def _format_treatment_name(name: str) -> str:
    s = str(name).strip().lower()
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if "control" in parts and len(parts) > 1:
        parts = [p for p in parts if p != "control"]
    if not parts:
        parts = ["control"]
    parts = sorted(parts)
    parts = [p[:1].upper() + p[1:] for p in parts]
    return ", ".join(parts)

def _stack_samples_first(da):
    da2 = da.stack(sample=("chain", "draw"))
    other = [d for d in da2.dims if d != "sample"]
    da2 = da2.transpose("sample", *other)
    return da2.values.astype(float)

def hdi_1d(x, prob=0.95):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    n = x.size
    if n == 0:
        return np.nan, np.nan
    x = np.sort(x)
    m = int(np.floor(prob * n))
    if m < 1:
        return np.nan, np.nan
    widths = x[m:] - x[: n - m]
    j = int(np.argmin(widths))
    return float(x[j]), float(x[j + m])

def hdi_along_samples(arr, prob=0.95):
    arr = np.asarray(arr, float)
    S, T = arr.shape
    lo = np.empty(T, dtype=float)
    hi = np.empty(T, dtype=float)
    for t in range(T):
        lo[t], hi[t] = hdi_1d(arr[:, t], prob=prob)
    return lo, hi

def format_polar(ax, *, rmax):
    ax.spines["polar"].set_visible(False)
    ax.set_xticks([])
    ax.set_xticklabels([])
    ax.grid(False)
    ax.set_yticks([])
    ax.set_yticklabels([])

    ax.plot([0, 0], [EDGE_RADIUS, rmax], color="0.80", lw=0.9, ls="-", zorder=0)

    ax.add_artist(Circle((0, 0), EDGE_RADIUS,       transform=ax.transData._b,
                         color="0.75", fill=False, linewidth=OUTER_LW))
    ax.add_artist(Circle((0, 0), EDGE_RADIUS + AMP, transform=ax.transData._b,
                         color="0.85", fill=False, linewidth=OUTER_LW))

# ============================================================
# MAIN
# ============================================================

# <-- CHANGE THIS IF NEEDED
post = idata_mix_full.posterior

# required base vars
required = ["mu0", "eta_kappa0", "logit_p0"]
missing = [v for v in required if v not in post.data_vars]
if missing:
    raise KeyError(f"Missing {missing}. Available: {list(post.data_vars)[:120]} ...")

# optional treatment vars (present in full model)
has_mu_treat    = "mu_treat" in post.data_vars
has_kappa_treat = "kappa_treat" in post.data_vars
has_beta_treat  = "beta_treat" in post.data_vars

# treatment levels/order (reference = first level)
treat_levels = (
    df_track["treatment"].cat.categories.tolist()
    if hasattr(df_track["treatment"], "cat")
    else sorted(df_track["treatment"].unique())
)
ref = treat_levels[0]
nonref = treat_levels[1:]

# labels
nice = {t: _format_treatment_name(t) for t in treat_levels}
if "treatmentName" in df_track.columns:
    name_map = df_track.groupby("treatment", observed=False)["treatmentName"].first().to_dict()
    for t in treat_levels:
        nice[t] = _format_treatment_name(name_map.get(t, t))

# panels: include all treatments if any treatment effect exists; else just control
has_any_treat = (has_mu_treat or has_kappa_treat or has_beta_treat) and (len(nonref) > 0)
if has_any_treat:
    panel_levels = [ref] + nonref
    panel_labels = ["Control"] + [nice[t] for t in nonref]
else:
    panel_levels = [ref]
    panel_labels = ["Control"]

# fixed 2x2 layout supports up to 4 panels
if len(panel_levels) > 4:
    raise ValueError(
        f"2x2 layout supports up to 4 panels, but got {len(panel_levels)}. "
        "Filter panel_levels or switch to a larger grid."
    )

# colors: seaborn colorblind (stable ordering by treat_levels)
palette = sns.color_palette("colorblind", n_colors=max(3, len(treat_levels)))
treat_to_color = {t: palette[i % len(palette)] for i, t in enumerate(treat_levels)}
panel_colors = [treat_to_color[t] for t in panel_levels]

# posterior draws (population-level only)
rng = np.random.default_rng(123)
theta = np.linspace(-np.pi, np.pi, THETA_N, endpoint=False)

mu0 = circular_modulo(_stack_samples_first(post["mu0"]))          # (S,)
ek0 = _stack_samples_first(post["eta_kappa0"]).reshape(-1)        # (S,)
lp0 = _stack_samples_first(post["logit_p0"]).reshape(-1)          # (S,)

if has_mu_treat:
    mu_tr = _stack_samples_first(post["mu_treat"])                # (S, Kmu)
    if mu_tr.ndim == 1:
        mu_tr = mu_tr[:, None]

if has_kappa_treat:
    kt = _stack_samples_first(post["kappa_treat"])                # (S, Kk)
    if kt.ndim == 1:
        kt = kt[:, None]

if has_beta_treat:
    bt = _stack_samples_first(post["beta_treat"])                 # (S, Kp)
    if bt.ndim == 1:
        bt = bt[:, None]

# subsample once
S = mu0.shape[0]
if MAX_DRAWS is not None and S > MAX_DRAWS:
    idx = rng.choice(S, size=MAX_DRAWS, replace=False)
    mu0 = mu0[idx]
    ek0 = ek0[idx]
    lp0 = lp0[idx]
    if has_mu_treat:
        mu_tr = mu_tr[idx, :]
    if has_kappa_treat:
        kt = kt[idx, :]
    if has_beta_treat:
        bt = bt[idx, :]
    S = MAX_DRAWS

# uniform component
pdf_unif = vonmises_pdf(theta[None, :], 0.0, KAPPA_UNIFORM)  # (1, T)

summ = {}
peak_max = 0.0

for t, lab in zip(panel_levels, panel_labels):
    is_ref = (t == ref)

    # mu(t): circular add on linear scale then wrap
    mu_t = mu0.copy()
    if (not is_ref) and has_mu_treat:
        j = nonref.index(t)
        mu_t = circular_modulo(mu_t + mu_tr[:, j])

    # kappa(t): eta_kappa0 + kappa_treat then softplus
    ek_t = ek0.copy()
    if (not is_ref) and has_kappa_treat:
        j = nonref.index(t)
        ek_t = ek_t + kt[:, j]
    k_t = softplus(ek_t)

    # p(t): logit_p0 + beta_treat then sigmoid
    lp_t = lp0.copy()
    if (not is_ref) and has_beta_treat:
        j = nonref.index(t)
        lp_t = lp_t + bt[:, j]
    p_goal = sigmoid(lp_t)

    # mixture pdf
    pdf_goal = vonmises_pdf(theta[None, :], mu_t[:, None], k_t[:, None])  # (S, T)
    pdf_mix  = p_goal[:, None] * pdf_goal + (1.0 - p_goal[:, None]) * pdf_unif  # (S, T)

    med = np.median(pdf_mix, axis=0)
    lo, hi = hdi_along_samples(pdf_mix, prob=HDI_PROB)

    summ[t] = dict(med=med, lo=lo, hi=hi, label=lab)
    peak_max = max(peak_max, float(np.nanmax(hi)))

def to_r(y):
    return EDGE_RADIUS + AMP * (y / (peak_max + 1e-12))

RMAX = EDGE_RADIUS + AMP * 1.03

# ----------------------------
# plot (fixed 2x2, compact)
# ----------------------------
k = len(panel_levels)
nrows, ncols = 2, 2

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(2.55, 2.35),  # compact overall figure
    dpi=DPI,
    subplot_kw={"projection": "polar"},
    squeeze=False
)
axes = axes.ravel()

# hide unused panels (if k < 4)
for ax in axes[k:]:
    ax.set_visible(False)

for ax, t, col, lab in zip(axes[:k], panel_levels, panel_colors, panel_labels):
    format_polar(ax, rmax=RMAX)
    ax.set_ylim(0, RMAX)

    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)

    res = summ[t]
    ax.fill_between(
        theta, to_r(res["lo"]), to_r(res["hi"]),
        facecolor=col, edgecolor="none", alpha=BAND_ALPHA, zorder=1
    )
    ax.plot(theta, to_r(res["med"]), color=col, lw=LINE_LW, zorder=3)

    # Panel title helps in a 2x2 layout
    # ax.set_title(lab, pad=2, fontsize=FONT_SIZE)

# legend (optional)
if SHOW_LEGEND:
    handles = [Line2D([0], [0], color=treat_to_color[t], lw=1.6) for t in panel_levels]
    labels = panel_labels
    fig.legend(
        handles, labels,
        loc="lower center", bbox_to_anchor=(0.5, 0.01),
        ncol=min(2, k), frameon=True, handlelength=1.2, columnspacing=0.8
    )
    bottom = 0.14
else:
    bottom = 0.05

fig.subplots_adjust(
    left=0.04, right=0.99, top=0.95, bottom=bottom,
    wspace=0.06, hspace=0.10
)

fig.savefig("../figures/bats/vmum_mixture_full_polar_pdfs_population_only.pdf",
            bbox_inches="tight", pad_inches=0.01)
fig.savefig("../figures/bats/vmum_mixture_full_polar_pdfs_population_only.svg",
            bbox_inches="tight", pad_inches=0.01)
plt.show()

Per-treatment population **mixture weight** $p$ (probability a step is goal-directed), goal-directed
alignment to $0^\circ$, and goal-directed $\kappa$, each with posterior means and HDIs.


In [ ]:
# ============================================================
# FULL MODEL: plot population-level
#   (1) mixture weight p(goal-directed) by treatment: mean ± HDI
#   (2) kappa (goal-directed concentration) by treatment: mean ± HDI
#   (3a) predictive density at theta=0 by treatment (FULL MIXTURE): mean ± HDI (+ uniform dashed line)
#   (3b) predictive density at theta=0 by treatment (DIRECTED COMPONENT ONLY): mean ± HDI (+ uniform dashed line)
#   (4) predictive mass within ±45° by treatment: mean ± HDI (+ uniform dashed line)
#   (5) alignment to 0 by treatment: mean ± HDI
#
# Color: seaborn colorblind palette
# Style: match your other plots (Arial + svg.fonttype none + white background)
#
# UPDATE:
#   - option to hide ONLY y-ticks + y-labels (keep y-axis spine)
#   - auto-narrow figure width when y-ticks + y-labels are hidden
#   - saves PDF + SVG (same basename)
#   - plots BOTH full-mixture predictive density and directed-component-only predictive density
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

DPI = 300
mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
})

KAPPA_UNIFORM = 1e-6
MASS_HALF_WIDTH_DEG = 45.0

# --- Bessel ratio A(kappa) = I1/I0 (robust) ---
try:
    from scipy.special import i0 as _i0, i1 as _i1
    def bessel_ratio(kappa):
        kappa = np.asarray(kappa, float)
        return _i1(kappa) / _i0(kappa)
except Exception:
    # fallback approximation (good enough for plotting/summary)
    def bessel_ratio(kappa):
        k = np.asarray(kappa, float)
        out = np.empty_like(k)

        small = k < 1e-3
        mid   = (k >= 1e-3) & (k <= 50.0)
        large = k > 50.0

        out[small] = 0.5 * k[small] - (k[small] ** 3) / 16.0
        km = k[mid]
        out[mid] = km / (2.0 + km)
        kl = k[large]
        out[large] = 1.0 - 1.0/(2.0*kl) - 1.0/(8.0*kl*kl)
        return out

# --- Optional SciPy CDF for circular interval mass ---
try:
    from scipy.stats import vonmises as _sp_vonmises
    _HAS_SCIPY_VONMISES_CDF = True
except Exception:
    _HAS_SCIPY_VONMISES_CDF = False

def sigmoid(x):
    x = np.asarray(x, float)
    return 1.0 / (1.0 + np.exp(-x))

def softplus(x):
    x = np.asarray(x, float)
    return np.where(x > 30, x, np.log1p(np.exp(x)))

def circular_modulo(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def _stack_samples_first(da):
    da2 = da.stack(sample=("chain", "draw"))
    other = [d for d in da2.dims if d != "sample"]
    da2 = da2.transpose("sample", *other)
    return da2.values.astype(float)

def hdi_1d(x, prob=0.95):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    n = x.size
    if n == 0:
        return np.nan, np.nan
    x = np.sort(x)
    m = int(np.floor(prob * n))
    if m < 1 or m >= n:
        return float(x[0]), float(x[-1])
    widths = x[m:] - x[: n - m]
    j = int(np.argmin(widths))
    return float(x[j]), float(x[j + m])

def vonmises_pdf_scalar(theta, mu, kappa):
    return np.exp(kappa * np.cos(theta - mu)) / (2 * np.pi * np.i0(kappa))

def vonmises_interval_mass(theta_lo, theta_hi, mu, kappa, *, grid_n=721):
    """
    P(theta_lo <= Theta <= theta_hi) for Von Mises(mu, kappa).

    Uses SciPy CDF when available; falls back to vectorized numerical integration.
    """
    mu = np.asarray(mu, float).reshape(-1)
    kappa = np.asarray(kappa, float).reshape(-1)

    if _HAS_SCIPY_VONMISES_CDF:
        # SciPy's circular cdf can jump by 1 depending on branch; correct by wrapping.
        c_hi = _sp_vonmises.cdf(theta_hi, kappa, loc=mu)
        c_lo = _sp_vonmises.cdf(theta_lo, kappa, loc=mu)
        mass = np.asarray(c_hi - c_lo, float)

        # Bring to [0, 1] for interval length < 2π (our ±45° interval is π/2).
        mass = np.where(mass < 0.0, mass + 1.0, mass)
        mass = np.where(mass > 1.0, mass - 1.0, mass)
        return np.clip(mass, 0.0, 1.0)

    # Fallback: direct integration of pdf over the interval (vectorized across draws)
    theta = np.linspace(theta_lo, theta_hi, int(grid_n), dtype=float)[:, None]  # (G,1)
    mu2 = mu[None, :]       # (1,S)
    k2  = kappa[None, :]    # (1,S)
    pdf = np.exp(k2 * np.cos(theta - mu2)) / (2 * np.pi * np.i0(k2))
    mass = np.trapz(pdf, theta[:, 0], axis=0)
    return np.clip(np.asarray(mass, float), 0.0, 1.0)

def _format_treatment_name(name: str) -> str:
    s = str(name).strip().lower()
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if "control" in parts and len(parts) > 1:
        parts = [p for p in parts if p != "control"]
    if not parts:
        parts = ["control"]
    parts = sorted(parts)
    parts = [p[:1].upper() + p[1:] for p in parts]
    return ", ".join(parts)

def _treat_levels_and_labels(df_fit, treat_col="treatment", treat_name_col="treatmentName"):
    if isinstance(df_fit[treat_col].dtype, pd.CategoricalDtype):
        treat_levels = list(df_fit[treat_col].cat.categories)
    else:
        treat_levels = sorted(df_fit[treat_col].dropna().unique().tolist())

    if treat_name_col in df_fit.columns:
        name_map = df_fit.groupby(treat_col, observed=False)[treat_name_col].first().to_dict()
        labels = [_format_treatment_name(name_map.get(t, t)) for t in treat_levels]
    else:
        labels = [_format_treatment_name(t) for t in treat_levels]

    return treat_levels, labels

def summarize_pop_p_kappa_mu_density0_mass45_alignment0_by_treatment(
    idata_mix_full,
    df_fit,
    *,
    treat_col="treatment",
    treat_name_col="treatmentName",
    hdi_prob=0.95,
    max_draws=2000,
    kappa_uniform=KAPPA_UNIFORM,
    mass_half_width_deg=MASS_HALF_WIDTH_DEG,
    mass_grid_n=721,
):
    """
    Summarize treatment-level posterior quantities for the full mixture model.

    Returns a DataFrame with:
      - p_*           : mixture weight (directed-component probability)
      - kappa_*       : directed-component concentration
      - dens0_mix_*   : full mixture predictive density at theta=0
      - dens0_dir_*   : directed-component-only predictive density at theta=0
      - dens0_*       : backward-compatible alias to dens0_mix_*
      - mass45_*      : full mixture predictive mass within ±mass_half_width_deg
      - align0_*      : full mixture E[cos(theta)]
    """
    post = idata_mix_full.posterior

    for v in ["mu0", "logit_p0", "eta_kappa0"]:
        if v not in post.data_vars:
            raise KeyError(f"Missing '{v}'. Available: {list(post.data_vars)[:120]} ...")

    has_beta = "beta_treat" in post.data_vars
    has_kt   = "kappa_treat" in post.data_vars
    has_mt   = "mu_treat" in post.data_vars

    treat_levels, labels = _treat_levels_and_labels(df_fit, treat_col=treat_col, treat_name_col=treat_name_col)
    ref = treat_levels[0]
    nonref = treat_levels[1:]

    rng = np.random.default_rng(123)

    mu0 = circular_modulo(_stack_samples_first(post["mu0"]).reshape(-1))   # (S,)
    lp0 = _stack_samples_first(post["logit_p0"]).reshape(-1)               # (S,)
    ek0 = _stack_samples_first(post["eta_kappa0"]).reshape(-1)             # (S,)

    if has_beta:
        beta = _stack_samples_first(post["beta_treat"])
        if beta.ndim == 1:
            beta = beta[:, None]

    if has_kt:
        kt = _stack_samples_first(post["kappa_treat"])
        if kt.ndim == 1:
            kt = kt[:, None]

    if has_mt:
        mt = _stack_samples_first(post["mu_treat"])
        if mt.ndim == 1:
            mt = mt[:, None]

    S = lp0.size
    if (max_draws is not None) and (S > max_draws):
        idx = rng.choice(S, size=max_draws, replace=False)
        mu0 = mu0[idx]
        lp0 = lp0[idx]
        ek0 = ek0[idx]
        if has_beta: beta = beta[idx, :]
        if has_kt:   kt   = kt[idx, :]
        if has_mt:   mt   = mt[idx, :]
        S = max_draws

    theta_half = np.deg2rad(float(mass_half_width_deg))
    theta_lo, theta_hi = -theta_half, theta_half

    f0_unif = float(vonmises_pdf_scalar(0.0, 0.0, kappa_uniform))  # ~ 1/(2π)
    A_unif  = float(bessel_ratio(kappa_uniform))
    mass_unif = float((theta_hi - theta_lo) / (2.0 * np.pi))        # exact uniform mass in interval

    rows = []
    for i, t in enumerate(treat_levels):
        is_ref = (t == ref)

        # Mixture weight p
        lp = lp0.copy()
        if (not is_ref) and has_beta:
            j = nonref.index(t)
            lp = lp + beta[:, j]
        p = sigmoid(lp)

        # Directed concentration kappa
        ek = ek0.copy()
        if (not is_ref) and has_kt:
            j = nonref.index(t)
            ek = ek + kt[:, j]
        kappa = softplus(ek)

        # Directed mean mu
        mu = mu0.copy()
        if (not is_ref) and has_mt:
            j = nonref.index(t)
            mu = circular_modulo(mu + mt[:, j])

        # Predictive density at theta=0
        # Directed component only (conditional on directed state)
        f0_dir = vonmises_pdf_scalar(0.0, mu, kappa)

        # Full mixture predictive density
        f0_mix = p * f0_dir + (1.0 - p) * f0_unif

        # Predictive mass within ±mass_half_width_deg (full mixture)
        mass_goal = vonmises_interval_mass(theta_lo, theta_hi, mu, kappa, grid_n=mass_grid_n)
        mass_pm = p * mass_goal + (1.0 - p) * mass_unif

        # Alignment to 0 (full mixture)
        A_goal = bessel_ratio(kappa)
        align0 = p * (A_goal * np.cos(mu)) + (1.0 - p) * A_unif

        # Summaries
        p_lo, p_hi = hdi_1d(p, prob=hdi_prob)
        k_lo, k_hi = hdi_1d(kappa, prob=hdi_prob)

        f_mix_lo, f_mix_hi = hdi_1d(f0_mix, prob=hdi_prob)
        f_dir_lo, f_dir_hi = hdi_1d(f0_dir, prob=hdi_prob)

        m_lo, m_hi = hdi_1d(mass_pm, prob=hdi_prob)
        a_lo, a_hi = hdi_1d(align0, prob=hdi_prob)

        rows.append({
            "treatment": t,
            "label": labels[i],

            "p_mean": float(np.mean(p)),
            "p_lo": p_lo,
            "p_hi": p_hi,

            "kappa_mean": float(np.mean(kappa)),
            "kappa_lo": k_lo,
            "kappa_hi": k_hi,

            # Full mixture predictive density at theta=0
            "dens0_mix_mean": float(np.mean(f0_mix)),
            "dens0_mix_lo": f_mix_lo,
            "dens0_mix_hi": f_mix_hi,

            # Directed-component-only predictive density at theta=0
            "dens0_dir_mean": float(np.mean(f0_dir)),
            "dens0_dir_lo": f_dir_lo,
            "dens0_dir_hi": f_dir_hi,

            # Backward-compatible alias (old columns = full mixture)
            "dens0_mean": float(np.mean(f0_mix)),
            "dens0_lo": f_mix_lo,
            "dens0_hi": f_mix_hi,

            "mass45_mean": float(np.mean(mass_pm)),
            "mass45_lo": m_lo,
            "mass45_hi": m_hi,

            "align0_mean": float(np.mean(align0)),
            "align0_lo": a_lo,
            "align0_hi": a_hi,
        })

    out = pd.DataFrame(rows)
    out.attrs["uniform_density_at_0"] = float(f0_unif)
    out.attrs["uniform_mass_pm_deg"] = float(mass_unif)
    out.attrs["mass_half_width_deg"] = float(mass_half_width_deg)
    return out

def plot_mean_hdi_horizontal(
    res, value_col, lo_col, hi_col, xlabel, *,
    xlim=None, savepath=None,
    cap=0.18,
    lw=1.2,
    ms=5,
    palette_name="colorblind",
    show_treatment_labels=True,  # show/hide y tick labels (treatment names)
    show_treatment_ticks=True,   # show/hide y tick marks
    fig_width=None,              # override width (optional)
    fig_width_with_labels=3.3,   # default width when labels/ticks shown
    fig_width_no_labels=2.3,     # default width when labels+ticks hidden
    vline_x=None,                # optional dashed vertical reference
    vline_color="black",
    vline_ls="--",
    vline_lw=1.0,
    save_svg_also=True,          # save same basename as .svg too
):
    colors = sns.color_palette(palette_name, n_colors=len(res))
    y = np.arange(len(res), dtype=float)

    # --- auto width: narrower when BOTH labels and ticks are hidden ---
    if fig_width is None:
        if (not show_treatment_labels) and (not show_treatment_ticks):
            fig_width = fig_width_no_labels
        else:
            fig_width = fig_width_with_labels

    fig_height = 0.35 * len(res)
    fig = plt.figure(figsize=(fig_width, fig_height), dpi=DPI)
    ax = plt.gca()

    for yy, (_, r), c in zip(y, res.iterrows(), colors):
        x  = float(r[value_col])
        lo = float(r[lo_col])
        hi = float(r[hi_col])

        ax.hlines(yy, lo, hi, color=c, linewidth=lw, zorder=2)
        ax.vlines([lo, hi], yy - cap, yy + cap, color=c, linewidth=lw, zorder=2)
        ax.plot(x, yy, "o", color=c, markersize=ms, markeredgewidth=0.0, zorder=3)

    if vline_x is not None:
        ax.axvline(float(vline_x), color=vline_color, linestyle=vline_ls, linewidth=vline_lw, zorder=1)

    # keep y-axis spine; hide only ticks/labels if requested
    ax.set_yticks(y)

    if show_treatment_labels:
        ax.set_yticklabels(res["label"].tolist(), fontsize=8)
    else:
        ax.set_yticklabels([""] * len(y))

    ax.tick_params(
        axis="y",
        which="both",
        left=show_treatment_ticks,
        length=3 if show_treatment_ticks else 0,
        labelleft=show_treatment_labels,
    )

    ax.invert_yaxis()
    ax.margins(y=0.12)

    ax.set_xlabel(xlabel)
    if xlim is not None:
        ax.set_xlim(*xlim)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

    plt.tight_layout()
    if savepath:
        fig.savefig(savepath, bbox_inches="tight", pad_inches=0.01)
        if save_svg_also:
            root, _ = os.path.splitext(savepath)
            fig.savefig(root + ".svg", bbox_inches="tight", pad_inches=0.01)
    plt.show()
    return fig, ax


# ======================
# RUN (FULL MODEL)
# ======================
res_full = summarize_pop_p_kappa_mu_density0_mass45_alignment0_by_treatment(
    idata_mix_full=idata_mix_full,
    df_fit=df_track,
    treat_col="treatment",
    treat_name_col="treatmentName",
    hdi_prob=0.95,
    max_draws=2000,
    kappa_uniform=KAPPA_UNIFORM,
    mass_half_width_deg=45.0,
)
# Toggle: hide ONLY ticks + labels (keep y-axis spine). Also auto-narrows width.
HIDE_TREATMENT_TICKS_AND_LABELS = True
show_labels = not HIDE_TREATMENT_TICKS_AND_LABELS
show_ticks  = not HIDE_TREATMENT_TICKS_AND_LABELS

# Uniform reference lines
uniform_density_at_0 = 1.0 / (2.0 * np.pi)     # exact uniform density
uniform_mass_pm45    = 90.0 / 360.0            # P(|theta| <= 45°) under uniform = 1/4

# (1) Mixture weight p (+ reference line at 0.5)
plot_mean_hdi_horizontal(
    res_full,
    value_col="p_mean",
    lo_col="p_lo",
    hi_col="p_hi",
    xlabel="Mixture weight",
    xlim=(0, 1),
    vline_x=0.5,
    show_treatment_labels=show_labels,
    show_treatment_ticks=show_ticks,
    savepath="../figures/bats/fullmodel_mixture_weight_by_treatment_mean_hdi.pdf",
)

# (2) Kappa
plot_mean_hdi_horizontal(
    res_full,
    value_col="kappa_mean",
    lo_col="kappa_lo",
    hi_col="kappa_hi",
    xlabel="Concentration " + rf"($\kappa$)",
    show_treatment_labels=show_labels,
    show_treatment_ticks=show_ticks,
    savepath="../figures/bats/fullmodel_kappa_by_treatment_mean_hdi.pdf",
)

# Shared x-limit so mixture and directed-density plots are directly comparable
dens_xmax = float(max(res_full["dens0_mix_hi"].max(), res_full["dens0_dir_hi"].max()) * 1.05)
dens_xlim = (0.0, dens_xmax)

# (3a) Predictive density at 0 (FULL MIXTURE) (+ uniform dashed line)
plot_mean_hdi_horizontal(
    res_full,
    value_col="dens0_mix_mean",
    lo_col="dens0_mix_lo",
    hi_col="dens0_mix_hi",
    xlabel="Predictive density (mixture)",
    xlim=dens_xlim,
    vline_x=uniform_density_at_0,
    show_treatment_labels=show_labels,
    show_treatment_ticks=show_ticks,
    savepath="../figures/bats/fullmodel_predictive_density_at_0_mixture_by_treatment_mean_hdi.pdf",
)

# (3b) Predictive density at 0 (DIRECTED COMPONENT ONLY)
# Uniform line remains a useful no-directional-preference reference.
plot_mean_hdi_horizontal(
    res_full,
    value_col="dens0_dir_mean",
    lo_col="dens0_dir_lo",
    hi_col="dens0_dir_hi",
    xlabel="Predictive density (directed component)",
    xlim=dens_xlim,
    vline_x=uniform_density_at_0,
    show_treatment_labels=show_labels,
    show_treatment_ticks=show_ticks,
    savepath="../figures/bats/fullmodel_predictive_density_at_0_directed_component_by_treatment_mean_hdi.pdf",
)

# (4) Predictive mass within ±45° (+ uniform dashed line at 0.25)
plot_mean_hdi_horizontal(
    res_full,
    value_col="mass45_mean",
    lo_col="mass45_lo",
    hi_col="mass45_hi",
    xlabel="Predictive mass (±45°)",
    xlim=(0, 1),
    vline_x=uniform_mass_pm45,
    show_treatment_labels=show_labels,
    show_treatment_ticks=show_ticks,
    savepath="../figures/bats/fullmodel_predictive_mass_pm45deg_by_treatment_mean_hdi.pdf",
)

# (5) Alignment to 0 (E[cos theta]) under mixture
plot_mean_hdi_horizontal(
    res_full,
    value_col="align0_mean",
    lo_col="align0_lo",
    hi_col="align0_hi",
    xlabel="Alignment",
    xlim=(0, 1),
    show_treatment_labels=show_labels,
    show_treatment_ticks=show_ticks,
    savepath="../figures/bats/fullmodel_alignment_to_zero_by_treatment_mean_hdi.pdf",
)

(<Figure size 690x420 with 1 Axes>, <Axes: xlabel='Alignment'>)

Combined 2x2 summary plot of the out-of-sample comparison against the population-level vMU
parameters.


In [ ]:
# ============================================================
# COMBINED FIGURE UPDATE (as requested)
#   - DROP old top ELPD panel
#   - MOVE ELPD-difference panel to the top (now "ax1")
#   - BELOW: 2x2 population-level parameter summaries by treatment:
#       (1) mu (mean angle, degrees)
#       (2) mixture weight
#       (3) kappa reparam as MRL via Bessel ratio I1/I0
#       (4) directed-component predictive density at 0 ("Predictive density")
#   - Bottom 2x2: NO treatment labels
#   - Bottom 2x2 spans the same overall figure width/margins as the top axis
#     (so the top bracket label space is preserved in the layout)
#
# ASSUMES you already have:
#   delta_df, treatment_order, idata_mix_full, df_track
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# ----------------------------
# Style
# ----------------------------
DPI = 300
mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
})

# ----------------------------
# Helpers
# ----------------------------
def sigmoid(x):
    x = np.asarray(x, float)
    return 1.0 / (1.0 + np.exp(-x))

def softplus(x):
    x = np.asarray(x, float)
    return np.where(x > 30, x, np.log1p(np.exp(x)))

def circular_modulo(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def circ_mean(x):
    """Circular mean (radians) for samples x."""
    x = np.asarray(x, float)
    return np.arctan2(np.mean(np.sin(x)), np.mean(np.cos(x)))

def _stack_samples_first(da):
    da2 = da.stack(sample=("chain", "draw"))
    other = [d for d in da2.dims if d != "sample"]
    da2 = da2.transpose("sample", *other)
    return da2.values.astype(float)

def hdi_1d(x, prob=0.95):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    n = x.size
    if n == 0:
        return np.nan, np.nan
    x = np.sort(x)
    m = int(np.floor(prob * n))
    if m < 1 or m >= n:
        return float(x[0]), float(x[-1])
    widths = x[m:] - x[: n - m]
    j = int(np.argmin(widths))
    return float(x[j]), float(x[j + m])

# --- Bessel ratio A(kappa)=I1/I0 used for MRL ---
try:
    from scipy.special import i0 as _i0, i1 as _i1
    def bessel_ratio(kappa):
        kappa = np.asarray(kappa, float)
        return _i1(kappa) / _i0(kappa)
except Exception:
    def bessel_ratio(kappa):
        # simple fallback (fine for plotting)
        k = np.asarray(kappa, float)
        out = np.empty_like(k)
        small = k < 1e-3
        mid   = (k >= 1e-3) & (k <= 50.0)
        large = k > 50.0
        out[small] = 0.5 * k[small] - (k[small] ** 3) / 16.0
        km = k[mid]
        out[mid] = km / (2.0 + km)
        kl = k[large]
        out[large] = 1.0 - 1.0/(2.0*kl) - 1.0/(8.0*kl*kl)
        return out

def vonmises_pdf_scalar(theta, mu, kappa):
    # theta, mu can be arrays; uses numpy i0
    theta = np.asarray(theta, float)
    mu = np.asarray(mu, float)
    kappa = np.asarray(kappa, float)
    return np.exp(kappa * np.cos(theta - mu)) / (2 * np.pi * np.i0(kappa))

def _format_treatment_name(name: str) -> str:
    s = str(name).strip().lower()
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if "control" in parts and len(parts) > 1:
        parts = [p for p in parts if p != "control"]
    if not parts:
        parts = ["control"]
    parts = sorted(parts)
    parts = [p[:1].upper() + p[1:] for p in parts]
    return ", ".join(parts)

def _treat_levels_and_labels(df_fit, treat_col="treatment", treat_name_col="treatmentName"):
    if isinstance(df_fit[treat_col].dtype, pd.CategoricalDtype):
        treat_levels = list(df_fit[treat_col].cat.categories)
    else:
        treat_levels = sorted(df_fit[treat_col].dropna().unique().tolist())

    if treat_name_col in df_fit.columns:
        name_map = df_fit.groupby(treat_col, observed=False)[treat_name_col].first().to_dict()
        labels = [_format_treatment_name(name_map.get(t, t)) for t in treat_levels]
    else:
        labels = [_format_treatment_name(t) for t in treat_levels]

    return treat_levels, labels

# ----------------------------
# Top panel: ELPD difference plot (unchanged)
# ----------------------------
def plot_delta_total_and_by_treatment_ax_colorblind_bracket(
    delta_df,
    ax,
    *,
    treatment_order=None,
    xlabel="ELPD difference",
    vline_at_zero=True,
    y_step=1.35,
    y_margins=0.10,
    bracket_label="Within-treatment",
    bracket_x=0.0,         # 0.0 touches y-axis spine
    bracket_cap=0.020,     # cap length INTO the plot
    bracket_tick=0.050,    # tick length OUT to the label (left)
    label_pad=0.006,
):
    out = delta_df.copy().reset_index(drop=True)

    treat_rows = out[out["row_type"] == "treatment"].copy()
    total_row = out[out["row_type"] == "total"].copy()

    if treatment_order is None:
        treatment_order = (
            treat_rows["treatment"].cat.categories.tolist()
            if len(treat_rows) and hasattr(treat_rows["treatment"], "cat")
            else sorted(treat_rows["treatment"].dropna().unique())
        )

    treat_rows["treatment"] = pd.Categorical(
        treat_rows["treatment"], categories=list(treatment_order), ordered=True
    )
    treat_rows = treat_rows.sort_values("treatment")
    out = pd.concat([total_row, treat_rows], ignore_index=True)

    n_treat = max(1, len(treatment_order))
    treat_colors = sns.color_palette("colorblind", n_colors=n_treat)
    treat_to_color = {t: treat_colors[i % len(treat_colors)] for i, t in enumerate(treatment_order)}

    y = np.arange(len(out), dtype=float) * float(y_step)

    if vline_at_zero:
        ax.axvline(0.0, linestyle="--", linewidth=1.0, color="k", zorder=0)

    for yy, (_, r) in zip(y, out.iterrows()):
        x = float(r["delta_elpd"])
        se = float(r["se_delta_elpd"]) if np.isfinite(r.get("se_delta_elpd", np.nan)) else np.nan
        color = "black" if r["row_type"] == "total" else treat_to_color.get(r["treatment"], "black")

        if np.isfinite(se):
            ax.errorbar(
                x, yy, xerr=se,
                fmt="none",
                ecolor=color,
                capsize=3, capthick=1,
                elinewidth=1,
                zorder=2
            )
        ax.plot(
            x, yy, "o",
            color=color,
            markersize=6,
            markeredgecolor="none",
            markeredgewidth=0,
            zorder=3
        )

    # Total tick+label (only tick we keep)
    if len(y):
        ax.set_yticks([float(y[0])])
        ax.set_yticklabels(["Total"], fontsize=8)
    else:
        ax.set_yticks([])
        ax.set_yticklabels([])

    # bracket over treatment rows
    if len(y) >= 2:
        y0 = float(y[1])     # first treatment row center
        y1 = float(y[-1])    # last treatment row center
        ymid = 0.5 * (y0 + y1)

        trans = ax.get_yaxis_transform()
        x = float(bracket_x)
        cap = float(bracket_cap)
        tick = float(bracket_tick)

        ax.plot([x, x], [y0, y1], transform=trans, color="black", lw=1.0, clip_on=False, zorder=10)
        ax.plot([x, x + cap], [y0, y0], transform=trans, color="black", lw=1.0, clip_on=False, zorder=10)
        ax.plot([x, x + cap], [y1, y1], transform=trans, color="black", lw=1.0, clip_on=False, zorder=10)
        ax.plot([x, x - tick], [ymid, ymid], transform=trans, color="black", lw=1.0, clip_on=False, zorder=10)

        ax.text(
            x - tick - label_pad, ymid, bracket_label,
            transform=trans,
            ha="right", va="center",
            fontsize=8, color="black",
            clip_on=False, zorder=11
        )

    ax.tick_params(axis="y", length=3)
    ax.margins(y=float(y_margins))
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

# ----------------------------
# Summaries for the 2x2 params (population-level, by treatment)
# ----------------------------
def summarize_pop_params_for_2x2_by_treatment(
    idata_mix_full,
    df_fit,
    *,
    treat_col="treatment",
    treat_name_col="treatmentName",
    treatment_order=None,
    hdi_prob=0.95,
    max_draws=2000,
):
    """
    Returns DataFrame with columns:
      mu_deg_mean, mu_deg_lo, mu_deg_hi  (mean angle, degrees; HDI computed around circular mean)
      p_mean, p_lo, p_hi                 (mixture weight)
      mrl_mean, mrl_lo, mrl_hi           (I1/I0(kappa))
      dens0_dir_mean, dens0_dir_lo, dens0_dir_hi  (directed-component density at 0)
    """
    post = idata_mix_full.posterior
    for v in ["mu0", "logit_p0", "eta_kappa0"]:
        if v not in post.data_vars:
            raise KeyError(f"Missing '{v}' in posterior. Found: {list(post.data_vars)[:80]} ...")

    has_beta = "beta_treat" in post.data_vars
    has_kt   = "kappa_treat" in post.data_vars
    has_mt   = "mu_treat" in post.data_vars

    treat_levels, labels = _treat_levels_and_labels(df_fit, treat_col=treat_col, treat_name_col=treat_name_col)

    # enforce explicit order if provided
    if treatment_order is not None:
        treat_levels = list(treatment_order)
        # map labels accordingly (fallback to formatted level)
        lab_map = dict(zip(*_treat_levels_and_labels(df_fit, treat_col=treat_col, treat_name_col=treat_name_col)))
        labels = [_format_treatment_name(lab_map.get(t, t)) for t in treat_levels]

    ref = treat_levels[0]
    nonref = treat_levels[1:]

    rng = np.random.default_rng(123)

    mu0 = circular_modulo(_stack_samples_first(post["mu0"]).reshape(-1))
    lp0 = _stack_samples_first(post["logit_p0"]).reshape(-1)
    ek0 = _stack_samples_first(post["eta_kappa0"]).reshape(-1)

    if has_beta:
        beta = _stack_samples_first(post["beta_treat"])
        if beta.ndim == 1:
            beta = beta[:, None]
    if has_kt:
        kt = _stack_samples_first(post["kappa_treat"])
        if kt.ndim == 1:
            kt = kt[:, None]
    if has_mt:
        mt = _stack_samples_first(post["mu_treat"])
        if mt.ndim == 1:
            mt = mt[:, None]

    S = lp0.size
    if (max_draws is not None) and (S > max_draws):
        idx = rng.choice(S, size=max_draws, replace=False)
        mu0 = mu0[idx]
        lp0 = lp0[idx]
        ek0 = ek0[idx]
        if has_beta: beta = beta[idx, :]
        if has_kt:   kt   = kt[idx, :]
        if has_mt:   mt   = mt[idx, :]
        S = max_draws

    rows = []
    for i, t in enumerate(treat_levels):
        is_ref = (t == ref)

        # mixture weight
        lp = lp0.copy()
        if (not is_ref) and has_beta:
            j = nonref.index(t)
            lp = lp + beta[:, j]
        p = sigmoid(lp)

        # kappa and MRL
        ek = ek0.copy()
        if (not is_ref) and has_kt:
            j = nonref.index(t)
            ek = ek + kt[:, j]
        kappa = softplus(ek)
        mrl = bessel_ratio(kappa)

        # mu (radians)
        mu = mu0.copy()
        if (not is_ref) and has_mt:
            j = nonref.index(t)
            mu = circular_modulo(mu + mt[:, j])

        # circular mean + HDI around the mean (avoid wrap issues)
        mu_bar = circ_mean(mu)
        dmu = circular_modulo(mu - mu_bar)            # centered deviations in (-pi, pi]
        dmu_deg = np.rad2deg(dmu)
        dlo, dhi = hdi_1d(dmu_deg, prob=hdi_prob)

        mu_bar_deg = float(np.rad2deg(mu_bar))
        mu_lo_deg = mu_bar_deg + float(dlo)
        mu_hi_deg = mu_bar_deg + float(dhi)

        # directed-component predictive density at 0
        f0_dir = vonmises_pdf_scalar(0.0, mu, kappa)

        # summarize
        p_lo, p_hi = hdi_1d(p, prob=hdi_prob)
        mrl_lo, mrl_hi = hdi_1d(mrl, prob=hdi_prob)
        f_lo, f_hi = hdi_1d(f0_dir, prob=hdi_prob)

        rows.append({
            "treatment": t,
            "label": labels[i],

            "mu_deg_mean": mu_bar_deg,
            "mu_deg_lo": mu_lo_deg,
            "mu_deg_hi": mu_hi_deg,

            "p_mean": float(np.mean(p)),
            "p_lo": p_lo,
            "p_hi": p_hi,

            "mrl_mean": float(np.mean(mrl)),
            "mrl_lo": mrl_lo,
            "mrl_hi": mrl_hi,

            "dens0_dir_mean": float(np.mean(f0_dir)),
            "dens0_dir_lo": f_lo,
            "dens0_dir_hi": f_hi,
        })

    return pd.DataFrame(rows)

# ----------------------------
# Plot helper for bottom 2x2 (on provided axes)
# ----------------------------
def plot_mean_hdi_horizontal_on_ax(
    ax,
    res,
    *,
    value_col,
    lo_col,
    hi_col,
    xlabel,
    xlim=None,
    vline_x=None,
    palette_name="colorblind",
    show_y=False,          # NO treatment labels/ticks
    cap=0.18,
    lw=1.2,
    ms=5,
):
    colors = sns.color_palette(palette_name, n_colors=len(res))
    y = np.arange(len(res), dtype=float)

    for yy, (_, r), c in zip(y, res.iterrows(), colors):
        x  = float(r[value_col])
        lo = float(r[lo_col])
        hi = float(r[hi_col])

        ax.hlines(yy, lo, hi, color=c, linewidth=lw, zorder=2)
        ax.vlines([lo, hi], yy - cap, yy + cap, color=c, linewidth=lw, zorder=2)
        ax.plot(x, yy, "o", color=c, markersize=ms, markeredgewidth=0.0, zorder=3)

    if vline_x is not None:
        ax.axvline(float(vline_x), color="black", linestyle="--", linewidth=1.0, zorder=1)

    ax.set_yticks(y)
    if show_y:
        ax.set_yticklabels(res["label"].tolist(), fontsize=8)
        ax.tick_params(axis="y", left=True, labelleft=True, length=3)
    else:
        ax.set_yticklabels([""] * len(y))
        ax.tick_params(axis="y", left=False, labelleft=False, length=0)

    ax.invert_yaxis()
    ax.margins(y=0.12)

    ax.set_xlabel(xlabel)
    if xlim is not None:
        ax.set_xlim(*xlim)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

# ============================================================
# BUILD THE NEW COMPOSITE FIGURE
# ============================================================

# --- 1) Summarize population-level params by treatment for 2x2 ---
res_params = summarize_pop_params_for_2x2_by_treatment(
    idata_mix_full=idata_mix_full,
    df_fit=df_track,
    treat_col="treatment",
    treat_name_col="treatmentName",
    treatment_order=treatment_order,
    hdi_prob=0.95,
    max_draws=2000,
)

# --- 2) Choose x-limits / reference lines ---
# mu: auto-pad from HDIs
mu_min = float(np.nanmin(res_params["mu_deg_lo"]))
mu_max = float(np.nanmax(res_params["mu_deg_hi"]))
mu_pad = 0.08 * (mu_max - mu_min + 1e-9)
mu_xlim = (mu_min - mu_pad, mu_max + mu_pad)

# mixture weight, MRL
p_xlim = (0.0, 1.0)
mrl_xlim = (0.0, 1.0)

# directed-component predictive density at 0
dens_xmax = float(np.nanmax(res_params["dens0_dir_hi"]) * 1.08)
dens_xlim = (0.0, max(0.5, dens_xmax))
uniform_density = 1.0 / (2.0 * np.pi)

# --- 3) Figure geometry (keeps left margin wide enough for bracket label) ---
n_treat = len(res_params)
y_step = 1.35

top_h = 1.45
row_h = 0.35 * n_treat + 0.45     # height per row of the 2x2 block
fig_h = top_h + 2.0 * row_h

fig_w = 5  # adjust to taste

fig = plt.figure(figsize=(fig_w, fig_h), dpi=DPI)

# Grid: 3 rows x 2 cols (top spans both cols; bottom = 2x2)
gs = fig.add_gridspec(
    nrows=3, ncols=2,
    height_ratios=[top_h, row_h, row_h],
    hspace=0.55, wspace=0.32
)

ax_top = fig.add_subplot(gs[0, :])
ax_mu   = fig.add_subplot(gs[1, 0])
ax_p    = fig.add_subplot(gs[1, 1], sharey=ax_mu)
ax_mrl  = fig.add_subplot(gs[2, 0], sharey=ax_mu)
ax_den  = fig.add_subplot(gs[2, 1], sharey=ax_mu)

# IMPORTANT: reserve left margin for the top bracket label so bottom plots
# "extend to match" the full width of the top axis + bracket label space.
fig.subplots_adjust(left=0.30, right=0.985, top=0.98, bottom=0.08)

# --- 4) Top: ELPD difference (was ax2; now ax1/top) ---
plot_delta_total_and_by_treatment_ax_colorblind_bracket(
    delta_df, ax_top,
    treatment_order=treatment_order,
    xlabel="ELPD difference",
    vline_at_zero=True,
    y_step=y_step,
    bracket_label="Within-treatment",
    bracket_x=0.0,       # bracket on the y-axis spine
    bracket_cap=0.020,
    bracket_tick=0.055,  # longer tick to label
    label_pad=0.012,
)

# --- 5) Bottom: 2x2 parameter summaries (NO treatment labels) ---
plot_mean_hdi_horizontal_on_ax(
    ax_mu, res_params,
    value_col="mu_deg_mean", lo_col="mu_deg_lo", hi_col="mu_deg_hi",
    xlabel="Mean angle (deg)",
    xlim=mu_xlim,
    vline_x=0.0,          # goal direction reference
    show_y=False,
)

plot_mean_hdi_horizontal_on_ax(
    ax_p, res_params,
    value_col="p_mean", lo_col="p_lo", hi_col="p_hi",
    xlabel="Mixture weight",
    xlim=p_xlim,
    vline_x=0.5,
    show_y=False,
)

plot_mean_hdi_horizontal_on_ax(
    ax_mrl, res_params,
    value_col="mrl_mean", lo_col="mrl_lo", hi_col="mrl_hi",
    xlabel="MRL (directed component)",
    xlim=mrl_xlim,
    vline_x=0.0,
    show_y=False,
)

plot_mean_hdi_horizontal_on_ax(
    ax_den, res_params,
    value_col="dens0_dir_mean", lo_col="dens0_dir_lo", hi_col="dens0_dir_hi",
    xlabel="Predictive density",
    xlim=dens_xlim,
    vline_x=uniform_density,   # uniform baseline density
    show_y=False,
)

# Optional: slightly reduce clutter on shared y across bottom
for ax in (ax_p, ax_mrl, ax_den):
    ax.spines["left"].set_visible(True)  # keep spine for alignment

# --- 6) Save ---
savepath = "../figures/bats/psisloo_plus_params_2x2.svg"
os.makedirs(os.path.dirname(savepath), exist_ok=True)
fig.savefig(savepath, bbox_inches="tight", pad_inches=0.01)

# also PDF
root, _ = os.path.splitext(savepath)
fig.savefig(root + ".pdf", bbox_inches="tight", pad_inches=0.01)

plt.show()

Per-treatment predicted posterior density at the goal direction for the vMU (analog of the vM
baseline figure above).


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import arviz as az

# ----------------------
# style (match reference ELPD plot)
# ----------------------
DPI = 300
mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
})

# ============================================================
# Helper: build res_treat (von Mises only) from idata_treatment / idata_treat
# Output columns:
#   dens0_mean, dens0_lo, dens0_hi, label
# (population-level predictive density at theta=0, no mixture)
# ============================================================
def build_res_treat_from_vonmises_idata(
    idata_treatment,
    df_track,
    *,
    theta0_deg=0.0,
    hdi_prob=0.95,
):
    post = idata_treatment.posterior
    theta0 = np.deg2rad(theta0_deg)

    # ---------- helpers ----------
    def _has(varname: str) -> bool:
        return varname in post.data_vars

    def circular_modulo(x):
        return (x + np.pi) % (2 * np.pi) - np.pi

    def softplus(x):
        x = np.asarray(x, float)
        return np.where(x > 30, x, np.log1p(np.exp(x)))

    def vonmises_pdf(theta, mu, kappa):
        # pdf(theta | mu, kappa) = exp(kappa cos(theta-mu)) / (2π I0(kappa))
        return np.exp(kappa * np.cos(theta - mu)) / (2.0 * np.pi * np.i0(kappa))

    def _stack_samples(da):
        return da.stack(sample=("chain", "draw")).values.astype(float)

    def _format_treatment_name(name: str):
        s = str(name).strip().lower()
        parts = [p.strip() for p in s.split(",") if p.strip()]
        if "control" in parts and len(parts) > 1:
            parts = [p for p in parts if p != "control"]
        if not parts:
            parts = ["control"]
        parts = sorted(parts)
        parts = [p[:1].upper() + p[1:] for p in parts]
        return ", ".join(parts)

    def _cat_dim(varname: str):
        da = post[varname]
        dims = [d for d in da.dims if d not in ("chain", "draw")]
        if len(dims) != 1:
            raise ValueError(f"Expected 1 non-(chain/draw) dim for '{varname}', got {da.dims}")
        return dims[0]

    def _sel_level(varname: str, lvl, nonref_levels):
        dim = _cat_dim(varname)
        coords = list(post[varname].coords[dim].values)
        if lvl in coords:
            return {dim: lvl}
        idx = nonref_levels.index(lvl)
        return {dim: coords[idx]}

    # ---------- require intercepts ----------
    if not _has("Intercept"):
        raise KeyError(f"Missing 'Intercept' (needed for μ). Available: {list(post.data_vars)[:40]} ...")
    if not _has("kappa_Intercept"):
        raise KeyError(f"Missing 'kappa_Intercept' (needed for κ). Available: {list(post.data_vars)[:40]} ...")

    mu_has_treat = _has("treatment")
    k_has_treat  = _has("kappa_treatment")

    # ---------- treatment levels ----------
    treat_levels = (
        df_track["treatment"].cat.categories.tolist()
        if hasattr(df_track["treatment"], "cat")
        else sorted(df_track["treatment"].unique())
    )
    ref = treat_levels[0]
    nonref = treat_levels[1:]

    # ---------- pretty labels ----------
    nice = {t: str(t) for t in treat_levels}
    if "treatmentName" in df_track.columns:
        name_map = df_track.groupby("treatment", observed=False)["treatmentName"].first().to_dict()
        for t in treat_levels:
            nice[t] = _format_treatment_name(name_map.get(t, t))
    else:
        for t in treat_levels:
            nice[t] = _format_treatment_name(t)

    # ---------- rows to plot ----------
    rows_have_treat = (mu_has_treat or k_has_treat) and (len(nonref) > 0)
    if rows_have_treat:
        labels = ["Control"] + [nice[t] for t in nonref]
        row_treatment = [ref] + nonref
    else:
        labels = ["Control"]
        row_treatment = [ref]

    # ---------- population-level intercept draws ----------
    b0_mu = _stack_samples(post["Intercept"])        # (S,)
    b0_k  = _stack_samples(post["kappa_Intercept"])  # (S,)

    # ---------- compute predictive density at theta0 ----------
    dens_draws_by_row = []
    for t in row_treatment:
        eta_mu = b0_mu.copy()
        eta_k  = b0_k.copy()

        if t != ref:
            if mu_has_treat:
                b_trt_mu = _stack_samples(post["treatment"].sel(**_sel_level("treatment", t, nonref)))
                eta_mu = eta_mu + b_trt_mu
            if k_has_treat:
                b_trt_k = _stack_samples(post["kappa_treatment"].sel(**_sel_level("kappa_treatment", t, nonref)))
                eta_k = eta_k + b_trt_k

        mu    = circular_modulo(eta_mu)
        kappa = softplus(eta_k)
        dens0 = vonmises_pdf(theta0, mu, kappa)  # (S,)
        dens_draws_by_row.append(dens0)

    # ---------- summaries ----------
    dens0_mean = np.array([d.mean() for d in dens_draws_by_row], float)
    hdi = np.array([az.hdi(d, hdi_prob=hdi_prob) for d in dens_draws_by_row], float)  # (K, 2)
    dens0_lo = hdi[:, 0]
    dens0_hi = hdi[:, 1]

    res_treat = pd.DataFrame({
        "label": labels,
        "dens0_mean": dens0_mean,
        "dens0_lo": dens0_lo,
        "dens0_hi": dens0_hi,
    })

    return res_treat


# ============================================================
# plotting helper (AX version)
# ============================================================
def plot_predictive_density_hdi_ax_colorblind(
    res,
    ax,
    *,
    xlabel="Predictive density",
    xlim=None,
    vline_x=None,
    title=None,
    title_fontsize=8,
    show_treatment_labels=True,
    show_treatment_ticks=True,
    y_step=1.35,
    y_margins=0.10,
    cap=0.18,
    lw=1.0,
    ms=6,
    palette_name="colorblind",
):
    out = res.copy().reset_index(drop=True)
    y = np.arange(len(out), dtype=float) * float(y_step)

    colors = sns.color_palette(palette_name, n_colors=len(out))

    if vline_x is not None:
        ax.axvline(float(vline_x), linestyle="--", linewidth=1.0, color="k", zorder=0)

    for yy, (_, r), c in zip(y, out.iterrows(), colors):
        x  = float(r["dens0_mean"])
        lo = float(r["dens0_lo"])
        hi = float(r["dens0_hi"])

        ax.hlines(yy, lo, hi, color=c, linewidth=lw, zorder=2)
        ax.vlines([lo, hi], yy - cap, yy + cap, color=c, linewidth=lw, zorder=2)
        ax.plot(
            x, yy, "o",
            color=c,
            markersize=ms,
            markeredgecolor="none",
            markeredgewidth=0,
            zorder=3
        )

    ax.set_yticks(y)
    if show_treatment_labels:
        ax.set_yticklabels(out["label"].tolist(), fontsize=8)
    else:
        ax.set_yticklabels([""] * len(y))

    ax.tick_params(
        axis="y",
        which="both",
        left=show_treatment_ticks,
        length=3 if show_treatment_ticks else 0,
        labelleft=show_treatment_labels,
    )

    ax.margins(y=float(y_margins))
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    if xlim is not None:
        ax.set_xlim(*xlim)

    if title is not None:
        ax.set_title(title, fontsize=title_fontsize, pad=3)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)


# ============================================================
# Required inputs
# - res_full from your vMU summary code
# - df_track
# - idata_treat or idata_treatment (vM model)
# ============================================================
if "res_full" not in globals():
    raise NameError(
        "`res_full` is missing. Run your full-model summary code first "
        "(summarize_pop_p_kappa_mu_density0_mass45_alignment0_by_treatment)."
    )

if "df_track" not in globals():
    raise NameError("`df_track` is required for treatment order/labels.")

if "idata_treat" in globals():
    _idata_vm = idata_treat
elif "idata_treatment" in globals():
    _idata_vm = idata_treatment
else:
    raise NameError("Could not find `idata_treat` or `idata_treatment` in the namespace.")

# ============================================================
# IMPORTANT: use DIRECTED-COMPONENT-ONLY predictive density from vMU
# res_full.dens0_* is the FULL MIXTURE alias in your summary code, so
# for matched comparison against vM we remap to dens0_dir_* here.
# ============================================================
required_dir_cols = ["label", "dens0_dir_mean", "dens0_dir_lo", "dens0_dir_hi"]
missing_dir_cols = [c for c in required_dir_cols if c not in res_full.columns]
if missing_dir_cols:
    raise KeyError(
        "res_full is missing directed-density columns needed for matched comparison: "
        f"{missing_dir_cols}. Make sure your full-model summary code produced dens0_dir_*."
    )

res_full_matched = res_full.copy()
res_full_matched["dens0_mean"] = res_full_matched["dens0_dir_mean"]
res_full_matched["dens0_lo"]   = res_full_matched["dens0_dir_lo"]
res_full_matched["dens0_hi"]   = res_full_matched["dens0_dir_hi"]

# Optional sanity check message (can remove)
print("Using vMU directed-component-only predictive density columns for matched comparison:")
print("  dens0_mean <- dens0_dir_mean")
print("  dens0_lo   <- dens0_dir_lo")
print("  dens0_hi   <- dens0_dir_hi")

# Build vM summaries
res_treat = build_res_treat_from_vonmises_idata(
    _idata_vm,
    df_track=df_track,
    theta0_deg=0.0,
    hdi_prob=0.95,
)

# ============================================================
# TWO-PANEL FIGURE (matched comparison: vMU directed component vs vM)
# ============================================================

# knobs (match reference)
HIDE_TREATMENT_TICKS_AND_LABELS = True
SHOW_TREATMENT_LABELS = not HIDE_TREATMENT_TICKS_AND_LABELS
SHOW_TREATMENT_TICKS  = not HIDE_TREATMENT_TICKS_AND_LABELS

y_step = 1.35
FIG_W = 3.2

# uniform reference density line
uniform_density_at_0 = 1.0 / (2.0 * np.pi)

# shared x-limits across panels (include uniform line)
xmin = min(
    float(np.nanmin(res_full_matched["dens0_lo"])),
    float(np.nanmin(res_treat["dens0_lo"])),
    uniform_density_at_0,
)
xmax = max(
    float(np.nanmax(res_full_matched["dens0_hi"])),
    float(np.nanmax(res_treat["dens0_hi"])),
    uniform_density_at_0,
)
pad = 0.10 * (xmax - xmin if xmax > xmin else 1.0)
shared_xlim = (xmin - pad, xmax + pad)

# panel heights
top_h = max(1.35, 0.20 * y_step * len(res_full_matched) + 0.20)
bottom_h = max(1.35, 0.20 * y_step * len(res_treat) + 0.20)
fig_h = top_h + bottom_h

fig, (ax1, ax2) = plt.subplots(
    nrows=2,
    figsize=(FIG_W, fig_h),
    dpi=DPI,
    sharex=True,
    gridspec_kw={"height_ratios": [top_h, bottom_h]},
)

# Top panel: vMU model (DIRECTED COMPONENT ONLY, matched comparison)
plot_predictive_density_hdi_ax_colorblind(
    res_full_matched,
    ax1,
    xlabel="Predictive density",
    xlim=shared_xlim,
    vline_x=uniform_density_at_0,
    title="von Mises-Uniform (directed component only)",
    title_fontsize=8,
    show_treatment_labels=SHOW_TREATMENT_LABELS,
    show_treatment_ticks=SHOW_TREATMENT_TICKS,
    y_step=y_step,
    y_margins=0.10,
    cap=0.18,
    lw=1.0,
    ms=6,
    palette_name="colorblind",
)

# Bottom panel: vM model (von Mises)
plot_predictive_density_hdi_ax_colorblind(
    res_treat,
    ax2,
    xlabel="Predictive density",
    xlim=shared_xlim,
    vline_x=uniform_density_at_0,
    title="von Mises",
    title_fontsize=8,
    show_treatment_labels=SHOW_TREATMENT_LABELS,
    show_treatment_ticks=SHOW_TREATMENT_TICKS,
    y_step=y_step,
    y_margins=0.10,
    cap=0.18,
    lw=1.0,
    ms=6,
    palette_name="colorblind",
)

plt.tight_layout()

# save both SVG + PDF (same basename)
savebase = "../figures/bats/predictive_density_two_model_comparison_matched_vmu_directed_vs_vm"
os.makedirs(os.path.dirname(savebase), exist_ok=True)
fig.savefig(savebase + ".svg", bbox_inches="tight", pad_inches=0.01)
fig.savefig(savebase + ".pdf", bbox_inches="tight", pad_inches=0.01)

plt.show()

Using vMU directed-component-only predictive density columns for matched comparison:
  dens0_mean <- dens0_dir_mean
  dens0_lo   <- dens0_dir_lo
  dens0_hi   <- dens0_dir_hi


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# Assumes these already exist:
# - res_total, delta_df, treatment_order
# - res_full   (predictive density summary for mixture model only)
# - plot_total_elpd_from_summary_ax_black(...)
# - plot_delta_total_and_by_treatment_ax_colorblind_bracket(...)
# - plot_predictive_density_hdi_ax_colorblind(...)

# ----------------------
# style (match reference ELPD plot)
# ----------------------
mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
})

# ----------------------
# knobs
# ----------------------
SHOW_MODEL_NAMES = True

# predictive panel labels/ticks
HIDE_PRED_TREATMENT_TICKS_AND_LABELS = True
SHOW_PRED_LABELS = not HIDE_PRED_TREATMENT_TICKS_AND_LABELS
SHOW_PRED_TICKS  = not HIDE_PRED_TREATMENT_TICKS_AND_LABELS

FIG_W = 3.2
DPI = 300
y_step = 1.35

# NEW: larger x-axis label/tick fonts (apply to all three panels)
X_LABEL_FS = 10
X_TICK_FS  = 9

# uniform reference density (for mixture predictive density panel)
uniform_density_at_0 = 1.0 / (2.0 * np.pi)

# x-limits for predictive panel (include uniform line)
xmin_pred = min(float(np.nanmin(res_full["dens0_lo"])), uniform_density_at_0)
xmax_pred = max(float(np.nanmax(res_full["dens0_hi"])), uniform_density_at_0)
pad_pred = 0.10 * (xmax_pred - xmin_pred if xmax_pred > xmin_pred else 1.0)
xlim_pred = (xmin_pred - pad_pred, xmax_pred + pad_pred)

# ----------------------
# panel heights (same style/formula as your ELPD figure)
# ----------------------
h1 = 1.25  # total ELPD panel
h2 = max(1.35, 0.20 * y_step * len(delta_df) + 0.20)   # ELPD delta panel
h3 = max(1.35, 0.20 * y_step * len(res_full) + 0.20)   # predictive density panel
fig_h = h1 + h2 + h3

fig, (ax1, ax2, ax3) = plt.subplots(
    nrows=3,
    figsize=(FIG_W, fig_h),
    dpi=DPI,
    gridspec_kw={"height_ratios": [h1, h2, h3]},
)

# ----------------------
# Panel 1: Total ELPD
# ----------------------
plot_total_elpd_from_summary_ax_black(
    res_total, ax1,
    xlabel="ELPD",
    vline_at_best=False,
    show_model_names=SHOW_MODEL_NAMES,
)

# ----------------------
# Panel 2: ELPD difference (total + within-treatment)
# ----------------------
plot_delta_total_and_by_treatment_ax_colorblind_bracket(
    delta_df, ax2,
    treatment_order=treatment_order,
    xlabel="ELPD difference",
    vline_at_zero=True,
    y_step=y_step,
    bracket_label="Within-treatment",
    bracket_x=-0.02,
    bracket_cap=0.020,
    bracket_tick=0.030,
    label_pad=0.006,
)

# ----------------------
# Panel 3: Predictive density (mixture model only; no title)
# ----------------------
plot_predictive_density_hdi_ax_colorblind(
    res_full,
    ax3,
    xlabel="Predictive density",
    xlim=xlim_pred,
    vline_x=uniform_density_at_0,
    title=None,  # <- no subplot title
    show_treatment_labels=SHOW_PRED_LABELS,
    show_treatment_ticks=SHOW_PRED_TICKS,
    y_step=y_step,
    y_margins=0.10,
    cap=0.18,
    lw=1.0,
    ms=6,
    palette_name="colorblind",
)

# ----------------------
# NEW: increase x-axis label + tick font sizes across all panels
# ----------------------
for ax in (ax1, ax2, ax3):
    ax.xaxis.label.set_size(X_LABEL_FS)
    ax.tick_params(axis="x", which="major", labelsize=X_TICK_FS)
    ax.tick_params(axis="x", which="minor", labelsize=X_TICK_FS)

plt.tight_layout()

# save SVG + PDF
savebase = "../figures/bats/psisloo_plus_predictive_density_3panel"
os.makedirs(os.path.dirname(savebase), exist_ok=True)
fig.savefig(savebase + ".svg", bbox_inches="tight", pad_inches=0.01)
fig.savefig(savebase + ".pdf", bbox_inches="tight", pad_inches=0.01)

plt.show()

## 8. Per-animal level

Helpers plus **per-animal (tag-level)** vMU summaries: each bat's mixture weight, goal-directed
alignment, and $\kappa$. This is where pronounced between-animal heterogeneity becomes visible.


In [ ]:
import numpy as np
import pandas as pd

KAPPA_UNIFORM = 1e-6

def sigmoid(x):
    x = np.asarray(x, float)
    return 1.0 / (1.0 + np.exp(-x))

def softplus(x):
    x = np.asarray(x, float)
    return np.where(x > 30, x, np.log1p(np.exp(x)))

def circular_modulo(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def _stack_samples_first(da):
    da2 = da.stack(sample=("chain", "draw"))
    other = [d for d in da2.dims if d != "sample"]
    da2 = da2.transpose("sample", *other)
    return da2.values.astype(float)

def hdi_1d(x, prob=0.95):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    n = x.size
    if n == 0:
        return np.nan, np.nan
    x = np.sort(x)
    m = int(np.floor(prob * n))
    if m < 1:
        return np.nan, np.nan
    widths = x[m:] - x[: n - m]
    j = int(np.argmin(widths))
    return float(x[j]), float(x[j + m])

def vonmises_pdf_scalar(theta, mu, kappa):
    return np.exp(kappa * np.cos(theta - mu)) / (2 * np.pi * np.i0(kappa))

# --- Bessel ratio A(kappa) = I1/I0 (robust) ---
try:
    from scipy.special import i0 as _i0, i1 as _i1
    def bessel_ratio(kappa):
        kappa = np.asarray(kappa, float)
        return _i1(kappa) / _i0(kappa)
except Exception:
    def bessel_ratio(kappa):
        k = np.asarray(kappa, float)
        out = np.empty_like(k)
        small = k < 1e-3
        mid   = (k >= 1e-3) & (k <= 50.0)
        large = k > 50.0
        out[small] = 0.5 * k[small] - (k[small] ** 3) / 16.0
        km = k[mid]
        out[mid] = km / (2.0 + km)
        kl = k[large]
        out[large] = 1.0 - 1.0/(2.0*kl) - 1.0/(8.0*kl*kl)
        return out

def _format_treatment_name(name: str) -> str:
    s = str(name).strip().lower()
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if "control" in parts and len(parts) > 1:
        parts = [p for p in parts if p.lower() != "control"]
    if not parts:
        parts = ["control"]
    parts = sorted(parts)
    parts = [p[:1].upper() + p[1:] for p in parts]
    return ", ".join(parts)

def _treat_levels_and_labels(df_fit, treat_col="treatment", treat_name_col="treatmentName"):
    if isinstance(df_fit[treat_col].dtype, pd.CategoricalDtype):
        treat_levels = list(df_fit[treat_col].cat.categories)
    else:
        treat_levels = sorted(df_fit[treat_col].dropna().unique().tolist())

    if treat_name_col in df_fit.columns:
        name_map = df_fit.groupby(treat_col, observed=False)[treat_name_col].first().to_dict()
        labels = [_format_treatment_name(name_map.get(t, t)) for t in treat_levels]
    else:
        labels = [_format_treatment_name(t) for t in treat_levels]

    return treat_levels, labels


def summarize_taglevel_from_model_terms(
    idata_mix_full,
    df_fit,
    *,
    treat_col="treatment",
    treat_name_col="treatmentName",
    individual_col="tag",          # your "individual" is tag
    hdi_prob=0.95,
    max_draws=2000,
    kappa_uniform=KAPPA_UNIFORM,
    seed=123,
):
    """
    Build tag-level params from pop + treatment + tag offsets:
      mu_it    = wrap(mu0 + mu_treat[t] + mu_tag[i])
      kappa_it = softplus(eta_kappa0 + kappa_treat[t] + eta_kappa_tag[i] * sigma_kappa_tag)
      p_it     = sigmoid(logit_p0 + beta_treat[t] + logit_p_tag[i] * sigma_p_tag)

    Then derived:
      dens0_it, align0_it
    """
    post = idata_mix_full.posterior

    # required vars in *your* model
    required = ["mu0", "mu_tag", "eta_kappa0", "eta_kappa_tag", "sigma_kappa_tag",
                "logit_p0", "logit_p_tag", "sigma_p_tag"]
    missing = [v for v in required if v not in post.data_vars]
    if missing:
        raise KeyError(f"Missing required posterior vars: {missing}. Available: {list(post.data_vars)[:120]} ...")

    # optional treatment contrasts
    has_mt   = "mu_treat" in post.data_vars
    has_kt   = "kappa_treat" in post.data_vars
    has_beta = "beta_treat" in post.data_vars

    # treatment levels
    treat_levels, treat_labels = _treat_levels_and_labels(df_fit, treat_col=treat_col, treat_name_col=treat_name_col)
    n_treat = len(treat_levels)

    ref = treat_levels[0]
    nonref = treat_levels[1:]
    treat_to_j = {t: nonref.index(t) for t in nonref}  # nonref -> 0..K-1

    # map each tag -> its (single) treatment (mode if messy)
    tmp = df_fit[[individual_col, treat_col]].copy()
    tmp[treat_col] = pd.Categorical(tmp[treat_col], categories=treat_levels, ordered=True)
    tag_to_treat = (tmp.groupby(individual_col, observed=False)[treat_col]
                      .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0])
                      .to_dict())

    tag_codes = np.array(sorted(tag_to_treat.keys(), key=lambda x: str(x)), dtype=object)
    n_tag = len(tag_codes)

    # IMPORTANT: tag index order must match the model's tag_idx ordering.
    # If you built tag_idx with np.unique(df_track["tag"], return_inverse=True), then
    # tag_codes_model is np.unique(df_track["tag"])[0] in sorted order.
    # We'll align to that by using np.unique on df_fit (same operation).
    tag_codes_model = np.unique(df_fit[individual_col].to_numpy())
    if len(tag_codes_model) != n_tag:
        raise ValueError("Tag code mismatch between df grouping and np.unique. Check df contents.")

    # posterior arrays
    rng = np.random.default_rng(seed)

    mu0 = circular_modulo(_stack_samples_first(post["mu0"]).reshape(-1))                 # (S,)
    mu_tag = _stack_samples_first(post["mu_tag"])                                        # (S, n_tag)

    ek0 = _stack_samples_first(post["eta_kappa0"]).reshape(-1)                           # (S,)
    ek_tag = _stack_samples_first(post["eta_kappa_tag"])                                 # (S, n_tag)
    sigk = _stack_samples_first(post["sigma_kappa_tag"]).reshape(-1)                     # (S,)

    lp0 = _stack_samples_first(post["logit_p0"]).reshape(-1)                             # (S,)
    lp_tag = _stack_samples_first(post["logit_p_tag"])                                   # (S, n_tag)
    sigp = _stack_samples_first(post["sigma_p_tag"]).reshape(-1)                         # (S,)

    # treatment arrays -> full (S, n_treat) with 0 at ref
    def _full_contr(arr_name):
        if arr_name not in post.data_vars:
            return None
        arr = _stack_samples_first(post[arr_name])
        if arr.ndim == 1:
            arr = arr[:, None]
        if arr.shape[1] != (n_treat - 1):
            raise ValueError(f"{arr_name} second dim {arr.shape[1]} != n_treat-1={n_treat-1}")
        z = np.zeros((arr.shape[0], 1), float)
        return np.concatenate([z, arr], axis=1)  # (S, n_treat)

    mu_treat_full = _full_contr("mu_treat")       # (S, n_treat) or None
    ktreat_full   = _full_contr("kappa_treat")    # (S, n_treat) or None
    beta_full     = _full_contr("beta_treat")     # (S, n_treat) or None

    # basic shape checks
    S = mu0.size
    if mu_tag.shape[1] != len(tag_codes_model):
        raise ValueError(f"mu_tag has n={mu_tag.shape[1]} but df unique tags has {len(tag_codes_model)}.")
    if ek_tag.shape[1] != len(tag_codes_model) or lp_tag.shape[1] != len(tag_codes_model):
        raise ValueError("Tag RE shapes don't match df unique tags; check tag indexing order.")

    # subsample draws once
    if (max_draws is not None) and (S > max_draws):
        idx = rng.choice(S, size=max_draws, replace=False)
        mu0 = mu0[idx]
        mu_tag = mu_tag[idx, :]
        ek0 = ek0[idx]
        ek_tag = ek_tag[idx, :]
        sigk = sigk[idx]
        lp0 = lp0[idx]
        lp_tag = lp_tag[idx, :]
        sigp = sigp[idx]
        if mu_treat_full is not None: mu_treat_full = mu_treat_full[idx, :]
        if ktreat_full   is not None: ktreat_full   = ktreat_full[idx, :]
        if beta_full     is not None: beta_full     = beta_full[idx, :]
        S = max_draws

    # constants for uniform component
    f0_unif = float(vonmises_pdf_scalar(0.0, 0.0, kappa_uniform))
    A_unif  = float(bessel_ratio(kappa_uniform))

    # tag index lookup in model order
    tag_to_index = {tag_codes_model[i]: i for i in range(len(tag_codes_model))}

    rows = []
    for tag_val in tag_codes_model:
        i_tag = tag_to_index[tag_val]
        t = tag_to_treat.get(tag_val, None)
        if t is None:
            continue
        t = str(t)
        if t not in treat_levels:
            # if df has different string repr, fall back to categorical compare
            t = tag_to_treat[tag_val]

        t_idx = treat_levels.index(t)

        # mu(tag,treat)
        mu_s = mu0 + mu_tag[:, i_tag]
        if mu_treat_full is not None:
            mu_s = mu_s + mu_treat_full[:, t_idx]
        mu_s = circular_modulo(mu_s)

        # kappa(tag,treat)
        eta_k_s = ek0 + (ek_tag[:, i_tag] * sigk)
        if ktreat_full is not None:
            eta_k_s = eta_k_s + ktreat_full[:, t_idx]
        kappa_s = softplus(eta_k_s)

        # p(tag,treat)
        logit_s = lp0 + (lp_tag[:, i_tag] * sigp)
        if beta_full is not None:
            logit_s = logit_s + beta_full[:, t_idx]
        p_s = sigmoid(logit_s)

        # derived
        f0_goal = vonmises_pdf_scalar(0.0, mu_s, kappa_s)
        dens0_s = p_s * f0_goal + (1.0 - p_s) * f0_unif

        A_goal = bessel_ratio(kappa_s)
        align0_s = p_s * (A_goal * np.cos(mu_s)) + (1.0 - p_s) * A_unif

        # summarize helper
        def _summ(x):
            lo, hi = hdi_1d(x, prob=hdi_prob)
            return float(np.mean(x)), lo, hi

        p_m, p_lo, p_hi = _summ(p_s)
        k_m, k_lo, k_hi = _summ(kappa_s)
        d_m, d_lo, d_hi = _summ(dens0_s)
        a_m, a_lo, a_hi = _summ(align0_s)

        rows.append({
            "treatment": t,
            "treatment_label": treat_labels[t_idx],
            "individual": tag_val,

            "p_mean": p_m, "p_lo": p_lo, "p_hi": p_hi,
            "kappa_mean": k_m, "kappa_lo": k_lo, "kappa_hi": k_hi,
            "dens0_mean": d_m, "dens0_lo": d_lo, "dens0_hi": d_hi,
            "align0_mean": a_m, "align0_lo": a_lo, "align0_hi": a_hi,
        })

    out = pd.DataFrame(rows)
    out["treatment"] = pd.Categorical(out["treatment"], categories=treat_levels, ordered=True)
    return out, treat_levels


def pop_summary_by_treatment_from_model_terms(
    idata_mix_full,
    df_fit,
    *,
    treat_col="treatment",
    treat_name_col="treatmentName",
    hdi_prob=0.95,
    max_draws=2000,
    kappa_uniform=KAPPA_UNIFORM,
    seed=123,
):
    post = idata_mix_full.posterior

    for v in ["mu0", "logit_p0", "eta_kappa0"]:
        if v not in post.data_vars:
            raise KeyError(f"Missing '{v}' in posterior. Available: {list(post.data_vars)[:120]} ...")

    treat_levels, labels = _treat_levels_and_labels(df_fit, treat_col=treat_col, treat_name_col=treat_name_col)
    n_treat = len(treat_levels)

    rng = np.random.default_rng(seed)

    mu0 = circular_modulo(_stack_samples_first(post["mu0"]).reshape(-1))    # (S,)
    lp0 = _stack_samples_first(post["logit_p0"]).reshape(-1)                # (S,)
    ek0 = _stack_samples_first(post["eta_kappa0"]).reshape(-1)              # (S,)

    def _full_contr(arr_name):
        if arr_name not in post.data_vars:
            return None
        arr = _stack_samples_first(post[arr_name])
        if arr.ndim == 1:
            arr = arr[:, None]
        if arr.shape[1] != (n_treat - 1):
            raise ValueError(f"{arr_name} dim {arr.shape[1]} != n_treat-1={n_treat-1}")
        return np.concatenate([np.zeros((arr.shape[0], 1)), arr], axis=1)   # (S, n_treat)

    mu_treat_full = _full_contr("mu_treat")
    ktreat_full   = _full_contr("kappa_treat")
    beta_full     = _full_contr("beta_treat")

    S = lp0.size
    if (max_draws is not None) and (S > max_draws):
        idx = rng.choice(S, size=max_draws, replace=False)
        mu0 = mu0[idx]; lp0 = lp0[idx]; ek0 = ek0[idx]
        if mu_treat_full is not None: mu_treat_full = mu_treat_full[idx, :]
        if ktreat_full   is not None: ktreat_full   = ktreat_full[idx, :]
        if beta_full     is not None: beta_full     = beta_full[idx, :]
        S = max_draws

    f0_unif = float(vonmises_pdf_scalar(0.0, 0.0, kappa_uniform))
    A_unif  = float(bessel_ratio(kappa_uniform))

    def _summ(x):
        lo, hi = hdi_1d(x, prob=hdi_prob)
        return float(np.mean(x)), lo, hi

    rows = []
    for t_idx, t in enumerate(treat_levels):
        mu_s = mu0.copy()
        if mu_treat_full is not None:
            mu_s = circular_modulo(mu_s + mu_treat_full[:, t_idx])
        else:
            mu_s = circular_modulo(mu_s)

        ek_s = ek0.copy()
        if ktreat_full is not None:
            ek_s = ek_s + ktreat_full[:, t_idx]
        kappa_s = softplus(ek_s)

        lp_s = lp0.copy()
        if beta_full is not None:
            lp_s = lp_s + beta_full[:, t_idx]
        p_s = sigmoid(lp_s)

        f0_goal = vonmises_pdf_scalar(0.0, mu_s, kappa_s)
        dens0_s = p_s * f0_goal + (1.0 - p_s) * f0_unif

        A_goal = bessel_ratio(kappa_s)
        align0_s = p_s * (A_goal * np.cos(mu_s)) + (1.0 - p_s) * A_unif

        p_m, p_lo, p_hi = _summ(p_s)
        k_m, k_lo, k_hi = _summ(kappa_s)
        d_m, d_lo, d_hi = _summ(dens0_s)
        a_m, a_lo, a_hi = _summ(align0_s)

        rows.append({
            "treatment": t,
            "treatment_label": labels[t_idx],

            "p_mean": p_m, "p_lo": p_lo, "p_hi": p_hi,
            "kappa_mean": k_m, "kappa_lo": k_lo, "kappa_hi": k_hi,
            "dens0_mean": d_m, "dens0_lo": d_lo, "dens0_hi": d_hi,
            "align0_mean": a_m, "align0_lo": a_lo, "align0_hi": a_hi,
        })

    out = pd.DataFrame(rows)
    out["treatment"] = pd.Categorical(out["treatment"], categories=treat_levels, ordered=True)
    return out


import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns  # <-- added for colorblind palette

mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

def plot_individual_subplots_by_treatment(
    res_ind,
    treatment_order,
    *,
    value_col,
    lo_col,
    hi_col,
    xlabel,
    pop_df=None,               # <- output from pop_summary_by_treatment_from_model_terms()
    pop_mean_col=None,
    pop_lo_col=None,
    pop_hi_col=None,
    savepath=None,
    sort_within="desc",
    lw=1.2,
    ms=5,
    y_step=1.25,
    y_margins=0.08,
    xlim=None,
    dpi=300,
    capsize=3,                 # caps in points
    capthick=1,
    band_color="0.85",
    band_alpha=0.6,
    vline_lw=1.0,
):
    res = res_ind.copy()
    res["treatment"] = pd.Categorical(res["treatment"], categories=list(treatment_order), ordered=True)

    # ---- swapped from tab10 to seaborn "colorblind" palette (same mapping logic) ----
    palette = sns.color_palette("colorblind", 10)
    treat_to_color = {t: palette[i % len(palette)] for i, t in enumerate(treatment_order)}
    k = len(treatment_order)

    # pop lookup: {treatment: (lo, mean, hi)}
    pop_lookup = {}
    if pop_df is not None and pop_mean_col and pop_lo_col and pop_hi_col:
        for _, r in pop_df.iterrows():
            pop_lookup[r["treatment"]] = (float(r[pop_lo_col]), float(r[pop_mean_col]), float(r[pop_hi_col]))

    # global xlim unless provided (include pop bands)
    if xlim is None:
        gmin, gmax = np.inf, -np.inf
        for t in treatment_order:
            sub = res[res["treatment"] == t]
            if not sub.empty:
                gmin = min(gmin, float(np.nanmin(sub[lo_col].to_numpy(float))))
                gmax = max(gmax, float(np.nanmax(sub[hi_col].to_numpy(float))))
            if t in pop_lookup:
                plo, pm, phi = pop_lookup[t]
                gmin = min(gmin, plo, pm, phi)
                gmax = max(gmax, plo, pm, phi)
        if not np.isfinite(gmin):
            gmin, gmax = -1.0, 1.0
        pad = 0.06 * (gmax - gmin if gmax > gmin else 1.0)
        xlim = (gmin - pad, gmax + pad)

    max_n = int(res.groupby("treatment").size().max()) if len(res) else 1
    fig_w = max(5.5, 1.6 * k)
    fig_h = max(2.0, 0.20 * y_step * max_n + 0.8)

    fig, axes = plt.subplots(1, k, figsize=(fig_w, fig_h), dpi=dpi, sharex=True)
    if k == 1:
        axes = [axes]

    for i, t in enumerate(treatment_order):
        ax = axes[i]
        sub = res[res["treatment"] == t].copy()

        if sort_within == "desc":
            sub = sub.sort_values(value_col, ascending=False)
        elif sort_within == "asc":
            sub = sub.sort_values(value_col, ascending=True)

        sub = sub.reset_index(drop=True)
        title = sub["treatment_label"].iloc[0] if len(sub) else str(t)
        ax.set_title(title, fontsize=9)
        ax.set_xlim(*xlim)

        col = treat_to_color.get(t, "black")

        # ---- POP band + dashed mean (behind everything) ----
        if t in pop_lookup:
            plo, pm, phi = pop_lookup[t]
            ax.axvspan(plo, phi, color=band_color, alpha=band_alpha, zorder=0)
            ax.axvline(pm, linestyle="--", linewidth=vline_lw, color="black", zorder=1)

        if len(sub):
            y = np.arange(len(sub), dtype=float) * float(y_step)

            for yy, r in zip(y, sub.to_dict("records")):
                m  = float(r[value_col])
                lo = float(r[lo_col])
                hi = float(r[hi_col])

                # fixed-size caps in points
                xerr = np.array([[m - lo], [hi - m]])
                ax.errorbar(
                    m, yy, xerr=xerr,
                    fmt="none",
                    ecolor=col,
                    elinewidth=lw,
                    capsize=capsize,
                    capthick=capthick,
                    zorder=2
                )

                # marker: NO edge
                ax.plot(
                    m, yy, "o",
                    color=col,
                    markersize=ms,
                    markeredgecolor="none",
                    markeredgewidth=0.0,
                    zorder=3
                )

            ax.set_yticks(y)
            ax.set_yticklabels(sub["individual"].astype(str).tolist(), fontsize=7)
            ax.invert_yaxis()
        else:
            ax.set_yticks([])

        ax.margins(y=float(y_margins))
        ax.tick_params(axis="x", labelsize=8)
        ax.grid(False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        if i != 0:
            ax.tick_params(axis="y", length=0)
            ax.set_ylabel("")

    fig.supxlabel(xlabel, fontsize=9)
    plt.tight_layout()
    if savepath:
        fig.savefig(savepath, dpi=dpi, bbox_inches="tight", pad_inches=0.01)
    plt.show()

In [ ]:
# ============================================================
# TAG-LEVEL plots by treatment (individual summaries)
# Updates:
#   - seaborn colorblind palette
#   - solid black line for population mean (with gray HDI band)
#   - optional dashed black reference line (e.g., 0.5 or uniform)
#   - save PDF + SVG (same basename)
#   - style matches other plots (Arial + svg.fonttype none + white bg)
#   - FIXED: endcaps are consistent (in points, not data units)
#   - FIXED: error bars / points are not visually clipped at x-lims
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

DPI = 300
mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
})

def plot_individual_subplots_by_treatment(
    res_ind,
    treatment_order,
    *,
    value_col,
    lo_col,
    hi_col,
    xlabel,
    pop_df=None,
    pop_mean_col=None,
    pop_lo_col=None,
    pop_hi_col=None,
    xlim=None,
    savepath=None,

    # optional dashed reference line (uniform / 0.5)
    ref_vline_x=None,
    ref_vline_color="black",
    ref_vline_ls="--",
    ref_vline_lw=1.0,

    # population mean line (solid black)
    pop_mean_line_color="black",
    pop_mean_line_ls="-",
    pop_mean_line_lw=1.0,

    # style
    palette_name="colorblind",
    point_ms=3.0,
    hdi_lw=0.9,
    capsize_pts=2.5,   # cap size in points (consistent across panels)
    title_fs=9,
    ytick_fs=7,
    xtick_fs=8,

    save_svg_also=True,
):
    """
    One row of subplots, one per treatment.
    Within each subplot: individuals on y-axis, horizontal mean ± HDI.
    Optional population HDI band + solid population mean line.
    Optional dashed reference line (e.g., 0.5 for mixture weight, 1/(2π) for density).

    Notes:
    - Error-bar caps are drawn in points (constant visual size).
    - Individual intervals/points are drawn with clip_on=False so caps/points
      are not visually cut off at fixed x-limits.
    """
    res = res_ind.copy()

    # robust treatment categorical ordering
    res["treatment"] = pd.Categorical(
        res["treatment"],
        categories=list(treatment_order),
        ordered=True
    )

    # seaborn colorblind palette (stable treatment order)
    colors = sns.color_palette(palette_name, n_colors=len(treatment_order))
    treat_to_color = {t: colors[i] for i, t in enumerate(treatment_order)}
    k = len(treatment_order)

    # infer label columns
    treat_label_col = "label" if "label" in res.columns else ("treatment_label" if "treatment_label" in res.columns else None)
    indiv_col = "individual" if "individual" in res.columns else ("tag" if "tag" in res.columns else None)

    # global x-limits (include all individual intervals + population band + ref line)
    gmin, gmax = np.inf, -np.inf
    for t in treatment_order:
        sub = res.loc[res["treatment"] == t].copy()
        if len(sub):
            sub = sub.sort_values(value_col, ascending=False)
            x = pd.to_numeric(sub[value_col], errors="coerce").to_numpy(float)
            lo = pd.to_numeric(sub[lo_col], errors="coerce").to_numpy(float)
            hi = pd.to_numeric(sub[hi_col], errors="coerce").to_numpy(float)
            finite = np.isfinite(x) & np.isfinite(lo) & np.isfinite(hi)
            if np.any(finite):
                gmin = min(gmin, float(np.min(lo[finite])))
                gmax = max(gmax, float(np.max(hi[finite])))

        if pop_df is not None and pop_mean_col and pop_lo_col and pop_hi_col:
            prow = pop_df.loc[pop_df["treatment"] == t]
            if len(prow):
                plo = float(prow.iloc[0][pop_lo_col])
                phi = float(prow.iloc[0][pop_hi_col])
                gmin = min(gmin, plo)
                gmax = max(gmax, phi)

    if ref_vline_x is not None:
        gmin = min(gmin, float(ref_vline_x))
        gmax = max(gmax, float(ref_vline_x))

    if not np.isfinite(gmin) or not np.isfinite(gmax):
        gmin, gmax = -1.0, 1.0

    if xlim is None:
        pad = 0.06 * (gmax - gmin if gmax > gmin else 1.0)
        xlim_use = (gmin - pad, gmax + pad)
    else:
        xlim_use = xlim

    # compact sizing
    max_n = 1
    if len(res):
        counts = res.groupby("treatment", observed=False).size()
        if len(counts):
            max_n = int(counts.max())
    fig_w = max(5.5, 1.6 * k)
    fig_h = max(2.0, 0.18 * max_n + 0.8)

    fig, axes = plt.subplots(1, k, figsize=(fig_w, fig_h), dpi=DPI, sharex=True)
    if k == 1:
        axes = [axes]

    for i, t in enumerate(treatment_order):
        ax = axes[i]
        sub = res.loc[res["treatment"] == t].copy().sort_values(value_col, ascending=False).reset_index(drop=True)

        # title label
        if len(sub) and (treat_label_col is not None):
            title = str(sub[treat_label_col].iloc[0])
        else:
            title = str(t)
        ax.set_title(title, fontsize=title_fs)

        # dashed reference line (uniform / 0.5)
        if ref_vline_x is not None:
            ax.axvline(
                float(ref_vline_x),
                color=ref_vline_color,
                linestyle=ref_vline_ls,
                linewidth=ref_vline_lw,
                zorder=0,
            )

        # population band + solid mean line
        if pop_df is not None and pop_mean_col and pop_lo_col and pop_hi_col:
            prow = pop_df.loc[pop_df["treatment"] == t]
            if len(prow):
                pm  = float(prow.iloc[0][pop_mean_col])
                plo = float(prow.iloc[0][pop_lo_col])
                phi = float(prow.iloc[0][pop_hi_col])

                ax.axvspan(plo, phi, color="0.85", alpha=1.0, zorder=0.5)
                ax.axvline(
                    pm,
                    color=pop_mean_line_color,
                    linestyle=pop_mean_line_ls,
                    linewidth=pop_mean_line_lw,
                    zorder=1,
                )

        # individual intervals + points
        if len(sub):
            y = np.arange(len(sub), dtype=float)
            x = pd.to_numeric(sub[value_col], errors="coerce").to_numpy(float)
            lo = pd.to_numeric(sub[lo_col], errors="coerce").to_numpy(float)
            hi = pd.to_numeric(sub[hi_col], errors="coerce").to_numpy(float)
            c = treat_to_color.get(t, "black")

            for yy, xx, xlo, xhi in zip(y, x, lo, hi):
                x_ok  = np.isfinite(xx)
                lo_ok = np.isfinite(xlo)
                hi_ok = np.isfinite(xhi)

                if x_ok and lo_ok and hi_ok:
                    # asymmetric x-errors from mean to HDI bounds
                    xerr_left  = max(0.0, xx - xlo)
                    xerr_right = max(0.0, xhi - xx)

                    eb = ax.errorbar(
                        xx, yy,
                        xerr=np.array([[xerr_left], [xerr_right]]),
                        fmt="o",
                        color=c,
                        ecolor=c,
                        markersize=point_ms,
                        markeredgecolor=c,
                        markeredgewidth=0.4,
                        elinewidth=hdi_lw,
                        capsize=capsize_pts,   # points => consistent cap size
                        capthick=hdi_lw,
                        zorder=3,
                    )

                    # Prevent visible clipping at fixed x-lims (keep limits unchanged)
                    try:
                        # marker line
                        if eb.lines and eb.lines[0] is not None:
                            eb.lines[0].set_clip_on(False)
                        # cap lines
                        if len(eb.lines) > 1:
                            for capline in eb.lines[1]:
                                capline.set_clip_on(False)
                        # errorbar line collections
                        if len(eb.lines) > 2:
                            for barcol in eb.lines[2]:
                                barcol.set_clip_on(False)
                    except Exception:
                        pass

                elif x_ok:
                    pt = ax.plot(
                        xx, yy, "o",
                        markersize=point_ms,
                        color=c, markeredgecolor=c,
                        markeredgewidth=0.4, zorder=3
                    )[0]
                    pt.set_clip_on(False)

            ax.set_yticks(y)
            if indiv_col is not None:
                ax.set_yticklabels(sub[indiv_col].astype(str).tolist(), fontsize=ytick_fs)
            else:
                ax.set_yticklabels([str(j) for j in range(len(sub))], fontsize=ytick_fs)
            ax.invert_yaxis()
        else:
            ax.set_yticks([])

        ax.set_xlim(*xlim_use)  # keep exact user-requested limits
        ax.tick_params(axis="x", labelsize=xtick_fs)
        ax.grid(False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        if i != 0:
            ax.tick_params(axis="y", length=0)
            ax.set_ylabel("")

    fig.supxlabel(xlabel, fontsize=9)
    plt.tight_layout()

    if savepath:
        fig.savefig(savepath, dpi=DPI, bbox_inches="tight", pad_inches=0.01)
        if save_svg_also:
            root, _ = os.path.splitext(savepath)
            fig.savefig(root + ".svg", bbox_inches="tight", pad_inches=0.01)

    plt.show()
    return fig, axes


# =========================
# RUN
# =========================

# tag-level individual summaries (pop + treatment + tag offsets)
res_ind, treatment_order = summarize_taglevel_from_model_terms(
    idata_mix_full=idata_mix_full,
    df_fit=df_track,
    treat_col="treatment",
    treat_name_col="treatmentName",
    individual_col="tag",
    hdi_prob=0.95,
    max_draws=2000,
    kappa_uniform=KAPPA_UNIFORM,
)

# population-only summaries (mean + HDI) for gray band + solid population mean line
pop_df = pop_summary_by_treatment_from_model_terms(
    idata_mix_full=idata_mix_full,
    df_fit=df_track,
    treat_col="treatment",
    treat_name_col="treatmentName",
    hdi_prob=0.95,
    max_draws=2000,
    kappa_uniform=KAPPA_UNIFORM,
)

# reference values
uniform_density_at_0 = 1.0 / (2.0 * np.pi)  # dashed line for predictive density
mixture_weight_ref   = 0.5                  # dashed line for mixture weight

# (1) p  -> dashed 0.5 line + xlim [0,1]
plot_individual_subplots_by_treatment(
    res_ind, treatment_order,
    value_col="p_mean", lo_col="p_lo", hi_col="p_hi",
    xlabel="Mixture weight",
    pop_df=pop_df, pop_mean_col="p_mean", pop_lo_col="p_lo", pop_hi_col="p_hi",
    xlim=(0, 1),
    ref_vline_x=mixture_weight_ref,
    savepath="../figures/bats/fullmodel_taglevel_mixture_weight_by_treatment_mean_hdi.pdf",
)

# (2) kappa
plot_individual_subplots_by_treatment(
    res_ind, treatment_order,
    value_col="kappa_mean", lo_col="kappa_lo", hi_col="kappa_hi",
    xlabel="Concentration " + rf"($\kappa$)",
    pop_df=pop_df, pop_mean_col="kappa_mean", pop_lo_col="kappa_lo", pop_hi_col="kappa_hi",
    savepath="../figures/bats/fullmodel_taglevel_kappa_by_treatment_mean_hdi.pdf",
)

# (3) dens0 -> dashed uniform line
plot_individual_subplots_by_treatment(
    res_ind, treatment_order,
    value_col="dens0_mean", lo_col="dens0_lo", hi_col="dens0_hi",
    xlabel="Predictive density",
    pop_df=pop_df, pop_mean_col="dens0_mean", pop_lo_col="dens0_lo", pop_hi_col="dens0_hi",
    ref_vline_x=uniform_density_at_0,
    savepath="../figures/bats/fullmodel_taglevel_predictive_density_at_0_by_treatment_mean_hdi.pdf",
)

# (4) align0
plot_individual_subplots_by_treatment(
    res_ind, treatment_order,
    value_col="align0_mean", lo_col="align0_lo", hi_col="align0_hi",
    xlabel="Alignment",
    pop_df=pop_df, pop_mean_col="align0_mean", pop_lo_col="align0_lo", pop_hi_col="align0_hi",
    xlim=(0, 1),
    savepath="../figures/bats/fullmodel_taglevel_alignment_to_zero_by_treatment_mean_hdi.pdf",
)

# Optional (if available): mass within ±45° -> dashed uniform line at 0.25
if all(col in res_ind.columns for col in ["mass45_mean", "mass45_lo", "mass45_hi"]) and \
   all(col in pop_df.columns for col in ["mass45_mean", "mass45_lo", "mass45_hi"]):
    plot_individual_subplots_by_treatment(
        res_ind, treatment_order,
        value_col="mass45_mean", lo_col="mass45_lo", hi_col="mass45_hi",
        xlabel="Predictive mass (±45°)",
        pop_df=pop_df, pop_mean_col="mass45_mean", pop_lo_col="mass45_lo", pop_hi_col="mass45_hi",
        xlim=(0, 1),
        ref_vline_x=0.25,
        savepath="../figures/bats/fullmodel_taglevel_predictive_mass_pm45deg_by_treatment_mean_hdi.pdf",
    )

In [ ]:
# ============================================================
# Bat vMU: population-level PPC by treatment (hierarchical bootstrap)
# Observed: hierarchical bootstrap KDE + histogram (by tag within treatment)
# PPC: simulate from posterior (INCLUDING tag REs) -> hierarchical bootstrap KDE + histogram
#
# Requires:
#   - df_track with columns: HeadingRadians, tag, treatment
#   - idata_mix_full from fit_vmum_model_full(...)
#
# Notes:
#   * All densities are UN-SCALED (proper circular pdfs, units: 1/radian).
#   * This PPC compares:
#       - Observed bootstrap distribution of summaries
#       - Posterior predictive distribution of the SAME summaries
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns  # <-- added for colorblind palette

from contextlib import contextmanager
from dataclasses import dataclass
from typing import Dict, List, Tuple

from joblib import Parallel, delayed
from tqdm.auto import tqdm

# ----------------------------
# Matplotlib style (match your small-figure style)
# ----------------------------
DPI = 300
AX_LW     = 0.8
TICK_LW   = 0.8
TICK_LEN  = 2.5
LINE_LW   = 0.8
FONT_SIZE = 6

mpl.rcParams.update({
    "font.size": FONT_SIZE,
    "axes.labelsize": FONT_SIZE,
    "axes.titlesize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,
    "font.family": "Arial",
    "svg.fonttype": "none",
    "axes.linewidth": AX_LW,
    "lines.linewidth": LINE_LW,
    "xtick.major.width": TICK_LW,
    "ytick.major.width": TICK_LW,
    "xtick.major.size":  TICK_LEN,
    "ytick.major.size":  TICK_LEN,
    "savefig.dpi": DPI,
    "figure.dpi": DPI,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# ----------------------------
# Knobs
# ----------------------------
HDI_PROB           = 0.95
KDE_KAPPA          = 30.0       # fixed kernel concentration for the bootstrap KDE
N_KDE_POINTS       = 360        # evaluation grid for KDE
N_HIST_BINS        = 16         # like your locust PPCs
N_BOOT_OBS         = 1000       # hierarchical bootstrap replicates for observed
N_PPC_DRAWS        = 4000       # posterior draws to use for PPC (subsample if needed)
PPC_BOOT_PER_DRAW  = 1          # hierarchical bootstrap reps per posterior draw (1 works well)
KAPPA_UNIFORM      = 1e-6       # the uniform-ish VM component in your model

EDGE_RADIUS        = 0.55       # radial offset for polar plot
BAND_ALPHA         = 0.25

N_JOBS             = max(1, (os.cpu_count() or 2) - 1)
RNG_SEED           = 123


# ============================================================
# Helpers: math
# ============================================================

def wrap_angle(x: np.ndarray) -> np.ndarray:
    """Wrap to (-pi, pi]."""
    return (x + np.pi) % (2 * np.pi) - np.pi

def von_mises_kde(data: np.ndarray, points: np.ndarray, kappa: float) -> np.ndarray:
    """
    Von Mises KDE (density per radian). Kernel is VM with concentration kappa.
    """
    data = wrap_angle(np.asarray(data, float))
    points = np.asarray(points, float)
    c = 1.0 / (2.0 * np.pi * np.i0(kappa))
    cos_diffs = np.cos(points[:, None] - data[None, :])
    return c * np.exp(kappa * cos_diffs).mean(axis=1)

def hist_density(data: np.ndarray, bin_edges: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Histogram density (per radian) + bin centers.
    """
    h, _ = np.histogram(data, bins=bin_edges, density=True)
    centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    return centers, h

def hdi_1d(x: np.ndarray, prob: float = 0.95) -> Tuple[float, float]:
    """
    1D HDI for a unimodal-ish sample. If you prefer quantiles, swap out.
    """
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    n = x.size
    if n == 0:
        return np.nan, np.nan
    x = np.sort(x)
    m = int(np.floor(prob * n))
    if m < 1:
        return np.nan, np.nan
    widths = x[m:] - x[: n - m]
    j = int(np.argmin(widths))
    return float(x[j]), float(x[j + m])

def hdi_along_samples(arr: np.ndarray, prob: float = 0.95) -> Tuple[np.ndarray, np.ndarray]:
    """
    HDI along axis=0 where arr shape = (S, T).
    """
    arr = np.asarray(arr, float)
    S, T = arr.shape
    lo = np.empty(T, float)
    hi = np.empty(T, float)
    for t in range(T):
        lo[t], hi[t] = hdi_1d(arr[:, t], prob=prob)
    return lo, hi


# ============================================================
# Helpers: tqdm + joblib
# ============================================================

@contextmanager
def tqdm_joblib(tqdm_object):
    """
    Context manager to patch joblib to report into tqdm progress bar.
    """
    import joblib
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old
        tqdm_object.close()


# ============================================================
# Grouping utilities
# ============================================================

@dataclass
class GroupData:
    treat_level: str
    tag_values: np.ndarray
    y_by_tag: List[np.ndarray]

def build_groups(
    df: pd.DataFrame,
    y_col: str = "HeadingRadians",
    tag_col: str = "tag",
    treat_col: str = "treatment",
) -> Tuple[List[str], Dict[str, GroupData]]:
    """
    Build per-treatment lists of per-tag y arrays for fast hierarchical bootstrap.
    """
    df = df.copy()

    # ensure treatment categorical (stable ordering)
    treat_cat = pd.Categorical(df[treat_col])
    df[treat_col] = treat_cat
    treat_levels = list(treat_cat.categories)

    groups: Dict[str, GroupData] = {}
    for t in treat_levels:
        sub = df[df[treat_col] == t]
        tags = sub[tag_col].to_numpy()
        y = wrap_angle(sub[y_col].to_numpy())

        uniq_tags = np.unique(tags)
        y_by_tag = []
        for tg in uniq_tags:
            y_by_tag.append(y[tags == tg])
        groups[t] = GroupData(treat_level=str(t), tag_values=uniq_tags, y_by_tag=y_by_tag)

    return treat_levels, groups

def hier_boot_resample_y(y_by_tag: List[np.ndarray], rng: np.random.Generator) -> np.ndarray:
    """
    Hierarchical bootstrap:
      - resample tags with replacement
      - within each selected tag, resample observations with replacement (same count as original tag)
    Returns a pooled y* array.
    """
    n_tags = len(y_by_tag)
    tag_idx_star = rng.integers(0, n_tags, size=n_tags)
    out = []
    for j in tag_idx_star:
        yj = y_by_tag[j]
        if yj.size == 0:
            continue
        out.append(rng.choice(yj, size=yj.size, replace=True))
    if not out:
        return np.array([], float)
    return wrap_angle(np.concatenate(out))


# ============================================================
# Observed bootstrap summaries
# ============================================================

def one_obs_boot_rep(
    y_by_tag: List[np.ndarray],
    angles: np.ndarray,
    bin_edges: np.ndarray,
    kde_kappa: float,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    y_star = hier_boot_resample_y(y_by_tag, rng)
    kde = von_mises_kde(y_star, angles, kappa=kde_kappa)
    _, h = hist_density(y_star, bin_edges)
    return kde, h

def bootstrap_observed(
    groups: Dict[str, GroupData],
    angles: np.ndarray,
    bin_edges: np.ndarray,
    n_boot: int,
    kde_kappa: float,
    n_jobs: int,
    seed: int,
) -> Dict[str, Dict[str, np.ndarray]]:
    """
    Returns dict:
      obs_boot[treat]["kde"]  shape (B, P)
      obs_boot[treat]["hist"] shape (B, BINS)
    """
    obs_boot: Dict[str, Dict[str, np.ndarray]] = {}
    base_rng = np.random.default_rng(seed)

    for t, gd in groups.items():
        seeds = base_rng.integers(1, 2**31 - 1, size=n_boot, dtype=np.int64)

        with tqdm_joblib(tqdm(total=n_boot, desc=f"Observed bootstrap ({t})", leave=False)):
            reps = Parallel(n_jobs=n_jobs)(
                delayed(one_obs_boot_rep)(gd.y_by_tag, angles, bin_edges, kde_kappa, int(s))
                for s in seeds
            )

        kde_mat = np.stack([r[0] for r in reps], axis=0)
        hist_mat = np.stack([r[1] for r in reps], axis=0)
        obs_boot[t] = {"kde": kde_mat, "hist": hist_mat}

    return obs_boot


# ============================================================
# Posterior extraction + simulation from fitted model (includes tag REs)
# ============================================================

def _stack_posterior(da) -> np.ndarray:
    """
    Stack chain/draw -> sample and return numpy array.
    """
    da2 = da.stack(sample=("chain", "draw"))
    other = [d for d in da2.dims if d != "sample"]
    da2 = da2.transpose("sample", *other)
    return np.asarray(da2.values)

def extract_simulation_arrays(idata, n_draws: int, seed: int = 123):
    """
    Extract mu, kappa, p from idata.posterior for simulation.

    Expects:
      - mu: shape (S, N)
      - kappa: shape (S, N)
      - p_goal: shape (S, N, 2) or equivalent; we take component 0 as p(directed)
    """
    post = idata.posterior

    if "mu" not in post.data_vars or "kappa" not in post.data_vars or "p_goal" not in post.data_vars:
        raise KeyError(
            "Need posterior vars 'mu', 'kappa', and 'p_goal' in idata. "
            f"Available: {list(post.data_vars)[:80]} ..."
        )

    mu = _stack_posterior(post["mu"])
    kappa = _stack_posterior(post["kappa"])
    p_goal = _stack_posterior(post["p_goal"])

    mu = mu.reshape(mu.shape[0], -1)
    kappa = kappa.reshape(kappa.shape[0], -1)

    if p_goal.ndim == 2:
        p_dir = p_goal.reshape(p_goal.shape[0], -1)
    elif p_goal.ndim == 3:
        if p_goal.shape[-1] == 2:
            p_dir = p_goal[..., 0].reshape(p_goal.shape[0], -1)
        elif p_goal.shape[1] == 2:
            p_dir = p_goal[:, 0, :].reshape(p_goal.shape[0], -1)
        elif p_goal.shape[2] == 2:
            p_dir = p_goal[:, :, 0].reshape(p_goal.shape[0], -1)
        else:
            raise ValueError(f"Unexpected p_goal shape: {p_goal.shape} (need a length-2 axis).")
    else:
        raise ValueError(f"Unexpected p_goal ndim={p_goal.ndim}, shape={p_goal.shape}")

    S = mu.shape[0]
    rng = np.random.default_rng(seed)

    if n_draws is not None and S > n_draws:
        idx = rng.choice(S, size=n_draws, replace=False)
        mu = mu[idx]
        kappa = kappa[idx]
        p_dir = p_dir[idx]

    return mu, kappa, p_dir

def simulate_one_draw(
    mu_s: np.ndarray,
    kappa_s: np.ndarray,
    p_s: np.ndarray,
    rng: np.random.Generator,
    kappa_uniform: float = 1e-6,
) -> np.ndarray:
    """
    Simulate y_rep for ONE posterior draw, for all observations (length N).
    Uses numpy's vonmises sampler.
    """
    N = mu_s.size
    u = rng.random(N)
    is_goal = u < p_s

    y_rep = np.empty(N, float)
    if is_goal.any():
        y_rep[is_goal] = rng.vonmises(mu_s[is_goal], kappa_s[is_goal])
    if (~is_goal).any():
        y_rep[~is_goal] = rng.vonmises(0.0, kappa_uniform, size=(~is_goal).sum())

    return wrap_angle(y_rep)


# ============================================================
# Posterior predictive summaries (simulate -> hierarchical bootstrap summaries)
# ============================================================

def one_ppc_rep_for_treatment(
    y_rep: np.ndarray,
    df: pd.DataFrame,
    treat_level: str,
    angles: np.ndarray,
    bin_edges: np.ndarray,
    kde_kappa: float,
    seed: int,
    n_boot_within: int = 1,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    For a fixed simulated dataset y_rep (length N), compute *n_boot_within* hierarchical-bootstrap
    summaries within one treatment. Returns the average over within-draw boot replicates.
    """
    rng = np.random.default_rng(seed)

    sub = df[df["treatment"] == treat_level]
    idx = sub.index.to_numpy()
    tags = sub["tag"].to_numpy()
    y_t = y_rep[idx]

    uniq_tags = np.unique(tags)
    y_by_tag = [y_t[tags == tg] for tg in uniq_tags]

    kde_acc = []
    hist_acc = []
    for _ in range(n_boot_within):
        y_star = hier_boot_resample_y(y_by_tag, rng)
        kde_acc.append(von_mises_kde(y_star, angles, kappa=kde_kappa))
        _, h = hist_density(y_star, bin_edges)
        hist_acc.append(h)

    return np.mean(np.stack(kde_acc, axis=0), axis=0), np.mean(np.stack(hist_acc, axis=0), axis=0)

def posterior_predictive_summaries(
    idata,
    df: pd.DataFrame,
    treat_levels: List[str],
    angles: np.ndarray,
    bin_edges: np.ndarray,
    kde_kappa: float,
    n_draws: int,
    n_boot_within_draw: int,
    n_jobs: int,
    seed: int,
) -> Dict[str, Dict[str, np.ndarray]]:
    """
    Returns dict:
      ppc[treat]["kde"]  shape (S, P)
      ppc[treat]["hist"] shape (S, BINS)
    where S = n_draws used.
    """
    mu, kappa, p_dir = extract_simulation_arrays(idata, n_draws=n_draws, seed=seed)
    S, N = mu.shape

    if N != len(df):
        raise ValueError(f"Posterior N={N} does not match df length={len(df)}. Check indexing/order.")

    base_rng = np.random.default_rng(seed)
    draw_seeds = base_rng.integers(1, 2**31 - 1, size=S, dtype=np.int64)

    def work_one_draw(s: int):
        rng = np.random.default_rng(int(draw_seeds[s]))
        y_rep = simulate_one_draw(mu[s], kappa[s], p_dir[s], rng, kappa_uniform=KAPPA_UNIFORM)

        out = {}
        treat_seeds = rng.integers(1, 2**31 - 1, size=len(treat_levels), dtype=np.int64)
        for j, t in enumerate(treat_levels):
            kde_j, hist_j = one_ppc_rep_for_treatment(
                y_rep=y_rep,
                df=df,
                treat_level=t,
                angles=angles,
                bin_edges=bin_edges,
                kde_kappa=kde_kappa,
                seed=int(treat_seeds[j]),
                n_boot_within=n_boot_within_draw,
            )
            out[t] = (kde_j, hist_j)
        return out

    with tqdm_joblib(tqdm(total=S, desc="Posterior predictive (draws)", leave=False)):
        draw_results = Parallel(n_jobs=n_jobs)(
            delayed(work_one_draw)(s) for s in range(S)
        )

    ppc: Dict[str, Dict[str, np.ndarray]] = {t: {"kde": [], "hist": []} for t in treat_levels}
    for out in draw_results:
        for t in treat_levels:
            kde_j, hist_j = out[t]
            ppc[t]["kde"].append(kde_j)
            ppc[t]["hist"].append(hist_j)

    for t in treat_levels:
        ppc[t]["kde"] = np.stack(ppc[t]["kde"], axis=0)
        ppc[t]["hist"] = np.stack(ppc[t]["hist"], axis=0)

    return ppc


# ============================================================
# Summaries + plot
# ============================================================

def summarize_band(mat: np.ndarray, prob: float) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    mat: (S, T)
    returns: median, lo, hi (T,)
    """
    med = np.median(mat, axis=0)
    lo, hi = hdi_along_samples(mat, prob=prob)
    return med, lo, hi

from matplotlib.patches import Circle

def format_polar(ax, edge_radius: float, outer_radius: float):
    ax.spines["polar"].set_visible(False)
    ax.grid(False)
    ax.set_yticks([])
    ax.set_yticklabels([])

    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)

    # Force full circle in [0, 2pi)
    ax.set_xlim(0, 2 * np.pi)

    ax.set_thetagrids(
        [0, 90, 180, 270],
        labels=[" 0°", "", "", ""]
    )

    # keep labels, but remove the tick marks (the little line at the rim)
    ax.tick_params(axis="x", which="major", pad=-2, length=0)

    # Reference circles
    ax.add_artist(Circle((0, 0), edge_radius,
                         transform=ax.transData._b,
                         color="0.35", fill=False, linewidth=0.7))
    ax.add_artist(Circle((0, 0), outer_radius,
                         transform=ax.transData._b,
                         color="0.85", fill=False, linewidth=0.7))

    # Reference ray at 0°
    ax.plot([0, 0], [edge_radius, outer_radius],
            color="0.85", lw=0.8, zorder=0)

def plot_ppc_by_treatment(
    treat_levels,
    angles,
    bin_centers,
    obs_boot,
    ppc,
    treatment_label_map=None,
    outpath_pdf="../figures/bats/bat_population_ppc_by_treatment.pdf",
    outpath_svg="../figures/bats/bat_population_ppc_by_treatment.svg",
):
    k = len(treat_levels)
    fig, axes = plt.subplots(
        1, k,
        figsize=(0.95 * k, 1.25),
        dpi=DPI,
        subplot_kw={"projection": "polar"},
        squeeze=True
    )
    if k == 1:
        axes = [axes]

    # --- swapped tab10 -> seaborn "colorblind" (keep cycling behavior) ---
    base_palette = sns.color_palette("colorblind", 10)
    colors = [base_palette[i % 10] for i in range(k)]

    # ---- convert theta to [0, 2pi) and sort for plotting ----
    theta_plot = (angles + 2 * np.pi) % (2 * np.pi)
    order = np.argsort(theta_plot)
    theta_plot = theta_plot[order]

    bins_plot = (bin_centers + 2 * np.pi) % (2 * np.pi)
    b_order = np.argsort(bins_plot)
    bins_plot = bins_plot[b_order]

    # ---- close the seam at 0° to avoid a thin renderer gap ----
    def _close_circle(theta, y):
        theta_c = np.concatenate([theta, [2.0 * np.pi]])
        y_c     = np.concatenate([y,     [y[0]]])
        return theta_c, y_c

    # Determine global density max (for consistent r scaling)
    global_hi = 0.0
    for t in treat_levels:
        _, o_lo, o_hi = summarize_band(obs_boot[t]["kde"], HDI_PROB)
        _, p_lo, p_hi = summarize_band(ppc[t]["kde"], HDI_PROB)
        global_hi = max(global_hi, float(np.nanmax(o_hi)), float(np.nanmax(p_hi)))

        _, _, o_hhi = summarize_band(obs_boot[t]["hist"], HDI_PROB)
        _, _, p_hhi = summarize_band(ppc[t]["hist"], HDI_PROB)
        global_hi = max(global_hi, float(np.nanmax(o_hhi)), float(np.nanmax(p_hhi)))

    # outer ring corresponds roughly to the upper density envelope (leave headroom)
    outer_ring = EDGE_RADIUS + 0.5
    rmax = EDGE_RADIUS + 0.5 * 1.15

    for ax, t, col in zip(axes, treat_levels, colors):
        format_polar(ax, edge_radius=EDGE_RADIUS, outer_radius=outer_ring)
        ax.set_ylim(0, rmax)

        # ---- Observed bootstrap KDE (black dashed + band) ----
        o_med, o_lo, o_hi = summarize_band(obs_boot[t]["kde"], HDI_PROB)
        o_med, o_lo, o_hi = o_med[order], o_lo[order], o_hi[order]

        theta_c, o_med_c = _close_circle(theta_plot, o_med)
        _,       o_lo_c  = _close_circle(theta_plot, o_lo)
        _,       o_hi_c  = _close_circle(theta_plot, o_hi)

        ax.fill_between(
            theta_c, EDGE_RADIUS + o_lo_c, EDGE_RADIUS + o_hi_c,
            color="black", alpha=BAND_ALPHA, edgecolor="none", zorder=1
        )
        ax.plot(
            theta_c, EDGE_RADIUS + o_med_c,
            color="black", ls="--", lw=LINE_LW, zorder=3
        )

        # ---- PPC KDE (color solid + band) ----
        p_med, p_lo, p_hi = summarize_band(ppc[t]["kde"], HDI_PROB)
        p_med, p_lo, p_hi = p_med[order], p_lo[order], p_hi[order]

        theta_c, p_med_c = _close_circle(theta_plot, p_med)
        _,       p_lo_c  = _close_circle(theta_plot, p_lo)
        _,       p_hi_c  = _close_circle(theta_plot, p_hi)

        ax.fill_between(
            theta_c, EDGE_RADIUS + p_lo_c, EDGE_RADIUS + p_hi_c,
            color=col, alpha=BAND_ALPHA, edgecolor="none", zorder=2
        )
        ax.plot(
            theta_c, EDGE_RADIUS + p_med_c,
            color=col, lw=LINE_LW, zorder=4
        )

        # # ---- Observed bootstrap histogram (points + error bars) ----
        # h_med, h_lo, h_hi = summarize_band(obs_boot[t]["hist"], HDI_PROB)
        # h_med, h_lo, h_hi = h_med[b_order], h_lo[b_order], h_hi[b_order]
        # yerr = np.vstack([h_med - h_lo, h_hi - h_med])
        #
        # # Optional seam-close for histogram too:
        # bins_c, h_med_c = _close_circle(bins_plot, h_med)
        # _,      h_lo_c  = _close_circle(bins_plot, h_lo)
        # _,      h_hi_c  = _close_circle(bins_plot, h_hi)
        # yerr_c = np.vstack([h_med_c - h_lo_c, h_hi_c - h_med_c])
        #
        # ax.errorbar(
        #     bins_c,
        #     EDGE_RADIUS + h_med_c,
        #     yerr=yerr_c,
        #     fmt="o",
        #     ecolor="black",
        #     markeredgecolor="black",
        #     markerfacecolor="black",
        #     lw=0.8,
        #     capsize=2,
        #     markersize=2,
        #     zorder=5
        # )

        title = str(t)
        if treatment_label_map is not None:
            title = format_treatment_title(t, treatment_label_map)

        ax.set_title(title, pad=6)

    # Shared legend
    handles = [
        mpl.lines.Line2D([0], [0], color="black", ls="--", lw=1.2, label="Data"),
        mpl.lines.Line2D([0], [0], color="black", lw=1.2, label="Model"),
    ]

    fig.legend(handles=handles, loc="center", bbox_to_anchor=(0.5, 0.02),
               ncol=3, frameon=True, handlelength=1.6, columnspacing=1.2)

    fig.subplots_adjust(left=0.03, right=0.99, top=0.93, bottom=0.18, wspace=0.22)

    fig.savefig(outpath_pdf, bbox_inches="tight", pad_inches=0.02)
    fig.savefig(outpath_svg, bbox_inches="tight", pad_inches=0.02)
    plt.show()


# ============================================================
# RUN IT
# ============================================================

# IMPORTANT: index must be 0..N-1 for PPC slicing
df_ppc = df_track.reset_index(drop=True).copy()

treat_levels, groups = build_groups(df_ppc, y_col="HeadingRadians", tag_col="tag", treat_col="treatment")

treatment_label_map = (
    df_ppc.groupby("treatment", observed=False)["treatmentName"]
    .first()
    .to_dict()
)

def format_treatment_title(code, label_map, drop_control_if_mixed=True):
    raw = str(label_map.get(code, code)).strip()
    if not raw:
        return str(code)

    parts = [p.strip() for p in raw.split(",") if p.strip()]

    if drop_control_if_mixed and len(parts) > 1:
        parts = [p for p in parts if p.lower() != "control"]

    parts = [p.lower().title() for p in parts]

    return ", ".join(parts) if parts else str(code)

angles = np.linspace(-np.pi, np.pi, N_KDE_POINTS, endpoint=False)
bin_edges = np.linspace(-np.pi, np.pi, N_HIST_BINS + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

obs_boot = bootstrap_observed(
    groups=groups,
    angles=angles,
    bin_edges=bin_edges,
    n_boot=N_BOOT_OBS,
    kde_kappa=KDE_KAPPA,
    n_jobs=N_JOBS,
    seed=RNG_SEED,
)

ppc = posterior_predictive_summaries(
    idata=idata_mix_full,
    df=df_ppc,                      # <-- use df_ppc
    treat_levels=treat_levels,
    angles=angles,
    bin_edges=bin_edges,
    kde_kappa=KDE_KAPPA,
    n_draws=N_PPC_DRAWS,
    n_boot_within_draw=PPC_BOOT_PER_DRAW,
    n_jobs=N_JOBS,
    seed=RNG_SEED,
)

plot_ppc_by_treatment(
    treat_levels=treat_levels,
    angles=angles,
    bin_centers=bin_centers,
    obs_boot=obs_boot,
    ppc=ppc,
    treatment_label_map=treatment_label_map,
    outpath_pdf="../figures/bats/bat_population_ppc_by_treatment.pdf",
    outpath_svg="../figures/bats/bat_population_ppc_by_treatment.svg",
)

### Posterior predictive checks

Build the PPC machinery (population and individual): resample / bootstrap the goal-relative headings
under the vMU, compare to the observed, per treatment and per animal.


In [ ]:
# ============================================================
# CELL 1 — BOOTSTRAP + PPC COMPUTATION (no plotting)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.lines import Line2D

from joblib import Parallel, delayed
from tqdm.auto import tqdm

# ----------------------------
# Styling (kept here so you can still save figures consistently)
# ----------------------------
DPI = 300
AX_LW     = 0.8
TICK_LW   = 0.8
TICK_LEN  = 2.5
LINE_LW   = 0.8
FONT_SIZE = 6

mpl.rcParams.update({
    "font.size": FONT_SIZE,
    "axes.labelsize": FONT_SIZE,
    "axes.titlesize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,
    "font.family": "Arial",
    "svg.fonttype": "none",
    "axes.linewidth": AX_LW,
    "lines.linewidth": LINE_LW,
    "xtick.major.width": TICK_LW,
    "ytick.major.width": TICK_LW,
    "xtick.major.size":  TICK_LEN,
    "ytick.major.size":  TICK_LEN,
    "savefig.dpi": DPI,
    "figure.dpi": DPI,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# ----------------------------
# Knobs (COMPUTE-side)
# ----------------------------
RNG_SEED        = 123
EDGE_RADIUS     = 0.55
AMP             = 1.00
THETA_N         = 360
HDI_PROB        = 0.95
KAPPA_KDE       = 15.0
N_BOOT          = 500
MAX_DRAWS       = 500
KAPPA_UNIFORM   = 1e-6

# Grid shape for panels (used later in plotting cell)
N_COLS          = 8

# Parallelization
N_JOBS          = -1
PREFER          = "threads"

# ----------------------------
# Helpers
# ----------------------------
def circular_modulo(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def wrap_0_2pi(x):
    return np.mod(x, 2.0 * np.pi)

def softplus(x):
    x = np.asarray(x, float)
    return np.where(x > 30, x, np.log1p(np.exp(x)))

def sigmoid(x):
    x = np.asarray(x, float)
    return 1.0 / (1.0 + np.exp(-x))

def von_mises_kde(data, points, kappa):
    data = np.asarray(data, float)
    points = np.asarray(points, float)
    c = 1.0 / (2.0 * np.pi * np.i0(kappa))
    return c * np.exp(kappa * np.cos(points[:, None] - data[None, :])).mean(axis=1)

def quantile_band(arr, prob=0.95):
    arr = np.asarray(arr, float)
    qlo = (1.0 - prob) / 2.0
    qhi = 1.0 - qlo
    med = np.nanmedian(arr, axis=0)
    lo  = np.nanquantile(arr, qlo, axis=0)
    hi  = np.nanquantile(arr, qhi, axis=0)
    return med, lo, hi

def _stack_samples_first(da):
    da2 = da.stack(sample=("chain", "draw"))
    other = [d for d in da2.dims if d != "sample"]
    da2 = da2.transpose("sample", *other)
    return da2.values.astype(float)

def format_treatment_nicely(name: str) -> str:
    s = str(name).strip()
    if not s:
        return "Control"
    parts = [p.strip() for p in s.split(",") if p.strip()]
    def titleish(x):
        return " ".join([w[:1].upper() + w[1:] if w else "" for w in x.lower().split(" ")])
    parts = [titleish(p) for p in parts]
    return ", ".join(parts)

# ============================================================
# Data + treatment maps
# ============================================================
df = df_track.copy()

treat_cat = pd.Categorical(df["treatment"])
df["treatment"] = treat_cat
treat_levels = list(treat_cat.categories)
n_treat = len(treat_levels)

if "treatmentName" in df.columns:
    treat_name_map = df.groupby("treatment", observed=False)["treatmentName"].first().to_dict()
else:
    treat_name_map = {t: str(t) for t in treat_levels}

nice_treat_label = {t: format_treatment_nicely(treat_name_map.get(t, t)) for t in treat_levels}

tag_col   = "tag"
y_col     = "HeadingRadians"
treat_col = "treatment"

tag_to_rows  = df.groupby(tag_col, observed=False).groups
tag_to_treat = df.groupby(tag_col, observed=False)[treat_col].first().to_dict()

tags = list(tag_to_rows.keys())

# Colors by treatment (used later)
cmap = mpl.colormaps.get_cmap("tab10")
treat_to_color = {t: cmap(i % 10) for i, t in enumerate(treat_levels)}

# Evaluation grid: full circle [0, 2π)
angles = np.linspace(0.0, 2.0 * np.pi, THETA_N, endpoint=False)

# ============================================================
# Extract posterior arrays needed to compute per-tag parameters
# ============================================================
post = idata_mix_full.posterior

required = ["mu0", "mu_tag", "eta_kappa0", "eta_kappa_tag", "sigma_kappa_tag",
            "logit_p0", "logit_p_tag", "sigma_p_tag"]
missing = [v for v in required if v not in post.data_vars]
if missing:
    raise KeyError(f"Missing {missing}. Available: {list(post.data_vars)[:80]} ...")

has_mu_treat    = ("mu_treat" in post.data_vars) and (n_treat > 1)
has_kappa_treat = ("kappa_treat" in post.data_vars) and (n_treat > 1)
has_beta_treat  = ("beta_treat" in post.data_vars) and (n_treat > 1)

mu0 = circular_modulo(_stack_samples_first(post["mu0"]).reshape(-1))
mu_tag = _stack_samples_first(post["mu_tag"])
eta_kappa0 = _stack_samples_first(post["eta_kappa0"]).reshape(-1)
eta_kappa_tag = _stack_samples_first(post["eta_kappa_tag"])
sigma_kappa_tag = _stack_samples_first(post["sigma_kappa_tag"]).reshape(-1)

logit_p0 = _stack_samples_first(post["logit_p0"]).reshape(-1)
logit_p_tag = _stack_samples_first(post["logit_p_tag"])
sigma_p_tag = _stack_samples_first(post["sigma_p_tag"]).reshape(-1)

if has_mu_treat:
    mu_treat = _stack_samples_first(post["mu_treat"])
    if mu_treat.ndim == 1:
        mu_treat = mu_treat[:, None]
if has_kappa_treat:
    kappa_treat = _stack_samples_first(post["kappa_treat"])
    if kappa_treat.ndim == 1:
        kappa_treat = kappa_treat[:, None]
if has_beta_treat:
    beta_treat = _stack_samples_first(post["beta_treat"])
    if beta_treat.ndim == 1:
        beta_treat = beta_treat[:, None]

# Subsample posterior draws for PPC
rng_global = np.random.default_rng(RNG_SEED)
S = mu0.shape[0]
if (MAX_DRAWS is not None) and (S > MAX_DRAWS):
    idx = rng_global.choice(S, size=MAX_DRAWS, replace=False)
    mu0 = mu0[idx]
    mu_tag = mu_tag[idx, :]
    eta_kappa0 = eta_kappa0[idx]
    eta_kappa_tag = eta_kappa_tag[idx, :]
    sigma_kappa_tag = sigma_kappa_tag[idx]
    logit_p0 = logit_p0[idx]
    logit_p_tag = logit_p_tag[idx, :]
    sigma_p_tag = sigma_p_tag[idx]
    if has_mu_treat:
        mu_treat = mu_treat[idx, :]
    if has_kappa_treat:
        kappa_treat = kappa_treat[idx, :]
    if has_beta_treat:
        beta_treat = beta_treat[idx, :]
    S = MAX_DRAWS

tag_codes_model = np.unique(df[tag_col].to_numpy())
tag_to_model_idx = {tag: i for i, tag in enumerate(tag_codes_model)}

# ============================================================
# Compute one individual's PPC (returns KDE summaries)
# ============================================================
def compute_one_tag(tag_value, seed=0):
    rng = np.random.default_rng(seed)

    rows = tag_to_rows[tag_value]
    y_obs = df.loc[rows, y_col].to_numpy(dtype=float)
    y_obs = wrap_0_2pi(y_obs)

    n_obs = y_obs.size
    treat = tag_to_treat[tag_value]

    if n_obs < 5:
        obs_mat = np.full((1, THETA_N), np.nan)
        ppc_mat = np.full((1, THETA_N), np.nan)
        obs_med, obs_lo, obs_hi = quantile_band(obs_mat, prob=HDI_PROB)
        ppc_med, ppc_lo, ppc_hi = quantile_band(ppc_mat, prob=HDI_PROB)
        return dict(
            tag=tag_value, treat=treat,
            obs_med=obs_med, obs_lo=obs_lo, obs_hi=obs_hi,
            ppc_med=ppc_med, ppc_lo=ppc_lo, ppc_hi=ppc_hi,
            peak=float(np.nanmax([np.nanmax(obs_hi), np.nanmax(ppc_hi)]))
        )

    treat_code = treat_levels.index(treat)

    if tag_value not in tag_to_model_idx:
        raise KeyError(f"Tag {tag_value} not found in model tag mapping.")
    j_tag = tag_to_model_idx[tag_value]

    mu_s = mu0 + mu_tag[:, j_tag]
    if has_mu_treat and (treat_code > 0):
        mu_s = mu_s + mu_treat[:, treat_code - 1]
    mu_s = circular_modulo(mu_s)

    eta_k_s = eta_kappa0 + eta_kappa_tag[:, j_tag] * sigma_kappa_tag
    if has_kappa_treat and (treat_code > 0):
        eta_k_s = eta_k_s + kappa_treat[:, treat_code - 1]
    kappa_s = softplus(eta_k_s)

    logit_p_s = logit_p0 + logit_p_tag[:, j_tag] * sigma_p_tag
    if has_beta_treat and (treat_code > 0):
        logit_p_s = logit_p_s + beta_treat[:, treat_code - 1]
    p_goal_s = sigmoid(logit_p_s)

    # Observed bootstrap KDE
    obs_kdes = np.empty((N_BOOT, THETA_N), dtype=float)
    for b in range(N_BOOT):
        boot = y_obs[rng.integers(0, n_obs, size=n_obs)]
        obs_kdes[b, :] = von_mises_kde(boot, angles, kappa=KAPPA_KDE)
    obs_med, obs_lo, obs_hi = quantile_band(obs_kdes, prob=HDI_PROB)

    # Posterior predictive KDE
    ppc_kdes = np.empty((S, THETA_N), dtype=float)
    for s in range(S):
        u = rng.uniform(size=n_obs)
        is_goal = (u < p_goal_s[s])

        n_goal = int(is_goal.sum())
        n_unif = n_obs - n_goal

        sim = np.empty(n_obs, dtype=float)
        if n_goal > 0:
            sim[is_goal] = rng.vonmises(mu_s[s], kappa_s[s], size=n_goal)
        if n_unif > 0:
            sim[~is_goal] = rng.vonmises(0.0, KAPPA_UNIFORM, size=n_unif)

        sim = wrap_0_2pi(sim)
        ppc_kdes[s, :] = von_mises_kde(sim, angles, kappa=KAPPA_KDE)

    ppc_med, ppc_lo, ppc_hi = quantile_band(ppc_kdes, prob=HDI_PROB)
    peak = float(np.nanmax([obs_hi.max(), ppc_hi.max()]))

    return dict(
        tag=tag_value, treat=treat,
        obs_med=obs_med, obs_lo=obs_lo, obs_hi=obs_hi,
        ppc_med=ppc_med, ppc_lo=ppc_lo, ppc_hi=ppc_hi,
        peak=peak
    )

# ============================================================
# Run compute (results used by plotting cell)
# ============================================================
results = Parallel(n_jobs=N_JOBS, prefer=PREFER)(
    delayed(compute_one_tag)(tag, seed=10_000 + i)
    for i, tag in enumerate(tqdm(tags, desc="Individual PPC (bootstrap KDE + PPC KDE)"))
)

treat_order = {t: i for i, t in enumerate(treat_levels)}
results = sorted(results, key=lambda r: treat_order.get(r["treat"], 10**9))

peak_max = max(r["peak"] for r in results if np.isfinite(r["peak"])) + 1e-12

def to_r(density):
    return EDGE_RADIUS + AMP * (density / peak_max)

RMAX = EDGE_RADIUS + AMP * 1.01


Individual PPC: bootstrap vs posterior KDE of the goal-directed component per animal.


In [ ]:
# ============================================================
# CELL 2 — PLOTTING ONLY (uses `results`, `angles`, `to_r`, etc.)
# ============================================================

import seaborn as sns

# Swap tab10 -> seaborn "colorblind" (keep everything else the same)
treat_to_color = {
    t: c for t, c in zip(
        treat_levels,
        sns.color_palette("colorblind", n_colors=len(treat_levels))
    )
}

def format_polar(ax, rmax):
    ax.spines["polar"].set_visible(False)
    ax.grid(False)
    ax.set_yticks([])
    ax.set_yticklabels([])

    # Only show 0° label, with NO tick mark/line
    ax.set_xticks([0.0])
    ax.set_xticklabels(["0°"])
    ax.tick_params(axis="x", pad=-3, length=0, width=0)

    ax.set_ylim(0, rmax)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)

    # Rings (outermost removed)
    ax.add_artist(Circle((0, 0), EDGE_RADIUS, transform=ax.transData._b,
                         color="0.80", fill=False, linewidth=0.6))
    ax.add_artist(Circle((0, 0), EDGE_RADIUS + 0.5 * AMP, transform=ax.transData._b,
                         color="0.86", fill=False, linewidth=0.6))

    # Gray radial reference line at 0°
    ax.plot([0, 0], [EDGE_RADIUS, rmax], color="0.85", lw=0.8, zorder=0)

# ============================================================
# Plot grid
# ============================================================
n = len(results)
ncols = N_COLS
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(0.86 * ncols, 0.86 * nrows),
    dpi=DPI,
    subplot_kw={"projection": "polar"},
    squeeze=False
)

# Delete unused axes
for k in range(n, nrows * ncols):
    fig.delaxes(axes[k // ncols, k % ncols])

for i, r in enumerate(results):
    ax = axes[i // ncols, i % ncols]
    format_polar(ax, rmax=RMAX)

    treat = r["treat"]
    col = treat_to_color.get(treat, "tab:blue")

    # Observed bootstrap KDE
    ax.fill_between(
        angles, to_r(r["obs_lo"]), to_r(r["obs_hi"]),
        facecolor="0.2", edgecolor="none", alpha=0.18, zorder=1
    )
    ax.plot(angles, to_r(r["obs_med"]), color="0.05", lw=0.9, ls="--", zorder=3)

    # PPC KDE
    ax.fill_between(
        angles, to_r(r["ppc_lo"]), to_r(r["ppc_hi"]),
        facecolor=col, edgecolor="none", alpha=0.28, zorder=2
    )
    ax.plot(angles, to_r(r["ppc_med"]), color=col, lw=1.0, ls="-", zorder=4)

    # Title (tight)
    ax.text(
        0.5, 1.2, f"{r['tag']}",
        transform=ax.transAxes, ha="center", va="bottom",
        fontsize=6, fontweight="bold",
    )

# Legend
handles_treat = [Line2D([0], [0], color=treat_to_color[t], lw=2.0) for t in treat_levels]
labels_treat  = [nice_treat_label[t] for t in treat_levels]

handles = [
    Line2D([0], [0], color="black", lw=1.6, ls="--"),
    Line2D([0], [0], color="black", lw=1.6, ls="-"),
] + handles_treat
labels = ["Data", "Model"] + labels_treat

fig.legend(
    handles, labels,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.012),
    ncol=min(6, len(labels)),
    frameon=True,
    handlelength=1.6,
    columnspacing=0.9
)

fig.subplots_adjust(
    left=0.015, right=0.985, top=0.985, bottom=0.085,
    wspace=0.25, hspace=0.45
)

out_pdf = "../figures/bats/bat_individual_ppc_bootstrap_vs_ppc_kde.pdf"
out_svg = "../figures/bats/bat_individual_ppc_bootstrap_vs_ppc_kde.svg"
fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.005)
fig.savefig(out_svg, bbox_inches="tight", pad_inches=0.005)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import arviz as az
import scipy.special as sps
# ---------- numerics ----------
LOG2PI = np.log(2.0 * np.pi)

def _stack_samples_first(da):
    """xarray DataArray -> numpy with shape (S, ...) where S=chain*draw"""
    da2 = da.stack(sample=("chain", "draw"))
    other = [d for d in da2.dims if d != "sample"]
    da2 = da2.transpose("sample", *other)
    return da2

def _as_sample_obs(arr_da, n_obs, *, want_components=False):
    """
    Convert posterior var to numpy with:
      - if want_components=True: (S, n_obs, 2)
      - else:                   (S, n_obs)
    Works even if dims are unnamed like *_dim_0, *_dim_1.
    """
    da = _stack_samples_first(arr_da)
    dims = [d for d in da.dims if d != "sample"]
    sizes = [da.sizes[d] for d in dims]

    x = da.values.astype(float)

    if not dims:
        # scalar -> broadcast
        if want_components:
            raise ValueError("Got scalar but want components (obs,2).")
        return np.broadcast_to(x.reshape(-1, 1), (x.shape[0], n_obs))

    if want_components:
        if len(dims) != 2:
            raise ValueError(f"Expected 2 dims (obs,comp) for mixture weights; got dims={da.dims} sizes={sizes}")
        # find obs dim (n_obs) and comp dim (2)
        if n_obs not in sizes or 2 not in sizes:
            raise ValueError(
                f"Can't identify obs/comp dims: need sizes include n_obs={n_obs} and 2; got dims={da.dims} sizes={sizes}"
            )
        obs_axis = sizes.index(n_obs)  # among the 2 dims
        comp_axis = sizes.index(2)

        # current x shape is (S, dim0, dim1)
        # we want (S, n_obs, 2)
        if obs_axis == 0 and comp_axis == 1:
            return x
        elif obs_axis == 1 and comp_axis == 0:
            return np.swapaxes(x, 1, 2)
        else:
            # shouldn't happen
            raise ValueError(f"Unexpected axis mapping dims={da.dims} sizes={sizes}")

    # want (S, n_obs)
    if len(dims) == 1:
        if sizes[0] == n_obs:
            return x
        # scalar-ish per tag etc; we can't safely align
        raise ValueError(f"Expected obs dim size {n_obs} but got {sizes[0]} for dims={da.dims}")

    # if more dims, try to find an obs axis and squeeze the rest
    if n_obs in sizes:
        obs_axis = sizes.index(n_obs)  # among dims
        # move obs axis to position 1 (after S)
        x2 = np.moveaxis(x, 1 + obs_axis, 1)
        # now x2 is (S, n_obs, ...)
        # if trailing dims are all 1, squeeze
        trailing = x2.shape[2:]
        if np.prod(trailing) == 1:
            return x2.reshape(x2.shape[0], x2.shape[1])
        raise ValueError(f"Variable has extra non-singleton dims beyond obs: shape={x2.shape} dims={da.dims}")

    raise ValueError(f"Could not find obs dim size {n_obs} in dims={da.dims} sizes={sizes}")

def _softplus(x):
    x = np.asarray(x, float)
    return np.where(x > 30, x, np.log1p(np.exp(x)))

def _vonmises_logpdf(theta, mu, kappa):
    """
    theta, mu, kappa broadcast to same shape.
    log f = kappa cos(theta-mu) - log(2π) - log(I0(kappa))
    """
    # numpy.i0 is stable for typical kappa; for huge kappa you may want scipy.special.i0e
    return kappa * np.cos(theta - mu) - LOG2PI - np.log(np.i0(kappa))

import numpy as np
import arviz as az
import scipy.special as sps

def _log_i0(kappa):
    """
    Stable log(I0(kappa)) for kappa >= 0 using i0e:
      i0(k) = i0e(k) * exp(k)  => log i0(k) = log i0e(k) + k
    """
    kappa = np.asarray(kappa, float)
    # i0e is defined for all real; VM expects kappa >= 0 anyway
    return np.log(sps.i0e(kappa)) + kappa

def posterior_prob_goal_per_step(
    idata,
    y_obs,
    *,
    max_draws=1200,
    hdi_prob=0.95,
    eps=1e-9,
    summarize_on="logit",  # "logit" or "prob"
):
    """
    Posterior responsibility for a 2-component mixture:
      goal ~ von Mises(mu, kappa)
      other ~ UniformCircle (density 1/(2π))

    r = p(goal | y, draw) = sigmoid( logit(w) + log f_goal(y) - log f_unif(y) )

    With proper uniform on the circle:
      log f_goal - log f_unif = kappa*cos(y-mu) - log I0(kappa)
    """
    post = idata.posterior
    y_obs = np.asarray(y_obs, float).reshape(-1)
    n_obs = y_obs.size

    # --- mixture weight w(goal) per obs per draw ---
    if "p_goal" in post.data_vars:
        w_full = _as_sample_obs(post["p_goal"], n_obs, want_components=True)  # (S, n_obs, 2)
        w = w_full[:, :, 0]
    else:
        if "logit_p" not in post.data_vars:
            raise KeyError("Need either posterior var 'p_goal' (preferred) or 'logit_p'.")
        logit_p = _as_sample_obs(post["logit_p"], n_obs, want_components=False)
        w = sps.expit(logit_p)

    # --- mu, kappa per obs per draw ---
    if "mu" in post.data_vars:
        mu = _as_sample_obs(post["mu"], n_obs, want_components=False)
    elif "mu0" in post.data_vars:
        mu0 = _stack_samples_first(post["mu0"]).values.astype(float).reshape(-1)  # (S,)
        mu = np.broadcast_to(mu0[:, None], (mu0.size, n_obs))
    else:
        raise KeyError("Need posterior var 'mu' or 'mu0'.")

    if "kappa" in post.data_vars:
        kappa = _as_sample_obs(post["kappa"], n_obs, want_components=False)
    elif "eta_kappa0" in post.data_vars:
        ek0 = _stack_samples_first(post["eta_kappa0"]).values.astype(float).reshape(-1)  # (S,)
        k0 = _softplus(ek0)
        kappa = np.broadcast_to(k0[:, None], (k0.size, n_obs))
    else:
        raise KeyError("Need posterior var 'kappa' or 'eta_kappa0'.")

    # --- subsample draws once ---
    S = w.shape[0]
    rng = np.random.default_rng(123)
    if (max_draws is not None) and (S > max_draws):
        idx = rng.choice(S, size=max_draws, replace=False)
        w = w[idx, :]
        mu = mu[idx, :]
        kappa = kappa[idx, :]
        S = max_draws

    # --- compute responsibility logit using exact uniform circle ---
    w = np.clip(w, eps, 1.0 - eps)
    logit_w = np.log(w) - np.log1p(-w)  # (S, n_obs)

    y = y_obs[None, :]
    ll_diff = kappa * np.cos(y - mu) - _log_i0(kappa)  # log f_VM - log f_unif

    logit_r = logit_w + ll_diff
    resp = sps.expit(logit_r)

    if summarize_on == "prob":
        mean_r = resp.mean(axis=0)
        hdi = az.hdi(resp, hdi_prob=hdi_prob, skipna=True)
        return mean_r, hdi, resp

    if summarize_on == "logit":
        mean_logit = logit_r.mean(axis=0)
        hdi_logit = az.hdi(logit_r, hdi_prob=hdi_prob, skipna=True)

        mean_r = sps.expit(mean_logit * 2)
        hdi = sps.expit(hdi_logit)
        return mean_r, hdi, resp

    raise ValueError("summarize_on must be 'prob' or 'logit'")

# ============================================================
# APPEND TO df_track
# ============================================================
# IMPORTANT: df_track must be the exact row order used in the fit.
df_track = df_track.copy()

# step index within each tag, using current row order
df_track["_step"] = df_track.groupby("tag", observed=False).cumcount()

y_obs = df_track["HeadingRadians"].to_numpy()

p_mean, p_hdi, _resp = posterior_prob_goal_per_step(
    idata_mix_full,
    y_obs,
    max_draws=2000,
    hdi_prob=0.95,
    summarize_on="logit",
)

df_track["p_goal_post_mean"] = p_mean
df_track["p_goal_post_lo"]   = p_hdi[:, 0]
df_track["p_goal_post_hi"]   = p_hdi[:, 1]

print(df_track[["tag", "treatment", "_step", "p_goal_post_mean", "p_goal_post_lo", "p_goal_post_hi"]].head())


/tmp/ipykernel_3898688/1720640019.py:179: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi_logit = az.hdi(logit_r, hdi_prob=hdi_prob, skipna=True)


    tag treatment  _step  p_goal_post_mean  p_goal_post_lo  p_goal_post_hi
1  7535         3      0      2.115701e-11    1.143538e-09        0.003496
2  7535         3      1      9.578935e-01    7.101751e-01        0.904318
3  7535         3      2      1.290928e-10    8.356906e-09        0.010242
4  7535         3      3      5.800259e-11    4.108252e-09        0.007829
5  7535         3      4      5.553072e-11    3.976896e-09        0.007764


## 9. Goal-directedness along the trajectory

Posterior probability that each step belongs to the **directed (goal-seeking) component**
($p_{goal}$), plotted along each trajectory (by distance / progress toward the goal). Directedness is
expected to be lower early in a flight (localization / exploration) and higher near the endpoint
(homing).


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
from matplotlib.collections import LineCollection
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import contextily as cx
from pyproj import Transformer

# NEW
import seaborn as sns

# NEW (for compass + precise placement)
from matplotlib.transforms import offset_copy
from matplotlib.patches import FancyArrowPatch, ArrowStyle

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
LAKE_GEOJSON = bats_lake_outline_path()
WEBMERC_EPSG = 3857

# Special locations
ROOST_LAT, ROOST_LON = 33.112373, 35.584135
TRANS_POINTS = [
    (33.135136, 35.592712),
    (33.081932, 35.604462),
]

# ------------------------------------------------------------
# SMALL UTILS
# ------------------------------------------------------------
def hide_axes(ax):
    ax.set_xlabel(""); ax.set_ylabel("")
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)

def _format_treatment_name(name: str) -> str:
    s = str(name).strip().lower()
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if "control" in parts and len(parts) > 1:
        parts = [p for p in parts if p != "control"]
    if not parts:
        parts = ["control"]
    parts = sorted(parts)
    parts = [p[:1].upper() + p[1:] for p in parts]
    return ", ".join(parts)

def _pick_order_col(df):
    for c in ["frame", "step", "t", "time", "timestamp"]:
        if c in df.columns:
            return c
    return None

def _ensure_step_index(df, tag_col="tag"):
    d = df.copy()
    order_col = _pick_order_col(d)
    if order_col is None:
        d["_step"] = d.groupby(tag_col, observed=False).cumcount()
    else:
        d = d.sort_values([tag_col, order_col]).copy()
        d["_step"] = d.groupby(tag_col, observed=False).cumcount()
    return d

def _make_points_gdf():
    pts = [{"site": "Roost", "geometry": Point(ROOST_LON, ROOST_LAT)}]
    pts += [{"site": "Translocation", "geometry": Point(lon, lat)} for (lat, lon) in TRANS_POINTS]
    return gpd.GeoDataFrame(pts, crs="EPSG:4326").to_crs(epsg=WEBMERC_EPSG)

# ------------------------------------------------------------
# SCALE BAR + COMPASS (BLACK/WHITE) + BOTTOM-RIGHT PLACEMENT
# ------------------------------------------------------------
def add_scale_bar_dual(
    ax,
    total_length_m=20.0,
    n_segments=2,
    segment_colors=("#ffffff", "#000000"),  # BLACK/WHITE
    segment_height_pt=8,
    edgecolor="#000000",                    # BLACK
    edgewidth=0.9,
    label=True,
    label_above=False,
    label_kwargs=None,
    # precise placement:
    ax_xy=(1.0, 0.0),       # anchor in axes coords
    offset_pt=(-22, 40),    # (dx, dy) in points from anchor
    align="right",
    direction="left",
    pad_text_pt=4,
    return_info=False
):
    """
    Two-tone horizontal scale bar (meters, data units).
    Placement uses axes coords + point offset. Rectangles drawn in data coords.
    """
    if label_kwargs is None:
        label_kwargs = dict(
            color="#000000", fontsize=10, ha="center", va="top",
            fontname="Arial", fontstretch="normal"
        )

    fig = ax.figure
    if fig.canvas is not None:
        fig.canvas.draw()

    base = offset_copy(ax.transAxes, fig=fig, x=offset_pt[0], y=offset_pt[1], units="points")

    x_disp, y_disp = base.transform(ax_xy)
    x_anchor, y_anchor = ax.transData.inverted().transform((x_disp, y_disp))

    seg_len = total_length_m / float(max(n_segments, 1))
    if direction == "left":
        x0 = (x_anchor if align == "left" else x_anchor - total_length_m)
        xdir = +1
    elif direction == "right":
        x0 = (x_anchor if align == "left" else x_anchor + total_length_m)
        if align == "right":
            x0 = x_anchor - total_length_m
        xdir = +1
    else:
        raise ValueError("direction must be 'left' or 'right'")

    # Convert bar height (points) to data units using axes pixel height
    bbox = ax.get_window_extent()
    axpix_h = bbox.height
    y_min, y_max = ax.get_ylim()
    pt_to_data_y = (y_max - y_min) / axpix_h
    half_h = 0.5 * segment_height_pt * pt_to_data_y
    yb, yt = y_anchor - half_h, y_anchor + half_h

    colors = (segment_colors * ((n_segments + len(segment_colors) - 1) // len(segment_colors)))[:n_segments]

    patches = []
    for i in range(n_segments):
        xa = x0 + xdir * (i * seg_len)
        xb = x0 + xdir * ((i + 1) * seg_len)
        p = plt.Rectangle(
            (min(xa, xb), yb),
            abs(xb - xa),
            yt - yb,
            facecolor=colors[i],
            edgecolor=edgecolor,
            linewidth=edgewidth,
            zorder=90
        )
        ax.add_patch(p)
        patches.append(p)

    x_center = x0 + xdir * (total_length_m / 2.0)
    txt = None
    if label:
        txt = ax.text(x_center, y_anchor, f"{int(total_length_m)} m", **label_kwargs)
        dy = pad_text_pt if label_above else -pad_text_pt
        txt.set_transform(offset_copy(ax.transData, fig=fig, x=0, y=dy, units="points"))
        txt.set_zorder(91)

    if return_info:
        return {
            "center_data": (x_center, y_anchor),
            "y_anchor_data": y_anchor,
            "segment_height_pt": segment_height_pt,
            "offset_pt": offset_pt,
            "ax_xy": ax_xy,
            "total_length_m": total_length_m,
            "patches": patches,
            "text": txt
        }
    return None


def add_compass(
    ax,
    ax_xy=(0.95, 0.12),
    offset_pt=(0, 0),
    arrow_len_pt=30,        # default smaller now
    head_width_pt=9,
    head_length_pt=10,
    tail_width_pt=2.0,
    color="#000000",
    edgecolor="#000000",
    edgewidth=1.0,
    alpha=1.0,
    draw_N=True,
    N_fontsize=10,
    N_offset_pt=(0, 3)
):
    """
    North arrow drawn in AXES coordinates (constant size, upright).
    """
    fig = ax.figure
    if fig.canvas is not None:
        fig.canvas.draw()

    base = offset_copy(ax.transAxes, fig=fig, x=offset_pt[0], y=offset_pt[1], units="points")

    bbox = ax.get_window_extent()
    axpix_h = bbox.height
    dy_axes = (arrow_len_pt * fig.dpi / 72.0) / axpix_h

    x_axes, y_axes = ax_xy
    start = (x_axes, y_axes)
    end   = (x_axes, y_axes + dy_axes)

    style = ArrowStyle.Simple(head_length=head_length_pt, head_width=head_width_pt, tail_width=tail_width_pt)
    patch = FancyArrowPatch(
        start, end,
        transform=ax.transAxes,
        arrowstyle=style,
        mutation_scale=1.0,
        linewidth=edgewidth,
        edgecolor=edgecolor,
        facecolor=color,
        alpha=alpha,
        joinstyle="miter", capstyle="butt",
        zorder=92
    )
    patch.set_transform(base)
    ax.add_patch(patch)

    if draw_N:
        tx = ax.text(
            x_axes, y_axes + dy_axes, "N",
            transform=ax.transAxes, ha="center", va="bottom",
            fontsize=N_fontsize, color="#000000", zorder=93,
            fontname="Arial"
        )
        tx.set_transform(offset_copy(
            ax.transAxes, fig=fig,
            x=N_offset_pt[0] + offset_pt[0],
            y=N_offset_pt[1] + offset_pt[1],
            units="points"
        ))


def _nice_scale_length_m(width_m: float) -> float:
    target = max(width_m * 0.2, 1.0)
    exp = np.floor(np.log10(target))
    base = target / (10 ** exp)
    if base < 1.5:
        nice = 1.0
    elif base < 3.5:
        nice = 2.0
    elif base < 7.5:
        nice = 5.0
    else:
        nice = 10.0
    return float(nice * (10 ** exp))


# --- CHANGE ONLY THESE TWO FUNCTIONS (drop-in replacements) ---
# (Everything else in your script can stay exactly the same.)

def _add_scale_and_compass_bottom_right_equal_pad(ax, corner_pad_px=22):
    """
    Adds a black/white scalebar + compass to the *bottom-right* of `ax`
    with equal padding to right & bottom borders (in pixels).
    Compass is centered above the scale bar.
    """
    fig = ax.figure
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    pad_pt = corner_pad_px * 72.0 / fig.dpi
    seg_h_pt = 8

    # ↓↓↓ make scale text closer to the bar (was 10)
    scale_text_breathe_pt = 6

    # scale length from axis width (meters; EPSG:3857)
    x0, x1 = ax.get_xlim()
    scale_len_m = _nice_scale_length_m(abs(x1 - x0))

    label_kwargs = dict(
        color="#000000", fontsize=10, ha="center", va="top",
        fontname="Arial", fontstretch="normal"
    )

    def _draw_scale(offset_y_pt):
        return add_scale_bar_dual(
            ax,
            total_length_m=scale_len_m,
            n_segments=2,
            segment_colors=("#ffffff", "#000000"),
            segment_height_pt=seg_h_pt,
            edgecolor="#000000",
            edgewidth=0.9,
            ax_xy=(1.0, 0.0),
            offset_pt=(-pad_pt, offset_y_pt),
            align="right",
            direction="left",
            label=True,
            label_above=False,
            # ↓↓↓ also tighten label offset (was 10 via breathe=10)
            pad_text_pt=scale_text_breathe_pt,
            label_kwargs=label_kwargs,
            return_info=True
        )

    # initial guess; then corrected so label bottom padding matches corner_pad_px
    offset_y_pt = pad_pt + (seg_h_pt / 2.0) + scale_text_breathe_pt + 6.0
    sb = _draw_scale(offset_y_pt)
    fig.canvas.draw()

    txt = sb["text"]
    if txt is not None:
        ax_bbox = ax.get_window_extent(renderer=renderer)
        txt_bbox = txt.get_window_extent(renderer=renderer)
        current_bottom_pad_px = txt_bbox.y0 - ax_bbox.y0
        dy_px = corner_pad_px - current_bottom_pad_px
        dy_pt = dy_px * 72.0 / fig.dpi

        if abs(dy_pt) > 0.25:
            for p in sb["patches"]:
                p.remove()
            txt.remove()
            sb = _draw_scale(offset_y_pt + dy_pt)
            fig.canvas.draw()

    # compass centered above scale bar
    (cx_data, y_anchor_data) = sb["center_data"]
    cx_disp, cy_disp = ax.transData.transform((cx_data, y_anchor_data))

    # ↓↓↓ bring compass slightly closer to the scale bar (was (seg_h/2)+14)
    compass_gap_pt = (seg_h_pt / 2.0)

    cy_disp_up = cy_disp + compass_gap_pt * fig.dpi / 72.0
    cx_axes, cy_axes = ax.transAxes.inverted().transform((cx_disp, cy_disp_up))

    add_compass(
        ax,
        ax_xy=(cx_axes, cy_axes),
        offset_pt=(0, 0),
        # ↓↓↓ make arrow smaller (was 42 / 12 / 14 / 2.5)
        arrow_len_pt=20,
        head_width_pt=9,
        head_length_pt=10,
        tail_width_pt=2.0,
        color="#000000",
        edgecolor="#000000",
        edgewidth=1.0,
        draw_N=True,
        # ↓↓↓ slightly smaller N to match
        N_fontsize=10,
        # ↓↓↓ keep N close to arrow head
        N_offset_pt=(0, 3),
    )



# ------------------------------------------------------------
# MAIN PLOT
# ------------------------------------------------------------
def plot_treatment_panels_tracks_colored_by_pgoal(
    tracks_df,
    *,
    tag_col="tag",
    lat_col="lat",
    lon_col="lon",
    treat_col="treatment",
    treat_name_col="treatmentName",
    p_col="p_goal_post_mean",
    gap_col="gap_before",
    df_prob=None,
    prob_tag_col="tag",
    prob_step_col="_step",
    prob_p_col="p_goal_post_mean",
    # style
    basemap_source=cx.providers.CartoDB.Positron,
    basemap_alpha=1.0,
    lake_facecolor="tab:cyan",
    lake_alpha=0.25,
    lake_edge_lw=1.0,
    seg_lw=0.9,
    seg_alpha=0.92,
    gap_lw=None,
    gap_alpha=0.80,
    gap_linestyle="--",
    # subsampling
    max_traj_per_panel=None,
    seed=123,
    out_pdf="../figures/bats/treatment_panels_tracks_pgoal_viridis.pdf",
    out_svg="../figures/bats/treatment_panels_tracks_pgoal_viridis.svg",
    # layout tweaks
    fig_w=8.2,
    fig_h=7.8,
    wspace=0.01,
    hspace=0.03,
    # colorbar tweaks
    cbar_width=0.012,
    cbar_pad=0.010,
    cbar_ticksize=8,
    cbar_labelsize=9,
):
    d = tracks_df.copy()
    for c in [tag_col, lat_col, lon_col, treat_col]:
        if c not in d.columns:
            raise KeyError(f"tracks_df missing required column '{c}'")

    d[lat_col] = pd.to_numeric(d[lat_col], errors="coerce")
    d[lon_col] = pd.to_numeric(d[lon_col], errors="coerce")
    d = d[np.isfinite(d[lat_col]) & np.isfinite(d[lon_col])].copy()
    d = d.dropna(subset=[tag_col, treat_col]).copy()
    d = _ensure_step_index(d, tag_col=tag_col)

    # attach p if needed
    if p_col not in d.columns:
        if df_prob is None:
            raise KeyError(
                f"tracks_df missing '{p_col}'. Either compute it on tracks_df "
                f"or pass df_prob (with columns {prob_tag_col}, {prob_step_col}, {prob_p_col})."
            )
        if prob_step_col not in df_prob.columns:
            raise KeyError(f"df_prob missing '{prob_step_col}'. Make sure it has the per-tag step index.")
        tmp = df_prob[[prob_tag_col, prob_step_col, prob_p_col]].copy()
        tmp = tmp.rename(columns={prob_tag_col: tag_col, prob_step_col: "_step", prob_p_col: p_col})
        d = d.merge(tmp, on=[tag_col, "_step"], how="left", validate="many_to_one")

    d[p_col] = pd.to_numeric(d[p_col], errors="coerce")
    d = d[np.isfinite(d[p_col])].copy()
    d[p_col] = d[p_col].clip(0.0, 1.0)

    has_gaps = gap_col in d.columns
    if has_gaps:
        d[gap_col] = pd.to_numeric(d[gap_col], errors="coerce").fillna(0).astype(int)

    # treatment order + labels
    if pd.api.types.is_categorical_dtype(d[treat_col]):
        treat_levels = list(d[treat_col].cat.categories)
    else:
        treat_levels = sorted(d[treat_col].dropna().unique().tolist())
    if len(treat_levels) == 0:
        raise ValueError("No treatment levels found.")

    nice = {t: _format_treatment_name(t) for t in treat_levels}
    if treat_name_col in d.columns:
        name_map = d.groupby(treat_col, observed=False)[treat_name_col].first().to_dict()
        for t in treat_levels:
            nice[t] = _format_treatment_name(name_map.get(t, t))

    # NEW: subplot border colors (seaborn colorblind, same order as treat_levels)
    treat_border_colors = sns.color_palette("colorblind", n_colors=len(treat_levels))

    # project coords to WebMercator
    transformer = Transformer.from_crs("EPSG:4326", f"EPSG:{WEBMERC_EPSG}", always_xy=True)
    xs, ys = transformer.transform(d[lon_col].to_numpy(), d[lat_col].to_numpy())
    d["_x"] = xs
    d["_y"] = ys

    # lake + points
    lake_gdf = gpd.read_file(LAKE_GEOJSON).to_crs(epsg=WEBMERC_EPSG)
    pts_gdf = _make_points_gdf()

    # common bounds
    layers_bounds = []
    if len(lake_gdf):
        layers_bounds.append(lake_gdf.total_bounds)
    if len(pts_gdf):
        layers_bounds.append(pts_gdf.total_bounds)
    if len(d):
        layers_bounds.append(np.array([d["_x"].min(), d["_y"].min(), d["_x"].max(), d["_y"].max()]))

    minx = min(b[0] for b in layers_bounds)
    miny = min(b[1] for b in layers_bounds)
    maxx = max(b[2] for b in layers_bounds)
    maxy = max(b[3] for b in layers_bounds)

    pad_x = (maxx - minx) * 0.06 if maxx > minx else 1000
    pad_y = (maxy - miny) * 0.06 if maxy > miny else 1000
    xlim = (minx - pad_x, maxx + pad_x)
    ylim = (miny - pad_y, maxy + pad_y)

    # layout: 1 row, 4 cols (as in your code)
    ncols = 4
    nrows = 1

    mpl.rcParams.update({
        "font.family": "Arial",
        "font.sans-serif": ["Arial"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
    })

    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), dpi=300)
    axes = np.atleast_1d(axes).ravel()

    cmap = mpl.colormaps.get_cmap("viridis")
    norm = mcolors.Normalize(vmin=0.0, vmax=1.0)

    if gap_lw is None:
        gap_lw = max(0.7, seg_lw * 0.95)

    rng = np.random.default_rng(seed)

    for i, t in enumerate(treat_levels):
        ax = axes[i]
        ax.set_xlim(*xlim); ax.set_ylim(*ylim)

        # basemap
        try:
            cx.add_basemap(ax, source=basemap_source, attribution=False, alpha=basemap_alpha)
        except Exception as e:
            print("Basemap failed (continuing without tiles):", e)

        # lake
        if len(lake_gdf):
            lake_gdf.plot(ax=ax, facecolor=lake_facecolor, edgecolor=lake_facecolor,
                          alpha=lake_alpha, linewidth=lake_edge_lw, zorder=12)
            lake_gdf.boundary.plot(ax=ax, color=lake_facecolor, linewidth=lake_edge_lw + 0.2,
                                   alpha=1.0, zorder=13)

        # roost + trans points
        ref = pts_gdf.loc[pts_gdf["site"] == "Roost"]
        ax.scatter(ref.geometry.x, ref.geometry.y, marker="*", s=200,
                   facecolors="white", edgecolors="black", linewidths=0.8, zorder=50)

        trans = pts_gdf.loc[pts_gdf["site"] == "Translocation"]
        ax.scatter(trans.geometry.x, trans.geometry.y, marker="X", s=120,
                   facecolors="white", edgecolors="black", linewidths=0.8, zorder=50)

        # tracks for this treatment
        dd = d.loc[d[treat_col] == t].copy()
        tags = sorted(dd[tag_col].unique().tolist())

        if (max_traj_per_panel is not None) and (len(tags) > max_traj_per_panel):
            tags = rng.choice(tags, size=max_traj_per_panel, replace=False).tolist()
            dd = dd.loc[dd[tag_col].isin(tags)].copy()

        all_segments, all_values = [], []
        gap_segments, gap_values = [], []

        order_col = _pick_order_col(dd)
        for tg, g in dd.groupby(tag_col, sort=False):
            g = g.sort_values(order_col) if order_col else g.sort_values("_step")

            x = g["_x"].to_numpy(float)
            y = g["_y"].to_numpy(float)
            p = g[p_col].to_numpy(float)

            if len(x) < 2:
                continue

            pts_xy = np.column_stack([x, y])
            segs = np.stack([pts_xy[:-1], pts_xy[1:]], axis=1)
            pseg = 0.5 * (p[:-1] + p[1:])

            if has_gaps and gap_col in g.columns:
                gb = g[gap_col].to_numpy(int)
                cross_gap = (gb[1:] == 1)

                if np.any(~cross_gap):
                    all_segments.append(segs[~cross_gap])
                    all_values.append(pseg[~cross_gap])

                if np.any(cross_gap):
                    gap_segments.append(segs[cross_gap])
                    gap_values.append(pseg[cross_gap])
            else:
                all_segments.append(segs)
                all_values.append(pseg)

        # dashed gap connectors (draw first)
        if len(gap_segments):
            gsegs = np.concatenate(gap_segments, axis=0)
            gvals = np.concatenate(gap_values, axis=0)
            glc = LineCollection(
                gsegs,
                cmap=cmap,
                norm=norm,
                linewidths=gap_lw,
                alpha=gap_alpha,
                linestyles=gap_linestyle,
                zorder=28,
            )
            glc.set_array(gvals)
            ax.add_collection(glc)

        # solid segments (colored by p)
        if len(all_segments):
            segs = np.concatenate(all_segments, axis=0)
            vals = np.concatenate(all_values, axis=0)
            lc = LineCollection(
                segs,
                cmap=cmap,
                norm=norm,
                linewidths=seg_lw,
                alpha=seg_alpha,
                zorder=30
            )
            lc.set_array(vals)
            ax.add_collection(lc)

        ax.set_aspect("equal", adjustable="box")

        # REMOVED: treatment names
        # ax.set_title(nice[t], loc="left", fontsize=11)

        hide_axes(ax)

        # NEW: colored box around each subplot (seaborn colorblind, same order)
        border_color = treat_border_colors[i]
        rect = plt.Rectangle(
            (0, 0), 1, 1,
            transform=ax.transAxes,
            fill=False,
            edgecolor=border_color,
            linewidth=2.2,
            zorder=99,
            clip_on=False
        )
        ax.add_patch(rect)

    # off unused axes
    for j in range(len(treat_levels), len(axes)):
        axes[j].axis("off")

    # --- add compass + scale bar to the RIGHTMOST SUBPLOT (bottom-right corner, equal pad) ---
    rightmost_idx = min(ncols - 1, max(len(treat_levels) - 1, 0))
    if axes[rightmost_idx].axison:
        _add_scale_and_compass_bottom_right_equal_pad(axes[rightmost_idx], corner_pad_px=22)

    # --- thin colorbar on the far right ---
    fig.subplots_adjust(
        left=0.01,
        right=0.91,
        top=0.99,
        bottom=0.06,
        wspace=wspace,
        hspace=hspace
    )

    cax_left = 0.91 + cbar_pad
    cax = fig.add_axes([cax_left, 0.20, cbar_width, 0.62])

    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label("Directed-component probability", fontsize=cbar_labelsize)
    cbar.ax.tick_params(labelsize=cbar_ticksize, width=0.6, length=2.0)
    cbar.outline.set_linewidth(0.6)

    # legend (+ optional gap entry)
    legend_handles = [
        Patch(facecolor=lake_facecolor, edgecolor=lake_facecolor, alpha=lake_alpha, label="Lake"),
        Line2D([0], [0], marker="*", color="none", markerfacecolor="white",
               markeredgecolor="black", markersize=11, label="Roost"),
        Line2D([0], [0], marker="X", color="none", markerfacecolor="white",
               markeredgecolor="black", markersize=8, label="Translocation"),
    ]
    if has_gaps and d[gap_col].sum() > 0:
        legend_handles.append(Line2D([0, 1], [0, 0], color="black", lw=gap_lw,
                                     linestyle=gap_linestyle, alpha=gap_alpha, label="Signal Dropout"))

    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=len(legend_handles),
        frameon=True,
        framealpha=0.9,
        bbox_to_anchor=(0.46, 0.10),
        borderpad=1.2,
        labelspacing=0.9,
        columnspacing=1.6,
        handletextpad=0.8,
        handlelength=2.0,
    )

    fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.01, dpi=300)
    fig.savefig(out_svg, bbox_inches="tight", pad_inches=0.01, dpi=300)
    plt.show()

    return fig, axes


# ------------------------------------------------------------
# RUN
# ------------------------------------------------------------
fig, axes = plot_treatment_panels_tracks_colored_by_pgoal(
    tracks_df=track_w_treat,
    p_col="p_goal_post_mean",
    df_prob=df_track,
    prob_tag_col="tag",
    prob_step_col="_step",
    prob_p_col="p_goal_post_mean",
    gap_col="gap_before",
    gap_linestyle="--",
    gap_alpha=0.80,
    seg_lw=1.0,
    wspace=0.10,
    hspace=0.10,
    fig_w=8.0,
    fig_h=6
)

/tmp/ipykernel_3898688/3470568040.py:432: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(d[treat_col]):
/home/jake/.venvs/unwrap-env/lib/python3.12/site-packages/contextily/tile.py:298: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tile_urls = [provider.build_url(x=tile.x, y=tile.y, z=tile.z) for tile in tiles]


/home/jake/.venvs/unwrap-env/lib/python3.12/site-packages/contextily/tile.py:298: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tile_urls = [provider.build_url(x=tile.x, y=tile.y, z=tile.z) for tile in tiles]


/home/jake/.venvs/unwrap-env/lib/python3.12/site-packages/contextily/tile.py:298: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tile_urls = [provider.build_url(x=tile.x, y=tile.y, z=tile.z) for tile in tiles]


/home/jake/.venvs/unwrap-env/lib/python3.12/site-packages/contextily/tile.py:298: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tile_urls = [provider.build_url(x=tile.x, y=tile.y, z=tile.z) for tile in tiles]


In [ ]:
# ------------------------------------------------------------
# SCALE BAR + COMPASS (BLACK/WHITE) + BOTTOM-RIGHT PLACEMENT
# (UPDATED: bigger scale/compass text for smaller figure)
# ------------------------------------------------------------
def _add_scale_and_compass_bottom_right_equal_pad(ax, corner_pad_px=14):
    """
    Adds a black/white scale bar + compass to the bottom-right of `ax`
    with equal padding to right & bottom borders (in pixels).
    Compass is centered above the scale bar.
    """
    fig = ax.figure
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    pad_pt = corner_pad_px * 72.0 / fig.dpi
    seg_h_pt = 8
    scale_text_breathe_pt = 5  # keep label close to bar

    # scale length from axis width (meters; EPSG:3857)
    x0, x1 = ax.get_xlim()
    scale_len_m = _nice_scale_length_m(abs(x1 - x0))

    # BIGGER than before
    label_kwargs = dict(
        color="#000000", fontsize=10, ha="center", va="top",
        fontname="Arial", fontstretch="normal"
    )

    def _draw_scale(offset_y_pt):
        return add_scale_bar_dual(
            ax,
            total_length_m=scale_len_m,
            n_segments=2,
            segment_colors=("#ffffff", "#000000"),
            segment_height_pt=seg_h_pt,
            edgecolor="#000000",
            edgewidth=0.8,
            ax_xy=(1.0, 0.0),
            offset_pt=(-pad_pt, offset_y_pt),
            align="right",
            direction="left",
            label=True,
            label_above=False,
            pad_text_pt=scale_text_breathe_pt,
            label_kwargs=label_kwargs,
            return_info=True
        )

    # initial guess; then correct so label bottom padding matches corner_pad_px
    offset_y_pt = pad_pt + (seg_h_pt / 2.0) + scale_text_breathe_pt + 4.0
    sb = _draw_scale(offset_y_pt)
    fig.canvas.draw()

    txt = sb["text"]
    if txt is not None:
        ax_bbox = ax.get_window_extent(renderer=renderer)
        txt_bbox = txt.get_window_extent(renderer=renderer)
        current_bottom_pad_px = txt_bbox.y0 - ax_bbox.y0
        dy_px = corner_pad_px - current_bottom_pad_px
        dy_pt = dy_px * 72.0 / fig.dpi

        if abs(dy_pt) > 0.25:
            for p in sb["patches"]:
                p.remove()
            txt.remove()
            sb = _draw_scale(offset_y_pt + dy_pt)
            fig.canvas.draw()

    # compass centered above scale bar
    (cx_data, y_anchor_data) = sb["center_data"]
    cx_disp, cy_disp = ax.transData.transform((cx_data, y_anchor_data))

    compass_gap_pt = (seg_h_pt / 2.0)
    cy_disp_up = cy_disp + compass_gap_pt * fig.dpi / 72.0
    cx_axes, cy_axes = ax.transAxes.inverted().transform((cx_disp, cy_disp_up))

    add_compass(
        ax,
        ax_xy=(cx_axes, cy_axes),
        offset_pt=(0, 0),
        arrow_len_pt=18,
        head_width_pt=8,
        head_length_pt=9,
        tail_width_pt=1.8,
        color="#000000",
        edgecolor="#000000",
        edgewidth=0.9,
        draw_N=True,
        N_fontsize=10,   # BIGGER
        N_offset_pt=(0, 2),
    )


# ------------------------------------------------------------
# MAIN PLOT (COMPACT / PAPER-READY)
#   - no legend
#   - horizontal colorbar on bottom
#   - UPDATED: smaller figure + bigger text
# ------------------------------------------------------------
def plot_treatment_panels_tracks_colored_by_pgoal(
    tracks_df,
    *,
    tag_col="tag",
    lat_col="lat",
    lon_col="lon",
    treat_col="treatment",
    treat_name_col="treatmentName",
    p_col="p_goal_post_mean",
    gap_col="gap_before",
    df_prob=None,
    prob_tag_col="tag",
    prob_step_col="_step",
    prob_p_col="p_goal_post_mean",
    # style
    basemap_source=cx.providers.CartoDB.Positron,
    basemap_alpha=1.0,
    lake_facecolor="tab:cyan",
    lake_alpha=0.25,
    lake_edge_lw=0.8,
    seg_lw=0.8,
    seg_alpha=0.90,
    gap_lw=None,
    gap_alpha=0.75,
    gap_linestyle="--",
    # subsampling
    max_traj_per_panel=None,
    seed=123,
    out_pdf="../figures/bats/treatment_panels_tracks_pgoal_compact_bottomcbar.pdf",
    out_svg="../figures/bats/treatment_panels_tracks_pgoal_compact_bottomcbar.svg",
    # layout tweaks (SMALLER than before, still ~4:3)
    fig_w=5.6,
    fig_h=4.2,
    wspace=0.01,
    hspace=0.02,
    # bottom colorbar (horizontal) -- bigger text/thicker bar
    cbar_height=0.034,
    cbar_bottom=0.060,
    cbar_left=0.20,
    cbar_width=0.60,
    cbar_ticksize=8,      # BIGGER
    cbar_labelsize=9,     # BIGGER
):
    d = tracks_df.copy()
    for c in [tag_col, lat_col, lon_col, treat_col]:
        if c not in d.columns:
            raise KeyError(f"tracks_df missing required column '{c}'")

    d[lat_col] = pd.to_numeric(d[lat_col], errors="coerce")
    d[lon_col] = pd.to_numeric(d[lon_col], errors="coerce")
    d = d[np.isfinite(d[lat_col]) & np.isfinite(d[lon_col])].copy()
    d = d.dropna(subset=[tag_col, treat_col]).copy()
    d = _ensure_step_index(d, tag_col=tag_col)

    # attach p if needed
    if p_col not in d.columns:
        if df_prob is None:
            raise KeyError(
                f"tracks_df missing '{p_col}'. Either compute it on tracks_df "
                f"or pass df_prob (with columns {prob_tag_col}, {prob_step_col}, {prob_p_col})."
            )
        if prob_step_col not in df_prob.columns:
            raise KeyError(f"df_prob missing '{prob_step_col}'. Make sure it has the per-tag step index.")
        tmp = df_prob[[prob_tag_col, prob_step_col, prob_p_col]].copy()
        tmp = tmp.rename(columns={prob_tag_col: tag_col, prob_step_col: "_step", prob_p_col: p_col})
        d = d.merge(tmp, on=[tag_col, "_step"], how="left", validate="many_to_one")

    d[p_col] = pd.to_numeric(d[p_col], errors="coerce")
    d = d[np.isfinite(d[p_col])].copy()
    d[p_col] = d[p_col].clip(0.0, 1.0)

    has_gaps = gap_col in d.columns
    if has_gaps:
        d[gap_col] = pd.to_numeric(d[gap_col], errors="coerce").fillna(0).astype(int)

    # treatment order + labels
    if pd.api.types.is_categorical_dtype(d[treat_col]):
        treat_levels = list(d[treat_col].cat.categories)
    else:
        treat_levels = sorted(d[treat_col].dropna().unique().tolist())
    if len(treat_levels) == 0:
        raise ValueError("No treatment levels found.")

    if treat_name_col in d.columns:
        name_map = d.groupby(treat_col, observed=False)[treat_name_col].first().to_dict()
        _ = {t: _format_treatment_name(name_map.get(t, t)) for t in treat_levels}

    # colored subplot borders (seaborn colorblind, stable order)
    treat_border_colors = sns.color_palette("colorblind", n_colors=len(treat_levels))

    # project coords to WebMercator
    transformer = Transformer.from_crs("EPSG:4326", f"EPSG:{WEBMERC_EPSG}", always_xy=True)
    xs, ys = transformer.transform(d[lon_col].to_numpy(), d[lat_col].to_numpy())
    d["_x"] = xs
    d["_y"] = ys

    # lake + points
    lake_gdf = gpd.read_file(LAKE_GEOJSON).to_crs(epsg=WEBMERC_EPSG)
    pts_gdf = _make_points_gdf()

    # common bounds
    layers_bounds = []
    if len(lake_gdf):
        layers_bounds.append(lake_gdf.total_bounds)
    if len(pts_gdf):
        layers_bounds.append(pts_gdf.total_bounds)
    if len(d):
        layers_bounds.append(np.array([d["_x"].min(), d["_y"].min(), d["_x"].max(), d["_y"].max()]))

    minx = min(b[0] for b in layers_bounds)
    miny = min(b[1] for b in layers_bounds)
    maxx = max(b[2] for b in layers_bounds)
    maxy = max(b[3] for b in layers_bounds)

    pad_x = (maxx - minx) * 0.05 if maxx > minx else 1000
    pad_y = (maxy - miny) * 0.05 if maxy > miny else 1000
    xlim = (minx - pad_x, maxx + pad_x)
    ylim = (miny - pad_y, maxy + pad_y)

    # layout: 1 row, 4 cols
    ncols = 4
    nrows = 1

    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), dpi=DPI)
    axes = np.atleast_1d(axes).ravel()

    cmap = mpl.colormaps.get_cmap("viridis")
    norm = mcolors.Normalize(vmin=0.0, vmax=1.0)

    if gap_lw is None:
        gap_lw = max(0.55, seg_lw * 0.9)

    rng = np.random.default_rng(seed)

    for i, t in enumerate(treat_levels):
        ax = axes[i]
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)

        # basemap
        try:
            cx.add_basemap(ax, source=basemap_source, attribution=False, alpha=basemap_alpha)
        except Exception as e:
            print("Basemap failed (continuing without tiles):", e)

        # lake
        if len(lake_gdf):
            lake_gdf.plot(
                ax=ax,
                facecolor=lake_facecolor,
                edgecolor=lake_facecolor,
                alpha=lake_alpha,
                linewidth=lake_edge_lw,
                zorder=12
            )
            lake_gdf.boundary.plot(
                ax=ax,
                color=lake_facecolor,
                linewidth=lake_edge_lw + 0.15,
                alpha=1.0,
                zorder=13
            )

        # roost + trans points
        ref = pts_gdf.loc[pts_gdf["site"] == "Roost"]
        ax.scatter(
            ref.geometry.x, ref.geometry.y,
            marker="*", s=105,
            facecolors="white", edgecolors="black",
            linewidths=0.7, zorder=50
        )

        trans = pts_gdf.loc[pts_gdf["site"] == "Translocation"]
        ax.scatter(
            trans.geometry.x, trans.geometry.y,
            marker="X", s=62,
            facecolors="white", edgecolors="black",
            linewidths=0.7, zorder=50
        )

        # tracks for this treatment
        dd = d.loc[d[treat_col] == t].copy()
        tags = sorted(dd[tag_col].unique().tolist())

        if (max_traj_per_panel is not None) and (len(tags) > max_traj_per_panel):
            tags = rng.choice(tags, size=max_traj_per_panel, replace=False).tolist()
            dd = dd.loc[dd[tag_col].isin(tags)].copy()

        all_segments, all_values = [], []
        gap_segments, gap_values = [], []

        order_col = _pick_order_col(dd)
        for tg, g in dd.groupby(tag_col, sort=False):
            g = g.sort_values(order_col) if order_col else g.sort_values("_step")

            x = g["_x"].to_numpy(float)
            y = g["_y"].to_numpy(float)
            p = g[p_col].to_numpy(float)

            if len(x) < 2:
                continue

            pts_xy = np.column_stack([x, y])
            segs = np.stack([pts_xy[:-1], pts_xy[1:]], axis=1)
            pseg = 0.5 * (p[:-1] + p[1:])

            if has_gaps and gap_col in g.columns:
                gb = g[gap_col].to_numpy(int)
                cross_gap = (gb[1:] == 1)

                if np.any(~cross_gap):
                    all_segments.append(segs[~cross_gap])
                    all_values.append(pseg[~cross_gap])

                if np.any(cross_gap):
                    gap_segments.append(segs[cross_gap])
                    gap_values.append(pseg[cross_gap])
            else:
                all_segments.append(segs)
                all_values.append(pseg)

        # dashed gap connectors (draw first)
        if len(gap_segments):
            gsegs = np.concatenate(gap_segments, axis=0)
            gvals = np.concatenate(gap_values, axis=0)
            glc = LineCollection(
                gsegs,
                cmap=cmap,
                norm=norm,
                linewidths=gap_lw,
                alpha=gap_alpha,
                linestyles=gap_linestyle,
                zorder=28,
            )
            glc.set_array(gvals)
            ax.add_collection(glc)

        # solid segments (colored by p)
        if len(all_segments):
            segs = np.concatenate(all_segments, axis=0)
            vals = np.concatenate(all_values, axis=0)
            lc = LineCollection(
                segs,
                cmap=cmap,
                norm=norm,
                linewidths=seg_lw,
                alpha=seg_alpha,
                zorder=30
            )
            lc.set_array(vals)
            ax.add_collection(lc)

        ax.set_aspect("equal", adjustable="box")
        hide_axes(ax)

        # colored border per treatment
        rect = plt.Rectangle(
            (0, 0), 1, 1,
            transform=ax.transAxes,
            fill=False,
            edgecolor=treat_border_colors[i],
            linewidth=1.5,
            zorder=99,
            clip_on=False
        )
        ax.add_patch(rect)

    # off unused axes
    for j in range(len(treat_levels), len(axes)):
        axes[j].axis("off")

    # scale bar + compass in rightmost active subplot
    rightmost_idx = min(ncols - 1, max(len(treat_levels) - 1, 0))
    if axes[rightmost_idx].axison:
        _add_scale_and_compass_bottom_right_equal_pad(axes[rightmost_idx], corner_pad_px=14)

    # compact layout + horizontal colorbar on bottom
    fig.subplots_adjust(
        left=0.01,
        right=0.99,
        top=0.995,
        bottom=max(0.13, cbar_bottom + cbar_height + 0.045),  # reserve room for larger text
        wspace=wspace,
        hspace=hspace
    )

    cax = fig.add_axes([cbar_left, cbar_bottom, cbar_width, cbar_height])

    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cbar.set_label("Directed-component probability", fontsize=cbar_labelsize, labelpad=2)
    cbar.ax.tick_params(labelsize=cbar_ticksize, width=0.6, length=2.2, pad=1)
    cbar.outline.set_linewidth(0.6)

    # no legend (paper-ready compact version)

    if out_pdf:
        os.makedirs(os.path.dirname(out_pdf), exist_ok=True)
        fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.01, dpi=DPI)
    if out_svg:
        os.makedirs(os.path.dirname(out_svg), exist_ok=True)
        fig.savefig(out_svg, bbox_inches="tight", pad_inches=0.01, dpi=DPI)

    plt.show()
    return fig, axes


# ------------------------------------------------------------
# RUN (SMALLER FIGURE + BIGGER FONT)
# ------------------------------------------------------------
fig, axes = plot_treatment_panels_tracks_colored_by_pgoal(
    tracks_df=track_w_treat,
    p_col="p_goal_post_mean",
    df_prob=df_track,
    prob_tag_col="tag",
    prob_step_col="_step",
    prob_p_col="p_goal_post_mean",
    gap_col="gap_before",
    gap_linestyle="--",
    gap_alpha=0.75,
    seg_lw=0.8,
    wspace=0.03,
    hspace=0.02,
    fig_w=5.6,   # smaller
    fig_h=4.2,   # smaller
    cbar_ticksize=8,   # bigger text
    cbar_labelsize=9,  # bigger text
    out_pdf="../figures/bats/treatment_panels_tracks_pgoal_small_bigfont.pdf",
    out_svg="../figures/bats/treatment_panels_tracks_pgoal_small_bigfont.svg",
)

/tmp/ipykernel_3898688/2553577505.py:176: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(d[treat_col]):
/home/jake/.venvs/unwrap-env/lib/python3.12/site-packages/contextily/tile.py:298: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tile_urls = [provider.build_url(x=tile.x, y=tile.y, z=tile.z) for tile in tiles]


/home/jake/.venvs/unwrap-env/lib/python3.12/site-packages/contextily/tile.py:298: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tile_urls = [provider.build_url(x=tile.x, y=tile.y, z=tile.z) for tile in tiles]


/home/jake/.venvs/unwrap-env/lib/python3.12/site-packages/contextily/tile.py:298: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tile_urls = [provider.build_url(x=tile.x, y=tile.y, z=tile.z) for tile in tiles]


/home/jake/.venvs/unwrap-env/lib/python3.12/site-packages/contextily/tile.py:298: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tile_urls = [provider.build_url(x=tile.x, y=tile.y, z=tile.z) for tile in tiles]


In [ ]:
# ------------------------------------------------------------
# OPTIONAL HELPERS FOR SELECTIVE RASTERIZATION (track layers only)
#   - keep lake/markers/text/colorbar/scale/compass vector where possible
# ------------------------------------------------------------
import os
from matplotlib.collections import LineCollection

def _set_rasterized(artists, rasterized=True):
    for a in artists:
        try:
            a.set_rasterized(rasterized)
        except Exception:
            pass

def _style_trajectory_artists(artists):
    """
    Make rasterized/vector track collections look smoother.
    """
    for a in artists:
        for method_name, val in [
            ("set_antialiased", True),
            ("set_antialiaseds", [True]),
            ("set_capstyle", "round"),
            ("set_joinstyle", "round"),
            ("set_snap", False),
        ]:
            if hasattr(a, method_name):
                try:
                    getattr(a, method_name)(val)
                except Exception:
                    pass

def _maybe_mkdir_for_file(path):
    if path:
        d = os.path.dirname(path)
        if d:
            os.makedirs(d, exist_ok=True)


# ------------------------------------------------------------
# SCALE BAR + COMPASS (BLACK/WHITE) + BOTTOM-RIGHT PLACEMENT
# (UPDATED: bigger scale/compass text for smaller figure)
# ------------------------------------------------------------
def _add_scale_and_compass_bottom_right_equal_pad(ax, corner_pad_px=14):
    """
    Adds a black/white scale bar + compass to the bottom-right of `ax`
    with equal padding to right & bottom borders (in pixels).
    Compass is centered above the scale bar.
    """
    fig = ax.figure
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    pad_pt = corner_pad_px * 72.0 / fig.dpi
    seg_h_pt = 8
    scale_text_breathe_pt = 5  # keep label close to bar

    # scale length from axis width (meters; EPSG:3857)
    x0, x1 = ax.get_xlim()
    scale_len_m = _nice_scale_length_m(abs(x1 - x0))

    # BIGGER than before
    label_kwargs = dict(
        color="#000000", fontsize=10, ha="center", va="top",
        fontname="Arial", fontstretch="normal"
    )

    def _draw_scale(offset_y_pt):
        return add_scale_bar_dual(
            ax,
            total_length_m=scale_len_m,
            n_segments=2,
            segment_colors=("#ffffff", "#000000"),
            segment_height_pt=seg_h_pt,
            edgecolor="#000000",
            edgewidth=0.8,
            ax_xy=(1.0, 0.0),
            offset_pt=(-pad_pt, offset_y_pt),
            align="right",
            direction="left",
            label=True,
            label_above=False,
            pad_text_pt=scale_text_breathe_pt,
            label_kwargs=label_kwargs,
            return_info=True
        )

    # initial guess; then correct so label bottom padding matches corner_pad_px
    offset_y_pt = pad_pt + (seg_h_pt / 2.0) + scale_text_breathe_pt + 4.0
    sb = _draw_scale(offset_y_pt)
    fig.canvas.draw()

    txt = sb["text"]
    if txt is not None:
        ax_bbox = ax.get_window_extent(renderer=renderer)
        txt_bbox = txt.get_window_extent(renderer=renderer)
        current_bottom_pad_px = txt_bbox.y0 - ax_bbox.y0
        dy_px = corner_pad_px - current_bottom_pad_px
        dy_pt = dy_px * 72.0 / fig.dpi

        if abs(dy_pt) > 0.25:
            for p in sb["patches"]:
                p.remove()
            txt.remove()
            sb = _draw_scale(offset_y_pt + dy_pt)
            fig.canvas.draw()

    # compass centered above scale bar
    (cx_data, y_anchor_data) = sb["center_data"]
    cx_disp, cy_disp = ax.transData.transform((cx_data, y_anchor_data))

    compass_gap_pt = (seg_h_pt / 2.0)
    cy_disp_up = cy_disp + compass_gap_pt * fig.dpi / 72.0
    cx_axes, cy_axes = ax.transAxes.inverted().transform((cx_disp, cy_disp_up))

    add_compass(
        ax,
        ax_xy=(cx_axes, cy_axes),
        offset_pt=(0, 0),
        arrow_len_pt=18,
        head_width_pt=8,
        head_length_pt=9,
        tail_width_pt=1.8,
        color="#000000",
        edgecolor="#000000",
        edgewidth=0.9,
        draw_N=True,
        N_fontsize=10,   # BIGGER
        N_offset_pt=(0, 2),
    )


# ------------------------------------------------------------
# MAIN PLOT (COMPACT / PAPER-READY)
#   - no legend
#   - horizontal colorbar on bottom
#   - UPDATED: smaller figure + bigger text
#   - UPDATED: selective rasterization of heavy track layers only
# ------------------------------------------------------------
def plot_treatment_panels_tracks_colored_by_pgoal(
    tracks_df,
    *,
    tag_col="tag",
    lat_col="lat",
    lon_col="lon",
    treat_col="treatment",
    treat_name_col="treatmentName",
    p_col="p_goal_post_mean",
    gap_col="gap_before",
    df_prob=None,
    prob_tag_col="tag",
    prob_step_col="_step",
    prob_p_col="p_goal_post_mean",
    # style
    basemap_source=cx.providers.CartoDB.Positron,
    basemap_alpha=1.0,
    lake_facecolor="tab:cyan",
    lake_alpha=0.25,
    lake_edge_lw=0.8,
    seg_lw=0.8,
    seg_alpha=0.90,
    gap_lw=None,
    gap_alpha=0.75,
    gap_linestyle="--",
    # subsampling
    max_traj_per_panel=None,
    seed=123,
    out_pdf="../figures/bats/treatment_panels_tracks_pgoal_compact_bottomcbar.pdf",
    out_svg="../figures/bats/treatment_panels_tracks_pgoal_compact_bottomcbar.svg",
    # layout tweaks (SMALLER than before, still ~4:3)
    fig_w=5.6,
    fig_h=4.2,
    wspace=0.01,
    hspace=0.02,
    # bottom colorbar (horizontal) -- bigger text/thicker bar
    cbar_height=0.034,
    cbar_bottom=0.060,
    cbar_left=0.20,
    cbar_width=0.60,
    cbar_ticksize=8,      # BIGGER
    cbar_labelsize=9,     # BIGGER
    # selective rasterization controls (NEW)
    rasterize_tracks_pdf=True,     # make PDF lighter/faster
    rasterize_tracks_svg=False,    # keep SVG tracks vector by default
    pdf_raster_dpi=300,            # dpi for rasterized track layers in PDF
    svg_raster_dpi=600,            # used only if rasterize_tracks_svg=True
):
    d = tracks_df.copy()
    for c in [tag_col, lat_col, lon_col, treat_col]:
        if c not in d.columns:
            raise KeyError(f"tracks_df missing required column '{c}'")

    d[lat_col] = pd.to_numeric(d[lat_col], errors="coerce")
    d[lon_col] = pd.to_numeric(d[lon_col], errors="coerce")
    d = d[np.isfinite(d[lat_col]) & np.isfinite(d[lon_col])].copy()
    d = d.dropna(subset=[tag_col, treat_col]).copy()
    d = _ensure_step_index(d, tag_col=tag_col)

    # attach p if needed
    if p_col not in d.columns:
        if df_prob is None:
            raise KeyError(
                f"tracks_df missing '{p_col}'. Either compute it on tracks_df "
                f"or pass df_prob (with columns {prob_tag_col}, {prob_step_col}, {prob_p_col})."
            )
        if prob_step_col not in df_prob.columns:
            raise KeyError(f"df_prob missing '{prob_step_col}'. Make sure it has the per-tag step index.")
        tmp = df_prob[[prob_tag_col, prob_step_col, prob_p_col]].copy()
        tmp = tmp.rename(columns={prob_tag_col: tag_col, prob_step_col: "_step", prob_p_col: p_col})
        d = d.merge(tmp, on=[tag_col, "_step"], how="left", validate="many_to_one")

    d[p_col] = pd.to_numeric(d[p_col], errors="coerce")
    d = d[np.isfinite(d[p_col])].copy()
    d[p_col] = d[p_col].clip(0.0, 1.0)

    has_gaps = gap_col in d.columns
    if has_gaps:
        d[gap_col] = pd.to_numeric(d[gap_col], errors="coerce").fillna(0).astype(int)

    # treatment order + labels
    if pd.api.types.is_categorical_dtype(d[treat_col]):
        treat_levels = list(d[treat_col].cat.categories)
    else:
        treat_levels = sorted(d[treat_col].dropna().unique().tolist())
    if len(treat_levels) == 0:
        raise ValueError("No treatment levels found.")

    if treat_name_col in d.columns:
        name_map = d.groupby(treat_col, observed=False)[treat_name_col].first().to_dict()
        _ = {t: _format_treatment_name(name_map.get(t, t)) for t in treat_levels}

    # colored subplot borders (seaborn colorblind, stable order)
    treat_border_colors = sns.color_palette("colorblind", n_colors=len(treat_levels))

    # project coords to WebMercator
    transformer = Transformer.from_crs("EPSG:4326", f"EPSG:{WEBMERC_EPSG}", always_xy=True)
    xs, ys = transformer.transform(d[lon_col].to_numpy(), d[lat_col].to_numpy())
    d["_x"] = xs
    d["_y"] = ys

    # lake + points
    lake_gdf = gpd.read_file(LAKE_GEOJSON).to_crs(epsg=WEBMERC_EPSG)
    pts_gdf = _make_points_gdf()

    # common bounds
    layers_bounds = []
    if len(lake_gdf):
        layers_bounds.append(lake_gdf.total_bounds)
    if len(pts_gdf):
        layers_bounds.append(pts_gdf.total_bounds)
    if len(d):
        layers_bounds.append(np.array([d["_x"].min(), d["_y"].min(), d["_x"].max(), d["_y"].max()]))

    minx = min(b[0] for b in layers_bounds)
    miny = min(b[1] for b in layers_bounds)
    maxx = max(b[2] for b in layers_bounds)
    maxy = max(b[3] for b in layers_bounds)

    pad_x = (maxx - minx) * 0.05 if maxx > minx else 1000
    pad_y = (maxy - miny) * 0.05 if maxy > miny else 1000
    xlim = (minx - pad_x, maxx + pad_x)
    ylim = (miny - pad_y, maxy + pad_y)

    # layout: 1 row, 4 cols
    ncols = 4
    nrows = 1

    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), dpi=DPI)
    axes = np.atleast_1d(axes).ravel()

    cmap = mpl.colormaps.get_cmap("viridis")
    norm = mcolors.Normalize(vmin=0.0, vmax=1.0)

    if gap_lw is None:
        gap_lw = max(0.55, seg_lw * 0.9)

    rng = np.random.default_rng(seed)

    # collect only heavy track artists (solid + dashed collections)
    track_artists = []

    for i, t in enumerate(treat_levels):
        ax = axes[i]
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)

        # basemap (already raster by nature)
        try:
            cx.add_basemap(ax, source=basemap_source, attribution=False, alpha=basemap_alpha)
        except Exception as e:
            print("Basemap failed (continuing without tiles):", e)

        # lake (keep vector)
        if len(lake_gdf):
            lake_gdf.plot(
                ax=ax,
                facecolor=lake_facecolor,
                edgecolor=lake_facecolor,
                alpha=lake_alpha,
                linewidth=lake_edge_lw,
                zorder=12
            )
            lake_gdf.boundary.plot(
                ax=ax,
                color=lake_facecolor,
                linewidth=lake_edge_lw + 0.15,
                alpha=1.0,
                zorder=13
            )

        # roost + trans points (keep vector)
        ref = pts_gdf.loc[pts_gdf["site"] == "Roost"]
        ax.scatter(
            ref.geometry.x, ref.geometry.y,
            marker="*", s=105,
            facecolors="white", edgecolors="black",
            linewidths=0.7, zorder=50
        )

        trans = pts_gdf.loc[pts_gdf["site"] == "Translocation"]
        ax.scatter(
            trans.geometry.x, trans.geometry.y,
            marker="X", s=62,
            facecolors="white", edgecolors="black",
            linewidths=0.7, zorder=50
        )

        # tracks for this treatment
        dd = d.loc[d[treat_col] == t].copy()
        tags = sorted(dd[tag_col].unique().tolist())

        if (max_traj_per_panel is not None) and (len(tags) > max_traj_per_panel):
            tags = rng.choice(tags, size=max_traj_per_panel, replace=False).tolist()
            dd = dd.loc[dd[tag_col].isin(tags)].copy()

        all_segments, all_values = [], []
        gap_segments, gap_values = [], []

        order_col = _pick_order_col(dd)
        for tg, g in dd.groupby(tag_col, sort=False):
            g = g.sort_values(order_col) if order_col else g.sort_values("_step")

            x = g["_x"].to_numpy(float)
            y = g["_y"].to_numpy(float)
            p = g[p_col].to_numpy(float)

            if len(x) < 2:
                continue

            pts_xy = np.column_stack([x, y])
            segs = np.stack([pts_xy[:-1], pts_xy[1:]], axis=1)
            pseg = 0.5 * (p[:-1] + p[1:])

            if has_gaps and gap_col in g.columns:
                gb = g[gap_col].to_numpy(int)
                cross_gap = (gb[1:] == 1)

                if np.any(~cross_gap):
                    all_segments.append(segs[~cross_gap])
                    all_values.append(pseg[~cross_gap])

                if np.any(cross_gap):
                    gap_segments.append(segs[cross_gap])
                    gap_values.append(pseg[cross_gap])
            else:
                all_segments.append(segs)
                all_values.append(pseg)

        # dashed gap connectors (draw first) -> heavy layer candidate for rasterization
        if len(gap_segments):
            gsegs = np.concatenate(gap_segments, axis=0)
            gvals = np.concatenate(gap_values, axis=0)
            glc = LineCollection(
                gsegs,
                cmap=cmap,
                norm=norm,
                linewidths=gap_lw,
                alpha=gap_alpha,
                linestyles=gap_linestyle,
                zorder=28,
            )
            glc.set_array(gvals)
            _style_trajectory_artists([glc])
            ax.add_collection(glc)
            track_artists.append(glc)

        # solid segments (colored by p) -> heavy layer candidate for rasterization
        if len(all_segments):
            segs = np.concatenate(all_segments, axis=0)
            vals = np.concatenate(all_values, axis=0)
            lc = LineCollection(
                segs,
                cmap=cmap,
                norm=norm,
                linewidths=seg_lw,
                alpha=seg_alpha,
                zorder=30
            )
            lc.set_array(vals)
            _style_trajectory_artists([lc])
            ax.add_collection(lc)
            track_artists.append(lc)

        ax.set_aspect("equal", adjustable="box")
        hide_axes(ax)

        # colored border per treatment (keep vector)
        rect = plt.Rectangle(
            (0, 0), 1, 1,
            transform=ax.transAxes,
            fill=False,
            edgecolor=treat_border_colors[i],
            linewidth=1.5,
            zorder=99,
            clip_on=False
        )
        ax.add_patch(rect)

    # off unused axes
    for j in range(len(treat_levels), len(axes)):
        axes[j].axis("off")

    # scale bar + compass in rightmost active subplot (keep vector)
    rightmost_idx = min(ncols - 1, max(len(treat_levels) - 1, 0))
    if axes[rightmost_idx].axison:
        _add_scale_and_compass_bottom_right_equal_pad(axes[rightmost_idx], corner_pad_px=14)

    # compact layout + horizontal colorbar on bottom
    fig.subplots_adjust(
        left=0.01,
        right=0.99,
        top=0.995,
        bottom=max(0.13, cbar_bottom + cbar_height + 0.045),  # reserve room for larger text
        wspace=wspace,
        hspace=hspace
    )

    cax = fig.add_axes([cbar_left, cbar_bottom, cbar_width, cbar_height])

    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cbar.set_label("Directed-component probability", fontsize=cbar_labelsize, labelpad=2)
    cbar.ax.tick_params(labelsize=cbar_ticksize, width=0.6, length=2.2, pad=1)
    cbar.outline.set_linewidth(0.6)

    # no legend (paper-ready compact version)

    # save with selective rasterization (tracks only)
    if out_pdf:
        _maybe_mkdir_for_file(out_pdf)
        _set_rasterized(track_artists, rasterize_tracks_pdf)
        fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.01, dpi=pdf_raster_dpi)

    if out_svg:
        _maybe_mkdir_for_file(out_svg)
        _set_rasterized(track_artists, rasterize_tracks_svg)
        fig.savefig(out_svg, bbox_inches="tight", pad_inches=0.01, dpi=svg_raster_dpi)

    plt.show()
    return fig, axes


# ------------------------------------------------------------
# RUN (SMALLER FIGURE + BIGGER FONT)
# ------------------------------------------------------------
fig, axes = plot_treatment_panels_tracks_colored_by_pgoal(
    tracks_df=track_w_treat,
    p_col="p_goal_post_mean",
    df_prob=df_track,
    prob_tag_col="tag",
    prob_step_col="_step",
    prob_p_col="p_goal_post_mean",
    gap_col="gap_before",
    gap_linestyle="--",
    gap_alpha=0.75,
    seg_lw=0.8,
    wspace=0.03,
    hspace=0.02,
    fig_w=5.6,
    fig_h=4.2,

    # keep fonts the same
    cbar_ticksize=8,
    cbar_labelsize=9,

    # smaller colorbar (half length)
    cbar_width=0.30,
    cbar_left=0.35,
    cbar_height=0.020,  # or 0.022

    out_pdf="../figures/bats/treatment_panels_tracks_pgoal_small_bigfont.pdf",
    out_svg="../figures/bats/treatment_panels_tracks_pgoal_small_bigfont.svg",

    rasterize_tracks_pdf=True,
    rasterize_tracks_svg=True,
)

/tmp/ipykernel_3898688/1385243575.py:221: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(d[treat_col]):
/home/jake/.venvs/unwrap-env/lib/python3.12/site-packages/contextily/tile.py:298: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tile_urls = [provider.build_url(x=tile.x, y=tile.y, z=tile.z) for tile in tiles]


/home/jake/.venvs/unwrap-env/lib/python3.12/site-packages/contextily/tile.py:298: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tile_urls = [provider.build_url(x=tile.x, y=tile.y, z=tile.z) for tile in tiles]


/home/jake/.venvs/unwrap-env/lib/python3.12/site-packages/contextily/tile.py:298: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tile_urls = [provider.build_url(x=tile.x, y=tile.y, z=tile.z) for tile in tiles]


/home/jake/.venvs/unwrap-env/lib/python3.12/site-packages/contextily/tile.py:298: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tile_urls = [provider.build_url(x=tile.x, y=tile.y, z=tile.z) for tile in tiles]


Population-level $p_{goal}$ as a function of distance from the flight's endpoint — lower early, higher
as the animal approaches its goal.


In [ ]:
# ============================================================
# FULL KERNEL-SMOOTHED PIPELINE (NO BINNING) — POPULATION CURVE
# UPDATED WITH:
#   - Per-tag truncation to observed x-range
#   - Optional per-tag truncation by kernel support (den_frac)
#   - Population post-smoothing across x (weighted by #tags contributing)
#   - Masking where too few tags contribute to avoid jagged/unstable tails
#   - Seaborn colorblind palette (match other plots)
#   - SVG save (same basename as PDF) + same style settings
#
# Pipeline:
#   - Compute posterior responsibilities per step: r_{s,i} = P(goal | y_i, draw s)
#   - For each treatment:
#       (1) For each tag: kernel smooth each posterior draw trajectory r_{s,i} vs x
#           and TRUNCATE it (mask to NaN) outside tag's data range and/or low support
#       (2) For each draw: average the truncated tag curves equally across tags
#           at each x (NaN-safe; only tags that exist at that x contribute)
#       (3) OPTIONAL: post-smooth each draw's population curve across x (weighted)
#       (4) Summarize across draws: posterior mean + HDI bands (NaN-safe)
#
# ASSUMES you already have:
#   - df_track (EXACT row order used in the mixture fit)
#   - idata_mix_full with posterior deterministic "p_goal"
#     and either per-obs "mu","kappa" OR pop "mu0","eta_kappa0"
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# ----------------------------
# Style (match other plots)
# ----------------------------
DPI = 300
mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
})

# ============================================================
# USER CONTROLS
# ============================================================
STEP_LENGTH_M = 60.0
DIST_COL = "dist_to_dest_grad_m"

BANDWIDTH_M = 120.0
N_GRID = 2000
CHUNK_SIZE = 4000

# --- truncation controls ---
TRUNCATE_TO_DATA_RANGE = True     # hard truncate to each tag's observed x-range
TRUNCATE_BY_WEIGHT = True         # additionally truncate where kernel support is tiny
DEN_FRAC = 0.01                   # den < DEN_FRAC * den.max()  (per tag)

# --- stabilize population curve after truncation ---
POP_POST_SMOOTH = True            # second-stage smoothing on population curve along x
POP_SMOOTH_BW = 240.0             # in x units; often same as BANDWIDTH_M works well
MIN_TAGS_AT_X = 2                 # mask x where fewer than this many tags contribute

# ----------------------------
# Small utilities
# ----------------------------
def circular_modulo(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def sigmoid(x):
    x = np.asarray(x, float)
    return 1.0 / (1.0 + np.exp(-x))

def softplus(x):
    x = np.asarray(x, float)
    return np.where(x > 30, x, np.log1p(np.exp(x)))

def _stack_samples_first(da):
    """xarray DataArray -> np array with sample first: (S, ...)"""
    da2 = da.stack(sample=("chain", "draw"))
    other = [d for d in da2.dims if d != "sample"]
    da2 = da2.transpose("sample", *other)
    return da2.values.astype(float)

def _format_treatment_name(name: str) -> str:
    s = str(name).strip().lower()
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if "control" in parts and len(parts) > 1:
        parts = [p for p in parts if p != "control"]
    if not parts:
        parts = ["control"]
    parts = sorted(parts)
    parts = [p[:1].upper() + p[1:] for p in parts]
    return ", ".join(parts)

def hdi_1d(x, prob=0.95):
    """Minimum-width interval (HDI) for 1D samples (NaN-safe)."""
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    n = x.size
    if n == 0:
        return np.nan, np.nan
    x = np.sort(x)
    m = int(np.floor(prob * n))
    if m < 1:
        return np.nan, np.nan
    widths = x[m:] - x[: n - m]
    j = int(np.argmin(widths))
    return float(x[j]), float(x[j + m])

def hdi_along_samples(arr, prob=0.95):
    """HDI across axis=0 for arr shape (S, N) -> (N,2). NaN-safe."""
    arr = np.asarray(arr, float)
    S, N = arr.shape
    out = np.empty((N, 2), float)
    for i in range(N):
        out[i, 0], out[i, 1] = hdi_1d(arr[:, i], prob=prob)
    return out

def log_i0_stable(kappa):
    """Stable-ish log(I0(kappa)) with asymptotic fallback."""
    kappa = np.asarray(kappa, float)
    out = np.empty_like(kappa)
    big = kappa > 700
    small = ~big
    out[small] = np.log(np.i0(kappa[small]))
    out[big] = kappa[big] - 0.5 * np.log(2 * np.pi * kappa[big])
    return out

def log_vonmises_pdf(y, mu, kappa):
    """log f(y|mu,kappa) = kappa*cos(y-mu) - log(2π) - log(I0(kappa))"""
    y = np.asarray(y, float)
    mu = np.asarray(mu, float)
    kappa = np.asarray(kappa, float)
    return kappa * np.cos(y - mu) - np.log(2 * np.pi) - log_i0_stable(kappa)

# ============================================================
# FFT weighted smoothing utilities (fast, grid-uniform)
# ============================================================
def _next_pow_two(n: int) -> int:
    return 1 << (int(n) - 1).bit_length()

def _fft_convolve_same_1d(x, k):
    """1D convolution (same) using FFT. x,k are 1D arrays."""
    x = np.asarray(x, float)
    k = np.asarray(k, float)
    nx = x.size
    nk = k.size
    n = nx + nk - 1
    nfft = _next_pow_two(n)

    X = np.fft.rfft(x, nfft)
    K = np.fft.rfft(k, nfft)
    y = np.fft.irfft(X * K, nfft)[:n]

    start = (nk - 1) // 2
    return y[start:start + nx]

def _fft_convolve_same_rows(A, k):
    """
    Row-wise convolution (same) using FFT.
    A: (S,G), k: (K,)
    returns: (S,G)
    """
    A = np.asarray(A, float)
    k = np.asarray(k, float)
    S, G = A.shape
    K = k.size
    n = G + K - 1
    nfft = _next_pow_two(n)

    Af = np.fft.rfft(A, nfft, axis=1)
    kf = np.fft.rfft(k, nfft)
    y = np.fft.irfft(Af * kf[None, :], nfft, axis=1)[:, :n]

    start = (K - 1) // 2
    return y[:, start:start + G]

def weighted_gaussian_smooth_draws(draw_curve_SxG, xg_G, weight_G, bw_x, eps=1e-12):
    """
    Smooth each draw curve along x with weighted gaussian smoothing:
      y_sm(x) = conv(y*w, k) / conv(w, k)
    where w is (weight_G) and k is gaussian kernel on grid.
    Does not plot beyond support if you mask afterward.
    """
    y = np.asarray(draw_curve_SxG, float)  # (S,G)
    xg = np.asarray(xg_G, float)
    w = np.asarray(weight_G, float)

    if y.ndim != 2 or xg.ndim != 1 or w.ndim != 1:
        raise ValueError("Bad shapes for weighted_gaussian_smooth_draws.")
    if y.shape[1] != xg.size or w.size != xg.size:
        raise ValueError("draw_curve, xg, weight must share grid length.")

    if (bw_x is None) or (not np.isfinite(bw_x)) or (bw_x <= 0):
        return y.copy()

    # uniform grid spacing assumed
    dx = float(np.nanmedian(np.diff(xg)))
    if not np.isfinite(dx) or dx <= 0:
        return y.copy()

    sigma = float(bw_x) / dx
    if sigma < 0.5:  # extremely narrow -> not helpful
        return y.copy()

    radius = int(np.ceil(4.0 * sigma))
    idx = np.arange(-radius, radius + 1, dtype=float)
    k = np.exp(-0.5 * (idx / sigma) ** 2)

    # numerator: y*w (NaNs treated as 0 with weight 0)
    y0 = np.where(np.isfinite(y), y, 0.0)
    num = y0 * w[None, :]

    # convolve numerator rows + denominator once
    num_sm = _fft_convolve_same_rows(num, k)
    den_sm = _fft_convolve_same_1d(w, k)

    out = num_sm / (den_sm[None, :] + eps)
    out[:, den_sm < eps] = np.nan
    return out

# ============================================================
# 1) Posterior responsibilities P(goal | y) per step
# ============================================================
def posterior_prob_goal_per_step(
    idata,
    y_obs,
    *,
    kappa_uniform=1e-6,
    goal_comp_index=0,
    max_draws=1200,
    seed=123,
    hdi_prob=0.95,
):
    post = idata.posterior
    y_obs = np.asarray(y_obs, float)
    n_obs = y_obs.size
    rng = np.random.default_rng(seed)

    if "p_goal" not in post.data_vars:
        raise KeyError("Expected deterministic 'p_goal' in posterior.")

    w = _stack_samples_first(post["p_goal"])  # (S, ...)

    if w.ndim == 2:
        axes = list(range(1, w.ndim))
        obs_axes = [ax for ax in axes if w.shape[ax] == n_obs]
        if len(obs_axes) != 1:
            if w.shape[1] != n_obs:
                raise ValueError(f"Can't identify obs axis for p_goal. shape={w.shape}, n_obs={n_obs}")
            w_goal = w
        else:
            w_goal = np.moveaxis(w, obs_axes[0], 1).reshape(w.shape[0], n_obs)

    elif w.ndim >= 3:
        axes = list(range(1, w.ndim))
        obs_axes = [ax for ax in axes if w.shape[ax] == n_obs]
        comp_axes = [ax for ax in axes if w.shape[ax] == 2]
        if len(obs_axes) != 1 or len(comp_axes) != 1:
            raise ValueError(f"Can't identify obs/comp dims for p_goal. shape={w.shape}, n_obs={n_obs}")

        w = np.moveaxis(w, obs_axes[0], 1)
        comp_candidates = [ax for ax in range(2, w.ndim) if w.shape[ax] == 2]
        if len(comp_candidates) != 1:
            raise ValueError(f"Can't identify comp axis after moving obs. shape={w.shape}")
        w = np.moveaxis(w, comp_candidates[0], 2)  # (S, n_obs, 2)

        if goal_comp_index not in (0, 1):
            raise ValueError("goal_comp_index must be 0 or 1 when comp dim exists.")
        w_goal = w[:, :, int(goal_comp_index)]
    else:
        raise ValueError(f"Unexpected p_goal shape={w.shape}")

    w_goal = np.clip(w_goal, 1e-12, 1 - 1e-12)  # (S, n_obs)

    # mu
    if "mu" in post.data_vars:
        mu = _stack_samples_first(post["mu"])
        axes = list(range(1, mu.ndim))
        obs_axes = [ax for ax in axes if mu.shape[ax] == n_obs]
        if len(obs_axes) != 1:
            raise ValueError(f"Can't find obs axis in 'mu'. shape={mu.shape}, n_obs={n_obs}")
        mu = np.moveaxis(mu, obs_axes[0], 1).reshape(mu.shape[0], n_obs)
        mu = circular_modulo(mu)
    elif "mu0" in post.data_vars:
        mu0 = _stack_samples_first(post["mu0"]).reshape(-1)
        mu = circular_modulo(mu0[:, None] + np.zeros((mu0.size, n_obs)))
    else:
        raise KeyError("Need either 'mu' or 'mu0' in posterior.")

    # kappa
    if "kappa" in post.data_vars:
        kappa = _stack_samples_first(post["kappa"])
        axes = list(range(1, kappa.ndim))
        obs_axes = [ax for ax in axes if kappa.shape[ax] == n_obs]
        if len(obs_axes) != 1:
            raise ValueError(f"Can't find obs axis in 'kappa'. shape={kappa.shape}, n_obs={n_obs}")
        kappa = np.moveaxis(kappa, obs_axes[0], 1).reshape(kappa.shape[0], n_obs)
        kappa = np.clip(kappa, 1e-12, np.inf)
    elif "eta_kappa0" in post.data_vars:
        ek0 = _stack_samples_first(post["eta_kappa0"]).reshape(-1)
        k0 = softplus(ek0)
        kappa = k0[:, None] + np.zeros((k0.size, n_obs))
    else:
        raise KeyError("Need either 'kappa' or 'eta_kappa0' in posterior.")

    # subsample draws
    S = w_goal.shape[0]
    if (max_draws is not None) and (S > max_draws):
        idx = rng.choice(S, size=int(max_draws), replace=False)
        w_goal = w_goal[idx]
        mu = mu[idx]
        kappa = kappa[idx]

    logit_w = np.log(w_goal) - np.log1p(-w_goal)
    log_f_goal = log_vonmises_pdf(y_obs[None, :], mu, kappa)
    log_f_unif = log_vonmises_pdf(y_obs[None, :], 0.0, kappa_uniform)

    logit_r = logit_w + (log_f_goal - log_f_unif)
    resp_draws = sigmoid(logit_r)

    resp_mean = resp_draws.mean(axis=0)
    resp_hdi = hdi_along_samples(resp_draws, prob=hdi_prob)

    return resp_draws, resp_mean, resp_hdi

def append_pgoal_post_to_df(
    df_track,
    idata,
    *,
    y_col="HeadingRadians",
    kappa_uniform=1e-6,
    goal_comp_index=0,
    max_draws=1200,
    hdi_prob=0.95,
):
    y_obs = pd.to_numeric(df_track[y_col], errors="coerce").to_numpy(float)
    resp_draws, resp_mean, resp_hdi = posterior_prob_goal_per_step(
        idata,
        y_obs,
        kappa_uniform=kappa_uniform,
        goal_comp_index=goal_comp_index,
        max_draws=max_draws,
        hdi_prob=hdi_prob
    )
    out = df_track.copy()
    out["p_goal_post_mean"] = resp_mean
    out["p_goal_post_lo"] = resp_hdi[:, 0]
    out["p_goal_post_hi"] = resp_hdi[:, 1]
    return out, resp_draws

# ============================================================
# 2) Population curve per treatment (per-tag smoothing + truncation + pop smoothing)
# ============================================================
def population_curve_kernel_per_tag(
    df,
    resp_draws,
    *,
    tag_col="tag",
    treat_col="treatment",
    treat_name_col="treatmentName",

    x_kind="distance",                 # "distance" or "steps_from_end"
    last_n_steps=150,
    steps_x_units="m",                 # "m" or "steps"
    step_length_m=60.0,

    dist_col="dist_to_dest_m",
    dist_clip_q=None,

    n_grid=2000,
    bandwidth=240.0,
    chunk_size=4000,

    hdi_prob=0.95,
    eps=1e-12,

    # truncation controls
    truncate_to_data_range=True,
    truncate_by_weight=True,
    den_frac=0.01,

    # population stabilization
    pop_post_smooth=True,
    pop_smooth_bandwidth=None,
    min_tags_at_x=1,

    return_tag_summaries=False,
):
    d = df.copy().reset_index(drop=True)
    n_obs = len(d)

    resp = np.asarray(resp_draws, float)
    if resp.ndim != 2 or resp.shape[1] != n_obs:
        raise ValueError(f"resp_draws must be (S, n_obs) matching df rows. Got {resp.shape} vs n_obs={n_obs}")

    d["_rowpos"] = np.arange(n_obs, dtype=int)

    # treatment order + labels
    if isinstance(d[treat_col].dtype, pd.CategoricalDtype):
        treat_levels = list(d[treat_col].cat.categories)
    else:
        treat_levels = sorted(d[treat_col].dropna().unique().tolist())

    if treat_name_col in d.columns:
        name_map = d.groupby(treat_col, observed=False)[treat_name_col].first().to_dict()
        labels = {t: _format_treatment_name(name_map.get(t, t)) for t in treat_levels}
    else:
        labels = {t: _format_treatment_name(t) for t in treat_levels}

    # ---- define x ----
    if x_kind == "steps_from_end":
        d["_step_fwd"] = d.groupby(tag_col, observed=False).cumcount()
        d["_last_fwd"] = d.groupby(tag_col, observed=False)["_step_fwd"].transform("max")
        d["_x_steps"] = d["_step_fwd"] - d["_last_fwd"]  # last=0; earlier negative

        d = d.loc[(d["_x_steps"] >= -int(last_n_steps)) & (d["_x_steps"] <= 0)].copy()

        if str(steps_x_units).lower() in ("m", "meter", "meters"):
            d["_x"] = d["_x_steps"].astype(float) * float(step_length_m)
        elif str(steps_x_units).lower() in ("step", "steps"):
            d["_x"] = d["_x_steps"].astype(float)
        else:
            raise ValueError("steps_x_units must be 'm' or 'steps'")

    elif x_kind == "distance":
        if dist_col not in d.columns:
            raise KeyError(f"Missing distance column '{dist_col}' in df.")
        xvals = pd.to_numeric(d[dist_col], errors="coerce").to_numpy(float)
        ok = np.isfinite(xvals)
        d = d.loc[ok].copy()
        xvals = pd.to_numeric(d[dist_col], errors="coerce").to_numpy(float)

        if dist_clip_q is not None:
            cap = float(np.nanquantile(xvals, float(dist_clip_q)))
            xvals = np.clip(xvals, a_min=None, a_max=cap)

        d["_x"] = xvals
    else:
        raise ValueError("x_kind must be 'distance' or 'steps_from_end'")

    # gaussian kernel
    def K(z):
        return np.exp(-0.5 * z * z)

    def _smooth_one_tag(resp_SxN, x_N, xg_G, h):
        S, N = resp_SxN.shape
        G = xg_G.size
        num = np.zeros((S, G), float)
        den = np.zeros((G,), float)

        for start in range(0, N, int(chunk_size)):
            stop = min(N, start + int(chunk_size))
            xs = x_N[start:stop]                       # (n,)
            rs = resp_SxN[:, start:stop]               # (S,n)
            z = (xs[:, None] - xg_G[None, :]) / h      # (n,G)
            w = K(z)                                   # (n,G)
            den += w.sum(axis=0)
            num += rs @ w

        den = den + eps
        return num / den[None, :], den

    out = {}
    tag_out = {} if return_tag_summaries else None

    for t in treat_levels:
        dt = d.loc[d[treat_col] == t].copy()
        if dt.empty:
            continue

        x_all = dt["_x"].to_numpy(float)
        xmin = float(np.nanmin(x_all))
        xmax = float(np.nanmax(x_all))
        if not np.isfinite(xmin) or not np.isfinite(xmax) or xmax <= xmin:
            continue

        xg = np.linspace(xmin, xmax, int(n_grid)).astype(float)
        h = float(bandwidth)
        if not np.isfinite(h) or h <= 0:
            raise ValueError("bandwidth must be positive and finite.")

        # accumulate per draw across tags, NaN-safe
        pop_sum = None                         # (S,G)
        pop_cnt = np.zeros((xg.size,), float)  # (#tags contributing at each x)
        n_tags_any = 0

        for tg, g in dt.groupby(tag_col, observed=False):
            j = g["_rowpos"].to_numpy(int)
            if j.size == 0:
                continue

            x_tg_full = g["_x"].to_numpy(float)
            x_tg = x_tg_full[np.isfinite(x_tg_full)]
            if x_tg.size < 2:
                continue

            x_min_tg = float(np.nanmin(x_tg))
            x_max_tg = float(np.nanmax(x_tg))

            resp_tg = resp[:, j]  # (S, N_tag)
            curve_SxG, den_G = _smooth_one_tag(resp_tg, x_tg_full, xg, h)

            # ---- truncation mask for this tag (same for all draws) ----
            mask = np.zeros_like(xg, dtype=bool)

            if truncate_to_data_range:
                mask |= (xg < x_min_tg) | (xg > x_max_tg)

            if truncate_by_weight:
                thr = float(den_frac) * float(np.nanmax(den_G))
                mask |= (den_G < thr)

            if mask.any():
                curve_SxG[:, mask] = np.nan

            if pop_sum is None:
                pop_sum = np.zeros_like(curve_SxG)

            good = np.isfinite(curve_SxG)  # (S,G) but mask identical across draws
            pop_sum += np.where(good, curve_SxG, 0.0)
            pop_cnt += np.isfinite(curve_SxG[0]).astype(float)

            n_tags_any += 1

            if return_tag_summaries:
                tg_mean = np.nanmean(curve_SxG, axis=0)
                tg_hdi = hdi_along_samples(curve_SxG, prob=hdi_prob)
                tag_out[(t, tg)] = {
                    "x_mid": xg,
                    "mean": tg_mean,
                    "lo": tg_hdi[:, 0],
                    "hi": tg_hdi[:, 1],
                    "x_min": x_min_tg,
                    "x_max": x_max_tg,
                }

        if pop_sum is None or n_tags_any == 0:
            continue

        # equal-weight mean across tags at each x (NaN where no tags)
        denom = np.maximum(pop_cnt, 1.0)
        draw_curve = pop_sum / denom[None, :]
        draw_curve[:, pop_cnt < 1] = np.nan

        # mask unstable x where too few tags contribute (removes jagged tails)
        min_k = int(max(1, min_tags_at_x))
        unstable = pop_cnt < float(min_k)
        if unstable.any():
            draw_curve[:, unstable] = np.nan

        # ---- population post-smoothing across x ----
        if pop_post_smooth:
            bw_pop = float(pop_smooth_bandwidth) if pop_smooth_bandwidth is not None else float(bandwidth)
            w = pop_cnt.copy()
            w[unstable] = 0.0
            draw_curve = weighted_gaussian_smooth_draws(draw_curve, xg, w, bw_pop, eps=eps)
            draw_curve[:, unstable] = np.nan
            draw_curve[:, pop_cnt < 1] = np.nan

        mean = np.nanmean(draw_curve, axis=0)
        hdi = hdi_along_samples(draw_curve, prob=hdi_prob)

        out[t] = {
            "label": labels.get(t, str(t)),
            "x_mid": xg,
            "mean": mean,
            "lo": hdi[:, 0],
            "hi": hdi[:, 1],
            "bandwidth": h,
            "n_tags": int(n_tags_any),
            "n_tags_by_x": pop_cnt,
            "min_tags_at_x": min_k,
            "pop_post_smooth": bool(pop_post_smooth),
            "pop_smooth_bandwidth": float(pop_smooth_bandwidth) if pop_smooth_bandwidth is not None else float(bandwidth),
        }

    if return_tag_summaries:
        return treat_levels, out, tag_out
    return treat_levels, out

# ============================================================
# 3) Smooth plotter
# ============================================================
def plot_population_curve_smooth(
    out_dict,
    treat_levels,
    *,
    xlabel,
    ylabel="Population mean p(goal | step)",
    band_alpha=0.20,
    line_lw=1.4,
    fig_w=4.4,
    fig_h=2.6,
    legend_fontsize=7,
    invert_x=False,
    xlim=None,
    x_tick_abs=False,
    savepath=None,
    palette_name="colorblind",
    save_svg_also=True,
):
    # seaborn colorblind palette (stable treatment order)
    colors = sns.color_palette(palette_name, n_colors=len(treat_levels))
    treat_to_color = {t: colors[i] for i, t in enumerate(treat_levels)}

    fig = plt.figure(figsize=(fig_w, fig_h), dpi=DPI)
    ax = plt.gca()
    ax.axhline(0.5, color="0.90", lw=0.9, ls="--", zorder=0)

    for t in treat_levels:
        if t not in out_dict:
            continue
        s = out_dict[t]
        x = np.asarray(s["x_mid"], float)
        m = np.asarray(s["mean"], float)
        lo = np.asarray(s["lo"], float)
        hi = np.asarray(s["hi"], float)

        order = np.argsort(x)
        x, m, lo, hi = x[order], m[order], lo[order], hi[order]

        c = treat_to_color[t]
        ax.fill_between(x, lo, hi, color=c, alpha=band_alpha, linewidth=0)
        ax.plot(x, m, color=c, lw=line_lw, label=s.get("label", str(t)))

    ax.set_ylim(0, 1)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    if xlim is not None:
        ax.set_xlim(*xlim)
    if invert_x:
        ax.invert_xaxis()

    if x_tick_abs:
        def _abs_tick_formatter(val, pos):
            aval = abs(val)
            if np.isfinite(aval) and np.isclose(aval, np.round(aval), atol=1e-9):
                return f"{int(np.round(aval)):,}"
            if np.isfinite(aval):
                return f"{aval:.2f}"
            return ""
        ax.xaxis.set_major_formatter(FuncFormatter(_abs_tick_formatter))

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

    ax.legend(
        frameon=True,
        fontsize=legend_fontsize,
        ncol=1,
        handlelength=1.4,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        borderaxespad=0.0,
    )
    fig.subplots_adjust(right=0.78)

    if savepath:
        fig.savefig(savepath, dpi=DPI, bbox_inches="tight", pad_inches=0.01)
        if save_svg_also:
            root, _ = os.path.splitext(savepath)
            fig.savefig(root + ".svg", bbox_inches="tight", pad_inches=0.01)

    plt.show()
    return fig, ax

# ============================================================
# RUN EVERYTHING
# ============================================================

# 1) compute and append per-step posterior P(goal|y)
df_track_with_p, resp = append_pgoal_post_to_df(
    df_track,
    idata_mix_full,
    y_col="HeadingRadians",
    max_draws=2000,
    hdi_prob=0.95,
    # goal_comp_index=0,
)

print(df_track_with_p[["p_goal_post_mean", "p_goal_post_lo", "p_goal_post_hi"]].head())

treat_levels = (
    list(df_track_with_p["treatment"].cat.categories)
    if isinstance(df_track_with_p["treatment"].dtype, pd.CategoricalDtype)
    else sorted(df_track_with_p["treatment"].dropna().unique().tolist())
)

# ----------------------------
# A) Distance-to-destination: pop mean+bnds (TRUNC + WEIGHT + POP SMOOTH)
# ----------------------------
_, out_dist_k = population_curve_kernel_per_tag(
    df_track_with_p, resp,
    x_kind="distance",
    dist_col=DIST_COL,
    dist_clip_q=None,
    n_grid=N_GRID,
    bandwidth=BANDWIDTH_M,
    chunk_size=CHUNK_SIZE,
    hdi_prob=0.95,
    truncate_to_data_range=TRUNCATE_TO_DATA_RANGE,
    truncate_by_weight=TRUNCATE_BY_WEIGHT,
    den_frac=DEN_FRAC,
    pop_post_smooth=POP_POST_SMOOTH,
    pop_smooth_bandwidth=POP_SMOOTH_BW,
    min_tags_at_x=MIN_TAGS_AT_X,
    return_tag_summaries=False,
)

plot_population_curve_smooth(
    out_dist_k,
    treat_levels,
    xlabel="Distance to destination (m)",
    ylabel="Mean directed-component probability",
    invert_x=True,
    savepath="../figures/bats/p_goal_pop_kernel_distance_TRUNC_SMOOTH.pdf",
)

# ----------------------------
# B) Steps-from-end (x in meters): pop mean+bnds (TRUNC + WEIGHT + POP SMOOTH)
# ----------------------------
LAST_N_STEPS = 110

_, out_steps_k = population_curve_kernel_per_tag(
    df_track_with_p, resp,
    x_kind="steps_from_end",
    last_n_steps=LAST_N_STEPS,
    steps_x_units="m",
    step_length_m=STEP_LENGTH_M,
    n_grid=N_GRID,
    bandwidth=BANDWIDTH_M,
    chunk_size=CHUNK_SIZE,
    hdi_prob=0.95,
    truncate_to_data_range=TRUNCATE_TO_DATA_RANGE,
    truncate_by_weight=TRUNCATE_BY_WEIGHT,
    den_frac=DEN_FRAC,
    pop_post_smooth=POP_POST_SMOOTH,
    pop_smooth_bandwidth=POP_SMOOTH_BW,
    min_tags_at_x=MIN_TAGS_AT_X,
    return_tag_summaries=False,
)

plot_population_curve_smooth(
    out_steps_k,
    treat_levels,
    xlabel="Cumulative path distance from end (m)",
    ylabel="Mean directed-component probability",
    invert_x=False,
    xlim=(-LAST_N_STEPS * STEP_LENGTH_M, 0),
    x_tick_abs=True,
    savepath="../figures/bats/p_goal_pop_kernel_distFromEndM_TRUNC_SMOOTH.pdf",
)

   p_goal_post_mean  p_goal_post_lo  p_goal_post_hi
1          0.001385    2.377827e-13        0.001339
2          0.821075    7.176318e-01        0.908276
3          0.001844    1.333362e-12        0.002415
4          0.001674    5.740170e-13        0.001843
5          0.001666    5.483800e-13        0.001822


/tmp/ipykernel_3898688/1803618528.py:570: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(draw_curve, axis=0)


/tmp/ipykernel_3898688/1803618528.py:570: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(draw_curve, axis=0)


(<Figure size 1320x780 with 1 Axes>,
 <Axes: xlabel='Cumulative path distance from end (m)', ylabel='Mean directed-component probability'>)

In [ ]:
# ============================================================
# FULL KERNEL-SMOOTHED PIPELINE (NO BINNING) — POPULATION CURVE
# COMPACT / PAPER-READY UPDATE:
#   - Smaller figure size
#   - Larger fonts
#   - NO legend
#   - Saves SVG alongside PDF
#   - Seaborn colorblind palette (style match)
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# ----------------------------
# Style (match other plots)
# ----------------------------
DPI = 300
mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
})

# ============================================================
# USER CONTROLS
# ============================================================
STEP_LENGTH_M = 60.0
DIST_COL = "dist_to_dest_grad_m"

BANDWIDTH_M = 120.0
N_GRID = 2000
CHUNK_SIZE = 4000

# --- truncation controls ---
TRUNCATE_TO_DATA_RANGE = True
TRUNCATE_BY_WEIGHT = True
DEN_FRAC = 0.01

# --- stabilize population curve after truncation ---
POP_POST_SMOOTH = True
POP_SMOOTH_BW = 240.0
MIN_TAGS_AT_X = 2

# ----------------------------
# Small utilities
# ----------------------------
def circular_modulo(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def sigmoid(x):
    x = np.asarray(x, float)
    return 1.0 / (1.0 + np.exp(-x))

def softplus(x):
    x = np.asarray(x, float)
    return np.where(x > 30, x, np.log1p(np.exp(x)))

def _stack_samples_first(da):
    """xarray DataArray -> np array with sample first: (S, ...)"""
    da2 = da.stack(sample=("chain", "draw"))
    other = [d for d in da2.dims if d != "sample"]
    da2 = da2.transpose("sample", *other)
    return da2.values.astype(float)

def _format_treatment_name(name: str) -> str:
    s = str(name).strip().lower()
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if "control" in parts and len(parts) > 1:
        parts = [p for p in parts if p != "control"]
    if not parts:
        parts = ["control"]
    parts = sorted(parts)
    parts = [p[:1].upper() + p[1:] for p in parts]
    return ", ".join(parts)

def hdi_1d(x, prob=0.95):
    """Minimum-width interval (HDI) for 1D samples (NaN-safe)."""
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    n = x.size
    if n == 0:
        return np.nan, np.nan
    x = np.sort(x)
    m = int(np.floor(prob * n))
    if m < 1:
        return np.nan, np.nan
    widths = x[m:] - x[: n - m]
    j = int(np.argmin(widths))
    return float(x[j]), float(x[j + m])

def hdi_along_samples(arr, prob=0.95):
    """HDI across axis=0 for arr shape (S, N) -> (N,2). NaN-safe."""
    arr = np.asarray(arr, float)
    S, N = arr.shape
    out = np.empty((N, 2), float)
    for i in range(N):
        out[i, 0], out[i, 1] = hdi_1d(arr[:, i], prob=prob)
    return out

def log_i0_stable(kappa):
    """Stable-ish log(I0(kappa)) with asymptotic fallback."""
    kappa = np.asarray(kappa, float)
    out = np.empty_like(kappa)
    big = kappa > 700
    small = ~big
    out[small] = np.log(np.i0(kappa[small]))
    out[big] = kappa[big] - 0.5 * np.log(2 * np.pi * kappa[big])
    return out

def log_vonmises_pdf(y, mu, kappa):
    """log f(y|mu,kappa) = kappa*cos(y-mu) - log(2π) - log(I0(kappa))"""
    y = np.asarray(y, float)
    mu = np.asarray(mu, float)
    kappa = np.asarray(kappa, float)
    return kappa * np.cos(y - mu) - np.log(2 * np.pi) - log_i0_stable(kappa)

# ============================================================
# FFT weighted smoothing utilities (fast, grid-uniform)
# ============================================================
def _next_pow_two(n: int) -> int:
    return 1 << (int(n) - 1).bit_length()

def _fft_convolve_same_1d(x, k):
    """1D convolution (same) using FFT. x,k are 1D arrays."""
    x = np.asarray(x, float)
    k = np.asarray(k, float)
    nx = x.size
    nk = k.size
    n = nx + nk - 1
    nfft = _next_pow_two(n)

    X = np.fft.rfft(x, nfft)
    K = np.fft.rfft(k, nfft)
    y = np.fft.irfft(X * K, nfft)[:n]

    start = (nk - 1) // 2
    return y[start:start + nx]

def _fft_convolve_same_rows(A, k):
    """
    Row-wise convolution (same) using FFT.
    A: (S,G), k: (K,)
    returns: (S,G)
    """
    A = np.asarray(A, float)
    k = np.asarray(k, float)
    S, G = A.shape
    K = k.size
    n = G + K - 1
    nfft = _next_pow_two(n)

    Af = np.fft.rfft(A, nfft, axis=1)
    kf = np.fft.rfft(k, nfft)
    y = np.fft.irfft(Af * kf[None, :], nfft, axis=1)[:, :n]

    start = (K - 1) // 2
    return y[:, start:start + G]

def weighted_gaussian_smooth_draws(draw_curve_SxG, xg_G, weight_G, bw_x, eps=1e-12):
    """
    Smooth each draw curve along x with weighted gaussian smoothing:
      y_sm(x) = conv(y*w, k) / conv(w, k)
    """
    y = np.asarray(draw_curve_SxG, float)  # (S,G)
    xg = np.asarray(xg_G, float)
    w = np.asarray(weight_G, float)

    if y.ndim != 2 or xg.ndim != 1 or w.ndim != 1:
        raise ValueError("Bad shapes for weighted_gaussian_smooth_draws.")
    if y.shape[1] != xg.size or w.size != xg.size:
        raise ValueError("draw_curve, xg, weight must share grid length.")

    if (bw_x is None) or (not np.isfinite(bw_x)) or (bw_x <= 0):
        return y.copy()

    dx = float(np.nanmedian(np.diff(xg)))
    if not np.isfinite(dx) or dx <= 0:
        return y.copy()

    sigma = float(bw_x) / dx
    if sigma < 0.5:
        return y.copy()

    radius = int(np.ceil(4.0 * sigma))
    idx = np.arange(-radius, radius + 1, dtype=float)
    k = np.exp(-0.5 * (idx / sigma) ** 2)

    y0 = np.where(np.isfinite(y), y, 0.0)
    num = y0 * w[None, :]

    num_sm = _fft_convolve_same_rows(num, k)
    den_sm = _fft_convolve_same_1d(w, k)

    out = num_sm / (den_sm[None, :] + eps)
    out[:, den_sm < eps] = np.nan
    return out

# ============================================================
# 1) Posterior responsibilities P(goal | y) per step
# ============================================================
def posterior_prob_goal_per_step(
    idata,
    y_obs,
    *,
    kappa_uniform=1e-6,
    goal_comp_index=0,
    max_draws=1200,
    seed=123,
    hdi_prob=0.95,
):
    post = idata.posterior
    y_obs = np.asarray(y_obs, float)
    n_obs = y_obs.size
    rng = np.random.default_rng(seed)

    if "p_goal" not in post.data_vars:
        raise KeyError("Expected deterministic 'p_goal' in posterior.")

    w = _stack_samples_first(post["p_goal"])  # (S, ...)

    if w.ndim == 2:
        axes = list(range(1, w.ndim))
        obs_axes = [ax for ax in axes if w.shape[ax] == n_obs]
        if len(obs_axes) != 1:
            if w.shape[1] != n_obs:
                raise ValueError(f"Can't identify obs axis for p_goal. shape={w.shape}, n_obs={n_obs}")
            w_goal = w
        else:
            w_goal = np.moveaxis(w, obs_axes[0], 1).reshape(w.shape[0], n_obs)

    elif w.ndim >= 3:
        axes = list(range(1, w.ndim))
        obs_axes = [ax for ax in axes if w.shape[ax] == n_obs]
        comp_axes = [ax for ax in axes if w.shape[ax] == 2]
        if len(obs_axes) != 1 or len(comp_axes) != 1:
            raise ValueError(f"Can't identify obs/comp dims for p_goal. shape={w.shape}, n_obs={n_obs}")

        w = np.moveaxis(w, obs_axes[0], 1)
        comp_candidates = [ax for ax in range(2, w.ndim) if w.shape[ax] == 2]
        if len(comp_candidates) != 1:
            raise ValueError(f"Can't identify comp axis after moving obs. shape={w.shape}")
        w = np.moveaxis(w, comp_candidates[0], 2)  # (S, n_obs, 2)

        if goal_comp_index not in (0, 1):
            raise ValueError("goal_comp_index must be 0 or 1 when comp dim exists.")
        w_goal = w[:, :, int(goal_comp_index)]
    else:
        raise ValueError(f"Unexpected p_goal shape={w.shape}")

    w_goal = np.clip(w_goal, 1e-12, 1 - 1e-12)

    # mu
    if "mu" in post.data_vars:
        mu = _stack_samples_first(post["mu"])
        axes = list(range(1, mu.ndim))
        obs_axes = [ax for ax in axes if mu.shape[ax] == n_obs]
        if len(obs_axes) != 1:
            raise ValueError(f"Can't find obs axis in 'mu'. shape={mu.shape}, n_obs={n_obs}")
        mu = np.moveaxis(mu, obs_axes[0], 1).reshape(mu.shape[0], n_obs)
        mu = circular_modulo(mu)
    elif "mu0" in post.data_vars:
        mu0 = _stack_samples_first(post["mu0"]).reshape(-1)
        mu = circular_modulo(mu0[:, None] + np.zeros((mu0.size, n_obs)))
    else:
        raise KeyError("Need either 'mu' or 'mu0' in posterior.")

    # kappa
    if "kappa" in post.data_vars:
        kappa = _stack_samples_first(post["kappa"])
        axes = list(range(1, kappa.ndim))
        obs_axes = [ax for ax in axes if kappa.shape[ax] == n_obs]
        if len(obs_axes) != 1:
            raise ValueError(f"Can't find obs axis in 'kappa'. shape={kappa.shape}, n_obs={n_obs}")
        kappa = np.moveaxis(kappa, obs_axes[0], 1).reshape(kappa.shape[0], n_obs)
        kappa = np.clip(kappa, 1e-12, np.inf)
    elif "eta_kappa0" in post.data_vars:
        ek0 = _stack_samples_first(post["eta_kappa0"]).reshape(-1)
        k0 = softplus(ek0)
        kappa = k0[:, None] + np.zeros((k0.size, n_obs))
    else:
        raise KeyError("Need either 'kappa' or 'eta_kappa0' in posterior.")

    # subsample draws
    S = w_goal.shape[0]
    if (max_draws is not None) and (S > max_draws):
        idx = rng.choice(S, size=int(max_draws), replace=False)
        w_goal = w_goal[idx]
        mu = mu[idx]
        kappa = kappa[idx]

    logit_w = np.log(w_goal) - np.log1p(-w_goal)
    log_f_goal = log_vonmises_pdf(y_obs[None, :], mu, kappa)
    log_f_unif = log_vonmises_pdf(y_obs[None, :], 0.0, kappa_uniform)

    logit_r = logit_w + (log_f_goal - log_f_unif)
    resp_draws = sigmoid(logit_r)

    resp_mean = resp_draws.mean(axis=0)
    resp_hdi = hdi_along_samples(resp_draws, prob=hdi_prob)

    return resp_draws, resp_mean, resp_hdi

def append_pgoal_post_to_df(
    df_track,
    idata,
    *,
    y_col="HeadingRadians",
    kappa_uniform=1e-6,
    goal_comp_index=0,
    max_draws=1200,
    hdi_prob=0.95,
):
    y_obs = pd.to_numeric(df_track[y_col], errors="coerce").to_numpy(float)
    resp_draws, resp_mean, resp_hdi = posterior_prob_goal_per_step(
        idata,
        y_obs,
        kappa_uniform=kappa_uniform,
        goal_comp_index=goal_comp_index,
        max_draws=max_draws,
        hdi_prob=hdi_prob
    )
    out = df_track.copy()
    out["p_goal_post_mean"] = resp_mean
    out["p_goal_post_lo"] = resp_hdi[:, 0]
    out["p_goal_post_hi"] = resp_hdi[:, 1]
    return out, resp_draws

# ============================================================
# 2) Population curve per treatment
#    (per-tag smoothing + truncation + pop smoothing)
# ============================================================
def population_curve_kernel_per_tag(
    df,
    resp_draws,
    *,
    tag_col="tag",
    treat_col="treatment",
    treat_name_col="treatmentName",

    x_kind="distance",                 # "distance" or "steps_from_end"
    last_n_steps=150,
    steps_x_units="m",                 # "m" or "steps"
    step_length_m=60.0,

    dist_col="dist_to_dest_m",
    dist_clip_q=None,

    n_grid=2000,
    bandwidth=240.0,
    chunk_size=4000,

    hdi_prob=0.95,
    eps=1e-12,

    # truncation controls
    truncate_to_data_range=True,
    truncate_by_weight=True,
    den_frac=0.01,

    # population stabilization
    pop_post_smooth=True,
    pop_smooth_bandwidth=None,
    min_tags_at_x=1,

    return_tag_summaries=False,
):
    d = df.copy().reset_index(drop=True)
    n_obs = len(d)

    resp = np.asarray(resp_draws, float)
    if resp.ndim != 2 or resp.shape[1] != n_obs:
        raise ValueError(f"resp_draws must be (S, n_obs) matching df rows. Got {resp.shape} vs n_obs={n_obs}")

    d["_rowpos"] = np.arange(n_obs, dtype=int)

    # treatment order + labels
    if isinstance(d[treat_col].dtype, pd.CategoricalDtype):
        treat_levels = list(d[treat_col].cat.categories)
    else:
        treat_levels = sorted(d[treat_col].dropna().unique().tolist())

    if treat_name_col in d.columns:
        name_map = d.groupby(treat_col, observed=False)[treat_name_col].first().to_dict()
        labels = {t: _format_treatment_name(name_map.get(t, t)) for t in treat_levels}
    else:
        labels = {t: _format_treatment_name(t) for t in treat_levels}

    # ---- define x ----
    if x_kind == "steps_from_end":
        d["_step_fwd"] = d.groupby(tag_col, observed=False).cumcount()
        d["_last_fwd"] = d.groupby(tag_col, observed=False)["_step_fwd"].transform("max")
        d["_x_steps"] = d["_step_fwd"] - d["_last_fwd"]  # last=0; earlier negative

        d = d.loc[(d["_x_steps"] >= -int(last_n_steps)) & (d["_x_steps"] <= 0)].copy()

        if str(steps_x_units).lower() in ("m", "meter", "meters"):
            d["_x"] = d["_x_steps"].astype(float) * float(step_length_m)
        elif str(steps_x_units).lower() in ("step", "steps"):
            d["_x"] = d["_x_steps"].astype(float)
        else:
            raise ValueError("steps_x_units must be 'm' or 'steps'")

    elif x_kind == "distance":
        if dist_col not in d.columns:
            raise KeyError(f"Missing distance column '{dist_col}' in df.")
        xvals = pd.to_numeric(d[dist_col], errors="coerce").to_numpy(float)
        ok = np.isfinite(xvals)
        d = d.loc[ok].copy()
        xvals = pd.to_numeric(d[dist_col], errors="coerce").to_numpy(float)

        if dist_clip_q is not None:
            cap = float(np.nanquantile(xvals, float(dist_clip_q)))
            xvals = np.clip(xvals, a_min=None, a_max=cap)

        d["_x"] = xvals
    else:
        raise ValueError("x_kind must be 'distance' or 'steps_from_end'")

    def K(z):
        return np.exp(-0.5 * z * z)

    def _smooth_one_tag(resp_SxN, x_N, xg_G, h):
        S, N = resp_SxN.shape
        G = xg_G.size
        num = np.zeros((S, G), float)
        den = np.zeros((G,), float)

        for start in range(0, N, int(chunk_size)):
            stop = min(N, start + int(chunk_size))
            xs = x_N[start:stop]                       # (n,)
            rs = resp_SxN[:, start:stop]               # (S,n)
            z = (xs[:, None] - xg_G[None, :]) / h      # (n,G)
            w = K(z)                                   # (n,G)
            den += w.sum(axis=0)
            num += rs @ w

        den = den + eps
        return num / den[None, :], den

    out = {}
    tag_out = {} if return_tag_summaries else None

    for t in treat_levels:
        dt = d.loc[d[treat_col] == t].copy()
        if dt.empty:
            continue

        x_all = dt["_x"].to_numpy(float)
        xmin = float(np.nanmin(x_all))
        xmax = float(np.nanmax(x_all))
        if not np.isfinite(xmin) or not np.isfinite(xmax) or xmax <= xmin:
            continue

        xg = np.linspace(xmin, xmax, int(n_grid)).astype(float)
        h = float(bandwidth)
        if not np.isfinite(h) or h <= 0:
            raise ValueError("bandwidth must be positive and finite.")

        pop_sum = None                         # (S,G)
        pop_cnt = np.zeros((xg.size,), float)  # tags contributing at each x
        n_tags_any = 0

        for tg, g in dt.groupby(tag_col, observed=False):
            j = g["_rowpos"].to_numpy(int)
            if j.size == 0:
                continue

            x_tg_full = g["_x"].to_numpy(float)
            x_tg = x_tg_full[np.isfinite(x_tg_full)]
            if x_tg.size < 2:
                continue

            x_min_tg = float(np.nanmin(x_tg))
            x_max_tg = float(np.nanmax(x_tg))

            resp_tg = resp[:, j]  # (S, N_tag)
            curve_SxG, den_G = _smooth_one_tag(resp_tg, x_tg_full, xg, h)

            # ---- truncation mask for this tag ----
            mask = np.zeros_like(xg, dtype=bool)

            if truncate_to_data_range:
                mask |= (xg < x_min_tg) | (xg > x_max_tg)

            if truncate_by_weight:
                thr = float(den_frac) * float(np.nanmax(den_G))
                mask |= (den_G < thr)

            if mask.any():
                curve_SxG[:, mask] = np.nan

            if pop_sum is None:
                pop_sum = np.zeros_like(curve_SxG)

            good = np.isfinite(curve_SxG)
            pop_sum += np.where(good, curve_SxG, 0.0)
            pop_cnt += np.isfinite(curve_SxG[0]).astype(float)

            n_tags_any += 1

            if return_tag_summaries:
                tg_mean = np.nanmean(curve_SxG, axis=0)
                tg_hdi = hdi_along_samples(curve_SxG, prob=hdi_prob)
                tag_out[(t, tg)] = {
                    "x_mid": xg,
                    "mean": tg_mean,
                    "lo": tg_hdi[:, 0],
                    "hi": tg_hdi[:, 1],
                    "x_min": x_min_tg,
                    "x_max": x_max_tg,
                }

        if pop_sum is None or n_tags_any == 0:
            continue

        denom = np.maximum(pop_cnt, 1.0)
        draw_curve = pop_sum / denom[None, :]
        draw_curve[:, pop_cnt < 1] = np.nan

        min_k = int(max(1, min_tags_at_x))
        unstable = pop_cnt < float(min_k)
        if unstable.any():
            draw_curve[:, unstable] = np.nan

        # ---- population post-smoothing across x ----
        if pop_post_smooth:
            bw_pop = float(pop_smooth_bandwidth) if pop_smooth_bandwidth is not None else float(bandwidth)
            w = pop_cnt.copy()
            w[unstable] = 0.0
            draw_curve = weighted_gaussian_smooth_draws(draw_curve, xg, w, bw_pop, eps=eps)
            draw_curve[:, unstable] = np.nan
            draw_curve[:, pop_cnt < 1] = np.nan

        mean = np.nanmean(draw_curve, axis=0)
        hdi = hdi_along_samples(draw_curve, prob=hdi_prob)

        out[t] = {
            "label": labels.get(t, str(t)),
            "x_mid": xg,
            "mean": mean,
            "lo": hdi[:, 0],
            "hi": hdi[:, 1],
            "bandwidth": h,
            "n_tags": int(n_tags_any),
            "n_tags_by_x": pop_cnt,
            "min_tags_at_x": min_k,
            "pop_post_smooth": bool(pop_post_smooth),
            "pop_smooth_bandwidth": float(pop_smooth_bandwidth) if pop_smooth_bandwidth is not None else float(bandwidth),
        }

    if return_tag_summaries:
        return treat_levels, out, tag_out
    return treat_levels, out

# ============================================================
# 3) Smooth plotter (COMPACT / PAPER-READY)
#    - smaller figure
#    - larger fonts
#    - NO legend
# ============================================================
def plot_population_curve_smooth(
    out_dict,
    treat_levels,
    *,
    xlabel,
    ylabel="Population mean p(goal | step)",
    band_alpha=0.18,
    line_lw=1.5,
    fig_w=3.6,              # compact
    fig_h=2.35,             # compact
    invert_x=False,
    xlim=None,
    x_tick_abs=False,
    savepath=None,
    palette_name="colorblind",
    save_svg_also=True,

    # larger text relative to figure size
    axis_label_fs=9,
    tick_label_fs=8,
    y_labelpad=2,
    x_labelpad=2,
):
    colors = sns.color_palette(palette_name, n_colors=len(treat_levels))
    treat_to_color = {t: colors[i] for i, t in enumerate(treat_levels)}

    fig = plt.figure(figsize=(fig_w, fig_h), dpi=DPI)
    ax = plt.gca()

    ax.axhline(0.5, color="0.88", lw=0.9, ls="--", zorder=0)

    for t in treat_levels:
        if t not in out_dict:
            continue
        s = out_dict[t]
        x = np.asarray(s["x_mid"], float)
        m = np.asarray(s["mean"], float)
        lo = np.asarray(s["lo"], float)
        hi = np.asarray(s["hi"], float)

        order = np.argsort(x)
        x, m, lo, hi = x[order], m[order], lo[order], hi[order]

        c = treat_to_color[t]
        ax.fill_between(x, lo, hi, color=c, alpha=band_alpha, linewidth=0)
        ax.plot(x, m, color=c, lw=line_lw)

    ax.set_ylim(0, 1)
    ax.set_xlabel(xlabel, fontsize=axis_label_fs, labelpad=x_labelpad)
    ax.set_ylabel(ylabel, fontsize=axis_label_fs, labelpad=y_labelpad)

    if xlim is not None:
        ax.set_xlim(*xlim)
    if invert_x:
        ax.invert_xaxis()

    if x_tick_abs:
        def _abs_tick_formatter(val, pos):
            aval = abs(val)
            if np.isfinite(aval) and np.isclose(aval, np.round(aval), atol=1e-9):
                return f"{int(np.round(aval)):,}"
            if np.isfinite(aval):
                return f"{aval:.2f}"
            return ""
        ax.xaxis.set_major_formatter(FuncFormatter(_abs_tick_formatter))

    ax.tick_params(axis="both", which="major", labelsize=tick_label_fs, length=3)
    ax.tick_params(axis="both", which="minor", labelsize=tick_label_fs, length=2)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

    # No legend
    plt.tight_layout(pad=0.35)

    if savepath:
        fig.savefig(savepath, dpi=DPI, bbox_inches="tight", pad_inches=0.01)
        if save_svg_also:
            root, _ = os.path.splitext(savepath)
            fig.savefig(root + ".svg", bbox_inches="tight", pad_inches=0.01)

    plt.show()
    return fig, ax

# ============================================================
# RUN EVERYTHING
# ============================================================

# 1) compute and append per-step posterior P(goal|y)
df_track_with_p, resp = append_pgoal_post_to_df(
    df_track,
    idata_mix_full,
    y_col="HeadingRadians",
    max_draws=2000,
    hdi_prob=0.95,
    # goal_comp_index=0,
)

print(df_track_with_p[["p_goal_post_mean", "p_goal_post_lo", "p_goal_post_hi"]].head())

treat_levels = (
    list(df_track_with_p["treatment"].cat.categories)
    if isinstance(df_track_with_p["treatment"].dtype, pd.CategoricalDtype)
    else sorted(df_track_with_p["treatment"].dropna().unique().tolist())
)

# ----------------------------
# A) Distance-to-destination
# ----------------------------
_, out_dist_k = population_curve_kernel_per_tag(
    df_track_with_p, resp,
    x_kind="distance",
    dist_col=DIST_COL,
    dist_clip_q=None,
    n_grid=N_GRID,
    bandwidth=BANDWIDTH_M,
    chunk_size=CHUNK_SIZE,
    hdi_prob=0.95,
    truncate_to_data_range=TRUNCATE_TO_DATA_RANGE,
    truncate_by_weight=TRUNCATE_BY_WEIGHT,
    den_frac=DEN_FRAC,
    pop_post_smooth=POP_POST_SMOOTH,
    pop_smooth_bandwidth=POP_SMOOTH_BW,
    min_tags_at_x=MIN_TAGS_AT_X,
    return_tag_summaries=False,
)

plot_population_curve_smooth(
    out_dist_k,
    treat_levels,
    xlabel="Distance to destination (m)",
    ylabel="Mean directed-component probability",
    invert_x=True,
    fig_w=3.6,
    fig_h=2.35,
    axis_label_fs=9,
    tick_label_fs=8,
    line_lw=1.5,
    band_alpha=0.18,
    savepath="../figures/bats/p_goal_pop_kernel_distance_TRUNC_SMOOTH_compact.pdf",
)

# ----------------------------
# B) Steps-from-end (x in meters)
# ----------------------------
LAST_N_STEPS = 110

_, out_steps_k = population_curve_kernel_per_tag(
    df_track_with_p, resp,
    x_kind="steps_from_end",
    last_n_steps=LAST_N_STEPS,
    steps_x_units="m",
    step_length_m=STEP_LENGTH_M,
    n_grid=N_GRID,
    bandwidth=BANDWIDTH_M,
    chunk_size=CHUNK_SIZE,
    hdi_prob=0.95,
    truncate_to_data_range=TRUNCATE_TO_DATA_RANGE,
    truncate_by_weight=TRUNCATE_BY_WEIGHT,
    den_frac=DEN_FRAC,
    pop_post_smooth=POP_POST_SMOOTH,
    pop_smooth_bandwidth=POP_SMOOTH_BW,
    min_tags_at_x=MIN_TAGS_AT_X,
    return_tag_summaries=False,
)

plot_population_curve_smooth(
    out_steps_k,
    treat_levels,
    xlabel="Cumulative path distance from end (m)",
    ylabel="Mean directed-component probability",
    invert_x=False,
    xlim=(-LAST_N_STEPS * STEP_LENGTH_M, 0),
    x_tick_abs=True,
    fig_w=3.6,
    fig_h=2.35,
    axis_label_fs=9,
    tick_label_fs=8,
    line_lw=1.5,
    band_alpha=0.18,
    savepath="../figures/bats/p_goal_pop_kernel_distFromEndM_TRUNC_SMOOTH_compact.pdf",
)

   p_goal_post_mean  p_goal_post_lo  p_goal_post_hi
1          0.001385    2.377827e-13        0.001339
2          0.821075    7.176318e-01        0.908276
3          0.001844    1.333362e-12        0.002415
4          0.001674    5.740170e-13        0.001843
5          0.001666    5.483800e-13        0.001822


/tmp/ipykernel_3898688/705294576.py:546: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(draw_curve, axis=0)


/tmp/ipykernel_3898688/705294576.py:546: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(draw_curve, axis=0)


(<Figure size 1080x705 with 1 Axes>,
 <Axes: xlabel='Cumulative path distance from end (m)', ylabel='Mean directed-component probability'>)

In [ ]:
# ============================================================
# FULL KERNEL-SMOOTHED PIPELINE (NO BINNING) — POPULATION CURVE
# LONG/SKINNY SWAP-IN VERSION (match ~1x4 polar figure footprint)
#   - Long + short figure (~3.4 x 1.2 in; matches 0.85*4 by 1.2)
#   - Slightly adjusted text/lines for tiny-height legibility
#   - NO legend
#   - Saves SVG alongside PDF
#   - Seaborn colorblind palette (style match)
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# ----------------------------
# Style (match other plots)
# ----------------------------
DPI = 300
AX_LW = 0.8
TICK_LW = 0.8
TICK_LEN = 2.5
LINE_LW_DEFAULT = 1.2
FONT_SIZE = 7

mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "font.size": FONT_SIZE,
    "axes.labelsize": FONT_SIZE,
    "axes.titlesize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
    "axes.linewidth": AX_LW,
    "xtick.major.width": TICK_LW,
    "ytick.major.width": TICK_LW,
    "xtick.major.size": TICK_LEN,
    "ytick.major.size": TICK_LEN,
})

# ============================================================
# USER CONTROLS
# ============================================================
STEP_LENGTH_M = 60.0
DIST_COL = "dist_to_dest_grad_m"

BANDWIDTH_M = 120.0
N_GRID = 2000
CHUNK_SIZE = 4000

# --- truncation controls ---
TRUNCATE_TO_DATA_RANGE = True
TRUNCATE_BY_WEIGHT = True
DEN_FRAC = 0.01

# --- stabilize population curve after truncation ---
POP_POST_SMOOTH = True
POP_SMOOTH_BW = 240.0
MIN_TAGS_AT_X = 2

# --- long/skinny target size (matches old 1x4 polar: 0.85*4 by 1.2) ---
SWAPIN_FIG_W = 3.40
SWAPIN_FIG_H = 1.20

# ----------------------------
# Small utilities
# ----------------------------
def circular_modulo(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def sigmoid(x):
    x = np.asarray(x, float)
    return 1.0 / (1.0 + np.exp(-x))

def softplus(x):
    x = np.asarray(x, float)
    return np.where(x > 30, x, np.log1p(np.exp(x)))

def _stack_samples_first(da):
    """xarray DataArray -> np array with sample first: (S, ...)"""
    da2 = da.stack(sample=("chain", "draw"))
    other = [d for d in da2.dims if d != "sample"]
    da2 = da2.transpose("sample", *other)
    return da2.values.astype(float)

def _format_treatment_name(name: str) -> str:
    s = str(name).strip().lower()
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if "control" in parts and len(parts) > 1:
        parts = [p for p in parts if p != "control"]
    if not parts:
        parts = ["control"]
    parts = sorted(parts)
    parts = [p[:1].upper() + p[1:] for p in parts]
    return ", ".join(parts)

def hdi_1d(x, prob=0.95):
    """Minimum-width interval (HDI) for 1D samples (NaN-safe)."""
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    n = x.size
    if n == 0:
        return np.nan, np.nan
    x = np.sort(x)
    m = int(np.floor(prob * n))
    if m < 1:
        return np.nan, np.nan
    widths = x[m:] - x[: n - m]
    j = int(np.argmin(widths))
    return float(x[j]), float(x[j + m])

def hdi_along_samples(arr, prob=0.95):
    """HDI across axis=0 for arr shape (S, N) -> (N,2). NaN-safe."""
    arr = np.asarray(arr, float)
    S, N = arr.shape
    out = np.empty((N, 2), float)
    for i in range(N):
        out[i, 0], out[i, 1] = hdi_1d(arr[:, i], prob=prob)
    return out

def log_i0_stable(kappa):
    """Stable-ish log(I0(kappa)) with asymptotic fallback."""
    kappa = np.asarray(kappa, float)
    out = np.empty_like(kappa)
    big = kappa > 700
    small = ~big
    out[small] = np.log(np.i0(kappa[small]))
    out[big] = kappa[big] - 0.5 * np.log(2 * np.pi * kappa[big])
    return out

def log_vonmises_pdf(y, mu, kappa):
    """log f(y|mu,kappa) = kappa*cos(y-mu) - log(2π) - log(I0(kappa))"""
    y = np.asarray(y, float)
    mu = np.asarray(mu, float)
    kappa = np.asarray(kappa, float)
    return kappa * np.cos(y - mu) - np.log(2 * np.pi) - log_i0_stable(kappa)

# ============================================================
# FFT weighted smoothing utilities (fast, grid-uniform)
# ============================================================
def _next_pow_two(n: int) -> int:
    return 1 << (int(n) - 1).bit_length()

def _fft_convolve_same_1d(x, k):
    """1D convolution (same) using FFT. x,k are 1D arrays."""
    x = np.asarray(x, float)
    k = np.asarray(k, float)
    nx = x.size
    nk = k.size
    n = nx + nk - 1
    nfft = _next_pow_two(n)

    X = np.fft.rfft(x, nfft)
    K = np.fft.rfft(k, nfft)
    y = np.fft.irfft(X * K, nfft)[:n]

    start = (nk - 1) // 2
    return y[start:start + nx]

def _fft_convolve_same_rows(A, k):
    """
    Row-wise convolution (same) using FFT.
    A: (S,G), k: (K,)
    returns: (S,G)
    """
    A = np.asarray(A, float)
    k = np.asarray(k, float)
    S, G = A.shape
    K = k.size
    n = G + K - 1
    nfft = _next_pow_two(n)

    Af = np.fft.rfft(A, nfft, axis=1)
    kf = np.fft.rfft(k, nfft)
    y = np.fft.irfft(Af * kf[None, :], nfft, axis=1)[:, :n]

    start = (K - 1) // 2
    return y[:, start:start + G]

def weighted_gaussian_smooth_draws(draw_curve_SxG, xg_G, weight_G, bw_x, eps=1e-12):
    """
    Smooth each draw curve along x with weighted gaussian smoothing:
      y_sm(x) = conv(y*w, k) / conv(w, k)
    """
    y = np.asarray(draw_curve_SxG, float)  # (S,G)
    xg = np.asarray(xg_G, float)
    w = np.asarray(weight_G, float)

    if y.ndim != 2 or xg.ndim != 1 or w.ndim != 1:
        raise ValueError("Bad shapes for weighted_gaussian_smooth_draws.")
    if y.shape[1] != xg.size or w.size != xg.size:
        raise ValueError("draw_curve, xg, weight must share grid length.")

    if (bw_x is None) or (not np.isfinite(bw_x)) or (bw_x <= 0):
        return y.copy()

    dx = float(np.nanmedian(np.diff(xg)))
    if not np.isfinite(dx) or dx <= 0:
        return y.copy()

    sigma = float(bw_x) / dx
    if sigma < 0.5:
        return y.copy()

    radius = int(np.ceil(4.0 * sigma))
    idx = np.arange(-radius, radius + 1, dtype=float)
    k = np.exp(-0.5 * (idx / sigma) ** 2)

    y0 = np.where(np.isfinite(y), y, 0.0)
    num = y0 * w[None, :]

    num_sm = _fft_convolve_same_rows(num, k)
    den_sm = _fft_convolve_same_1d(w, k)

    out = num_sm / (den_sm[None, :] + eps)
    out[:, den_sm < eps] = np.nan
    return out

# ============================================================
# 1) Posterior responsibilities P(goal | y) per step
# ============================================================
def posterior_prob_goal_per_step(
    idata,
    y_obs,
    *,
    kappa_uniform=1e-6,
    goal_comp_index=0,
    max_draws=1200,
    seed=123,
    hdi_prob=0.95,
):
    post = idata.posterior
    y_obs = np.asarray(y_obs, float)
    n_obs = y_obs.size
    rng = np.random.default_rng(seed)

    if "p_goal" not in post.data_vars:
        raise KeyError("Expected deterministic 'p_goal' in posterior.")

    w = _stack_samples_first(post["p_goal"])  # (S, ...)

    if w.ndim == 2:
        axes = list(range(1, w.ndim))
        obs_axes = [ax for ax in axes if w.shape[ax] == n_obs]
        if len(obs_axes) != 1:
            if w.shape[1] != n_obs:
                raise ValueError(f"Can't identify obs axis for p_goal. shape={w.shape}, n_obs={n_obs}")
            w_goal = w
        else:
            w_goal = np.moveaxis(w, obs_axes[0], 1).reshape(w.shape[0], n_obs)

    elif w.ndim >= 3:
        axes = list(range(1, w.ndim))
        obs_axes = [ax for ax in axes if w.shape[ax] == n_obs]
        comp_axes = [ax for ax in axes if w.shape[ax] == 2]
        if len(obs_axes) != 1 or len(comp_axes) != 1:
            raise ValueError(f"Can't identify obs/comp dims for p_goal. shape={w.shape}, n_obs={n_obs}")

        w = np.moveaxis(w, obs_axes[0], 1)
        comp_candidates = [ax for ax in range(2, w.ndim) if w.shape[ax] == 2]
        if len(comp_candidates) != 1:
            raise ValueError(f"Can't identify comp axis after moving obs. shape={w.shape}")
        w = np.moveaxis(w, comp_candidates[0], 2)  # (S, n_obs, 2)

        if goal_comp_index not in (0, 1):
            raise ValueError("goal_comp_index must be 0 or 1 when comp dim exists.")
        w_goal = w[:, :, int(goal_comp_index)]
    else:
        raise ValueError(f"Unexpected p_goal shape={w.shape}")

    w_goal = np.clip(w_goal, 1e-12, 1 - 1e-12)

    # mu
    if "mu" in post.data_vars:
        mu = _stack_samples_first(post["mu"])
        axes = list(range(1, mu.ndim))
        obs_axes = [ax for ax in axes if mu.shape[ax] == n_obs]
        if len(obs_axes) != 1:
            raise ValueError(f"Can't find obs axis in 'mu'. shape={mu.shape}, n_obs={n_obs}")
        mu = np.moveaxis(mu, obs_axes[0], 1).reshape(mu.shape[0], n_obs)
        mu = circular_modulo(mu)
    elif "mu0" in post.data_vars:
        mu0 = _stack_samples_first(post["mu0"]).reshape(-1)
        mu = circular_modulo(mu0[:, None] + np.zeros((mu0.size, n_obs)))
    else:
        raise KeyError("Need either 'mu' or 'mu0' in posterior.")

    # kappa
    if "kappa" in post.data_vars:
        kappa = _stack_samples_first(post["kappa"])
        axes = list(range(1, kappa.ndim))
        obs_axes = [ax for ax in axes if kappa.shape[ax] == n_obs]
        if len(obs_axes) != 1:
            raise ValueError(f"Can't find obs axis in 'kappa'. shape={kappa.shape}, n_obs={n_obs}")
        kappa = np.moveaxis(kappa, obs_axes[0], 1).reshape(kappa.shape[0], n_obs)
        kappa = np.clip(kappa, 1e-12, np.inf)
    elif "eta_kappa0" in post.data_vars:
        ek0 = _stack_samples_first(post["eta_kappa0"]).reshape(-1)
        k0 = softplus(ek0)
        kappa = k0[:, None] + np.zeros((k0.size, n_obs))
    else:
        raise KeyError("Need either 'kappa' or 'eta_kappa0' in posterior.")

    # subsample draws
    S = w_goal.shape[0]
    if (max_draws is not None) and (S > max_draws):
        idx = rng.choice(S, size=int(max_draws), replace=False)
        w_goal = w_goal[idx]
        mu = mu[idx]
        kappa = kappa[idx]

    logit_w = np.log(w_goal) - np.log1p(-w_goal)
    log_f_goal = log_vonmises_pdf(y_obs[None, :], mu, kappa)
    log_f_unif = log_vonmises_pdf(y_obs[None, :], 0.0, kappa_uniform)

    logit_r = logit_w + (log_f_goal - log_f_unif)
    resp_draws = sigmoid(logit_r)

    resp_mean = resp_draws.mean(axis=0)
    resp_hdi = hdi_along_samples(resp_draws, prob=hdi_prob)

    return resp_draws, resp_mean, resp_hdi

def append_pgoal_post_to_df(
    df_track,
    idata,
    *,
    y_col="HeadingRadians",
    kappa_uniform=1e-6,
    goal_comp_index=0,
    max_draws=1200,
    hdi_prob=0.95,
):
    y_obs = pd.to_numeric(df_track[y_col], errors="coerce").to_numpy(float)
    resp_draws, resp_mean, resp_hdi = posterior_prob_goal_per_step(
        idata,
        y_obs,
        kappa_uniform=kappa_uniform,
        goal_comp_index=goal_comp_index,
        max_draws=max_draws,
        hdi_prob=hdi_prob
    )
    out = df_track.copy()
    out["p_goal_post_mean"] = resp_mean
    out["p_goal_post_lo"] = resp_hdi[:, 0]
    out["p_goal_post_hi"] = resp_hdi[:, 1]
    return out, resp_draws

# ============================================================
# 2) Population curve per treatment
#    (per-tag smoothing + truncation + pop smoothing)
# ============================================================
def population_curve_kernel_per_tag(
    df,
    resp_draws,
    *,
    tag_col="tag",
    treat_col="treatment",
    treat_name_col="treatmentName",

    x_kind="distance",                 # "distance" or "steps_from_end"
    last_n_steps=150,
    steps_x_units="m",                 # "m" or "steps"
    step_length_m=60.0,

    dist_col="dist_to_dest_m",
    dist_clip_q=None,

    n_grid=2000,
    bandwidth=240.0,
    chunk_size=4000,

    hdi_prob=0.95,
    eps=1e-12,

    # truncation controls
    truncate_to_data_range=True,
    truncate_by_weight=True,
    den_frac=0.01,

    # population stabilization
    pop_post_smooth=True,
    pop_smooth_bandwidth=None,
    min_tags_at_x=1,

    return_tag_summaries=False,
):
    d = df.copy().reset_index(drop=True)
    n_obs = len(d)

    resp = np.asarray(resp_draws, float)
    if resp.ndim != 2 or resp.shape[1] != n_obs:
        raise ValueError(f"resp_draws must be (S, n_obs) matching df rows. Got {resp.shape} vs n_obs={n_obs}")

    d["_rowpos"] = np.arange(n_obs, dtype=int)

    # treatment order + labels
    if isinstance(d[treat_col].dtype, pd.CategoricalDtype):
        treat_levels = list(d[treat_col].cat.categories)
    else:
        treat_levels = sorted(d[treat_col].dropna().unique().tolist())

    if treat_name_col in d.columns:
        name_map = d.groupby(treat_col, observed=False)[treat_name_col].first().to_dict()
        labels = {t: _format_treatment_name(name_map.get(t, t)) for t in treat_levels}
    else:
        labels = {t: _format_treatment_name(t) for t in treat_levels}

    # ---- define x ----
    if x_kind == "steps_from_end":
        d["_step_fwd"] = d.groupby(tag_col, observed=False).cumcount()
        d["_last_fwd"] = d.groupby(tag_col, observed=False)["_step_fwd"].transform("max")
        d["_x_steps"] = d["_step_fwd"] - d["_last_fwd"]  # last=0; earlier negative

        d = d.loc[(d["_x_steps"] >= -int(last_n_steps)) & (d["_x_steps"] <= 0)].copy()

        if str(steps_x_units).lower() in ("m", "meter", "meters"):
            d["_x"] = d["_x_steps"].astype(float) * float(step_length_m)
        elif str(steps_x_units).lower() in ("step", "steps"):
            d["_x"] = d["_x_steps"].astype(float)
        else:
            raise ValueError("steps_x_units must be 'm' or 'steps'")

    elif x_kind == "distance":
        if dist_col not in d.columns:
            raise KeyError(f"Missing distance column '{dist_col}' in df.")
        xvals = pd.to_numeric(d[dist_col], errors="coerce").to_numpy(float)
        ok = np.isfinite(xvals)
        d = d.loc[ok].copy()
        xvals = pd.to_numeric(d[dist_col], errors="coerce").to_numpy(float)

        if dist_clip_q is not None:
            cap = float(np.nanquantile(xvals, float(dist_clip_q)))
            xvals = np.clip(xvals, a_min=None, a_max=cap)

        d["_x"] = xvals
    else:
        raise ValueError("x_kind must be 'distance' or 'steps_from_end'")

    def K(z):
        return np.exp(-0.5 * z * z)

    def _smooth_one_tag(resp_SxN, x_N, xg_G, h):
        S, N = resp_SxN.shape
        G = xg_G.size
        num = np.zeros((S, G), float)
        den = np.zeros((G,), float)

        for start in range(0, N, int(chunk_size)):
            stop = min(N, start + int(chunk_size))
            xs = x_N[start:stop]                       # (n,)
            rs = resp_SxN[:, start:stop]               # (S,n)
            z = (xs[:, None] - xg_G[None, :]) / h      # (n,G)
            w = K(z)                                   # (n,G)
            den += w.sum(axis=0)
            num += rs @ w

        den = den + eps
        return num / den[None, :], den

    out = {}
    tag_out = {} if return_tag_summaries else None

    for t in treat_levels:
        dt = d.loc[d[treat_col] == t].copy()
        if dt.empty:
            continue

        x_all = dt["_x"].to_numpy(float)
        xmin = float(np.nanmin(x_all))
        xmax = float(np.nanmax(x_all))
        if not np.isfinite(xmin) or not np.isfinite(xmax) or xmax <= xmin:
            continue

        xg = np.linspace(xmin, xmax, int(n_grid)).astype(float)
        h = float(bandwidth)
        if not np.isfinite(h) or h <= 0:
            raise ValueError("bandwidth must be positive and finite.")

        pop_sum = None                         # (S,G)
        pop_cnt = np.zeros((xg.size,), float)  # tags contributing at each x
        n_tags_any = 0

        for tg, g in dt.groupby(tag_col, observed=False):
            j = g["_rowpos"].to_numpy(int)
            if j.size == 0:
                continue

            x_tg_full = g["_x"].to_numpy(float)
            x_tg = x_tg_full[np.isfinite(x_tg_full)]
            if x_tg.size < 2:
                continue

            x_min_tg = float(np.nanmin(x_tg))
            x_max_tg = float(np.nanmax(x_tg))

            resp_tg = resp[:, j]  # (S, N_tag)
            curve_SxG, den_G = _smooth_one_tag(resp_tg, x_tg_full, xg, h)

            # ---- truncation mask for this tag ----
            mask = np.zeros_like(xg, dtype=bool)

            if truncate_to_data_range:
                mask |= (xg < x_min_tg) | (xg > x_max_tg)

            if truncate_by_weight:
                thr = float(den_frac) * float(np.nanmax(den_G))
                mask |= (den_G < thr)

            if mask.any():
                curve_SxG[:, mask] = np.nan

            if pop_sum is None:
                pop_sum = np.zeros_like(curve_SxG)

            good = np.isfinite(curve_SxG)
            pop_sum += np.where(good, curve_SxG, 0.0)
            pop_cnt += np.isfinite(curve_SxG[0]).astype(float)

            n_tags_any += 1

            if return_tag_summaries:
                tg_mean = np.nanmean(curve_SxG, axis=0)
                tg_hdi = hdi_along_samples(curve_SxG, prob=hdi_prob)
                tag_out[(t, tg)] = {
                    "x_mid": xg,
                    "mean": tg_mean,
                    "lo": tg_hdi[:, 0],
                    "hi": tg_hdi[:, 1],
                    "x_min": x_min_tg,
                    "x_max": x_max_tg,
                }

        if pop_sum is None or n_tags_any == 0:
            continue

        denom = np.maximum(pop_cnt, 1.0)
        draw_curve = pop_sum / denom[None, :]
        draw_curve[:, pop_cnt < 1] = np.nan

        min_k = int(max(1, min_tags_at_x))
        unstable = pop_cnt < float(min_k)
        if unstable.any():
            draw_curve[:, unstable] = np.nan

        # ---- population post-smoothing across x ----
        if pop_post_smooth:
            bw_pop = float(pop_smooth_bandwidth) if pop_smooth_bandwidth is not None else float(bandwidth)
            w = pop_cnt.copy()
            w[unstable] = 0.0
            draw_curve = weighted_gaussian_smooth_draws(draw_curve, xg, w, bw_pop, eps=eps)
            draw_curve[:, unstable] = np.nan
            draw_curve[:, pop_cnt < 1] = np.nan

        mean = np.nanmean(draw_curve, axis=0)
        hdi = hdi_along_samples(draw_curve, prob=hdi_prob)

        out[t] = {
            "label": labels.get(t, str(t)),
            "x_mid": xg,
            "mean": mean,
            "lo": hdi[:, 0],
            "hi": hdi[:, 1],
            "bandwidth": h,
            "n_tags": int(n_tags_any),
            "n_tags_by_x": pop_cnt,
            "min_tags_at_x": min_k,
            "pop_post_smooth": bool(pop_post_smooth),
            "pop_smooth_bandwidth": float(pop_smooth_bandwidth) if pop_smooth_bandwidth is not None else float(bandwidth),
        }

    if return_tag_summaries:
        return treat_levels, out, tag_out
    return treat_levels, out

# ============================================================
# 3) Smooth plotter (LONG / SKINNY SWAP-IN)
#    - long + short footprint for figure replacement
#    - no legend
# ============================================================
def plot_population_curve_smooth(
    out_dict,
    treat_levels,
    *,
    xlabel,
    ylabel="Population mean p(goal | step)",
    band_alpha=0.18,
    line_lw=LINE_LW_DEFAULT,
    fig_w=SWAPIN_FIG_W,      # long/skinny (~1x4 polar width)
    fig_h=SWAPIN_FIG_H,      # short (~1x4 polar height)
    invert_x=False,
    xlim=None,
    x_tick_abs=False,
    savepath=None,
    palette_name="colorblind",
    save_svg_also=True,

    # tiny-height friendly text
    axis_label_fs=7,
    tick_label_fs=6,
    y_labelpad=1,
    x_labelpad=1,
):
    colors = sns.color_palette(palette_name, n_colors=len(treat_levels))
    treat_to_color = {t: colors[i] for i, t in enumerate(treat_levels)}

    fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=DPI)

    ax.axhline(0.5, color="0.88", lw=0.8, ls="--", zorder=0)

    for t in treat_levels:
        if t not in out_dict:
            continue
        s = out_dict[t]
        x = np.asarray(s["x_mid"], float)
        m = np.asarray(s["mean"], float)
        lo = np.asarray(s["lo"], float)
        hi = np.asarray(s["hi"], float)

        order = np.argsort(x)
        x, m, lo, hi = x[order], m[order], lo[order], hi[order]

        c = treat_to_color[t]
        ax.fill_between(x, lo, hi, color=c, alpha=band_alpha, linewidth=0)
        ax.plot(x, m, color=c, lw=line_lw)

    ax.set_ylim(0, 1)
    ax.set_xlabel(xlabel, fontsize=axis_label_fs, labelpad=x_labelpad)
    ax.set_ylabel(ylabel, fontsize=axis_label_fs, labelpad=y_labelpad)

    if xlim is not None:
        ax.set_xlim(*xlim)
    if invert_x:
        ax.invert_xaxis()

    if x_tick_abs:
        def _abs_tick_formatter(val, pos):
            aval = abs(val)
            if np.isfinite(aval) and np.isclose(aval, np.round(aval), atol=1e-9):
                return f"{int(np.round(aval)):,}"
            if np.isfinite(aval):
                return f"{aval:.2f}"
            return ""
        ax.xaxis.set_major_formatter(FuncFormatter(_abs_tick_formatter))

    ax.tick_params(axis="both", which="major", labelsize=tick_label_fs, length=2.5, pad=1)
    ax.tick_params(axis="both", which="minor", labelsize=tick_label_fs, length=2.0, pad=1)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

    # No legend; use manual margins because tight_layout can struggle at tiny height
    fig.subplots_adjust(left=0.13, right=0.995, bottom=0.30, top=0.98)

    if savepath:
        fig.savefig(savepath, dpi=DPI, bbox_inches="tight", pad_inches=0.01)
        if save_svg_also:
            root, _ = os.path.splitext(savepath)
            fig.savefig(root + ".svg", bbox_inches="tight", pad_inches=0.01)

    plt.show()
    return fig, ax

# ============================================================
# RUN EVERYTHING
# ============================================================

# 1) compute and append per-step posterior P(goal|y)
df_track_with_p, resp = append_pgoal_post_to_df(
    df_track,
    idata_mix_full,
    y_col="HeadingRadians",
    max_draws=2000,
    hdi_prob=0.95,
    # goal_comp_index=0,
)

print(df_track_with_p[["p_goal_post_mean", "p_goal_post_lo", "p_goal_post_hi"]].head())

treat_levels = (
    list(df_track_with_p["treatment"].cat.categories)
    if isinstance(df_track_with_p["treatment"].dtype, pd.CategoricalDtype)
    else sorted(df_track_with_p["treatment"].dropna().unique().tolist())
)

# ----------------------------
# A) Distance-to-destination
# ----------------------------
_, out_dist_k = population_curve_kernel_per_tag(
    df_track_with_p, resp,
    x_kind="distance",
    dist_col=DIST_COL,
    dist_clip_q=None,
    n_grid=N_GRID,
    bandwidth=BANDWIDTH_M,
    chunk_size=CHUNK_SIZE,
    hdi_prob=0.95,
    truncate_to_data_range=TRUNCATE_TO_DATA_RANGE,
    truncate_by_weight=TRUNCATE_BY_WEIGHT,
    den_frac=DEN_FRAC,
    pop_post_smooth=POP_POST_SMOOTH,
    pop_smooth_bandwidth=POP_SMOOTH_BW,
    min_tags_at_x=MIN_TAGS_AT_X,
    return_tag_summaries=False,
)

plot_population_curve_smooth(
    out_dist_k,
    treat_levels,
    xlabel="Distance to destination (m)",
    ylabel="Directed-component\nprobability",
    invert_x=True,
    fig_w=SWAPIN_FIG_W,
    fig_h=SWAPIN_FIG_H,
    axis_label_fs=7,
    tick_label_fs=6,
    line_lw=1.2,
    band_alpha=0.18,
    savepath="../figures/bats/p_goal_pop_kernel_distance_TRUNC_SMOOTH_compact.pdf",
)

# ----------------------------
# B) Steps-from-end (x in meters)
# ----------------------------
LAST_N_STEPS = 110

_, out_steps_k = population_curve_kernel_per_tag(
    df_track_with_p, resp,
    x_kind="steps_from_end",
    last_n_steps=LAST_N_STEPS,
    steps_x_units="m",
    step_length_m=STEP_LENGTH_M,
    n_grid=N_GRID,
    bandwidth=BANDWIDTH_M,
    chunk_size=CHUNK_SIZE,
    hdi_prob=0.95,
    truncate_to_data_range=TRUNCATE_TO_DATA_RANGE,
    truncate_by_weight=TRUNCATE_BY_WEIGHT,
    den_frac=DEN_FRAC,
    pop_post_smooth=POP_POST_SMOOTH,
    pop_smooth_bandwidth=POP_SMOOTH_BW,
    min_tags_at_x=MIN_TAGS_AT_X,
    return_tag_summaries=False,
)

plot_population_curve_smooth(
    out_steps_k,
    treat_levels,
    xlabel="Cumulative path distance from end (m)",
    ylabel="Directed-component\nprobability",
    invert_x=False,
    xlim=(-LAST_N_STEPS * STEP_LENGTH_M, 0),
    x_tick_abs=True,
    fig_w=SWAPIN_FIG_W,
    fig_h=SWAPIN_FIG_H,
    axis_label_fs=7,
    tick_label_fs=6,
    line_lw=1.2,
    band_alpha=0.18,
    savepath="../figures/bats/p_goal_pop_kernel_distFromEndM_TRUNC_SMOOTH_compact.pdf",
)

   p_goal_post_mean  p_goal_post_lo  p_goal_post_hi
1          0.001385    2.377827e-13        0.001339
2          0.821075    7.176318e-01        0.908276
3          0.001844    1.333362e-12        0.002415
4          0.001674    5.740170e-13        0.001843
5          0.001666    5.483800e-13        0.001822


/tmp/ipykernel_3898688/325684253.py:567: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(draw_curve, axis=0)


/tmp/ipykernel_3898688/325684253.py:567: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(draw_curve, axis=0)


(<Figure size 1020x360 with 1 Axes>,
 <Axes: xlabel='Cumulative path distance from end (m)', ylabel='Directed-component\nprobability'>)

Per-animal (faceted) $p_{goal}$ along the flight, showing the heterogeneity in when each bat commits
to goal-directed travel.


In [ ]:
# ============================================================
# FULL KERNEL-SMOOTHED PIPELINE (NO BINNING) — INDIVIDUAL TAGS
# WITH TRUNCATION (NO SMOOTHING PAST END OF EACH INDIVIDUAL):
#
# For each treatment (faceted 2x2 = 4 subplots per figure):
#   - For each tag (individual):
#       kernel-smooth r_{s,i}=P(goal|y_i, draw s) vs x on a treatment grid
#       summarize within-tag across draws -> mean ± HDI
#       TRUNCATE (mask to NaN) outside the tag's observed x-range
#         so nothing is plotted where the individual has no data.
#
# Runs BOTH:
#   A) distance-to-destination
#   B) steps_from_end (in meters or steps)
#
# UPDATED:
#   - seaborn colorblind palette
#   - style matches other plots (Arial + svg.fonttype none + white bg)
#   - saves SVG alongside PDF (same basename)
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# ----------------------------
# Style
# ----------------------------
DPI = 300
mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
})

# ============================================================
# USER CONTROLS
# ============================================================
STEP_LENGTH_M = 60.0                 # meters per step (used for steps-from-end if x in meters)
DIST_COL = "dist_to_dest_grad_m"     # <-- set to your distance column name

# --- bandwidths (in SAME units as x) ---
BANDWIDTH_M = 240.0                  # ~240 (meters if x in meters)

N_GRID = 2000                        # grid resolution for smooth curves
CHUNK_SIZE = 4000                    # memory control inside kernel smoothing

# Truncation controls (IMPORTANT)
TRUNCATE_TO_DATA_RANGE = True        # mask outside [min(x_tag), max(x_tag)] for each tag
TRUNCATE_BY_WEIGHT = True            # optional extra masking where kernel support is tiny
DEN_FRAC = 0.01                      # threshold: den < DEN_FRAC * den.max()

# ----------------------------
# Small utilities
# ----------------------------
def circular_modulo(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def sigmoid(x):
    x = np.asarray(x, float)
    return 1.0 / (1.0 + np.exp(-x))

def softplus(x):
    x = np.asarray(x, float)
    return np.where(x > 30, x, np.log1p(np.exp(x)))

def _stack_samples_first(da):
    """xarray DataArray -> np array with sample first: (S, ...)"""
    da2 = da.stack(sample=("chain", "draw"))
    other = [d for d in da2.dims if d != "sample"]
    da2 = da2.transpose("sample", *other)
    return da2.values.astype(float)

def _format_treatment_name(name: str) -> str:
    s = str(name).strip().lower()
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if "control" in parts and len(parts) > 1:
        parts = [p for p in parts if p != "control"]
    if not parts:
        parts = ["control"]
    parts = sorted(parts)
    parts = [p[:1].upper() + p[1:] for p in parts]
    return ", ".join(parts)

def hdi_1d(x, prob=0.95):
    """Minimum-width interval (HDI) for 1D samples."""
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    n = x.size
    if n == 0:
        return np.nan, np.nan
    x = np.sort(x)
    m = int(np.floor(prob * n))
    if m < 1:
        return np.nan, np.nan
    widths = x[m:] - x[: n - m]
    j = int(np.argmin(widths))
    return float(x[j]), float(x[j + m])

def hdi_along_samples(arr, prob=0.95):
    """HDI across axis=0 for arr shape (S, N) -> (N,2). Handles NaNs."""
    arr = np.asarray(arr, float)
    S, N = arr.shape
    out = np.empty((N, 2), float)
    for i in range(N):
        out[i, 0], out[i, 1] = hdi_1d(arr[:, i], prob=prob)
    return out

def log_i0_stable(kappa):
    """Stable-ish log(I0(kappa)) with asymptotic fallback."""
    kappa = np.asarray(kappa, float)
    out = np.empty_like(kappa)
    big = kappa > 700
    small = ~big
    out[small] = np.log(np.i0(kappa[small]))
    out[big] = kappa[big] - 0.5 * np.log(2 * np.pi * kappa[big])
    return out

def log_vonmises_pdf(y, mu, kappa):
    """
    y, mu, kappa broadcastable.
    log f(y|mu,kappa) = kappa*cos(y-mu) - log(2π) - log(I0(kappa))
    """
    y = np.asarray(y, float)
    mu = np.asarray(mu, float)
    kappa = np.asarray(kappa, float)
    return kappa * np.cos(y - mu) - np.log(2 * np.pi) - log_i0_stable(kappa)

# ============================================================
# 1) Posterior responsibilities P(goal | y) per step
# ============================================================
def posterior_prob_goal_per_step(
    idata,
    y_obs,
    *,
    kappa_uniform=1e-6,
    goal_comp_index=0,
    max_draws=1200,
    seed=123,
    hdi_prob=0.95,
):
    """
    Returns:
      resp_draws : (S, n_obs) posterior draws of P(goal | y_i)
      resp_mean  : (n_obs,)
      resp_hdi   : (n_obs, 2)
    """
    post = idata.posterior
    y_obs = np.asarray(y_obs, float)
    n_obs = y_obs.size
    rng = np.random.default_rng(seed)

    if "p_goal" not in post.data_vars:
        raise KeyError("Expected deterministic 'p_goal' in posterior.")

    w = _stack_samples_first(post["p_goal"])

    if w.ndim == 2:
        axes = list(range(1, w.ndim))
        obs_axes = [ax for ax in axes if w.shape[ax] == n_obs]
        if len(obs_axes) != 1:
            if w.shape[1] != n_obs:
                raise ValueError(f"Can't identify obs axis for p_goal. shape={w.shape}, n_obs={n_obs}")
            w_goal = w
        else:
            w_goal = np.moveaxis(w, obs_axes[0], 1).reshape(w.shape[0], n_obs)

    elif w.ndim >= 3:
        axes = list(range(1, w.ndim))
        obs_axes = [ax for ax in axes if w.shape[ax] == n_obs]
        comp_axes = [ax for ax in axes if w.shape[ax] == 2]
        if len(obs_axes) != 1 or len(comp_axes) != 1:
            raise ValueError(f"Can't identify obs/comp dims for p_goal. shape={w.shape}, n_obs={n_obs}")

        w = np.moveaxis(w, obs_axes[0], 1)
        comp_candidates = [ax for ax in range(2, w.ndim) if w.shape[ax] == 2]
        if len(comp_candidates) != 1:
            raise ValueError(f"Can't identify comp axis after moving obs. shape={w.shape}")
        w = np.moveaxis(w, comp_candidates[0], 2)  # (S, n_obs, 2)

        if goal_comp_index not in (0, 1):
            raise ValueError("goal_comp_index must be 0 or 1 when comp dim exists.")
        w_goal = w[:, :, int(goal_comp_index)]
    else:
        raise ValueError(f"Unexpected p_goal shape={w.shape}")

    w_goal = np.clip(w_goal, 1e-12, 1 - 1e-12)

    # mu
    if "mu" in post.data_vars:
        mu = _stack_samples_first(post["mu"])
        axes = list(range(1, mu.ndim))
        obs_axes = [ax for ax in axes if mu.shape[ax] == n_obs]
        if len(obs_axes) != 1:
            raise ValueError(f"Can't find obs axis in 'mu'. shape={mu.shape}, n_obs={n_obs}")
        mu = np.moveaxis(mu, obs_axes[0], 1).reshape(mu.shape[0], n_obs)
        mu = circular_modulo(mu)
    elif "mu0" in post.data_vars:
        mu0 = _stack_samples_first(post["mu0"]).reshape(-1)
        mu = circular_modulo(mu0[:, None] + np.zeros((mu0.size, n_obs)))
    else:
        raise KeyError("Need either 'mu' or 'mu0' in posterior.")

    # kappa
    if "kappa" in post.data_vars:
        kappa = _stack_samples_first(post["kappa"])
        axes = list(range(1, kappa.ndim))
        obs_axes = [ax for ax in axes if kappa.shape[ax] == n_obs]
        if len(obs_axes) != 1:
            raise ValueError(f"Can't find obs axis in 'kappa'. shape={kappa.shape}, n_obs={n_obs}")
        kappa = np.moveaxis(kappa, obs_axes[0], 1).reshape(kappa.shape[0], n_obs)
        kappa = np.clip(kappa, 1e-12, np.inf)
    elif "eta_kappa0" in post.data_vars:
        ek0 = _stack_samples_first(post["eta_kappa0"]).reshape(-1)
        k0 = softplus(ek0)
        kappa = k0[:, None] + np.zeros((k0.size, n_obs))
    else:
        raise KeyError("Need either 'kappa' or 'eta_kappa0' in posterior.")

    # subsample draws once
    S = w_goal.shape[0]
    if (max_draws is not None) and (S > max_draws):
        idx = rng.choice(S, size=int(max_draws), replace=False)
        w_goal = w_goal[idx]
        mu = mu[idx]
        kappa = kappa[idx]

    logit_w = np.log(w_goal) - np.log1p(-w_goal)
    log_f_goal = log_vonmises_pdf(y_obs[None, :], mu, kappa)
    log_f_unif = log_vonmises_pdf(y_obs[None, :], 0.0, kappa_uniform)

    logit_r = logit_w + (log_f_goal - log_f_unif)
    resp_draws = sigmoid(logit_r)

    resp_mean = resp_draws.mean(axis=0)
    resp_hdi = hdi_along_samples(resp_draws, prob=hdi_prob)

    return resp_draws, resp_mean, resp_hdi

def append_pgoal_post_to_df(
    df_track,
    idata,
    *,
    y_col="HeadingRadians",
    kappa_uniform=1e-6,
    goal_comp_index=0,
    max_draws=1200,
    hdi_prob=0.95,
):
    y_obs = pd.to_numeric(df_track[y_col], errors="coerce").to_numpy(float)
    resp_draws, resp_mean, resp_hdi = posterior_prob_goal_per_step(
        idata,
        y_obs,
        kappa_uniform=kappa_uniform,
        goal_comp_index=goal_comp_index,
        max_draws=max_draws,
        hdi_prob=hdi_prob
    )
    out = df_track.copy()
    out["p_goal_post_mean"] = resp_mean
    out["p_goal_post_lo"] = resp_hdi[:, 0]
    out["p_goal_post_hi"] = resp_hdi[:, 1]
    return out, resp_draws

# ============================================================
# 2) Kernel smoothing PER TAG, summarized within tag (mean ± HDI)
#    with TRUNCATION per tag so curves/bands do not extend past data
# ============================================================
def tag_curves_kernel_by_treatment(
    df,
    resp_draws,
    *,
    tag_col="tag",
    treat_col="treatment",
    treat_name_col="treatmentName",

    x_kind="distance",                 # "distance" or "steps_from_end"

    # steps-from-end x definition
    last_n_steps=150,
    steps_x_units="m",                 # "m" or "steps"
    step_length_m=60.0,

    # distance x definition
    dist_col="dist_to_dest_m",
    dist_clip_q=None,                  # None for full range; or e.g. 0.999 to cap outliers

    # smoothing controls
    n_grid=2000,
    bandwidth=240.0,
    chunk_size=4000,

    hdi_prob=0.95,
    eps=1e-12,

    # TRUNCATION CONTROLS
    truncate_to_data_range=True,
    truncate_by_weight=False,
    den_frac=0.01,
):
    """
    Returns:
      treat_levels, out_by_treat

    out_by_treat[t] = {
        "label": str,
        "x_mid": (G,),
        "bandwidth": float,
        "tags": {
            tag_value: {
                "mean": (G,), "lo": (G,), "hi": (G,),
                "n_obs": int, "x_min": float, "x_max": float
            }, ...
        }
      }
    """
    d = df.copy().reset_index(drop=True)
    n_obs = len(d)

    resp = np.asarray(resp_draws, float)
    if resp.ndim != 2 or resp.shape[1] != n_obs:
        raise ValueError(f"resp_draws must be (S, n_obs) matching df rows. Got {resp.shape} vs n_obs={n_obs}")

    d["_rowpos"] = np.arange(n_obs, dtype=int)

    if isinstance(d[treat_col].dtype, pd.CategoricalDtype):
        treat_levels = list(d[treat_col].cat.categories)
    else:
        treat_levels = sorted(d[treat_col].dropna().unique().tolist())

    if treat_name_col in d.columns:
        name_map = d.groupby(treat_col, observed=False)[treat_name_col].first().to_dict()
        labels = {t: _format_treatment_name(name_map.get(t, t)) for t in treat_levels}
    else:
        labels = {t: _format_treatment_name(t) for t in treat_levels}

    # ---- define x for each row ----
    if x_kind == "steps_from_end":
        d["_step_fwd"] = d.groupby(tag_col, observed=False).cumcount()
        d["_last_fwd"] = d.groupby(tag_col, observed=False)["_step_fwd"].transform("max")
        d["_x_steps"] = d["_step_fwd"] - d["_last_fwd"]  # last=0; earlier negative

        d = d.loc[(d["_x_steps"] >= -int(last_n_steps)) & (d["_x_steps"] <= 0)].copy()

        if str(steps_x_units).lower() in ("m", "meter", "meters"):
            d["_x"] = d["_x_steps"].astype(float) * float(step_length_m)
        elif str(steps_x_units).lower() in ("step", "steps"):
            d["_x"] = d["_x_steps"].astype(float)
        else:
            raise ValueError("steps_x_units must be 'm' or 'steps'")

    elif x_kind == "distance":
        if dist_col not in d.columns:
            raise KeyError(f"Missing distance column '{dist_col}' in df.")

        xvals = pd.to_numeric(d[dist_col], errors="coerce").to_numpy(float)
        ok = np.isfinite(xvals)
        d = d.loc[ok].copy()
        xvals = pd.to_numeric(d[dist_col], errors="coerce").to_numpy(float)

        if dist_clip_q is not None:
            cap = float(np.nanquantile(xvals, float(dist_clip_q)))
            xvals = np.clip(xvals, a_min=None, a_max=cap)

        d["_x"] = xvals
    else:
        raise ValueError("x_kind must be 'distance' or 'steps_from_end'")

    def K(z):
        return np.exp(-0.5 * z * z)

    def _smooth_one_tag(resp_SxN, x_N, xg_G, h):
        """
        resp_SxN: (S,N), x_N: (N,), xg_G: (G,)
        returns:
          curve_SxG: (S,G)
          den_G: (G,) kernel weight sum at each grid point (same for all draws)
        """
        S, N = resp_SxN.shape
        G = xg_G.size
        num = np.zeros((S, G), float)
        den = np.zeros((G,), float)

        for start in range(0, N, int(chunk_size)):
            stop = min(N, start + int(chunk_size))
            xs = x_N[start:stop]                       # (n,)
            rs = resp_SxN[:, start:stop]               # (S,n)
            z = (xs[:, None] - xg_G[None, :]) / h      # (n,G)
            w = K(z)                                   # (n,G)
            den += w.sum(axis=0)
            num += rs @ w

        den = den + eps
        return num / den[None, :], den

    out = {}

    for t in treat_levels:
        dt = d.loc[d[treat_col] == t].copy()
        if dt.empty:
            continue

        # common x-grid per treatment (align all tags in panel)
        x_all = dt["_x"].to_numpy(float)
        xmin = float(np.nanmin(x_all))
        xmax = float(np.nanmax(x_all))
        if not np.isfinite(xmin) or not np.isfinite(xmax) or xmax <= xmin:
            continue

        xg = np.linspace(xmin, xmax, int(n_grid)).astype(float)
        h = float(bandwidth)
        if not np.isfinite(h) or h <= 0:
            raise ValueError("bandwidth must be positive and finite.")

        out[t] = {
            "label": labels.get(t, str(t)),
            "x_mid": xg,
            "bandwidth": h,
            "tags": {},
        }

        for tg, g in dt.groupby(tag_col, observed=False):
            j = g["_rowpos"].to_numpy(int)
            if j.size == 0:
                continue

            x_tg = g["_x"].to_numpy(float)
            x_tg = x_tg[np.isfinite(x_tg)]
            if x_tg.size < 2:
                continue

            x_min_tg = float(np.nanmin(x_tg))
            x_max_tg = float(np.nanmax(x_tg))

            resp_tg = resp[:, j]  # (S, N_tag)
            curve_SxG, den_G = _smooth_one_tag(resp_tg, g["_x"].to_numpy(float), xg, h)

            # ---- TRUNCATION (critical): mask outside the individual's observed support ----
            mask = np.zeros_like(xg, dtype=bool)

            if truncate_to_data_range:
                mask |= (xg < x_min_tg) | (xg > x_max_tg)

            if truncate_by_weight:
                thr = float(den_frac) * float(np.nanmax(den_G))
                mask |= (den_G < thr)

            if mask.any():
                curve_SxG[:, mask] = np.nan  # breaks line & band beyond data

            tg_mean = np.nanmean(curve_SxG, axis=0)
            tg_hdi = hdi_along_samples(curve_SxG, prob=hdi_prob)

            out[t]["tags"][tg] = {
                "mean": tg_mean,
                "lo": tg_hdi[:, 0],
                "hi": tg_hdi[:, 1],
                "n_obs": int(j.size),
                "x_min": x_min_tg,
                "x_max": x_max_tg,
            }

    return treat_levels, out

# ============================================================
# 3) Faceted plotter: 4 subplots per figure (2x2), one per treatment
#    Each panel shows ALL tags with mean ± HDI (truncated regions not drawn)
# ============================================================
def plot_tag_curves_faceted_4panel(
    out_by_treat,
    treat_levels,
    *,
    xlabel,
    ylabel="p(goal | step)",
    invert_x=False,
    xlim=None,
    x_tick_abs=False,
    band_alpha=0.10,
    line_alpha=0.75,
    line_lw=0.9,
    max_tags_per_panel=None,  # None = all tags; else randomly sample
    seed=123,
    savepath=None,
    fig_w=10.5,
    fig_h=7.6,
    palette_name="colorblind",
    save_svg_also=True,
):
    rng = np.random.default_rng(seed)

    # seaborn colorblind palette (stable treatment order)
    colors = sns.color_palette(palette_name, n_colors=len(treat_levels))
    treat_to_color = {t: colors[i] for i, t in enumerate(treat_levels)}

    tl = [t for t in treat_levels if t in out_by_treat]
    pages = [tl[i:i+4] for i in range(0, len(tl), 4)]
    figs = []

    for p_i, page in enumerate(pages, start=1):
        fig, axes = plt.subplots(2, 2, figsize=(fig_w, fig_h), dpi=DPI, sharey=True)
        axes = axes.ravel()

        for ax_i, ax in enumerate(axes):
            if ax_i >= len(page):
                ax.axis("off")
                continue

            t = page[ax_i]
            block = out_by_treat[t]
            x = np.asarray(block["x_mid"], float)
            order = np.argsort(x)
            x = x[order]

            ax.axhline(0.5, color="0.90", lw=0.9, ls="--", zorder=0)

            c = treat_to_color[t]
            tags = list(block["tags"].items())
            tags.sort(key=lambda kv: kv[1].get("n_obs", 0), reverse=True)

            if (max_tags_per_panel is not None) and (len(tags) > int(max_tags_per_panel)):
                idx = rng.choice(len(tags), size=int(max_tags_per_panel), replace=False)
                idx = np.sort(idx)
                tags = [tags[i] for i in idx]

            for tg, s in tags:
                m = np.asarray(s["mean"], float)[order]
                lo = np.asarray(s["lo"], float)[order]
                hi = np.asarray(s["hi"], float)[order]

                # NaNs (from truncation) naturally break lines and bands
                ax.fill_between(x, lo, hi, color=c, alpha=band_alpha, linewidth=0)
                ax.plot(x, m, color=c, alpha=line_alpha, lw=line_lw)

            ax.set_ylim(0, 1)

            if xlim is not None:
                ax.set_xlim(*xlim)
            if invert_x:
                ax.invert_xaxis()

            if x_tick_abs:
                def _abs_tick_formatter(val, pos):
                    aval = abs(val)
                    if np.isfinite(aval) and np.isclose(aval, np.round(aval), atol=1e-9):
                        return f"{int(np.round(aval)):,}"
                    if np.isfinite(aval):
                        return f"{aval:,.2f}"
                    return ""
                ax.xaxis.set_major_formatter(FuncFormatter(_abs_tick_formatter))

            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            ax.grid(False)

            # --- font sizes ---
            AXIS_LABEL_FS = 8
            TICK_LABEL_FS = 6
            TITLE_FS      = 11

            ax.set_title(block.get("label", str(t)), fontsize=TITLE_FS)
            ax.set_xlabel(xlabel, fontsize=AXIS_LABEL_FS)
            if ax_i % 2 == 0:
                ax.set_ylabel(ylabel, fontsize=AXIS_LABEL_FS)

            ax.tick_params(axis="both", which="major", labelsize=TICK_LABEL_FS)
            ax.tick_params(axis="both", which="minor", labelsize=TICK_LABEL_FS)

        fig.tight_layout()

        if savepath:
            if len(pages) > 1:
                root, ext = os.path.splitext(savepath)
                out_path = f"{root}_p{p_i}{ext}"
            else:
                out_path = savepath

            fig.savefig(out_path, dpi=DPI, bbox_inches="tight", pad_inches=0.01)

            if save_svg_also:
                root_page, _ = os.path.splitext(out_path)
                fig.savefig(f"{root_page}.svg", bbox_inches="tight", pad_inches=0.01)

        plt.show()
        figs.append(fig)

    return figs

# ============================================================
# RUN EVERYTHING
# ============================================================

# 1) compute and append per-step posterior P(goal|y)
df_track_with_p, resp = append_pgoal_post_to_df(
    df_track,
    idata_mix_full,
    y_col="HeadingRadians",
    max_draws=2000,
    hdi_prob=0.95,
    # goal_comp_index=0,  # change to 1 if your comp index differs
)

print(df_track_with_p[["p_goal_post_mean", "p_goal_post_lo", "p_goal_post_hi"]].head())

# treatment order (for consistent panel order)
treat_levels = (
    list(df_track_with_p["treatment"].cat.categories)
    if isinstance(df_track_with_p["treatment"].dtype, pd.CategoricalDtype)
    else sorted(df_track_with_p["treatment"].dropna().unique().tolist())
)

# ----------------------------
# A) Distance-to-destination: PER TAG mean ± HDI (TRUNCATED)
# ----------------------------
_, out_tags_dist = tag_curves_kernel_by_treatment(
    df_track_with_p,
    resp,
    x_kind="distance",
    dist_col=DIST_COL,
    dist_clip_q=None,
    n_grid=N_GRID,
    bandwidth=BANDWIDTH_M,
    chunk_size=CHUNK_SIZE,
    hdi_prob=0.95,
    truncate_to_data_range=TRUNCATE_TO_DATA_RANGE,
    truncate_by_weight=TRUNCATE_BY_WEIGHT,
    den_frac=DEN_FRAC,
)

plot_tag_curves_faceted_4panel(
    out_tags_dist,
    treat_levels,
    xlabel="Distance to destination (m)",
    ylabel="Directed-component probability",
    invert_x=True,  # far -> near to the right
    savepath="../figures/bats/p_goal_tag_kernel_distance_faceted_TRUNC.pdf",
    # max_tags_per_panel=50,  # optionally cap if too dense
    fig_h=6,
    fig_w=8,
)

# ----------------------------
# B) Steps-from-end (x in meters): PER TAG mean ± HDI (TRUNCATED)
# ----------------------------
LAST_N_STEPS = 110

_, out_tags_steps = tag_curves_kernel_by_treatment(
    df_track_with_p,
    resp,
    x_kind="steps_from_end",
    last_n_steps=LAST_N_STEPS,
    steps_x_units="m",
    step_length_m=STEP_LENGTH_M,
    n_grid=N_GRID,
    bandwidth=BANDWIDTH_M,
    chunk_size=CHUNK_SIZE,
    hdi_prob=0.95,
    truncate_to_data_range=TRUNCATE_TO_DATA_RANGE,
    truncate_by_weight=TRUNCATE_BY_WEIGHT,
    den_frac=DEN_FRAC,
)

plot_tag_curves_faceted_4panel(
    out_tags_steps,
    treat_levels,
    xlabel="Cumulative path distance from end (m)",
    ylabel="Directed-component probability",
    invert_x=False,
    xlim=(-LAST_N_STEPS * STEP_LENGTH_M, 0),
    x_tick_abs=True,  # show positive tick labels while keeping x negative-to-zero
    savepath="../figures/bats/p_goal_tag_kernel_distFromEndM_faceted_TRUNC.pdf",
    # max_tags_per_panel=50,  # optionally cap if too dense
    fig_h=6,
    fig_w=8,
)

   p_goal_post_mean  p_goal_post_lo  p_goal_post_hi
1          0.001385    2.377827e-13        0.001339
2          0.821075    7.176318e-01        0.908276
3          0.001844    1.333362e-12        0.002415
4          0.001674    5.740170e-13        0.001843
5          0.001666    5.483800e-13        0.001822


/tmp/ipykernel_3898688/1050404124.py:462: RuntimeWarning: Mean of empty slice
  tg_mean = np.nanmean(curve_SxG, axis=0)


/tmp/ipykernel_3898688/1050404124.py:462: RuntimeWarning: Mean of empty slice
  tg_mean = np.nanmean(curve_SxG, axis=0)


[<Figure size 2400x1800 with 4 Axes>]